# 🔥🛰️ HELIOS-SHIELD — обучение ИИ на 26 годах пожаров NASA (MODIS + VIIRS)

**Задача:** гармонизировать 26-летнюю историю активных пожаров MODIS (1 км, с 2000 г.) и VIIRS (375 м, с 2012 г.),
отсеять шум (факелы, заводы, блики, одиночные артефакты) и обучить три ИИ:

| ИИ | Где работает | Что делает |
|---|---|---|
| **№1 Edge Detector** | на борту спутника | по первому снимку: настоящий ли это пожар и разрастётся ли он → приоритет тревоги |
| **№2 Link AI** | на борту спутника | заранее предсказывает потерю связи (дождь, облака, мерцания, уход за горизонт) и выбирает канал радио/лазер |
| **№3 Ground Forecast** | на Земле («суперкомпьютер») | прогноз риска на 1–4 недели, погода NASA POWER, распространение по ветру, задержки спутников |

**Структура ноутбука**
1. **Historical Intelligence** — MODIS + VIIRS, H3, фильтрация постоянных источников, гармонизация 26 лет.
2. **Edge AI Simulation** — небольшая модель оценивает новые наблюдения, сравнивает с историей и формирует короткие приоритетные тревоги.
3. **Smart Alert Delivery** — окна связи, задержки, сначала важные сообщения, результат на карте.

> ⚙️ На Kaggle включите **Settings → Internet: On**. Ключи не обязательны: всё основное работает на открытых данных NASA.
> Секреты (по желанию) — **Add-ons → Secrets**: `GEMINI_API_KEY`, `FIRMS_MAP_KEY`, `SUPABASE_URL`, `SUPABASE_PUBLISHABLE_KEY`, `HELIOS_INGEST_TOKEN`.

Фильтрация основана на открытых проектах **flare-chronos** (MIT), **MODIS-VIIRS-fires** (Franklin et al. 2025, MIT) и идеях **Kiln-Watch** — и улучшена (см. `helios/processing/filters.py`).

In [ ]:
!pip install -q h3 sgp4 onnx onnxruntime onnxscript google-genai python-dotenv pyarrow duckdb h3ronpy lightgbm shapely polars ziglang

In [ ]:
import os, sys, pathlib
ROOT = pathlib.Path('/kaggle/working') if pathlib.Path('/kaggle').exists() else pathlib.Path.cwd()
os.chdir(ROOT)
os.environ['HELIOS_ROOT'] = str(ROOT)
for d in ['config', 'helios/apis', 'helios/processing', 'helios/comms', 'helios/models', 'helios/db', 'helios/world', 'helios/onboard']:
    (ROOT / d).mkdir(parents=True, exist_ok=True)
# секреты Kaggle -> переменные окружения (если заданы)
try:
    from kaggle_secrets import UserSecretsClient
    sec = UserSecretsClient()
    for k in ['GEMINI_API_KEY', 'FIRMS_MAP_KEY', 'SUPABASE_URL', 'SUPABASE_PUBLISHABLE_KEY', 'HELIOS_INGEST_TOKEN',
              'SUPABASE_SERVICE_ROLE_KEY', 'EARTHDATA_TOKEN', 'NASA_API_KEY']:
        try:
            os.environ[k] = sec.get_secret(k)
        except Exception:
            pass
except ImportError:
    pass
print('working dir:', ROOT)

## 0. Код пакета `helios` (записывается в рабочую папку)

### Конфигурация

In [ ]:
%%writefile config/settings.yaml
# HELIOS-SHIELD — глобальные настройки пайплайна.
# Регион по умолчанию: Казахстан. Чтобы сменить регион, поменяйте блок `region`
# (имя страны в архиве FIRMS + bbox) и перезапустите `python -m helios.cli all`.

project:
  name: HELIOS-SHIELD
  version: 0.1.0

region:
  id: kz
  name: Казахстан
  firms_country: Kazakhstan        # имя страны в годовом архиве FIRMS (data/country/...)
  firms_open_region: Russia_Asia   # региональная NRT-лента FIRMS без ключа
  iso3: KAZ                        # для контура страны (geoBoundaries)
  bbox: [46.4, 40.5, 87.4, 55.5]   # [запад, юг, восток, север]
  utc_offset_h: 5

history:
  start_year: 2000
  end_year: 2025                   # последний полный год научного архива FIRMS
  streams:                         # поток -> (папка архива FIRMS, первый год)
    modis: {archive: modis, first_year: 2000}
    viirs_snpp: {archive: viirs-snpp, first_year: 2012}
    viirs_noaa20: {archive: viirs-jpss1, first_year: 2018}
    viirs_noaa21: {archive: viirs-jpss2, first_year: 2024}

h3:
  res_fine: 8        # реестр постоянных источников (~0.74 км²)
  res_cell: 7        # «сгоревшие гексагоны» (cell-days, ~5.2 км²)
  res_model: 5       # прогноз ИИ №3 и бортовая таблица истории (~252 км²)
  res_weather: 3     # сетка погоды NASA POWER (~12 400 км²)

filters:
  modis_min_conf: 30            # ниже — low confidence (рекомендация FIRMS, как в kiln-watch)
  viirs_drop_low: true
  p_fire_keep: 0.45             # порог унифицированной вероятности пожара P(fire)
  static_min_months: 10         # >=10 разных месяцев в году -> постоянный источник (flare-chronos)
  static_min_years: 3           # ...и так минимум 3 года подряд/суммарно
  static_type_share: 0.5        # доля type∈{1,2,3} в ячейке (маска статичных источников)
  hdbscan_min_cluster: 5        # годовая кластеризация факелов (MODIS-VIIRS-fires)
  hdbscan_recurrence_years: 4   # кластер на том же месте >=4 лет -> постоянный источник
  isolation_radius_km: 3.0      # одиночная точка без соседей в радиусе/окне -> шум
  isolation_window_days: 2
  glint_angle_deg: 12.0         # угол солнечного блика, ниже — риск ложной точки днём
  saa_bbox: [-110.0, -55.0, 11.0, 7.0]   # Южно-Атлантическая аномалия [W, S, E, N]

harmonization:
  reference: MODIS_A            # общая шкала — Aqua MODIS (летает с 2002 г. всё время)
  bridges:                      # поток -> (опорный поток, окно перекрытия)
    MODIS_T: {ref: MODIS_A, start: "2003-01", end: "2019-12"}
    VIIRS_SNPP: {ref: MODIS_A, start: "2012-03", end: "2020-12"}
    VIIRS_NOAA20: {ref: VIIRS_SNPP, start: "2018-06", end: "2025-12"}
    VIIRS_NOAA21: {ref: VIIRS_NOAA20, start: "2024-02", end: "2026-12"}
  denial_train: ["2012-03", "2016-12"]   # тест «MODIS выключили»: учимся до 2016...
  denial_test: ["2017-01", "2020-12"]    # ...и восстанавливаем 2017-2020 только по VIIRS

models:
  edge:                         # ИИ №1 (на борту): критичность обнаружения
    history_until_year: 2018    # бортовая таблица истории строится только по 2000-2018
    train_years: [2019, 2023]
    val_year: 2024
    test_year: 2025
    escalation_window_h: 72
    escalation_radius_km: 10
    escalation_min_celldays: 6
  link:                         # ИИ №2 (на борту): прогноз потери связи
    sim_days: 10
    step_s: 30
    min_elevation_deg: 5
  ground:                       # ИИ №3 (Земля): прогноз пожарного риска
    train_years: [2013, 2022]
    val_year: 2023
    test_years: [2024, 2025]
    horizon_days: 7
    negative_sample_rate: 0.15

apis:
  firms: https://firms.modaps.eosdis.nasa.gov
  power: https://power.larc.nasa.gov/api/temporal/daily/point
  open_meteo_forecast: https://api.open-meteo.com/v1/forecast
  open_meteo_archive: https://archive-api.open-meteo.com/v1/archive
  celestrak: https://celestrak.org/NORAD/elements/gp.php
  swpc: https://services.swpc.noaa.gov
  gfz: https://kp.gfz-potsdam.de/app/json/
  donki: https://api.nasa.gov/DONKI
  donki_mirror: https://kauai.ccmc.gsfc.nasa.gov/DONKI/WS/get
  cmr: https://cmr.earthdata.nasa.gov/search
  eonet: https://eonet.gsfc.nasa.gov/api/v3
  gibs_wmts: https://gibs.earthdata.nasa.gov/wmts/epsg3857/best
  geoboundaries: https://www.geoboundaries.org/api/current/gbOpen
  gemini_model: gemini-2.5-flash
  gemini_fallback_models: [gemini-flash-latest, gemini-2.0-flash]

### Ядро и коннекторы к API (NASA FIRMS, POWER, CMR, EONET, GIBS, DONKI; CelesTrak, NOAA SWPC, GFZ, Open-Meteo, Gemini)

In [ ]:
%%writefile helios/__init__.py
"""HELIOS-SHIELD: гармонизация пожаров MODIS + VIIRS, три ИИ и умная доставка алертов.

ИИ №1 (борт)  — models.edge_detector: критичность нового обнаружения.
ИИ №2 (борт)  — models.link_ai: прогноз потери связи и выбор канала (радио/лазер).
ИИ №3 (Земля) — models.ground_forecast: прогноз пожарного риска, погода, задержки спутников.
"""

__version__ = "0.1.0"

In [ ]:
%%writefile helios/config.py
"""Настройки проекта: YAML-конфиг + переменные окружения (.env)."""

from __future__ import annotations

import os
from functools import lru_cache
from pathlib import Path

import yaml
from dotenv import load_dotenv

ROOT = Path(os.getenv("HELIOS_ROOT", Path(__file__).resolve().parent.parent))
load_dotenv(ROOT / ".env")

DATA = Path(os.getenv("HELIOS_DATA_DIR", ROOT / "data"))
RAW = DATA / "raw"
PROCESSED = DATA / "processed"
CACHE = DATA / "cache"
ARTIFACTS = Path(os.getenv("HELIOS_ARTIFACTS_DIR", ROOT / "artifacts"))
MODELS_DIR = ARTIFACTS / "models"
REPORTS_DIR = ARTIFACTS / "reports"


@lru_cache(maxsize=1)
def settings() -> dict:
    path = Path(os.getenv("HELIOS_CONFIG", ROOT / "config" / "settings.yaml"))
    with open(path, encoding="utf-8") as f:
        return yaml.safe_load(f)


def env(name: str, default: str | None = None) -> str | None:
    value = os.getenv(name)
    return value if value not in (None, "") else default


def region() -> dict:
    return settings()["region"]


def ensure_dirs() -> None:
    for d in (RAW, PROCESSED, CACHE, MODELS_DIR, REPORTS_DIR):
        d.mkdir(parents=True, exist_ok=True)

In [ ]:
%%writefile helios/utils.py
"""Общие утилиты: логирование, HTTP с повторами и кэшем, работа со временем."""

from __future__ import annotations

import hashlib
import json
import logging
import sys
import time
from pathlib import Path
from typing import Any

import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

from .config import CACHE

_LOG_READY = False


def get_logger(name: str = "helios") -> logging.Logger:
    global _LOG_READY
    if not _LOG_READY:
        try:  # Windows-консоль: печатаем кириллицу без ошибок
            sys.stdout.reconfigure(encoding="utf-8")  # type: ignore[attr-defined]
            sys.stderr.reconfigure(encoding="utf-8")  # type: ignore[attr-defined]
        except Exception:
            pass
        logging.basicConfig(
            level=logging.INFO,
            format="%(asctime)s %(levelname)s %(name)s: %(message)s",
            datefmt="%H:%M:%S",
        )
        _LOG_READY = True
    return logging.getLogger(name)


log = get_logger()
_SESSION: requests.Session | None = None


def http() -> requests.Session:
    """Общая HTTP-сессия с повторами на 429/5xx и нейтральным User-Agent."""
    global _SESSION
    if _SESSION is None:
        s = requests.Session()
        retry = Retry(
            total=5,
            backoff_factor=1.5,
            status_forcelist=(429, 500, 502, 503, 504),
            allowed_methods=frozenset(["GET", "POST"]),
            raise_on_status=False,
        )
        adapter = HTTPAdapter(max_retries=retry, pool_maxsize=16)
        s.mount("https://", adapter)
        s.mount("http://", adapter)
        s.headers["User-Agent"] = "HELIOS-SHIELD/0.1 (research client)"
        _SESSION = s
    return _SESSION


def _cache_path(key: str, suffix: str = ".json") -> Path:
    digest = hashlib.sha1(key.encode("utf-8")).hexdigest()[:20]
    CACHE.mkdir(parents=True, exist_ok=True)
    return CACHE / f"{digest}{suffix}"


def get_json(
    url: str,
    params: dict | None = None,
    timeout: float = 60,
    ttl_s: float | None = None,
    headers: dict | None = None,
) -> Any:
    """GET → JSON. При ttl_s ответ кэшируется на диск (повторные запуски не дёргают API)."""
    key = url + "?" + json.dumps(params or {}, sort_keys=True)
    path = _cache_path(key)
    if ttl_s is not None and path.exists() and time.time() - path.stat().st_mtime < ttl_s:
        return json.loads(path.read_text(encoding="utf-8"))
    r = http().get(url, params=params, timeout=timeout, headers=headers)
    r.raise_for_status()
    data = r.json()
    if ttl_s is not None:
        path.write_text(json.dumps(data), encoding="utf-8")
    return data


def get_text(url: str, params: dict | None = None, timeout: float = 60, ttl_s: float | None = None) -> str:
    key = url + "?" + json.dumps(params or {}, sort_keys=True)
    path = _cache_path(key, ".txt")
    if ttl_s is not None and path.exists() and time.time() - path.stat().st_mtime < ttl_s:
        return path.read_text(encoding="utf-8")
    r = http().get(url, params=params, timeout=timeout)
    r.raise_for_status()
    if ttl_s is not None:
        path.write_text(r.text, encoding="utf-8")
    return r.text


def download(url: str, dest: Path, timeout: float = 300, skip_existing: bool = True) -> Path | None:
    """Скачать файл. Возвращает None, если на сервере файла нет (404)."""
    dest = Path(dest)
    if skip_existing and dest.exists() and dest.stat().st_size > 0:
        return dest
    dest.parent.mkdir(parents=True, exist_ok=True)
    r = http().get(url, timeout=timeout, stream=True)
    if r.status_code == 404:
        return None
    r.raise_for_status()
    tmp = dest.with_suffix(dest.suffix + ".part")
    with open(tmp, "wb") as f:
        for chunk in r.iter_content(chunk_size=1 << 20):
            f.write(chunk)
    tmp.replace(dest)
    return dest


def save_json(obj: Any, path: Path) -> Path:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(obj, ensure_ascii=False, indent=2, default=_json_default), encoding="utf-8")
    return path


def _json_default(o: Any) -> Any:
    try:
        import numpy as np

        if isinstance(o, np.integer):
            return int(o)
        if isinstance(o, np.floating):
            return None if np.isnan(o) else float(o)
        if isinstance(o, np.ndarray):
            return o.tolist()
        if isinstance(o, np.bool_):
            return bool(o)
    except ImportError:
        pass
    if hasattr(o, "isoformat"):
        return o.isoformat()
    return str(o)

In [ ]:
%%writefile helios/apis/__init__.py
"""Коннекторы к внешним API (NASA FIRMS, POWER, CMR, EONET, GIBS, CelesTrak, NOAA SWPC, GFZ, Open-Meteo, Gemini)."""

In [ ]:
%%writefile helios/apis/firms.py
"""NASA FIRMS — активные пожары MODIS (1 км) и VIIRS (375 м).

Три способа получить данные:
1. Годовой архив по стране (научное качество, с полем `type`) — БЕЗ ключа.
   https://firms.modaps.eosdis.nasa.gov/data/country/{stream}/{year}/{stream}_{year}_{Country}.csv
2. Открытые NRT-ленты (24h / 48h / 7d по регионам) — БЕЗ ключа.
3. Area API с MAP_KEY (любой bbox, любые даты, NRT и SP) — ключ бесплатный:
   https://firms.modaps.eosdis.nasa.gov/api/map_key/

Все источники приводятся к единой схеме (`normalize`), чтобы MODIS и VIIRS шли
через одни и те же фильтры.
"""

from __future__ import annotations

import io
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import date, timedelta

import numpy as np
import pandas as pd

from ..config import PROCESSED, RAW, env, settings
from ..utils import download, get_logger, get_json, http

log = get_logger("helios.firms")

# Открытые NRT-ленты FIRMS (без ключа): папка -> шаблон имени файла
OPEN_FEEDS = {
    "MODIS": ("modis-c6.1", "MODIS_C6_1_{region}_{period}.csv"),
    "VIIRS_SNPP": ("suomi-npp-viirs-c2", "SUOMI_VIIRS_C2_{region}_{period}.csv"),
    "VIIRS_NOAA20": ("noaa-20-viirs-c2", "J1_VIIRS_C2_{region}_{period}.csv"),
    "VIIRS_NOAA21": ("noaa-21-viirs-c2", "J2_VIIRS_C2_{region}_{period}.csv"),
}

# Источники Area API (MAP_KEY)
AREA_SOURCES = {
    ("MODIS", "nrt"): "MODIS_NRT",
    ("MODIS", "sp"): "MODIS_SP",
    ("VIIRS_SNPP", "nrt"): "VIIRS_SNPP_NRT",
    ("VIIRS_SNPP", "sp"): "VIIRS_SNPP_SP",
    ("VIIRS_NOAA20", "nrt"): "VIIRS_NOAA20_NRT",
    ("VIIRS_NOAA20", "sp"): "VIIRS_NOAA20_SP",
    ("VIIRS_NOAA21", "nrt"): "VIIRS_NOAA21_NRT",
}

ARCHIVE_TO_STREAM = {
    "modis": "MODIS",
    "viirs-snpp": "VIIRS_SNPP",
    "viirs-jpss1": "VIIRS_NOAA20",
    "viirs-jpss2": "VIIRS_NOAA21",
}

SCHEMA = [
    "lat", "lon", "acq_dt", "sensor", "stream", "bright_mir", "bright_tir", "frp",
    "scan", "track", "conf_pct", "conf_cat", "daynight", "version", "type", "quality",
]


# ----------------------------------------------------------------------------- нормализация
def normalize(df: pd.DataFrame, stream: str, quality: str) -> pd.DataFrame:
    """Привести CSV FIRMS (любой источник) к единой схеме SCHEMA."""
    if df is None or df.empty:
        return pd.DataFrame(columns=SCHEMA)
    is_modis = "brightness" in df.columns
    out = pd.DataFrame(index=df.index)
    out["lat"] = pd.to_numeric(df["latitude"], errors="coerce")
    out["lon"] = pd.to_numeric(df["longitude"], errors="coerce")
    hhmm = pd.to_numeric(df["acq_time"], errors="coerce").fillna(0).astype(int).astype(str).str.zfill(4)
    out["acq_dt"] = pd.to_datetime(
        df["acq_date"].astype(str) + " " + hhmm, format="%Y-%m-%d %H%M", utc=True, errors="coerce"
    )
    if is_modis:
        sat = df["satellite"].astype(str).str.strip().str[0].str.upper()  # Terra/Aqua или T/A
        out["sensor"] = "MODIS"
        out["stream"] = np.where(sat == "A", "MODIS_A", "MODIS_T")
        out["bright_mir"] = pd.to_numeric(df["brightness"], errors="coerce")
        out["bright_tir"] = pd.to_numeric(df["bright_t31"], errors="coerce")
        conf = pd.to_numeric(df["confidence"], errors="coerce")
        out["conf_pct"] = conf
        out["conf_cat"] = np.select([conf < 30, conf < 80], ["l", "n"], "h")
    else:
        out["sensor"] = "VIIRS"
        out["stream"] = stream
        out["bright_mir"] = pd.to_numeric(df["bright_ti4"], errors="coerce")
        out["bright_tir"] = pd.to_numeric(df["bright_ti5"], errors="coerce")
        out["conf_pct"] = np.nan
        # архив: l/n/h; NRT: low/nominal/high
        out["conf_cat"] = df["confidence"].astype(str).str.strip().str.lower().str[0]
    out["frp"] = pd.to_numeric(df["frp"], errors="coerce").clip(lower=0)
    out["scan"] = pd.to_numeric(df["scan"], errors="coerce")
    out["track"] = pd.to_numeric(df["track"], errors="coerce")
    out["daynight"] = df["daynight"].astype(str).str.strip().str.upper().str[0]
    out["version"] = df["version"].astype(str)
    out["type"] = pd.to_numeric(df["type"], errors="coerce") if "type" in df.columns else np.nan
    out["quality"] = quality
    out = out.dropna(subset=["lat", "lon", "acq_dt"])
    for c in ("bright_mir", "bright_tir", "frp", "scan", "track", "conf_pct"):
        out[c] = out[c].astype("float32")
    out["type"] = out["type"].astype("float32")
    for c in ("sensor", "stream", "conf_cat", "daynight", "quality"):
        out[c] = out[c].astype("category")
    return out[SCHEMA].reset_index(drop=True)


def clip_bbox(df: pd.DataFrame, bbox: list[float] | None = None) -> pd.DataFrame:
    w, s, e, n = bbox or settings()["region"]["bbox"]
    m = df["lon"].between(w, e) & df["lat"].between(s, n)
    return df.loc[m].reset_index(drop=True)


# ----------------------------------------------------------------------------- годовой архив
def archive_url(archive: str, year: int, country: str) -> str:
    base = settings()["apis"]["firms"]
    return f"{base}/data/country/{archive}/{year}/{archive}_{year}_{country}.csv"


def download_country_archive(
    country: str | None = None,
    start_year: int | None = None,
    end_year: int | None = None,
    workers: int = 4,
) -> list[tuple[str, int, str]]:
    """Скачать годовые CSV по стране для всех потоков (MODIS, VIIRS SNPP/NOAA-20/NOAA-21)."""
    cfg = settings()
    country = country or cfg["region"]["firms_country"]
    start_year = start_year or cfg["history"]["start_year"]
    end_year = end_year or cfg["history"]["end_year"]
    jobs = []
    for meta in cfg["history"]["streams"].values():
        for year in range(max(start_year, meta["first_year"]), end_year + 1):
            jobs.append((meta["archive"], year))

    done: list[tuple[str, int, str]] = []

    def _one(archive: str, year: int):
        dest = RAW / "firms" / country / archive / f"{year}.csv"
        path = download(archive_url(archive, year, country), dest)
        return archive, year, ("ok" if path else "missing")

    with ThreadPoolExecutor(max_workers=workers) as pool:
        futs = [pool.submit(_one, a, y) for a, y in jobs]
        for f in as_completed(futs):
            try:
                archive, year, status = f.result()
            except Exception as exc:  # сеть/сервер — не валим весь пайплайн
                log.warning("archive download failed: %s", exc)
                continue
            done.append((archive, year, status))
            log.info("FIRMS %s %s: %s", archive, year, status)
    return sorted(done)


def load_country_archive(country: str | None = None, cache: bool = True) -> pd.DataFrame:
    """Собрать все скачанные годовые CSV в один DataFrame единой схемы (+ parquet-кэш)."""
    country = country or settings()["region"]["firms_country"]
    out_path = PROCESSED / f"firms_archive_{country}.parquet"
    if cache and out_path.exists():
        return pd.read_parquet(out_path)
    frames = []
    for archive, stream in ARCHIVE_TO_STREAM.items():
        folder = RAW / "firms" / country / archive
        for csv in sorted(folder.glob("*.csv")) if folder.exists() else []:
            raw = pd.read_csv(csv, low_memory=False)
            frames.append(normalize(raw, stream, "sp"))
    if not frames:
        raise FileNotFoundError("Архив FIRMS не скачан: запустите `python -m helios.cli fetch-history`")
    df = pd.concat(frames, ignore_index=True)
    for c in ("sensor", "stream", "conf_cat", "daynight", "quality"):
        df[c] = df[c].astype("category")
    df = df.sort_values("acq_dt").reset_index(drop=True)
    PROCESSED.mkdir(parents=True, exist_ok=True)
    df.to_parquet(out_path, index=False)
    log.info("FIRMS archive: %d detections -> %s", len(df), out_path)
    return df


# ----------------------------------------------------------------------------- NRT без ключа
def _get_with_deadline(url: str, deadline_s: float = 180.0) -> tuple[int, str]:
    """GET с общим лимитом времени: сервер иногда отдаёт файл «по капле», и read-timeout не срабатывает."""
    t0 = time.time()
    with http().get(url, timeout=(15, 60), stream=True) as r:
        if r.status_code != 200:
            return r.status_code, ""
        chunks = []
        for chunk in r.iter_content(chunk_size=1 << 16):
            chunks.append(chunk)
            if time.time() - t0 > deadline_s:
                raise TimeoutError(f"загрузка дольше {deadline_s:.0f} с — пропускаю")
        return 200, b"".join(chunks).decode("utf-8", errors="replace")


def fetch_open_nrt(period: str = "48h", region: str | None = None, clip: bool = True) -> pd.DataFrame:
    """Свежие пожары из открытых лент FIRMS (24h | 48h | 7d) по всем спутникам."""
    region = region or settings()["region"]["firms_open_region"]
    base = settings()["apis"]["firms"]
    frames = []
    for stream, (folder, pattern) in OPEN_FEEDS.items():
        url = f"{base}/data/active_fire/{folder}/csv/{pattern.format(region=region, period=period)}"
        try:
            status, text = _get_with_deadline(url)
            if status != 200 or len(text) < 120:
                log.info("NRT %s: нет данных (%s)", stream, status)
                continue
            raw = pd.read_csv(io.StringIO(text))
        except Exception as exc:
            log.warning("NRT %s failed: %s", stream, exc)
            continue
        frames.append(normalize(raw, stream, "nrt"))
    if not frames:
        return pd.DataFrame(columns=SCHEMA)
    df = pd.concat(frames, ignore_index=True)
    return clip_bbox(df) if clip else df


# ----------------------------------------------------------------------------- Area API (MAP_KEY)
def map_key() -> str | None:
    return env("FIRMS_MAP_KEY")


def mapkey_status(key: str | None = None) -> dict:
    key = key or map_key()
    if not key:
        return {"ok": False, "reason": "FIRMS_MAP_KEY не задан в .env"}
    base = settings()["apis"]["firms"]
    data = get_json(f"{base}/mapserver/mapkey_status/", params={"MAP_KEY": key}, timeout=30)
    return {"ok": True, **data}


def fetch_area(
    source: str,
    day_range: int = 5,
    on_date: date | None = None,
    bbox: list[float] | None = None,
    key: str | None = None,
) -> pd.DataFrame:
    """Один запрос Area API. source — например VIIRS_NOAA20_NRT; day_range 1..5."""
    key = key or map_key()
    if not key:
        raise RuntimeError("Нужен FIRMS_MAP_KEY (бесплатно: firms.modaps.eosdis.nasa.gov/api/map_key/)")
    w, s, e, n = bbox or settings()["region"]["bbox"]
    base = settings()["apis"]["firms"]
    url = f"{base}/api/area/csv/{key}/{source}/{w},{s},{e},{n}/{day_range}"
    if on_date:
        url += f"/{on_date.isoformat()}"
    r = http().get(url, timeout=180)
    r.raise_for_status()
    if r.text.lower().startswith("invalid") or "error" in r.text[:80].lower():
        raise RuntimeError(f"FIRMS Area API: {r.text[:200]}")
    raw = pd.read_csv(io.StringIO(r.text))
    stream = next((k[0] for k, v in AREA_SOURCES.items() if v == source), "VIIRS_NOAA20")
    quality = "sp" if source.endswith("_SP") else "nrt"
    return normalize(raw, stream, quality)


def fetch_area_range(start: date, end: date, streams: list[str] | None = None, quality: str = "nrt") -> pd.DataFrame:
    """Area API за произвольный период (кусками по 5 дней) — например, весь 2026 год."""
    streams = streams or ["MODIS", "VIIRS_NOAA20", "VIIRS_NOAA21", "VIIRS_SNPP"]
    frames = []
    for stream in streams:
        source = AREA_SOURCES.get((stream, quality))
        if not source:
            continue
        day = start
        while day <= end:
            span = min(5, (end - day).days + 1)
            try:
                frames.append(fetch_area(source, day_range=span, on_date=day))
            except Exception as exc:
                log.warning("Area API %s %s: %s", source, day, exc)
            day += timedelta(days=span)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame(columns=SCHEMA)

In [ ]:
%%writefile helios/apis/boundaries.py
"""Контур региона (geoBoundaries, OSM, ODbL) и маска H3-ячеек внутри страны."""

from __future__ import annotations

import json
from functools import lru_cache

import h3
import numpy as np

from ..config import PROCESSED, RAW, settings
from ..utils import download, get_json, get_logger

log = get_logger("helios.boundaries")


def region_geojson() -> dict:
    """GeoJSON-геометрия страны. Если сеть недоступна — прямоугольник bbox."""
    reg = settings()["region"]
    path = RAW / "boundaries" / f"{reg['iso3']}_ADM0_simplified.geojson"
    try:
        if not path.exists():
            meta = get_json(f"{settings()['apis']['geoboundaries']}/{reg['iso3']}/ADM0/", ttl_s=30 * 86400)
            download(meta["simplifiedGeometryGeoJSON"], path)
        fc = json.loads(path.read_text(encoding="utf-8"))
        geom = fc["features"][0]["geometry"] if fc.get("type") == "FeatureCollection" else fc
        return geom
    except Exception as exc:
        log.warning("geoBoundaries недоступен (%s) — использую bbox", exc)
        w, s, e, n = reg["bbox"]
        return {"type": "Polygon", "coordinates": [[[w, s], [e, s], [e, n], [w, n], [w, s]]]}


@lru_cache(maxsize=8)
def region_cells(res: int) -> frozenset[str]:
    """Все H3-ячейки разрешения `res`, центр которых внутри страны (с кэшем на диск)."""
    reg = settings()["region"]
    path = PROCESSED / f"region_cells_{reg['id']}_r{res}.json"
    if path.exists():
        return frozenset(json.loads(path.read_text(encoding="utf-8")))
    cells = h3.geo_to_cells(region_geojson(), res)
    PROCESSED.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(sorted(cells)), encoding="utf-8")
    log.info("region mask res %d: %d cells", res, len(cells))
    return frozenset(cells)


def in_region(lat: np.ndarray, lon: np.ndarray, res: int = 5) -> np.ndarray:
    """Маска точек внутри страны (с буфером в одно кольцо H3, чтобы не терять приграничные пожары)."""
    base = region_cells(res)
    buffered = set(base)
    for c in base:
        buffered.update(h3.grid_disk(c, 1))
    cells = [h3.latlng_to_cell(float(a), float(b), res) for a, b in zip(lat, lon)]
    return np.fromiter((c in buffered for c in cells), dtype=bool, count=len(cells))

In [ ]:
%%writefile helios/apis/nasa_power.py
"""NASA POWER (LaRC) — суточная погода и облачность для любой точки с 1981 г., без ключа.

Используем для ИИ №3 (погодные драйверы пожаров) и детектора слепых зон (CLOUD_AMT).
Параметры:
  T2M_MAX      — макс. температура, °C
  RH2M         — относительная влажность, %
  WS10M        — ветер 10 м, м/с
  PRECTOTCORR  — осадки, мм/сут
  CLOUD_AMT    — облачность, %
  GWETTOP      — влажность верхнего слоя почвы (0..1)
"""

from __future__ import annotations

from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import date

import numpy as np
import pandas as pd

from ..config import RAW, settings
from ..utils import get_json, get_logger

log = get_logger("helios.power")

PARAMS = ["T2M_MAX", "RH2M", "WS10M", "PRECTOTCORR", "CLOUD_AMT", "GWETTOP"]


def fetch_point(lat: float, lon: float, start: date, end: date, params: list[str] | None = None) -> pd.DataFrame:
    params = params or PARAMS
    data = get_json(
        settings()["apis"]["power"],
        params={
            "parameters": ",".join(params),
            "community": "AG",
            "latitude": round(lat, 4),
            "longitude": round(lon, 4),
            "start": start.strftime("%Y%m%d"),
            "end": end.strftime("%Y%m%d"),
            "format": "JSON",
        },
        timeout=180,
    )
    block = data["properties"]["parameter"]
    df = pd.DataFrame({p: pd.Series(block.get(p, {}), dtype="float64") for p in params})
    df.index = pd.to_datetime(df.index, format="%Y%m%d")
    df.index.name = "date"
    return df.replace(-999.0, np.nan).astype("float32")


def fetch_grid(points: dict[str, tuple[float, float]], start: date, end: date, workers: int = 4) -> pd.DataFrame:
    """Погода для набора точек (обычно центры H3 res-3). Возобновляемая: готовые точки берутся с диска."""
    folder = RAW / "power" / f"{start:%Y%m%d}_{end:%Y%m%d}"
    folder.mkdir(parents=True, exist_ok=True)

    def _one(cell: str, lat: float, lon: float) -> pd.DataFrame:
        path = folder / f"{cell}.parquet"
        if path.exists():
            return pd.read_parquet(path)
        df = fetch_point(lat, lon, start, end).reset_index()
        df.insert(0, "cell", cell)
        df.to_parquet(path, index=False)
        return df

    frames, failed = [], 0
    with ThreadPoolExecutor(max_workers=workers) as pool:
        futs = {pool.submit(_one, c, la, lo): c for c, (la, lo) in points.items()}
        for i, f in enumerate(as_completed(futs), 1):
            try:
                frames.append(f.result())
            except Exception as exc:
                failed += 1
                log.warning("POWER %s: %s", futs[f], exc)
            if i % 25 == 0:
                log.info("POWER: %d/%d points", i, len(futs))
    if failed:
        log.warning("POWER: %d points failed (перезапуск докачает)", failed)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

In [ ]:
%%writefile helios/apis/open_meteo.py
"""Open-Meteo — почасовой прогноз и архив (ERA5) без ключа.

Нужен там, где NASA POWER слишком груб по времени:
* погода у наземных станций (дождь → замирания Ka-диапазона, облака → лазерный канал);
* ветер над активными пожарами (направление распространения);
* облачность в часы пролётов спутников (слепые зоны).
"""

from __future__ import annotations

from datetime import date

import pandas as pd

from ..config import settings
from ..utils import get_json

HOURLY_DEFAULT = [
    "precipitation", "cloud_cover", "cloud_cover_low", "relative_humidity_2m",
    "temperature_2m", "wind_speed_10m", "wind_direction_10m",
]


def _frames(payload, n: int) -> list[pd.DataFrame]:
    items = payload if isinstance(payload, list) else [payload]
    out = []
    for item in items[:n]:
        h = item.get("hourly", {})
        df = pd.DataFrame(h)
        if "time" in df:
            df["time"] = pd.to_datetime(df["time"], utc=True)
            df = df.set_index("time")
        out.append(df)
    return out


def forecast(lats: list[float], lons: list[float], hourly: list[str] | None = None, days: int = 7) -> list[pd.DataFrame]:
    hourly = hourly or HOURLY_DEFAULT
    data = get_json(
        settings()["apis"]["open_meteo_forecast"],
        params={
            "latitude": ",".join(f"{x:.4f}" for x in lats),
            "longitude": ",".join(f"{x:.4f}" for x in lons),
            "hourly": ",".join(hourly),
            "forecast_days": days,
            "wind_speed_unit": "kmh",
            "timezone": "GMT",
        },
        timeout=60,
        ttl_s=1800,
    )
    return _frames(data, len(lats))


def archive(lats: list[float], lons: list[float], start: date, end: date, hourly: list[str] | None = None) -> list[pd.DataFrame]:
    hourly = hourly or HOURLY_DEFAULT
    data = get_json(
        settings()["apis"]["open_meteo_archive"],
        params={
            "latitude": ",".join(f"{x:.4f}" for x in lats),
            "longitude": ",".join(f"{x:.4f}" for x in lons),
            "start_date": start.isoformat(),
            "end_date": end.isoformat(),
            "hourly": ",".join(hourly),
            "wind_speed_unit": "kmh",
            "timezone": "GMT",
        },
        timeout=120,
        ttl_s=7 * 86400,
    )
    return _frames(data, len(lats))

In [ ]:
%%writefile helios/apis/celestrak.py
"""CelesTrak — актуальные орбитальные элементы (TLE) спутников, без ключа.

Спутники наблюдения пожаров (MODIS/VIIRS) и ретрансляторы:
* STPSat-6 несёт LCRD — лазерный ретранслятор NASA на ГСО (оптическая связь до 1.2 Гбит/с);
* TDRS — радио-ретрансляторы NASA (S/Ka).
"""

from __future__ import annotations

import time
from pathlib import Path

from ..config import CACHE, settings
from ..utils import get_logger, http

log = get_logger("helios.celestrak")

# имя -> (NORAD ID, роль, сенсор)
SATELLITES: dict[str, tuple[int, str, str | None]] = {
    "TERRA": (25994, "observer", "MODIS"),
    "AQUA": (27424, "observer", "MODIS"),
    "SUOMI NPP": (37849, "observer", "VIIRS"),
    "NOAA 20": (43013, "observer", "VIIRS"),
    "NOAA 21": (54234, "observer", "VIIRS"),
    "STPSAT-6 (LCRD)": (49817, "relay_optical", None),
    "TDRS 13": (42915, "relay_radio", None),
}

# поток FIRMS -> имя спутника
STREAM_TO_SAT = {
    "MODIS_T": "TERRA",
    "MODIS_A": "AQUA",
    "VIIRS_SNPP": "SUOMI NPP",
    "VIIRS_NOAA20": "NOAA 20",
    "VIIRS_NOAA21": "NOAA 21",
}


def fetch_tle(norad: int, max_age_h: float = 12) -> tuple[str, str, str]:
    path = Path(CACHE) / f"tle_{norad}.txt"
    if path.exists() and time.time() - path.stat().st_mtime < max_age_h * 3600:
        lines = path.read_text(encoding="utf-8").splitlines()
    else:
        r = http().get(settings()["apis"]["celestrak"], params={"CATNR": norad, "FORMAT": "TLE"}, timeout=60)
        r.raise_for_status()
        lines = [ln.rstrip() for ln in r.text.splitlines() if ln.strip()]
        if len(lines) < 3 or not lines[1].startswith("1 "):
            if path.exists():  # CelesTrak недоступен — берём последний известный TLE
                lines = path.read_text(encoding="utf-8").splitlines()
            else:
                raise RuntimeError(f"TLE {norad}: {r.text[:120]}")
        else:
            path.parent.mkdir(parents=True, exist_ok=True)
            path.write_text("\n".join(lines[:3]), encoding="utf-8")
    return lines[0].strip(), lines[1], lines[2]


def fetch_all(roles: tuple[str, ...] = ("observer", "relay_optical", "relay_radio")) -> dict[str, tuple[str, str]]:
    out = {}
    for name, (norad, role, _) in SATELLITES.items():
        if role not in roles:
            continue
        try:
            _, l1, l2 = fetch_tle(norad)
            out[name] = (l1, l2)
        except Exception as exc:
            log.warning("TLE %s (%s): %s", name, norad, exc)
    return out

In [ ]:
%%writefile helios/apis/space_weather.py
"""Космическая погода: NOAA SWPC (сейчас), GFZ Potsdam (история), NASA DONKI (вспышки).

* Kp — геомагнитная активность (бури → ионосферные мерцания S/X-диапазона у полярных станций);
* поток рентгена GOES и класс вспышки (A/B/C/M/X);
* F10.7 — радиопоток Солнца (растёт шумовая температура антенны, когда Солнце рядом с лучом).
"""

from __future__ import annotations

import math
from datetime import date, datetime, timezone

import pandas as pd
import requests

from ..config import env, settings
from ..utils import get_json, get_logger

log = get_logger("helios.spaceweather")

_CLASS = {"A": 1e-8, "B": 1e-7, "C": 1e-6, "M": 1e-5, "X": 1e-4}


def flare_class_to_flux(cls: str | None) -> float:
    """'M1.2' -> 1.2e-5 Вт/м² (диапазон 0.1–0.8 нм)."""
    if not cls:
        return 1e-8
    cls = cls.strip().upper()
    try:
        return _CLASS[cls[0]] * float(cls[1:] or 1.0)
    except (KeyError, ValueError):
        return 1e-8


def flux_to_flare_class(flux: float) -> str:
    flux = max(float(flux), 1e-9)
    for letter in ("X", "M", "C", "B", "A"):
        if flux >= _CLASS[letter]:
            return f"{letter}{flux / _CLASS[letter]:.1f}"
    return "A0.1"


def kp_recent() -> pd.DataFrame:
    data = get_json(f"{settings()['apis']['swpc']}/products/noaa-planetary-k-index.json", ttl_s=900)
    if data and isinstance(data[0], list):  # старый формат: первая строка — заголовок
        df = pd.DataFrame(data[1:], columns=data[0])
    else:
        df = pd.DataFrame(data)
    df["time"] = pd.to_datetime(df["time_tag"], utc=True)
    df["kp"] = pd.to_numeric(df["Kp"] if "Kp" in df else df["kp"], errors="coerce")
    return df[["time", "kp"]]


def xray_flux_recent() -> pd.DataFrame:
    data = get_json(f"{settings()['apis']['swpc']}/json/goes/primary/xrays-1-day.json", ttl_s=600)
    df = pd.DataFrame(data)
    df = df[df["energy"] == "0.1-0.8nm"].copy()
    df["time"] = pd.to_datetime(df["time_tag"], utc=True)
    df["flux"] = pd.to_numeric(df["flux"], errors="coerce")
    return df[["time", "flux"]]


def flares_recent() -> list[dict]:
    return get_json(f"{settings()['apis']['swpc']}/json/goes/primary/xray-flares-7-day.json", ttl_s=900)


def gfz_index(index: str, start: datetime, end: datetime) -> pd.DataFrame:
    """История индексов GFZ: Kp (3 ч), ap, Fobs (F10.7, сутки), SN (число Вольфа)."""
    data = get_json(
        settings()["apis"]["gfz"],
        params={
            "start": start.strftime("%Y-%m-%dT%H:%M:%SZ"),
            "end": end.strftime("%Y-%m-%dT%H:%M:%SZ"),
            "index": index,
        },
        timeout=90,
        ttl_s=6 * 3600,
    )
    df = pd.DataFrame({"time": pd.to_datetime(data["datetime"], utc=True), index.lower(): data[index]})
    return df


def donki_flares(start: date, end: date) -> list[dict]:
    """Вспышки из NASA DONKI (ключ NASA_API_KEY или DEMO_KEY; при сбое — зеркало CCMC)."""
    key = env("NASA_API_KEY", "DEMO_KEY")
    params = {"startDate": start.isoformat(), "endDate": end.isoformat()}
    errors = []
    for url, extra in (
        (f"{settings()['apis']['donki']}/FLR", {"api_key": key}),
        (f"{settings()['apis']['donki_mirror']}/FLR", {}),
    ):
        try:  # без повторов и с коротким таймаутом: DONKI часто недоступен, не ждём минутами
            r = requests.get(url, params={**params, **extra}, timeout=15)
            r.raise_for_status()
            return r.json() or []
        except Exception as exc:
            log.info("DONKI %s: %s", url, exc)
            errors.append(f"{url.split('/')[2]}: {type(exc).__name__}")
    # не маскируем отказ пустым списком: вспышки за 7 дней всё равно есть в NOAA SWPC (flares_recent)
    raise RuntimeError("DONKI недоступен (" + "; ".join(errors) + ") — используем NOAA SWPC")


def current_conditions() -> dict:
    """Сводка «прямо сейчас» для ИИ №2 и симулятора лазер/радио."""
    out = {"time": datetime.now(timezone.utc).isoformat(), "kp": None, "xray_flux": None, "flare_class": None}
    try:
        kp = kp_recent().dropna()
        out["kp"] = float(kp["kp"].iloc[-1])
    except Exception as exc:
        log.warning("SWPC Kp: %s", exc)
    try:
        xr = xray_flux_recent().dropna()
        flux = float(xr["flux"].iloc[-1])
        out["xray_flux"] = flux
        out["flare_class"] = flux_to_flare_class(flux)
    except Exception as exc:
        log.warning("SWPC X-ray: %s", exc)
    try:
        now = datetime.now(timezone.utc)
        f107 = gfz_index("Fobs", now.replace(day=1) - pd.Timedelta(days=40), now).dropna()
        out["f107"] = float(f107["fobs"].iloc[-1])
    except Exception as exc:
        log.info("GFZ F10.7: %s", exc)
    out["solar_activity_index"] = (
        None if out["xray_flux"] is None else round(max(1.0, math.log10(out["xray_flux"] / 1e-8) * 25), 1)
    )
    return out

In [ ]:
%%writefile helios/apis/earthdata.py
"""NASA Earthdata CMR — метаданные гранул (поиск без ключа).

Здесь CMR используется для мониторинга ЗАДЕРЖЕК спутников (задача ИИ №3 из разговора
с ментором): для каждой гранулы активных пожаров берём время съёмки и время появления
в архиве. Если у спутника задержка растёт — на Земле могли «не увидеть» пожар вовремя.

EARTHDATA_TOKEN (необязательно) нужен только для скачивания самих файлов гранул.
"""

from __future__ import annotations

from datetime import datetime, timedelta, timezone

import numpy as np
import pandas as pd

from ..config import env, settings
from ..utils import get_json, get_logger

log = get_logger("helios.earthdata")

# поток -> короткое имя коллекции NRT активных пожаров в CMR
NRT_COLLECTIONS = {
    "MODIS_T": "MOD14",
    "MODIS_A": "MYD14",
    "VIIRS_SNPP": "VNP14IMG_NRT",
    "VIIRS_NOAA20": "VJ114IMG_NRT",
    "VIIRS_NOAA21": "VJ214IMG_NRT",
}


def search_granules(short_name: str, start: datetime, end: datetime, bbox: list[float] | None = None,
                    page_size: int = 500) -> list[dict]:
    params = {
        "short_name": short_name,
        "temporal": f"{start:%Y-%m-%dT%H:%M:%SZ},{end:%Y-%m-%dT%H:%M:%SZ}",
        "page_size": page_size,
        "sort_key": "-start_date",
    }
    if bbox:
        params["bounding_box"] = ",".join(str(x) for x in bbox)
    headers = {"Authorization": f"Bearer {env('EARTHDATA_TOKEN')}"} if env("EARTHDATA_TOKEN") else None
    data = get_json(f"{settings()['apis']['cmr']}/granules.json", params=params, timeout=60, headers=headers)
    return data.get("feed", {}).get("entry", [])


def granule_latency(hours: int = 24, bbox: list[float] | None = None) -> pd.DataFrame:
    """Задержка доставки (мин) по гранулам за последние `hours` часов для каждого спутника."""
    end = datetime.now(timezone.utc)
    start = end - timedelta(hours=hours)
    rows = []
    for stream, short in NRT_COLLECTIONS.items():
        try:
            entries = search_granules(short, start, end, bbox=bbox)
        except Exception as exc:
            log.warning("CMR %s: %s", short, exc)
            continue
        for e in entries:
            gid = e.get("producer_granule_id") or e.get("title", "")
            if short in ("MOD14", "MYD14") and "NRT" not in gid:
                continue  # берём только NRT-гранулы MODIS
            t_end = pd.to_datetime(e.get("time_end") or e.get("time_start"), utc=True)
            upd = pd.to_datetime(e.get("updated"), utc=True)
            rows.append({
                "stream": stream,
                "granule": gid,
                "time_start": pd.to_datetime(e.get("time_start"), utc=True),
                "available_at": upd,
                "latency_min": (upd - t_end).total_seconds() / 60.0,
                "day_night": e.get("day_night_flag"),
            })
    df = pd.DataFrame(rows)
    if df.empty:
        return df
    return df[df["latency_min"].between(0, 24 * 60)].reset_index(drop=True)


def latency_report(df: pd.DataFrame) -> pd.DataFrame:
    """Медиана/перцентили задержки и флаг аномалии (робастный z-score по свежим гранулам)."""
    if df.empty:
        return df
    out = []
    for stream, g in df.groupby("stream"):
        lat = g.sort_values("time_start")["latency_min"].to_numpy()
        med = float(np.median(lat))
        mad = float(np.median(np.abs(lat - med))) or 1.0
        recent = float(np.median(lat[-5:]))
        z = (recent - med) / (1.4826 * mad)
        out.append({
            "stream": stream,
            "granules": int(len(lat)),
            "median_latency_min": round(med, 1),
            "p90_latency_min": round(float(np.percentile(lat, 90)), 1),
            "recent_latency_min": round(recent, 1),
            "robust_z": round(float(z), 2),
            "status": "DELAYED" if z > 3 and recent > med + 30 else "OK",
            "last_granule": g["time_start"].max(),
        })
    return pd.DataFrame(out)

In [ ]:
%%writefile helios/apis/eonet.py
"""NASA EONET — каталог природных событий (лесные пожары) для сверки наших алертов."""

from __future__ import annotations

from ..config import settings
from ..utils import get_json


def wildfire_events(days: int = 30, bbox: list[float] | None = None, status: str = "all", limit: int = 200) -> list[dict]:
    w, s, e, n = bbox or settings()["region"]["bbox"]
    data = get_json(
        f"{settings()['apis']['eonet']}/events",
        params={"category": "wildfires", "days": days, "status": status, "limit": limit, "bbox": f"{w},{n},{e},{s}"},
        timeout=60,
        ttl_s=3600,
    )
    out = []
    for ev in data.get("events", []):
        for g in ev.get("geometry", []):
            if g.get("type") != "Point":
                continue
            lon, lat = g["coordinates"][:2]
            out.append({"id": ev["id"], "title": ev["title"], "date": g.get("date"), "lat": lat, "lon": lon,
                        "sources": [s_.get("url") for s_ in ev.get("sources", [])]})
    return out

In [ ]:
%%writefile helios/apis/gibs.py
"""NASA GIBS — готовые слои снимков для карты фронтенда (WMTS, без ключа)."""

from __future__ import annotations

from datetime import date

from ..config import settings

# слой -> (описание, формат, max zoom)
LAYERS = {
    "MODIS_Terra_CorrectedReflectance_TrueColor": ("Terra MODIS, истинные цвета", "jpg", 9),
    "VIIRS_NOAA20_CorrectedReflectance_TrueColor": ("NOAA-20 VIIRS, истинные цвета", "jpg", 9),
    "VIIRS_NOAA20_Thermal_Anomalies_375m_All": ("NOAA-20 VIIRS, тепловые аномалии 375 м", "png", 9),
    "VIIRS_NOAA21_Thermal_Anomalies_375m_All": ("NOAA-21 VIIRS, тепловые аномалии 375 м", "png", 9),
    "MODIS_Combined_Thermal_Anomalies_All": ("MODIS Terra+Aqua, тепловые аномалии 1 км", "png", 9),
    "MODIS_Terra_Cloud_Top_Temp_Day": ("Terra MODIS, температура верхней границы облаков", "png", 6),
}


def tile_url(layer: str, day: date | None = None) -> str:
    """Шаблон XYZ-тайла для Leaflet/MapLibre: {z}/{y}/{x}."""
    _, fmt, zoom = LAYERS.get(layer, ("", "png", 9))
    base = settings()["apis"]["gibs_wmts"]
    when = (day or date.today()).isoformat()
    matrix = f"GoogleMapsCompatible_Level{zoom}"
    return f"{base}/{layer}/default/{when}/{matrix}/{{z}}/{{y}}/{{x}}.{fmt}"


def layer_catalog(day: date | None = None) -> list[dict]:
    return [
        {"id": lid, "title": title, "max_zoom": zoom, "url": tile_url(lid, day)}
        for lid, (title, _, zoom) in LAYERS.items()
    ]

In [ ]:
%%writefile helios/apis/gemini.py
"""Gemini API — текстовый анализ рисков по клику на регион (идея Kiln-Watch, своя реализация).

Принцип «модель пишет слова, числа — из пайплайна»: в промпт уходит готовая
статистическая сводка (JSON), и модели запрещено придумывать цифры. Без ключа
GEMINI_API_KEY работает офлайн-аналитик по шаблону (ответ того же формата).
"""

from __future__ import annotations

import json
from typing import Any

from ..config import env, settings
from ..utils import get_logger

log = get_logger("helios.gemini")

SYSTEM_PROMPT = """Ты — аналитик системы HELIOS-SHIELD (гармонизированные данные пожаров NASA MODIS + VIIRS).
Правила:
- Используй ТОЛЬКО числа из JSON-сводки. Ничего не придумывай и не экстраполируй.
- Пиши по-русски, кратко: вердикт (уровень риска), 3–5 пунктов «почему», 2–3 рекомендации.
- Различай природу горения: постоянные источники (факелы, заводы) — это не лесной пожар.
- Если в сводке есть слепые зоны (облачность, мало пролётов) — обязательно предупреди:
  «Недостаточно данных, возможны пропущенные очаги».
- Прогноз ИИ — это вероятность, а не гарантия; укажи качество модели, если оно есть в сводке.
- Формат Markdown: ключевые числа и уровень риска выделяй **жирным**.
"""

RISK_WORDS = [(0.15, "низкий"), (0.35, "умеренный"), (0.6, "высокий"), (1.01, "очень высокий")]


def _risk_word(p: float | None) -> str:
    if p is None:
        return "неизвестен"
    return next(word for thr, word in RISK_WORDS if p < thr)


def offline_analysis(stats: dict[str, Any]) -> str:
    """Шаблонный анализ без внешнего API (резерв, когда нет ключа или сети)."""
    p = stats.get("risk_probability_next_7d")
    lines = [f"**Уровень риска на 7 дней: {_risk_word(p)}**" + (f" (вероятность **{p:.0%}**)" if p is not None else "")]
    why = []
    if stats.get("detections_last_7d") is not None:
        why.append(f"За последние 7 дней отфильтрованных очагов: **{stats['detections_last_7d']}**.")
    if stats.get("climatology_probability") is not None:
        why.append(f"Исторически в эту неделю года пожары здесь бывают с вероятностью **{stats['climatology_probability']:.0%}**.")
    if stats.get("fire_dna"):
        dna = stats["fire_dna"]
        why.append(f"Характер горения (Fire DNA): **{dna.get('label_ru', dna.get('class'))}**, уверенность {dna.get('confidence', 0):.0%}.")
    if stats.get("weather"):
        w = stats["weather"]
        why.append(f"Погода: t° макс {w.get('t2m_max')}°C, влажность {w.get('rh2m')}%, ветер {w.get('ws10m')} м/с, осадки {w.get('precip_7d')} мм за 7 дней.")
    if stats.get("season_shift_days_per_decade") is not None:
        why.append(f"Сдвиг пика сезона: **{stats['season_shift_days_per_decade']:+.1f} дн./10 лет**.")
    lines += [f"- {w}" for w in why] or ["- Недостаточно статистики для пояснений."]
    if stats.get("blind_spot"):
        lines.append("> ⚠️ Недостаточно данных: облачность/мало пролётов. Возможны пропущенные очаги.")
    rec = ["Сверить свежие снимки VIIRS (слой GIBS) перед выездом."]
    if p is not None and p >= 0.35:
        rec.insert(0, "Предупредить лесхоз/МЧС района, проверить готовность техники.")
    if stats.get("fire_dna", {}).get("class") == "industrial":
        rec.append("Точки похожи на промышленный источник — не учитывать как лесной пожар.")
    lines.append("**Рекомендации:**")
    lines += [f"- {r}" for r in rec]
    return "\n".join(lines)


def analyze_region(stats: dict[str, Any], question: str | None = None) -> dict[str, Any]:
    """Сводка региона -> текст анализа. Возвращает {text, model, source}."""
    key = env("GEMINI_API_KEY")
    if not key:
        return {"text": offline_analysis(stats), "model": "offline-template", "source": "offline"}
    try:
        from google import genai
        from google.genai import types
    except ImportError:
        return {"text": offline_analysis(stats), "model": "offline-template", "source": "offline"}

    client = genai.Client(api_key=key)
    user = "Сводка по региону (JSON):\n" + json.dumps(stats, ensure_ascii=False, default=str)
    if question:
        user += f"\n\nВопрос пользователя: {question}"
    cfg = settings()["apis"]
    models = [env("GEMINI_MODEL", cfg["gemini_model"]), *cfg.get("gemini_fallback_models", [])]
    last_err = None
    for model in models:
        try:
            resp = client.models.generate_content(
                model=model,
                contents=user,
                config=types.GenerateContentConfig(system_instruction=SYSTEM_PROMPT, temperature=0.3,
                                                   max_output_tokens=900),
            )
            text = (resp.text or "").strip()
            if text:
                return {"text": text, "model": model, "source": "gemini"}
        except Exception as exc:  # неверная модель/квота — пробуем следующую
            last_err = exc
            log.warning("Gemini %s: %s", model, exc)
    return {"text": offline_analysis(stats), "model": "offline-template", "source": f"offline ({last_err})"}

In [ ]:
%%writefile helios/apis/health.py
"""Проверка всех внешних API и ключей одной командой: python -m helios.cli check-apis"""

from __future__ import annotations

from datetime import date, datetime, timedelta, timezone

from ..config import env


def _try(fn):
    try:
        detail = fn()
        return {"ok": True, "detail": detail}
    except Exception as exc:
        return {"ok": False, "detail": f"{type(exc).__name__}: {str(exc)[:140]}"}


def check_all() -> dict[str, dict]:
    from . import celestrak, earthdata, eonet, firms, nasa_power, open_meteo, space_weather

    today = date.today()
    report = {
        "FIRMS archive (no key)": _try(lambda: "годовой архив по стране доступен"
                                       if firms.http().head(firms.archive_url("modis", 2024, "Kazakhstan"), timeout=30).status_code == 200
                                       else (_ for _ in ()).throw(RuntimeError("archive 404"))),
        "FIRMS NRT feeds": _try(lambda: f"{len(firms.fetch_open_nrt('24h'))} свежих точек в bbox"),
        "FIRMS MAP_KEY": _try(lambda: str(firms.mapkey_status())[:120] if env("FIRMS_MAP_KEY")
                              else (_ for _ in ()).throw(RuntimeError("FIRMS_MAP_KEY не задан (нужен для Area API)"))),
        "NASA POWER": _try(lambda: f"{len(nasa_power.fetch_point(51.1, 71.4, today - timedelta(days=10), today - timedelta(days=3)))} дней"),
        "NASA CMR latency": _try(lambda: f"{len(earthdata.granule_latency(hours=6))} гранул за 6 ч"),
        "NASA EONET": _try(lambda: f"{len(eonet.wildfire_events(days=30))} событий в регионе за 30 дней"),
        "NASA DONKI": _try(lambda: f"{len(space_weather.donki_flares(today - timedelta(days=7), today))} вспышек за 7 дней"),
        "NOAA SWPC": _try(lambda: str(space_weather.current_conditions())[:120]),
        "GFZ Kp/F10.7": _try(lambda: f"{len(space_weather.gfz_index('Kp', datetime.now(timezone.utc) - timedelta(days=2), datetime.now(timezone.utc)))} значений Kp"),
        "CelesTrak TLE": _try(lambda: ", ".join(celestrak.fetch_all().keys())),
        "Open-Meteo": _try(lambda: f"{len(open_meteo.forecast([51.1], [71.4], days=1)[0])} часов прогноза"),
        "Gemini key": _try(lambda: "задан" if env("GEMINI_API_KEY")
                           else (_ for _ in ()).throw(RuntimeError("GEMINI_API_KEY не задан — работает офлайн-аналитик"))),
        "Supabase": _try(_supabase_detail),
    }
    return report


def _supabase_detail() -> str:
    from ..db.supabase_sync import SupabaseClient

    client = SupabaseClient()
    return client.ping()

### Обработка: фильтрация шума, H3, гармонизация, аналитика

In [ ]:
%%writefile helios/processing/__init__.py
"""Обработка: фильтры шума, H3, гармонизация, аналитика."""

In [ ]:
%%writefile helios/processing/solar.py
"""Солнечная геометрия (векторно, numpy).

* Зенитный угол Солнца (SZA) и день/ночь по физике (SZA < 85°), а не по флагу NRT —
  формулы NOAA/Spencer, идея и коэффициенты из flare-chronos (MIT, (c) 2026 Xiro The Dev).
* НОВОЕ: азимут Солнца и угол солнечного блика (sun glint) — днём зеркальное отражение
  от воды/крыш даёт ложные «горячие» пиксели. Сторона полосы обзора в FIRMS не известна,
  поэтому считаем блик для обеих сторон и берём минимум (консервативно).
"""

from __future__ import annotations

import numpy as np
import pandas as pd

DAYNIGHT_SZA_DEG = 85.0


def solar_position(lat: np.ndarray, lon: np.ndarray, dt_utc: pd.Series | pd.DatetimeIndex):
    """Вернуть (sza_deg, sun_azimuth_deg) для массивов координат и времени UTC."""
    dt = pd.DatetimeIndex(dt_utc)
    doy = dt.dayofyear.to_numpy(dtype=np.float64)
    hours = dt.hour.to_numpy() + dt.minute.to_numpy() / 60.0 + dt.second.to_numpy() / 3600.0
    gamma = 2.0 * np.pi / 365.25 * (doy - 1.0 + (hours - 12.0) / 24.0)
    eqtime = 229.18 * (0.000075 + 0.001868 * np.cos(gamma) - 0.032077 * np.sin(gamma)
                       - 0.014615 * np.cos(2 * gamma) - 0.040849 * np.sin(2 * gamma))
    decl = (0.006918 - 0.399912 * np.cos(gamma) + 0.070257 * np.sin(gamma)
            - 0.006758 * np.cos(2 * gamma) + 0.000907 * np.sin(2 * gamma)
            - 0.002697 * np.cos(3 * gamma) + 0.001480 * np.sin(3 * gamma))
    tst = np.mod(hours * 60.0 + eqtime + 4.0 * np.asarray(lon, dtype=np.float64), 1440.0)
    ha = np.radians(tst / 4.0 - 180.0)
    phi = np.radians(np.asarray(lat, dtype=np.float64))
    cos_sza = np.clip(np.sin(phi) * np.sin(decl) + np.cos(phi) * np.cos(decl) * np.cos(ha), -1, 1)
    sza = np.degrees(np.arccos(cos_sza))
    az = np.degrees(np.arctan2(np.sin(ha), np.cos(ha) * np.sin(phi) - np.tan(decl) * np.cos(phi))) + 180.0
    return sza.astype(np.float32), np.mod(az, 360.0).astype(np.float32)


def daynight(sza: np.ndarray) -> np.ndarray:
    return np.where(sza < DAYNIGHT_SZA_DEG, "D", "N")


def glint_angle(sza: np.ndarray, sun_az: np.ndarray, vza: np.ndarray, track_az: np.ndarray) -> np.ndarray:
    """Минимальный угол блика (°) для двух возможных сторон полосы обзора.

    cos(g) = cos(θs)cos(θv) − sin(θs)sin(θv)cos(φ),  φ = азимут Солнца − азимут сенсора.
    Сенсор смотрит перпендикулярно трассе спутника: азимут сенсора = трасса ± 90°.
    """
    ts, tv = np.radians(sza), np.radians(vza)
    out = np.full(np.shape(sza), 180.0, dtype=np.float32)
    for side in (90.0, -90.0):
        phi = np.radians(sun_az - (track_az + side))
        cos_g = np.cos(ts) * np.cos(tv) - np.sin(ts) * np.sin(tv) * np.cos(phi)
        out = np.minimum(out, np.degrees(np.arccos(np.clip(cos_g, -1, 1))).astype(np.float32))
    return out

In [ ]:
%%writefile helios/processing/footprint.py
"""Геометрия пикселя MODIS (1 км) и VIIRS (375 м) по углу сканирования.

Формулы сферической геометрии сканирования (Giglio et al. 2016; Wolfe et al. 2013;
Schroeder et al. 2014) и обращение «размер пикселя -> угол скана» взяты из
flare-chronos (MIT, (c) 2026 Xiro The Dev) и переписаны на numpy (векторно, быстро
для миллионов точек).
"""

from __future__ import annotations

import numpy as np

R_EARTH_KM = 6378.137
MODIS_ORBIT_KM = R_EARTH_KM + 705.0
MODIS_SAMPLING_RAD = 0.0014184397
VIIRS_ORBIT_KM = R_EARTH_KM + 829.0
VIIRS_TRACK_RAD = 0.0004340
VIIRS_MAX_SCAN_DEG = 56.28


def _modis_scan_km(theta: np.ndarray) -> np.ndarray:
    ratio_sq = (R_EARTH_KM / MODIS_ORBIT_KM) ** 2
    root = np.sqrt(np.maximum(1e-9, ratio_sq - np.sin(theta) ** 2))
    return R_EARTH_KM * MODIS_SAMPLING_RAD * (np.cos(theta) / root - 1.0)


def _viirs_track_km(theta: np.ndarray) -> np.ndarray:
    ratio_sq = (R_EARTH_KM / VIIRS_ORBIT_KM) ** 2
    root = np.sqrt(np.maximum(1e-9, ratio_sq - np.sin(theta) ** 2))
    return VIIRS_ORBIT_KM * VIIRS_TRACK_RAD * (np.cos(theta) - root)


def _bisect(fn, target: np.ndarray, hi_rad: float, iters: int = 40) -> np.ndarray:
    lo = np.zeros_like(target, dtype=np.float64)
    hi = np.full_like(target, hi_rad, dtype=np.float64)
    for _ in range(iters):
        mid = 0.5 * (lo + hi)
        below = fn(mid) < target
        lo = np.where(below, mid, lo)
        hi = np.where(below, hi, mid)
    return 0.5 * (lo + hi)


def scan_angle_deg(sensor: np.ndarray, scan_km: np.ndarray, track_km: np.ndarray) -> np.ndarray:
    """Угол сканирования (°) из размеров пикселя, отдельно для MODIS и VIIRS."""
    sensor = np.asarray(sensor).astype(str)
    scan_km = np.nan_to_num(np.asarray(scan_km, dtype=np.float64), nan=1.0)
    track_km = np.nan_to_num(np.asarray(track_km, dtype=np.float64), nan=0.4)
    out = np.zeros(len(sensor), dtype=np.float64)
    m = sensor == "MODIS"
    if m.any():
        tgt = np.maximum(scan_km[m], _modis_scan_km(np.array([0.0]))[0])
        out[m] = _bisect(_modis_scan_km, tgt, np.radians(55.5))
    v = ~m
    if v.any():
        tgt = np.clip(track_km[v], _viirs_track_km(np.array([0.0]))[0], _viirs_track_km(np.array([np.radians(VIIRS_MAX_SCAN_DEG)]))[0])
        out[v] = _bisect(_viirs_track_km, tgt, np.radians(VIIRS_MAX_SCAN_DEG))
    return np.degrees(out).astype(np.float32)


def view_zenith_deg(sensor: np.ndarray, scan_deg: np.ndarray) -> np.ndarray:
    """Зенитный угол визирования на земле: sin(vza) = (r / R_e) * sin(scan)."""
    sensor = np.asarray(sensor).astype(str)
    r = np.where(sensor == "MODIS", MODIS_ORBIT_KM, VIIRS_ORBIT_KM)
    s = np.clip(r / R_EARTH_KM * np.sin(np.radians(scan_deg)), -1, 1)
    return np.degrees(np.arcsin(s)).astype(np.float32)


def track_azimuth_deg(lat: np.ndarray, stream: np.ndarray, daynight: np.ndarray) -> np.ndarray:
    """Азимут подспутниковой трассы солнечно-синхронной орбиты (адаптировано из flare-chronos)."""
    stream = np.asarray(stream).astype(str)
    inc = np.where(np.char.startswith(stream, "MODIS"), 98.2, 98.7)
    cos_lat = np.maximum(0.15, np.cos(np.radians(lat)))
    base = np.degrees(np.arcsin(np.clip(np.cos(np.radians(inc)) / cos_lat, -1, 1)))
    terra = stream == "MODIS_T"
    day = np.asarray(daynight).astype(str) == "D"
    ascending = np.where(terra, ~day, day)  # Terra — утренняя нисходящая, остальные — дневные восходящие
    return np.where(ascending, np.mod(360.0 + base, 360.0), np.mod(180.0 - base, 360.0)).astype(np.float32)


def pixel_area_km2(scan_km: np.ndarray, track_km: np.ndarray) -> np.ndarray:
    return (np.asarray(scan_km, dtype=np.float32) * np.asarray(track_km, dtype=np.float32)).astype(np.float32)

In [ ]:
%%writefile helios/processing/persistence.py
"""Реестр постоянных источников тепла (факелы нефтепромыслов, заводы, ТЭЦ, котельные).

Собрано из трёх проектов и усилено:
1. Маска по полю FIRMS `type` (1 вулкан, 2 статичный наземный источник, 3 офшор):
   ячейка статична, если ≥50% типизированных точек — 1/2/3 (идея Kiln-Watch, своя реализация).
   Маска переносится на NRT-данные, где поля `type` нет.
2. Временная устойчивость: точка горит ≥10 разных месяцев в году (flare-chronos, MIT)
   ИЛИ ≥6 месяцев в году на протяжении ≥3 лет (НОВОЕ: ловит факелы с перерывами).
3. Пространственная повторяемость: годовая кластеризация HDBSCAN (MODIS-VIIRS-fires,
   Franklin et al. 2025, MIT); плотный компактный кластер на том же месте ≥4 лет —
   постоянный источник (НОВОЕ: проверка повторяемости между годами и компактности).
4. НОВОЕ: итоговый static_score 0..1 из всех улик + доля ночных обнаружений
   (факелы горят и ночью, а сельхозпалы — днём) + стабильность FRP.
"""

from __future__ import annotations

import h3
import numpy as np
import pandas as pd
from sklearn.cluster import HDBSCAN

from ..config import settings
from ..utils import get_logger

log = get_logger("helios.persistence")


def _project_km(lat: np.ndarray, lon: np.ndarray, lat0: float) -> np.ndarray:
    return np.column_stack([np.radians(lon) * 6371.0 * np.cos(np.radians(lat0)), np.radians(lat) * 6371.0])


def hdbscan_recurrent_cells(df: pd.DataFrame, min_cluster: int, min_years: int, res: int = 7) -> pd.DataFrame:
    """Годовая HDBSCAN-кластеризация плотных точек -> ячейки res-7, где кластер повторяется ≥ min_years лет."""
    rows = []
    rng = np.random.default_rng(42)
    for year, g in df.groupby(df["acq_dt"].dt.year):
        # кандидаты: res-8 ячейки, горевшие ≥3 РАЗНЫХ дней за год (пал поля горит 1–2 дня,
        # факел — постоянно). Порог поднимается, пока точек не станет ≤40k: sklearn-HDBSCAN O(n²).
        day = g["acq_dt"].dt.normalize()
        days_per_cell = pd.DataFrame({"h3_r8": g["h3_r8"], "day": day}).drop_duplicates().groupby("h3_r8").size()
        days = g["h3_r8"].map(days_per_cell).to_numpy()
        # не больше 6 точек на ячейку: плотность кластера сохраняется, а n падает в разы
        rank = pd.Series(rng.random(len(g)), index=g.index).groupby(g["h3_r8"]).rank(method="first").to_numpy()
        min_days = 3
        cand = g[(days >= min_days) & (rank <= 6)]
        while len(cand) > 40_000 and min_days < 30:
            min_days += 2
            cand = g[(days >= min_days) & (rank <= 6)]
        if len(cand) < min_cluster:
            continue
        xy = _project_km(cand["lat"].to_numpy(), cand["lon"].to_numpy(), float(cand["lat"].mean()))
        try:  # параметр copy есть в свежих sklearn; на старых (Kaggle) — без него
            model = HDBSCAN(min_cluster_size=min_cluster, min_samples=3, copy=True)
        except TypeError:
            model = HDBSCAN(min_cluster_size=min_cluster, min_samples=3)
        labels = model.fit_predict(xy)
        cand = cand.assign(cl=labels)
        for cl, c in cand[cand.cl >= 0].groupby("cl"):
            spread = float(np.sqrt(np.var(xy[labels == cl], axis=0).sum()))
            if spread > 1.5:  # размазанный кластер — это не точечный источник
                continue
            rows.append({"year": int(year), "lat": float(c.lat.mean()), "lon": float(c.lon.mean()),
                         "n": int(len(c)), "spread_km": spread})
    if not rows:
        return pd.DataFrame(columns=["h3_r7", "hdb_years"])
    clusters = pd.DataFrame(rows)
    clusters["h3_r7"] = [h3.latlng_to_cell(a, b, res) for a, b in zip(clusters.lat, clusters.lon)]
    # повторяемость: учитываем соседние ячейки (k=1), центроиды «гуляют» на ~1 км
    years_by_cell: dict[str, set[int]] = {}
    for cell, year in zip(clusters.h3_r7, clusters.year):
        for nb in h3.grid_disk(cell, 1):
            years_by_cell.setdefault(nb, set()).add(year)
    centers = clusters.groupby("h3_r7").agg(n_clusters=("n", "size")).reset_index()
    centers["hdb_years"] = [len(years_by_cell.get(c, ())) for c in centers.h3_r7]
    return centers[centers.hdb_years >= min_years][["h3_r7", "hdb_years"]]


def build_static_registry(df: pd.DataFrame) -> pd.DataFrame:
    """Реестр статичных res-8 ячеек. df: нормализованные точки + h3_r8, h3_r7, daynight_sza."""
    f = settings()["filters"]
    d = df[["h3_r8", "h3_r7", "lat", "lon", "acq_dt", "type", "frp", "daynight_sza"]].copy()
    d["year"] = d["acq_dt"].dt.year
    d["month"] = d["acq_dt"].dt.month
    d["is_typed"] = d["type"].notna()
    d["is_static_type"] = d["type"].isin([1.0, 2.0, 3.0])
    d["is_night"] = d["daynight_sza"] == "N"

    g = d.groupby("h3_r8")
    reg = pd.DataFrame({
        "n": g.size(),
        "lat": g["lat"].mean(),
        "lon": g["lon"].mean(),
        "n_typed": g["is_typed"].sum(),
        "n_static_type": g["is_static_type"].sum(),
        "night_share": g["is_night"].mean(),
        "frp_median": g["frp"].median(),
        "frp_cv": g["frp"].std() / g["frp"].mean().replace(0, np.nan),
        "first_year": g["year"].min(),
        "last_year": g["year"].max(),
        "n_years": g["year"].nunique(),
    })
    # (1) маска по полю type
    reg["static_type_share"] = (reg.n_static_type / reg.n_typed.replace(0, np.nan)).fillna(0)
    reg["ev_type"] = (reg.static_type_share >= f["static_type_share"]) & (reg.n_typed >= 3)

    # (2) временная устойчивость: месяцы в году
    months = d.groupby(["h3_r8", "year"])["month"].nunique().rename("months").reset_index()
    max_months = months.groupby("h3_r8")["months"].max()
    years6 = months[months.months >= 6].groupby("h3_r8").size()
    reg["max_months_in_year"] = max_months
    reg["years_with_6m"] = years6.reindex(reg.index).fillna(0).astype(int)
    reg["ev_temporal"] = (reg.max_months_in_year >= f["static_min_months"]) | (reg.years_with_6m >= f["static_min_years"])

    # (3) HDBSCAN-повторяемость
    rec = hdbscan_recurrent_cells(d, f["hdbscan_min_cluster"], f["hdbscan_recurrence_years"])
    rec_cells = set(rec.h3_r7)
    parent = pd.Series([h3.cell_to_parent(c, 7) for c in reg.index], index=reg.index)
    reg["ev_hdbscan"] = parent.isin(rec_cells) & (reg.n_years >= 3)

    # (4) итоговый балл
    score = (
        0.45 * reg.ev_type.astype(float)
        + 0.30 * reg.ev_temporal.astype(float)
        + 0.25 * reg.ev_hdbscan.astype(float)
        + 0.10 * np.clip((reg.night_share - 0.25) / 0.5, 0, 1)
        + 0.05 * np.clip((reg.n_years - 3) / 10, 0, 1)
    )
    reg["static_score"] = np.clip(score, 0, 1).round(3)
    reg["is_static"] = (reg.static_score >= 0.45) & (reg.n >= 4)
    reg.index.name = "h3_r8"
    out = reg.reset_index()
    log.info(
        "static registry: %d static of %d cells (type=%d, temporal=%d, hdbscan=%d)",
        int(out.is_static.sum()), len(out), int(out.ev_type.sum()), int(out.ev_temporal.sum()), int(out.ev_hdbscan.sum()),
    )
    return out


def static_cell_set(registry: pd.DataFrame, buffer_k: int = 1) -> set[str]:
    """Статичные res-8 ячейки + буфер в k колец (пиксель MODIS 1 км «гуляет»)."""
    cells = set()
    for c in registry.loc[registry.is_static, "h3_r8"]:
        cells.update(h3.grid_disk(c, buffer_k))
    return cells

In [ ]:
%%writefile helios/processing/filters.py
"""Единый фильтр шума MODIS + VIIRS.

Что берём у других (с указанием авторства) и что улучшаем:

| Шаг | Источник | Улучшение HELIOS |
|---|---|---|
| Унифицированная вероятность P(fire) из MODIS 0–100% и VIIRS l/n/h + контраст ΔT(4–11 мкм) | flare-chronos (MIT) | + штраф за солнечный блик, + подтверждение соседями |
| Фильтр Южно-Атлантической аномалии (ночные «протонные» пиксели) | flare-chronos (MIT) | без изменений (важно для глобального режима) |
| День/ночь по зенитному углу Солнца (SZA < 85°) | flare-chronos (MIT) | векторно |
| Отсев low confidence (MODIS < 30, VIIRS «l») | рекомендации FIRMS / Kiln-Watch | low confidence НЕ выкидываем вслепую: если точку подтвердили соседи/другой спутник — оставляем |
| Постоянные источники (факелы, заводы) | flare-chronos + MODIS-VIIRS-fires + Kiln-Watch | единый реестр с 4 уликами (см. persistence.py) |
| Солнечный блик | — | НОВОЕ: угол блика по геометрии Солнца и сенсора |
| Одиночные точки | — | НОВОЕ: пространственно-временная поддержка (KD-дерево 3 км × ±2 сут) |
| Кросс-сенсорное подтверждение | идея эпизодов flare-chronos | НОВОЕ: другой спутник видел то же место ±60 мин |

На выходе у каждой точки: p_fire, флаги и причина отсева noise_reason; keep_veg — то,
что дальше считается растительным/природным пожаром.
"""

from __future__ import annotations

import h3
import numpy as np
import pandas as pd
from scipy.spatial import cKDTree

from ..config import settings
from ..utils import get_logger
from .footprint import pixel_area_km2, scan_angle_deg, track_azimuth_deg, view_zenith_deg
from .solar import daynight, glint_angle, solar_position

log = get_logger("helios.filters")


# ----------------------------------------------------------------------------- геометрия и H3
def add_geometry(df: pd.DataFrame) -> pd.DataFrame:
    """SZA, день/ночь, угол скана, VZA, площадь пикселя, угол блика, H3-ячейки."""
    out = df.copy()
    sza, sun_az = solar_position(out["lat"].to_numpy(), out["lon"].to_numpy(), out["acq_dt"])
    out["sza"] = sza
    out["daynight_sza"] = pd.Categorical(daynight(sza), categories=["D", "N"])
    sensor = out["sensor"].astype(str).to_numpy()
    out["scan_deg"] = scan_angle_deg(sensor, out["scan"].to_numpy(), out["track"].to_numpy())
    out["vza"] = view_zenith_deg(sensor, out["scan_deg"].to_numpy())
    out["pixel_km2"] = pixel_area_km2(out["scan"].to_numpy(), out["track"].to_numpy())
    trk = track_azimuth_deg(out["lat"].to_numpy(), out["stream"].astype(str).to_numpy(), out["daynight_sza"].astype(str).to_numpy())
    out["glint_deg"] = np.where(out["daynight_sza"] == "D", glint_angle(sza, sun_az, out["vza"].to_numpy(), trk), 180.0).astype(np.float32)
    res = settings()["h3"]
    lat, lon = out["lat"].to_numpy(), out["lon"].to_numpy()
    out["h3_r8"] = [h3.latlng_to_cell(a, b, res["res_fine"]) for a, b in zip(lat, lon)]
    out["h3_r7"] = [h3.cell_to_parent(c, res["res_cell"]) for c in out["h3_r8"]]
    out["h3_r5"] = [h3.cell_to_parent(c, res["res_model"]) for c in out["h3_r7"]]
    return out


# ----------------------------------------------------------------------------- P(fire)
def unified_probability(df: pd.DataFrame) -> np.ndarray:
    """MODIS: логистика по confidence% (якоря Giglio 2003/2016); VIIRS: якоря по l/n/h.
    Оба усиливаются контрастом ΔT = T(4 мкм) − T(11 мкм). Коэффициенты — flare-chronos (MIT)."""
    dt45 = np.maximum(0.0, (df["bright_mir"] - df["bright_tir"]).fillna(0).to_numpy(dtype=np.float64))
    is_day = (df["daynight_sza"] == "D").to_numpy()
    p = np.zeros(len(df), dtype=np.float64)
    m = (df["sensor"] == "MODIS").to_numpy()
    conf = np.clip(df["conf_pct"].fillna(50).to_numpy(dtype=np.float64), 0, 100)
    z = (conf - 38.0) / 19.0 + 0.015 * np.clip(dt45 - 18.0, -10, 40)
    p[m] = 1.0 / (1.0 + np.exp(-z[m]))
    cat = df["conf_cat"].astype(str).to_numpy()
    anchor = np.where(cat == "h", 0.94, np.where(cat == "n", np.where(is_day, 0.78, 0.84), np.where(is_day, 0.28, 0.38)))
    pv = np.clip(anchor + 0.06 * np.tanh((dt45 - 20.0) / 15.0), 0.05, 0.99)
    p[~m] = pv[~m]
    return p


def saa_flag(df: pd.DataFrame, p: np.ndarray) -> np.ndarray:
    """Ночные ложные пиксели в Южно-Атлантической аномалии (flare-chronos, MIT)."""
    w, s, e, n = settings()["filters"]["saa_bbox"]
    in_saa = df["lon"].between(w, e).to_numpy() & df["lat"].between(s, n).to_numpy()
    night = (df["daynight_sza"] == "N").to_numpy()
    dt45 = (df["bright_mir"] - df["bright_tir"]).fillna(0).to_numpy()
    frp = df["frp"].fillna(0).to_numpy()
    return in_saa & night & (((dt45 < 16.5) & (frp < 2.2)) | ((p < 0.45) & (dt45 < 19.5)))


# ----------------------------------------------------------------------------- поддержка соседями
def _xyz(lat: np.ndarray, lon: np.ndarray, t_days: np.ndarray, km_per_day: float) -> np.ndarray:
    lat0 = float(np.nanmean(lat)) if len(lat) else 0.0
    x = np.radians(lon) * 6371.0 * np.cos(np.radians(lat0))
    y = np.radians(lat) * 6371.0
    return np.column_stack([x, y, t_days * km_per_day])


def spatiotemporal_support(df: pd.DataFrame) -> tuple[np.ndarray, np.ndarray]:
    """(число соседей в 3 км и ±2 сут, подтверждён ли другим спутником в 2 км и ±60 мин)."""
    f = settings()["filters"]
    r_km, win = f["isolation_radius_km"], f["isolation_window_days"]
    neighbors = np.zeros(len(df), dtype=np.int32)
    confirmed = np.zeros(len(df), dtype=bool)
    t_days = (df["acq_dt"] - pd.Timestamp("2000-01-01", tz="UTC")).dt.total_seconds().to_numpy() / 86400.0
    lat, lon = df["lat"].to_numpy(), df["lon"].to_numpy()
    stream = df["stream"].astype(str).to_numpy()
    year = df["acq_dt"].dt.year.to_numpy()
    for y in np.unique(year):
        idx = np.flatnonzero(year == y)
        # окно ±2 сут масштабируем так, чтобы оно совпало с радиусом 3 км (эллипсоид)
        pts = _xyz(lat[idx], lon[idx], t_days[idx], r_km / win)
        tree = cKDTree(pts)
        neighbors[idx] = tree.query_ball_point(pts, r=r_km, return_length=True) - 1
        # кросс-сенсорное подтверждение: 2 км и ±60 мин, другой спутник
        pts2 = _xyz(lat[idx], lon[idx], t_days[idx], 2.0 / (60.0 / 1440.0))
        for s in np.unique(stream[idx]):
            mine = stream[idx] == s
            if mine.all() or not mine.any():
                continue
            other = cKDTree(pts2[~mine])
            cnt = other.query_ball_point(pts2[mine], r=2.0, return_length=True)
            confirmed[idx[mine]] = cnt > 0
    return neighbors, confirmed


# ----------------------------------------------------------------------------- главный фильтр
def apply_filters(df: pd.DataFrame, static_cells: set[str] | None = None) -> pd.DataFrame:
    """Полный фильтр шума. df должен пройти add_geometry(). Возвращает копию с флагами."""
    f = settings()["filters"]
    out = df.copy()
    p = unified_probability(out)
    out["p_fire_raw"] = p.astype(np.float32)

    out["is_low_conf"] = np.where(
        out["sensor"] == "MODIS", out["conf_pct"].fillna(0) < f["modis_min_conf"],
        (out["conf_cat"].astype(str) == "l") & bool(f["viirs_drop_low"]),
    )
    out["is_saa"] = saa_flag(out, p)
    glint = (out["daynight_sza"] == "D").to_numpy() & (out["glint_deg"].to_numpy() < f["glint_angle_deg"])
    out["is_glint_risk"] = glint & (out["conf_cat"].astype(str) != "h").to_numpy()

    nb, conf = spatiotemporal_support(out)
    out["neighbors"] = nb
    out["cross_confirmed"] = conf

    # корректировка вероятности (НОВОЕ)
    p = np.where(out["is_glint_risk"], p * 0.55, p)
    p = np.where(out["is_saa"], p * 0.12, p)
    p = np.where(nb >= 2, 1 - (1 - p) * 0.7, p)               # есть соседи — увереннее
    p = np.where(conf, 1 - (1 - p) * 0.5, p)                  # другой спутник подтвердил
    isolated = (nb == 0) & ~conf & (out["conf_cat"].astype(str) != "h").to_numpy()
    p = np.where(isolated, p * 0.75, p)                       # одиночка без подтверждения
    out["p_fire"] = np.clip(p, 0, 1).astype(np.float32)
    out["is_isolated"] = isolated

    # постоянные источники
    typed_static = out["type"].isin([1.0, 2.0, 3.0]).to_numpy()
    in_registry = out["h3_r8"].isin(static_cells).to_numpy() if static_cells else np.zeros(len(out), bool)
    out["is_static"] = typed_static | in_registry

    reason = np.full(len(out), "", dtype=object)
    reason[out["p_fire"].to_numpy() < f["p_fire_keep"]] = "low_probability"
    reason[out["is_glint_risk"].to_numpy() & (out["p_fire"].to_numpy() < f["p_fire_keep"])] = "sun_glint"
    reason[isolated & (out["p_fire"].to_numpy() < f["p_fire_keep"])] = "isolated"
    reason[out["is_saa"].to_numpy()] = "saa"
    reason[out["is_static"].to_numpy()] = "static_source"
    out["noise_reason"] = pd.Categorical(np.where(reason == "", "kept", reason))
    out["keep_veg"] = out["noise_reason"] == "kept"
    return out


def filter_report(df: pd.DataFrame) -> dict:
    """Сколько точек отсеяно и почему — по сенсорам (для слайдов и README)."""
    rep = (
        df.groupby(["sensor", "noise_reason"], observed=True).size().unstack(fill_value=0)
        if len(df) else pd.DataFrame()
    )
    total = df.groupby("sensor", observed=True).size()
    return {
        "total": {k: int(v) for k, v in total.items()},
        "by_reason": {s: {k: int(v) for k, v in row.items()} for s, row in rep.iterrows()},
        "kept_share": {s: round(float(rep.loc[s].get("kept", 0) / total[s]), 4) for s in rep.index},
    }

In [ ]:
%%writefile helios/processing/harmonize.py
"""Гармонизация 26 лет MODIS + VIIRS в одну непрерывную летопись.

Идея (как у Flare-Chronos): вместо сырых точек считаем «сгоревшие гексагоны» —
число пар (H3 res-7 ячейка, местные сутки) с пожаром. Пиксель 1 км и три пикселя
375 м в одном гексагоне дают одну единицу, это гасит разницу разрешений.

Дальше (идея Kiln-Watch, своя реализация + улучшения):
* общая шкала — Aqua MODIS (единственный спутник, летающий с 2002 г. через все эпохи VIIRS);
* мост «поток → опорный поток» в окне перекрытия: отношение сумм по каждому
  календарному месяцу, отдельно день/ночь, со сжатием к общему отношению по
  эмпирическому Байесу (меньше шума в «тихих» месяцах);
* НОВОЕ: доля рабочих дней спутника (coverage) берётся по «маякам» — постоянным
  факелам, которые видны почти каждый день; пропуски из-за сбоев спутника не
  выглядят как «нет пожаров»;
* НОВОЕ: бутстреп по годам → неопределённость калибровки и 95% интервал на каждое значение;
* проверки: ступенька при появлении SNPP (2012) и NOAA-20 (2018) до/после,
  тест «MODIS выключили» (учимся на 2012–2016, восстанавливаем 2017–2020 только по VIIRS).
"""

from __future__ import annotations

import math

import numpy as np
import pandas as pd
from scipy.stats import spearmanr

from ..config import settings
from ..utils import get_logger

log = get_logger("helios.harmonize")

STREAMS = ["MODIS_A", "MODIS_T", "VIIRS_SNPP", "VIIRS_NOAA20", "VIIRS_NOAA21"]
DN = ["D", "N"]


# ----------------------------------------------------------------------------- базовые величины
def local_date(df: pd.DataFrame) -> pd.Series:
    """Местные солнечные сутки (UTC + долгота/15 ч): ночной пролёт 01:30 относится к новым суткам."""
    shifted = df["acq_dt"].dt.tz_convert(None) + pd.to_timedelta(df["lon"] / 15.0, unit="h")
    return shifted.dt.normalize()


def stream_coverage(all_det: pd.DataFrame) -> pd.DataFrame:
    """Доля суток в месяце, когда поток вообще что-то видел (маяки — постоянные факелы)."""
    d = pd.DataFrame({"stream": all_det["stream"].astype(str), "date": local_date(all_det)}).drop_duplicates()
    d["month"] = d["date"].dt.to_period("M")
    cov = d.groupby(["stream", "month"]).size().rename("active_days").reset_index()
    cov["coverage"] = cov["active_days"] / cov["month"].dt.days_in_month
    return cov


def celldays(df: pd.DataFrame) -> pd.DataFrame:
    """Уникальные (поток, сутки, день/ночь, H3 res-7) среди отфильтрованных растительных пожаров."""
    d = df.loc[df["keep_veg"], ["stream", "daynight_sza", "h3_r7", "h3_r5", "frp", "lat", "lon", "acq_dt"]].copy()
    d["date"] = local_date(d)
    d["stream"] = d["stream"].astype(str)
    d["dn"] = d["daynight_sza"].astype(str)
    g = d.groupby(["stream", "date", "dn", "h3_r7"], observed=True).agg(
        h3_r5=("h3_r5", "first"), n_det=("frp", "size"), frp_sum=("frp", "sum")
    ).reset_index()
    return g


def monthly_stream_table(cd: pd.DataFrame, cov: pd.DataFrame) -> pd.DataFrame:
    """Помесячно: cell-days по потокам и дню/ночи + coverage."""
    t = cd.assign(month=cd["date"].dt.to_period("M")).groupby(["stream", "month", "dn"]).size().rename("celldays").reset_index()
    t = t.merge(cov[["stream", "month", "coverage"]], on=["stream", "month"], how="left")
    return t


# ----------------------------------------------------------------------------- калибровка
def ratio_calibration_eb(pairs: pd.DataFrame) -> dict:
    """pairs: [cal_month, ref, src] (уже нормированы на coverage). Отношение сумм по месяцам,
    сжатое к общему отношению эмпирическим Байесом (метод моментов для τ²)."""
    num = pairs.groupby("cal_month")["ref"].sum().reindex(range(1, 13), fill_value=0.0)
    den = pairs.groupby("cal_month")["src"].sum().reindex(range(1, 13), fill_value=0.0)
    N, D = float(num.sum()), float(den.sum())
    if N <= 0 or D <= 0:
        return {"beta": [1.0] * 12, "pooled": 1.0, "tau2": 0.0, "calibrated": False}
    pooled = N / D
    L = math.log(pooled)
    ok = (num > 0) & (den > 0)
    logs = np.log(num[ok] / den[ok])
    v = 1.0 / num[ok] + 1.0 / den[ok]
    tau2 = max(0.0, float(np.mean((logs - L) ** 2) - np.mean(v))) if ok.any() else 0.0
    beta = []
    for m in range(1, 13):
        if num[m] > 0 and den[m] > 0:
            vm = 1.0 / num[m] + 1.0 / den[m]
            lam = tau2 / (tau2 + vm) if tau2 > 0 else 0.0
            beta.append(math.exp(lam * math.log(num[m] / den[m]) + (1 - lam) * L))
        else:
            beta.append(pooled)
    return {"beta": beta, "pooled": pooled, "tau2": tau2, "calibrated": True}


def fit_bridge(mt: pd.DataFrame, src: str, ref: str, start: str, end: str, dn: str,
               n_boot: int = 200, min_cov: float = 0.8) -> dict:
    """Калибровка src -> ref (один день/ночь) + бутстреп по годам."""
    a = mt[(mt.stream == ref) & (mt.dn == dn)].set_index("month")
    b = mt[(mt.stream == src) & (mt.dn == dn)].set_index("month")
    months = pd.period_range(start, end, freq="M")
    rows = []
    for m in months:
        if m in a.index and m in b.index and a.loc[m, "coverage"] >= min_cov and b.loc[m, "coverage"] >= min_cov:
            rows.append({"month": m, "cal_month": m.month,
                         "ref": a.loc[m, "celldays"] / a.loc[m, "coverage"],
                         "src": b.loc[m, "celldays"] / b.loc[m, "coverage"]})
    pairs = pd.DataFrame(rows)
    if pairs.empty:
        return {"beta": [1.0] * 12, "pooled": 1.0, "sigma2": [0.25] * 12, "n_months": 0, "calibrated": False}
    fit = ratio_calibration_eb(pairs)
    # бутстреп по годам — неопределённость калибровки по месяцам
    rng = np.random.default_rng(7)
    years = pairs["month"].dt.year.unique()
    boot = []
    for _ in range(n_boot):
        pick = rng.choice(years, size=len(years), replace=True)
        sample = pd.concat([pairs[pairs["month"].dt.year == y] for y in pick])
        boot.append(np.log(ratio_calibration_eb(sample)["beta"]))
    sigma2 = np.var(np.array(boot), axis=0).tolist()
    # сверхпуассоновский разброс «месяц к месяцу» (то, что не объясняет счётный шум) —
    # без него 95% интервалы на тесте «MODIS выключили» накрывали только ~63% месяцев
    beta_m = np.array(fit["beta"])[pairs["cal_month"].to_numpy() - 1]
    resid = np.log((pairs["ref"] + 0.5) / (beta_m * pairs["src"] + 0.5))
    noise = 1.0 / (pairs["ref"] + 0.5) + 1.0 / (pairs["src"] + 0.5)
    sigma2_extra = max(0.0025, float(np.var(resid) - np.mean(noise))) if len(pairs) >= 6 else 0.05
    return {**fit, "sigma2": [max(s, 0.0025) for s in sigma2], "sigma2_extra": sigma2_extra,
            "n_months": int(len(pairs))}


def fit_all_bridges(mt: pd.DataFrame) -> dict:
    """Все мосты из конфига, затем цепочки до опорного потока (Aqua MODIS)."""
    cfg = settings()["harmonization"]
    ref0 = cfg["reference"]
    bridges = {}
    for src, b in cfg["bridges"].items():
        if src not in set(mt.stream):
            continue
        bridges[src] = {"ref": b["ref"], **{dn: fit_bridge(mt, src, b["ref"], b["start"], b["end"], dn) for dn in DN}}
    # цепочки: beta_total(src) = beta(src->ref) * beta_total(ref)
    to_ref: dict[str, dict] = {ref0: {dn: {"beta": [1.0] * 12, "sigma2": [0.0] * 12, "sigma2_extra": 0.0} for dn in DN}}
    pending = dict(bridges)
    while pending:
        progressed = False
        for src, b in list(pending.items()):
            if b["ref"] in to_ref:
                to_ref[src] = {
                    dn: {
                        "beta": [x * y for x, y in zip(b[dn]["beta"], to_ref[b["ref"]][dn]["beta"])],
                        "sigma2": [x + y for x, y in zip(b[dn]["sigma2"], to_ref[b["ref"]][dn]["sigma2"])],
                        "sigma2_extra": b[dn].get("sigma2_extra", 0.05) + to_ref[b["ref"]][dn]["sigma2_extra"],
                    }
                    for dn in DN
                }
                pending.pop(src)
                progressed = True
        if not progressed:
            break
    return {"bridges": bridges, "to_ref": to_ref}


# ----------------------------------------------------------------------------- гармонизированный ряд
def harmonized_monthly(mt: pd.DataFrame, cal: dict, min_cov: float = 0.5) -> pd.DataFrame:
    """Помесячный ряд на шкале Aqua MODIS с 95% интервалом (обратно-дисперсионное среднее потоков)."""
    to_ref = cal["to_ref"]
    rows = []
    for (month, dn), g in mt.groupby(["month", "dn"]):
        est = []
        for _, r in g.iterrows():
            if r.stream not in to_ref or not (r.coverage >= min_cov):
                continue
            beta = to_ref[r.stream][dn]["beta"][month.month - 1]
            s2 = to_ref[r.stream][dn]["sigma2"][month.month - 1] + to_ref[r.stream][dn]["sigma2_extra"]
            val = beta * r.celldays / r.coverage
            var = val ** 2 * s2 + beta ** 2 * max(r.celldays, 1.0) / r.coverage ** 2
            est.append((val, var, r.stream))
        if not est:
            continue
        w = np.array([1.0 / max(v, 1e-6) for _, v, _ in est])
        vals = np.array([x for x, _, _ in est])
        rows.append({"month": month, "dn": dn, "value": float((w * vals).sum() / w.sum()),
                     "var": float(1.0 / w.sum()), "streams": ",".join(sorted(s for _, _, s in est))})
    h = pd.DataFrame(rows)
    wide = h.pivot_table(index="month", columns="dn", values=["value", "var"], aggfunc="first")
    out = pd.DataFrame(index=wide.index)
    out["harmonized_day"] = wide[("value", "D")] if ("value", "D") in wide else np.nan
    out["harmonized_night"] = wide[("value", "N")] if ("value", "N") in wide else np.nan
    out["harmonized"] = out[["harmonized_day", "harmonized_night"]].sum(axis=1, min_count=1)
    var = wide["var"].sum(axis=1, min_count=1)
    out["lo95"] = (out["harmonized"] - 1.96 * np.sqrt(var)).clip(lower=0)
    out["hi95"] = out["harmonized"] + 1.96 * np.sqrt(var)
    streams = h.groupby("month")["streams"].agg(lambda s: ",".join(sorted(set(",".join(s).split(",")))))
    out["streams"] = streams
    return out.reset_index()


def raw_monthly(cd: pd.DataFrame, filtered: pd.DataFrame) -> pd.DataFrame:
    """Наивные метрики для сравнения: сумма точек всех спутников и union cell-days."""
    det = filtered.loc[filtered["keep_veg"]]
    raw_det = det.groupby(local_date(det).dt.to_period("M")).size().rename("raw_detections")
    union = cd[["date", "h3_r7"]].drop_duplicates()
    raw_union = union.groupby(union["date"].dt.to_period("M")).size().rename("raw_union_celldays")
    per_stream = cd.assign(month=cd["date"].dt.to_period("M")).groupby(["month", "stream"]).size().unstack(fill_value=0)
    per_stream.columns = [f"celldays_{c}" for c in per_stream.columns]
    out = pd.concat([raw_det, raw_union, per_stream], axis=1)
    out.index.name = "month"
    return out.reset_index()


# ----------------------------------------------------------------------------- проверки
def step_ratio(series: pd.Series, transition: str, window: int = 24) -> float | None:
    """Ступенька при смене состава спутников на десезонализированном ряду: mean(после)/mean(до)."""
    s = series.dropna()
    if s.empty:
        return None
    clim = s.groupby(s.index.month).transform("mean").replace(0, np.nan)
    anom = (s / clim).dropna()
    t = pd.Period(transition, freq="M")
    before = anom[(anom.index < t) & (anom.index >= t - window)]
    after = anom[(anom.index >= t) & (anom.index < t + window)]
    if len(before) < 12 or len(after) < 12:
        return None
    return round(float(after.mean() / before.mean()), 3)


def denial_test(mt: pd.DataFrame) -> dict:
    """«MODIS выключили»: калибруем SNPP->Aqua на 2012–2016, восстанавливаем Aqua 2017–2020 по SNPP."""
    cfg = settings()["harmonization"]
    (tr0, tr1), (te0, te1) = cfg["denial_train"], cfg["denial_test"]
    preds, truth, inside = [], [], []
    for dn in DN:
        fit = fit_bridge(mt, "VIIRS_SNPP", "MODIS_A", tr0, tr1, dn, n_boot=100)
        a = mt[(mt.stream == "MODIS_A") & (mt.dn == dn)].set_index("month")
        b = mt[(mt.stream == "VIIRS_SNPP") & (mt.dn == dn)].set_index("month")
        for m in pd.period_range(te0, te1, freq="M"):
            if m in a.index and m in b.index and a.loc[m, "coverage"] >= 0.8 and b.loc[m, "coverage"] >= 0.8:
                beta = fit["beta"][m.month - 1]
                p = beta * b.loc[m, "celldays"] / b.loc[m, "coverage"]
                t = a.loc[m, "celldays"] / a.loc[m, "coverage"]
                s2 = fit["sigma2"][m.month - 1] + fit["sigma2_extra"]
                sd = math.sqrt(p ** 2 * s2 + beta ** 2 * max(b.loc[m, "celldays"], 1) + max(t, 1))
                preds.append((m, dn, p))
                truth.append(t)
                inside.append(abs(p - t) <= 1.96 * sd)
    if not preds:
        return {"available": False}
    pdf = pd.DataFrame(preds, columns=["month", "dn", "pred"]).assign(truth=truth)
    tot = pdf.groupby("month")[["pred", "truth"]].sum()
    tot = tot[tot.truth > 0]
    err = (tot.pred.sum() - tot.truth.sum()) / tot.truth.sum()
    rho = spearmanr(tot.pred, tot.truth).statistic if len(tot) > 3 else None
    return {
        "available": True,
        "train": [tr0, tr1], "test": [te0, te1],
        "months": int(len(tot)),
        "total_error_pct": round(100 * float(err), 1),
        "median_abs_pct_error_monthly": round(100 * float(np.median(np.abs(tot.pred - tot.truth) / tot.truth)), 1),
        "spearman_rho": None if rho is None else round(float(rho), 3),
        "interval_coverage_95": round(float(np.mean(inside)), 3),
    }


# ----------------------------------------------------------------------------- календарь
def burning_calendar(hm: pd.DataFrame, raw: pd.DataFrame) -> pd.DataFrame:
    """Календарь горения год × месяц: гармонизированное значение, сырые метрики, аномалия."""
    cal = hm.merge(raw, on="month", how="outer").sort_values("month")
    cal["year"] = cal["month"].dt.year
    cal["mon"] = cal["month"].dt.month
    med = cal.groupby("mon")["harmonized"].transform("median")
    mad = cal.groupby("mon")["harmonized"].transform(lambda s: (s - s.median()).abs().median())
    cal["anomaly_z"] = ((cal["harmonized"] - med) / (1.4826 * mad.replace(0, np.nan))).round(2)
    cal["pct_rank"] = cal.groupby("mon")["harmonized"].rank(pct=True).round(3)
    return cal


def run(filtered: pd.DataFrame) -> dict:
    """Полный цикл гармонизации. filtered — выход filters.apply_filters()."""
    cov = stream_coverage(filtered)
    cd = celldays(filtered)
    mt = monthly_stream_table(cd, cov)
    cal = fit_all_bridges(mt)
    hm = harmonized_monthly(mt, cal)
    raw = raw_monthly(cd, filtered)
    calendar = burning_calendar(hm, raw)

    series = calendar.set_index("month")
    aqua = mt[(mt.stream == "MODIS_A")].groupby("month")["celldays"].sum()
    steps = {}
    for name, when in (("SNPP_2012", "2012-01"), ("NOAA20_2018", "2018-04")):
        steps[name] = {
            "raw_detections": step_ratio(series["raw_detections"], when),
            "raw_union_celldays": step_ratio(series["raw_union_celldays"], when),
            "harmonized": step_ratio(series["harmonized"], when),
            "aqua_only": step_ratio(aqua, when),
        }
    # дрейф орбиты Aqua (после 2022 пролёт позже 13:30): отношение Aqua/SNPP по годам
    yearly = mt.assign(year=mt["month"].dt.year).groupby(["year", "stream"])["celldays"].sum().unstack()
    drift = {}
    if {"MODIS_A", "VIIRS_SNPP"} <= set(yearly.columns):
        ratio = (yearly["MODIS_A"] / yearly["VIIRS_SNPP"]).dropna()
        drift = {int(y): round(float(v), 3) for y, v in ratio.items()}
    validation = {"denial_test": denial_test(mt), "step_tests": steps, "aqua_to_snpp_ratio_by_year": drift}
    log.info("harmonization: denial=%s steps=%s", validation["denial_test"], steps)
    calib_rows = []
    for src, b in cal["bridges"].items():
        for dn in DN:
            for m in range(12):
                calib_rows.append({"stream": src, "ref": b["ref"], "dn": dn, "cal_month": m + 1,
                                   "beta": round(b[dn]["beta"][m], 4), "sigma2": round(b[dn]["sigma2"][m], 5),
                                   "sigma2_extra": round(b[dn].get("sigma2_extra", 0.0), 5),
                                   "beta_to_aqua": round(cal["to_ref"][src][dn]["beta"][m], 4),
                                   "n_months": b[dn].get("n_months", 0)})
    return {
        "celldays": cd,
        "coverage": cov,
        "monthly_streams": mt,
        "calibration": pd.DataFrame(calib_rows),
        "calendar": calendar,
        "validation": validation,
        "to_ref": cal["to_ref"],
    }


def harmonized_cell_weekly(cd: pd.DataFrame, to_ref: dict, res_col: str = "h3_r5") -> pd.DataFrame:
    """Неделя × ячейка res-5: union cell-days (любой спутник) и Aqua-шкала (среднее по доступным потокам)."""
    d = cd.copy()
    d["week"] = d["date"].dt.to_period("W-SUN").dt.start_time
    union = d[["week", res_col, "h3_r7", "date"]].drop_duplicates().groupby(["week", res_col]).size().rename("union_celldays")
    d["beta"] = [to_ref.get(s, {}).get(dn, {"beta": [1.0] * 12})["beta"][m - 1]
                 for s, dn, m in zip(d["stream"], d["dn"], d["date"].dt.month)]
    per_stream = d.groupby(["week", res_col, "stream"]).agg(cdays=("h3_r7", "size"), beta=("beta", "mean")).reset_index()
    per_stream["aqua_scale"] = per_stream["cdays"] * per_stream["beta"]
    harm = per_stream.groupby(["week", res_col])["aqua_scale"].mean().rename("harmonized_celldays")
    frp = d.groupby(["week", res_col])["frp_sum"].sum().rename("frp_sum")
    out = pd.concat([union, harm, frp], axis=1).reset_index()
    return out

In [ ]:
%%writefile helios/processing/fire_dna.py
"""Fire DNA — «характер горения» каждой ячейки H3 res-7 по её 26-летней истории.

Классы (вероятность + уверенность; истинную причину без полевых данных установить нельзя):
* industrial   — постоянный источник: факел, завод, ТЭЦ (горит круглый год, и днём, и ночью);
* agricultural — сельхозпалы: повторяются почти каждый год в узкое окно весной/осенью, днём, мелкие;
* wildfire     — природный/степной пожар: эпизодический, летний, крупный, быстро растёт по площади;
* sporadic     — мало наблюдений, характер не ясен.
Подход: прозрачные признаки + мягкая балльная модель (softmax) — объяснимо для жюри.
"""

from __future__ import annotations

import numpy as np
import pandas as pd

LABELS_RU = {"industrial": "постоянный промышленный источник", "agricultural": "сельскохозяйственные палы",
             "wildfire": "природный (степной/лесной) пожар", "sporadic": "единичные случаи"}


def cell_features(det: pd.DataFrame, registry: pd.DataFrame) -> pd.DataFrame:
    d = det[["h3_r7", "h3_r8", "acq_dt", "frp", "daynight_sza", "keep_veg", "is_static"]].copy()
    d["date"] = d["acq_dt"].dt.normalize()
    d["year"] = d["acq_dt"].dt.year
    d["doy"] = d["acq_dt"].dt.dayofyear
    d["night"] = d["daynight_sza"].astype(str) == "N"
    g = d.groupby("h3_r7")
    ang = 2 * np.pi * d["doy"] / 365.25
    feats = pd.DataFrame({
        "n_det": g.size(),
        "n_days": g["date"].nunique(),
        "n_years": g["year"].nunique(),
        "first_year": g["year"].min(),
        "last_year": g["year"].max(),
        "night_share": g["night"].mean(),
        "static_share": g["is_static"].mean(),
        "frp_median": g["frp"].median(),
        "frp_p90": g["frp"].quantile(0.9),
        # сезонная концентрация (длина среднего вектора дня года): 1 = всё в одну неделю, 0 = круглый год
        "season_R": np.hypot(np.cos(ang).groupby(d["h3_r7"]).mean(), np.sin(ang).groupby(d["h3_r7"]).mean()),
        "peak_doy": (np.degrees(np.arctan2(np.sin(ang).groupby(d["h3_r7"]).mean(),
                                           np.cos(ang).groupby(d["h3_r7"]).mean())) % 360 / 360 * 365.25).round(0),
    })
    # «эпизодичность»: дней с огнём на один активный год
    feats["days_per_year"] = feats["n_days"] / feats["n_years"]
    feats["recurrence"] = feats["n_years"] / (feats["last_year"] - feats["first_year"] + 1)
    # максимальный «рост» события: соседние res-7 ячейки, горевшие в те же сутки
    daily = d[d["keep_veg"]].groupby(["date", "h3_r7"]).size().reset_index()
    import h3

    daily["r5"] = [h3.cell_to_parent(c, 5) for c in daily["h3_r7"]]
    spread = daily.groupby(["date", "r5"])["h3_r7"].transform("size")
    feats["max_event_cells"] = daily.assign(s=spread).groupby("h3_r7")["s"].max().reindex(feats.index).fillna(0)
    reg = registry.set_index("h3_r8")
    st = reg.loc[reg["is_static"], "static_score"]
    st7 = st.groupby([h3.cell_to_parent(c, 7) for c in st.index]).max()
    feats["static_score"] = st7.reindex(feats.index).fillna(0)
    return feats


def classify_cells(det: pd.DataFrame, registry: pd.DataFrame, temperature: float = 0.6) -> pd.DataFrame:
    f = cell_features(det, registry)
    month = ((f["peak_doy"] / 30.44).astype(int) % 12) + 1
    spring_autumn = month.isin([3, 4, 5, 9, 10, 11]).astype(float)
    summer = month.isin([6, 7, 8]).astype(float)
    s_ind = (3.0 * f["static_score"] + 2.0 * f["static_share"] + 1.5 * (1 - f["season_R"]) + 1.5 * f["night_share"]
             + 0.8 * np.clip((f["days_per_year"] - 5) / 20, 0, 1))
    s_agr = (2.0 * f["recurrence"] * (f["n_years"] >= 3) + 1.5 * f["season_R"] + 1.2 * spring_autumn
             + 1.0 * (f["night_share"] < 0.2) + 0.8 * (f["frp_median"] < 15) - 2.0 * f["static_share"])
    s_wild = (1.5 * summer + 1.5 * np.clip(np.log1p(f["max_event_cells"]) / 3, 0, 1.5) + 1.0 * (f["frp_p90"] > 30)
              + 1.0 * (1 - f["recurrence"]) + 0.5 * f["season_R"] - 2.0 * f["static_share"])
    s_spo = 2.5 * (f["n_days"] <= 2) + 1.0 * (f["n_years"] <= 1)
    scores = np.column_stack([s_ind, s_agr, s_wild, s_spo]) / temperature
    p = np.exp(scores - scores.max(1, keepdims=True))
    p /= p.sum(1, keepdims=True)
    classes = np.array(["industrial", "agricultural", "wildfire", "sporadic"])
    out = f.reset_index().rename(columns={"index": "h3_r7"})
    for i, c in enumerate(classes):
        out[f"p_{c}"] = p[:, i].round(3)
    out["dna_class"] = classes[p.argmax(1)]
    out["confidence"] = p.max(1).round(3)
    out["label_ru"] = out["dna_class"].map(LABELS_RU)
    import h3

    ll = np.array([h3.cell_to_latlng(c) for c in out["h3_r7"]])
    out["lat"], out["lon"] = ll[:, 0].round(5), ll[:, 1].round(5)
    return out

In [ ]:
%%writefile helios/processing/season_shift.py
"""Fire Season Shift Detector — сдвигается ли пожарный сезон и не артефакт ли это смены спутников.

Для каждого года: начало (10% годовой активности), пик (центр масс 30-дневного окна максимума),
конец (90%). Тренд — наклон Тейла–Сена + тест Манна–Кендалла (непараметрические, устойчивы к
выбросам). Проверка на артефакт: тот же расчёт ТОЛЬКО по Aqua MODIS (один и тот же сенсор
с 2003 г.). Если сдвиг значим и в смешанном ряду, и в Aqua-only — он реальный.
"""

from __future__ import annotations

import h3
import numpy as np
import pandas as pd
from scipy.stats import kendalltau, theilslopes


def season_metrics(daily: pd.Series) -> dict | None:
    """daily: индекс — даты одного года, значения — активность (cell-days)."""
    if daily.sum() < 20:
        return None
    s = daily.reindex(pd.date_range(f"{daily.index[0].year}-01-01", f"{daily.index[0].year}-12-31"), fill_value=0)
    c = s.cumsum() / s.sum()
    doy = np.arange(1, len(s) + 1)
    roll = s.rolling(30, center=True, min_periods=1).sum().to_numpy()
    k = int(np.argmax(roll))
    lo, hi = max(0, k - 15), min(len(s), k + 15)
    w = s.to_numpy()[lo:hi]
    peak = float((doy[lo:hi] * w).sum() / max(w.sum(), 1))
    return {"start_doy": int(doy[np.searchsorted(c.to_numpy(), 0.10)]), "peak_doy": round(peak, 1),
            "end_doy": int(doy[min(np.searchsorted(c.to_numpy(), 0.90), len(doy) - 1)]), "total": float(s.sum())}


def trend(years: np.ndarray, values: np.ndarray) -> dict:
    if len(years) < 8:
        return {"n_years": int(len(years))}
    slope, _, lo, hi = theilslopes(values, years)
    tau, p = kendalltau(years, values)
    return {"n_years": int(len(years)), "days_per_decade": round(10 * slope, 1),
            "ci95_days_per_decade": [round(10 * lo, 1), round(10 * hi, 1)], "kendall_tau": round(float(tau), 3),
            "p_value": round(float(p), 4), "significant": bool(p < 0.05)}


def _yearly(cd: pd.DataFrame) -> pd.DataFrame:
    rows = []
    union = cd[["date", "h3_r7"]].drop_duplicates()
    for y, g in union.groupby(union["date"].dt.year):
        m = season_metrics(g.groupby("date").size())
        if m:
            rows.append({"year": int(y), **m})
    return pd.DataFrame(rows)


def region_season_shift(cd: pd.DataFrame, start_year: int = 2003, end_year: int = 2025) -> dict:
    cd = cd[cd["date"].dt.year.between(start_year, end_year)]
    mixed = _yearly(cd)
    aqua = _yearly(cd[cd["stream"] == "MODIS_A"])
    out = {"period": [start_year, end_year]}
    for name, df in (("all_sensors", mixed), ("aqua_only", aqua)):
        out[name] = {m: trend(df["year"].to_numpy(), df[m].to_numpy()) for m in ("start_doy", "peak_doy", "end_doy")} if len(df) else {}
        out[name]["yearly"] = df.to_dict("records")
    verdict = {}
    for m in ("start_doy", "peak_doy", "end_doy"):
        a, b = out["all_sensors"].get(m, {}), out["aqua_only"].get(m, {})
        if a.get("significant") and b.get("significant"):
            verdict[m] = "реальный сдвиг (значим и в Aqua-only)"
        elif a.get("significant") and not b.get("significant"):
            verdict[m] = "вероятный артефакт смены спутников"
        elif b.get("significant"):
            verdict[m] = "сдвиг по Aqua-only, смешанный ряд шумнее"
        else:
            verdict[m] = "значимого сдвига нет"
    out["verdict"] = verdict
    return out


def cell_season_shift(cd: pd.DataFrame, res: int = 4, start_year: int = 2003, end_year: int = 2025,
                      min_years: int = 10) -> pd.DataFrame:
    """Карта сдвига пика сезона по ячейкам H3 res-4 (~1770 км²), только Aqua MODIS (без артефактов)."""
    d = cd[(cd["stream"] == "MODIS_A") & cd["date"].dt.year.between(start_year, end_year)].copy()
    d["r4"] = [h3.cell_to_parent(c, res) for c in d["h3_r7"]]
    d = d[["date", "h3_r7", "r4"]].drop_duplicates()
    rows = []
    for cell, g in d.groupby("r4"):
        yrs = []
        for y, gy in g.groupby(g["date"].dt.year):
            doy = gy["date"].dt.dayofyear.to_numpy()
            if len(doy) >= 3:
                ang = 2 * np.pi * doy / 365.25
                yrs.append((y, (np.degrees(np.arctan2(np.sin(ang).mean(), np.cos(ang).mean())) % 360) / 360 * 365.25, len(doy)))
        if len(yrs) < min_years:
            continue
        arr = np.array(yrs)
        t = trend(arr[:, 0], arr[:, 1])
        lat, lon = h3.cell_to_latlng(cell)
        rows.append({"h3_r4": cell, "lat": round(lat, 4), "lon": round(lon, 4), "years": int(len(arr)),
                     "mean_peak_doy": round(float(arr[:, 1].mean()), 1), **{k: v for k, v in t.items() if k != "ci95_days_per_decade"}})
    return pd.DataFrame(rows)

In [ ]:
%%writefile helios/processing/blind_spots.py
"""Fire Blind Spot Detector — «пустая карта» не всегда означает «пожаров нет».

Если в ячейке пожары исторически вероятны, а во время пролётов спутников было облачно,
отсутствие точек ничего не доказывает. Показываем: «Недостаточно данных. Возможны
пропущенные очаги».

* История (2012–2025): ожидаемая вероятность пожара (климатология прошлых лет) × облачность
  NASA POWER CLOUD_AMT за неделю. Проверка: после «слепых» недель пожары находятся
  чаще, чем после ясных недель с той же климатологией (значит, часть очагов действительно
  пропускалась).
* Сейчас: облачность Open-Meteo в часы пролётов (≈13:30 и ≈01:30 местного) + ожидаемый риск
  (ИИ №3 или климатология) по сетке H3 res-3.
"""

from __future__ import annotations

import h3
import numpy as np
import pandas as pd

from ..config import settings
from ..utils import get_logger

log = get_logger("helios.blindspots")

MESSAGE_RU = "Недостаточно данных: во время пролётов было облачно. Возможны пропущенные очаги."


def historical_blind_spots(threshold: float = 0.2, start: str = "2012-01-01", end: str = "2025-12-31") -> pd.DataFrame | None:
    """Порог 0.2: проверка показала, что облака «прячут» очаги именно в ячейках с высоким
    ожидаемым риском (p ≥ 0.5: +14% находок после облачных недель), а в малорисковых облака
    приходят вместе с дождём и пожаров, наоборот, меньше."""
    from ..models.features import Panel

    pn = Panel()
    ww = pn.weather_weekly
    if ww is None:
        log.warning("нет погоды NASA POWER — исторические слепые зоны не посчитать")
        return None
    years, woys = pn.iso
    rows = []
    bins = np.array([0.05, 0.1, 0.2, 0.35, 0.5, 1.01])
    # счётчики по корзинам ожидаемого риска: [облачно-n, облачно-пожар, ясно-n, ясно-пожар]
    strat = np.zeros((len(bins) - 1, 4))
    for w in pn.week_index(start, end):
        if w + 1 >= len(pn.weeks):
            break
        exp_p = pn.clim[int(years[w])][int(woys[w]) - 1]
        cloud = np.nan_to_num(ww["cloud"][w], nan=50.0) / 100.0
        score = exp_p * np.clip((cloud - 0.4) / 0.5, 0, 1) ** 1.5
        observed = pn.B[w]
        nxt = pn.B[w + 1]
        flag = (score >= threshold) & ~observed
        # проверка: при ОДИНАКОВОМ ожидаемом риске сравниваем «облачно и пусто» с «ясно и пусто»
        b = np.digitize(exp_p, bins) - 1
        ok = (b >= 0) & (b < len(bins) - 1) & ~observed
        cloudy, clear = ok & (cloud >= 0.7), ok & (cloud < 0.4)
        np.add.at(strat[:, 0], b[cloudy], 1)
        np.add.at(strat[:, 1], b[cloudy], nxt[cloudy])
        np.add.at(strat[:, 2], b[clear], 1)
        np.add.at(strat[:, 3], b[clear], nxt[clear])
        for i in np.flatnonzero(flag):
            rows.append((pn.cells[i], pn.weeks[w], float(exp_p[i]), float(cloud[i]), float(score[i]), bool(nxt[i])))
    df = pd.DataFrame(rows, columns=["h3_r5", "week", "expected_fire_p", "cloud_frac", "blind_score", "fire_next_week"])
    table = []
    for k in range(len(bins) - 1):
        cn, cf, sn, sf = strat[k]
        if cn >= 50 and sn >= 50:
            table.append({"expected_p_bin": f"{bins[k]:.2f}-{min(bins[k + 1], 1):.2f}", "cloudy_n": int(cn),
                          "fire_next_after_cloudy": round(cf / cn, 4), "clear_n": int(sn),
                          "fire_next_after_clear": round(sf / sn, 4), "lift": round((cf / cn) / max(sf / sn, 1e-6), 3)})
    w_lift = (sum(t["lift"] * t["cloudy_n"] for t in table) / max(sum(t["cloudy_n"] for t in table), 1)) if table else None
    df.attrs["validation"] = {"flagged_cell_weeks": int(len(df)), "stratified": table,
                              "weighted_lift": None if w_lift is None else round(w_lift, 3),
                              "interpretation": "lift>1: после облачных «пустых» недель пожары находят чаще, "
                                                "чем после ясных при том же ожидаемом риске — часть очагов была скрыта облаками"}
    log.info("blind spots history: lift %s", df.attrs["validation"]["weighted_lift"])
    return df


def overpass_hours_utc(lon: float) -> list[int]:
    """Часы UTC дневного (~13:30) и ночного (~01:30) пролётов VIIRS/Aqua для долготы."""
    off = lon / 15.0
    return sorted({int((13.5 - off) % 24), int((1.5 - off) % 24)})


def recent_blind_spots(risk_r5: pd.DataFrame | None = None, recent_fire_cells: set[str] | None = None,
                       threshold: float = 0.2) -> pd.DataFrame:
    """Слепые зоны за последние ~2 суток по сетке res-3 (облачность Open-Meteo в часы пролётов)."""
    from ..apis.boundaries import region_cells
    from ..apis.open_meteo import forecast

    res_w = settings()["h3"]["res_weather"]
    cells = sorted(region_cells(res_w))
    ll = np.array([h3.cell_to_latlng(c) for c in cells])
    clouds = []
    for i in range(0, len(cells), 90):  # пачками по 90 точек
        frames = forecast(list(ll[i:i + 90, 0]), list(ll[i:i + 90, 1]), hourly=["cloud_cover"], days=1)
        clouds += frames
    rows = []
    now = pd.Timestamp.now(tz="UTC")
    for c, (lat, lon), f in zip(cells, ll, clouds):
        if f is None or f.empty:
            continue
        past = f[(f.index <= now) & (f.index >= now - pd.Timedelta(hours=48))]
        hrs = overpass_hours_utc(lon)
        sel = past[past.index.hour.isin(hrs)]["cloud_cover"]
        cloud = float(sel.mean() / 100) if len(sel) else float(past["cloud_cover"].mean() / 100)
        rows.append({"h3_r3": c, "lat": round(lat, 4), "lon": round(lon, 4), "cloud_at_overpasses": round(cloud, 3)})
    df = pd.DataFrame(rows)
    if risk_r5 is not None and len(risk_r5):
        r = risk_r5.assign(h3_r3=[h3.cell_to_parent(c, res_w) for c in risk_r5["h3_r5"]])
        exp = r.groupby("h3_r3")["p_fire"].quantile(0.9).rename("expected_fire_p")
        df = df.merge(exp, on="h3_r3", how="left")
    else:
        df["expected_fire_p"] = np.nan
    df["expected_fire_p"] = df["expected_fire_p"].fillna(0.0)
    df["blind_score"] = (df["expected_fire_p"] * np.clip((df["cloud_at_overpasses"] - 0.4) / 0.5, 0, 1) ** 1.5).round(4)
    seen = recent_fire_cells or set()
    df["fires_seen_48h"] = df["h3_r3"].isin(seen)
    df["is_blind"] = (df["blind_score"] >= threshold) & ~df["fires_seen_48h"]
    df["message"] = np.where(df["is_blind"], MESSAGE_RU, "")
    df["computed_at"] = now
    return df

In [ ]:
%%writefile helios/processing/spread.py
"""Прогноз распространения активного пожара по погоде (часть ИИ №3 «на Земле»).

Казахстан — в основном степь, поэтому берём травяную модель Макартура Mk5:
  GFDI = exp(−1.523 + 1.027·ln Q − 0.009432·(100−C)^1.536 + 0.02764·T − 0.2205·√H + 0.6422·√V)
  (Purton 1982; Noble et al. 1980), Q — запас топлива т/га, C — степень высыхания травы %,
  T — °C, H — влажность %, V — ветер км/ч. Скорость фронта R ≈ 0.13·GFDI км/ч (Noble 1980).
Форма — эллипс по ветру, отношение длины к ширине LB = 1.1·V^0.464 (Alexander 1985).
"""

from __future__ import annotations

import math

import numpy as np
import pandas as pd

DANGER = [(3, "низкая"), (8, "умеренная"), (20, "высокая"), (50, "очень высокая"), (100, "чрезвычайная"),
          (150, "экстремальная"), (1e9, "катастрофическая")]


def curing_pct(month: int, precip_30d_mm: float | None = None) -> float:
    base = {1: 100, 2: 100, 3: 60, 4: 50, 5: 55, 6: 75, 7: 90, 8: 95, 9: 95, 10: 90, 11: 95, 12: 100}[month]
    if precip_30d_mm is not None:
        base -= min(25.0, max(0.0, precip_30d_mm - 25.0) * 0.5)
    return float(np.clip(base, 20, 100))


def gfdi(temp_c: float, rh: float, wind_kmh: float, curing: float = 90.0, fuel_t_ha: float = 4.5) -> float:
    return math.exp(-1.523 + 1.027 * math.log(fuel_t_ha) - 0.009432 * (100 - curing) ** 1.536 + 0.02764 * temp_c
                    - 0.2205 * math.sqrt(max(rh, 1.0)) + 0.6422 * math.sqrt(max(wind_kmh, 0.0)))


def danger_class(index: float) -> str:
    return next(name for thr, name in DANGER if index < thr)


def spread_ellipse(lat: float, lon: float, ros_kmh: float, wind_from_deg: float, wind_kmh: float, hours: float,
                   n: int = 36) -> list[list[float]]:
    """Полигон зоны возможного распространения за `hours` (GeoJSON-порядок lon, lat)."""
    head = ros_kmh * hours
    back = 0.15 * head
    lb = max(1.0, 1.1 * max(wind_kmh, 1.0) ** 0.464)
    a = (head + back) / 2
    b = a / lb
    center_shift = (head - back) / 2
    to_dir = math.radians((wind_from_deg + 180) % 360)
    pts = []
    for k in range(n + 1):
        t = 2 * math.pi * k / n
        x_al, y_cr = center_shift + a * math.cos(t), b * math.sin(t)
        east = x_al * math.sin(to_dir) + y_cr * math.cos(to_dir)
        north = x_al * math.cos(to_dir) - y_cr * math.sin(to_dir)
        pts.append([round(lon + east / (111.32 * math.cos(math.radians(lat))), 5), round(lat + north / 111.32, 5)])
    return pts


def forecast_spread(clusters: pd.DataFrame, horizons_h=(6, 12, 24)) -> pd.DataFrame:
    """clusters: [cluster_id, lat, lon, frp_sum] → прогноз по погоде Open-Meteo на ближайшие сутки."""
    from ..apis.open_meteo import forecast

    if clusters.empty:
        return clusters
    frames = forecast(clusters["lat"].tolist(), clusters["lon"].tolist(),
                      hourly=["temperature_2m", "relative_humidity_2m", "wind_speed_10m", "wind_direction_10m", "precipitation"],
                      days=2)
    now = pd.Timestamp.now(tz="UTC").floor("h")
    rows = []
    for (_, c), f in zip(clusters.iterrows(), frames):
        nxt = f[(f.index >= now) & (f.index < now + pd.Timedelta(hours=24))]
        if nxt.empty:
            continue
        worst = nxt.iloc[int(np.argmax(nxt["wind_speed_10m"].to_numpy() - 0.3 * nxt["relative_humidity_2m"].to_numpy()))]
        cur = curing_pct(now.month)
        idx = gfdi(worst["temperature_2m"], worst["relative_humidity_2m"], worst["wind_speed_10m"], cur)
        ros = 0.13 * idx
        row = {"cluster_id": c["cluster_id"], "lat": c["lat"], "lon": c["lon"], "gfdi": round(idx, 1),
               "danger_ru": danger_class(idx), "ros_kmh": round(ros, 2), "wind_kmh": round(float(worst["wind_speed_10m"]), 1),
               "wind_from_deg": round(float(worst["wind_direction_10m"]), 0),
               "spread_to_deg": round((float(worst["wind_direction_10m"]) + 180) % 360, 0),
               "rain_next_24h_mm": round(float(nxt["precipitation"].sum()), 1)}
        for hz in horizons_h:
            row[f"polygon_{hz}h"] = spread_ellipse(c["lat"], c["lon"], ros, row["wind_from_deg"], row["wind_kmh"], hz)
        rows.append(row)
    return pd.DataFrame(rows)

### Связь: орбиты, станции, физика радио/лазер, пакеты, планировщик, доставка

In [ ]:
%%writefile helios/comms/__init__.py
"""Связь: орбиты, станции, физика радио/лазер, пакеты тревог, планировщик и доставка."""

In [ ]:
%%writefile helios/comms/orbits.py
"""Орбитальная механика: SGP4 по TLE → положение спутника, пролёты над станциями, полосы обзора.

Используется ИИ №2 (когда будет связь), планировщиком связи и «сетью спутников»
(какой спутник следующим пролетит над очагом и подтвердит его).
"""

from __future__ import annotations

from dataclasses import dataclass

import numpy as np
import pandas as pd
from sgp4.api import Satrec

R_E = 6378.137
E2 = 6.69437999014e-3  # эксцентриситет² WGS-84
GEO_NORTH_POLE = (80.8, -72.8)  # геомагнитный северный полюс (IGRF, ~2025)


def unix_seconds(times) -> np.ndarray:
    """Секунды UNIX независимо от единиц хранения (pandas 3 часто хранит время в мкс)."""
    t = pd.DatetimeIndex(times)
    t = t.tz_convert("UTC") if t.tz is not None else t.tz_localize("UTC")
    return t.as_unit("ns").asi8 / 1e9


def julian(times: pd.DatetimeIndex) -> tuple[np.ndarray, np.ndarray]:
    unix = unix_seconds(times)
    jd_full = unix / 86400.0 + 2440587.5
    jd = np.floor(jd_full - 0.5) + 0.5
    return jd, jd_full - jd


def gmst_rad(jd: np.ndarray, fr: np.ndarray) -> np.ndarray:
    T = (jd - 2451545.0 + fr) / 36525.0
    sec = 67310.54841 + (876600.0 * 3600 + 8640184.812866) * T + 0.093104 * T ** 2 - 6.2e-6 * T ** 3
    return np.radians(np.mod(sec / 240.0, 360.0))


def rot_z(vec: np.ndarray, theta: np.ndarray) -> np.ndarray:
    c, s = np.cos(theta), np.sin(theta)
    x, y, z = vec[:, 0], vec[:, 1], vec[:, 2]
    return np.column_stack([c * x + s * y, -s * x + c * y, z])


def geodetic_to_ecef(lat: float, lon: float, alt_km: float = 0.0) -> np.ndarray:
    phi, lam = np.radians(lat), np.radians(lon)
    n = R_E / np.sqrt(1 - E2 * np.sin(phi) ** 2)
    return np.array([(n + alt_km) * np.cos(phi) * np.cos(lam), (n + alt_km) * np.cos(phi) * np.sin(lam),
                     (n * (1 - E2) + alt_km) * np.sin(phi)])


def ecef_to_latlon(xyz: np.ndarray) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    x, y, z = xyz[:, 0], xyz[:, 1], xyz[:, 2]
    lon = np.degrees(np.arctan2(y, x))
    p = np.sqrt(x ** 2 + y ** 2)
    lat = np.arctan2(z, p * (1 - E2))
    for _ in range(3):
        n = R_E / np.sqrt(1 - E2 * np.sin(lat) ** 2)
        lat = np.arctan2(z + E2 * n * np.sin(lat), p)
    n = R_E / np.sqrt(1 - E2 * np.sin(lat) ** 2)
    alt = p / np.cos(lat) - n
    return np.degrees(lat), lon, alt


def sun_ecef_unit(times: pd.DatetimeIndex) -> np.ndarray:
    """Направление на Солнце (Astronomical Almanac, точность ~0.01°) в ECEF."""
    jd, fr = julian(times)
    n = jd + fr - 2451545.0
    L = np.radians(np.mod(280.460 + 0.9856474 * n, 360))
    g = np.radians(np.mod(357.528 + 0.9856003 * n, 360))
    lam = L + np.radians(1.915) * np.sin(g) + np.radians(0.020) * np.sin(2 * g)
    eps = np.radians(23.439 - 4e-7 * n)
    eci = np.column_stack([np.cos(lam), np.cos(eps) * np.sin(lam), np.sin(eps) * np.sin(lam)])
    return rot_z(eci, gmst_rad(jd, fr))


def geomagnetic_lat(lat: float, lon: float) -> float:
    """Геомагнитная широта по дипольной модели (для ионосферных мерцаний)."""
    pl, pn = np.radians(GEO_NORTH_POLE[0]), np.radians(GEO_NORTH_POLE[1])
    p, l = np.radians(lat), np.radians(lon)
    return float(np.degrees(np.arcsin(np.sin(p) * np.sin(pl) + np.cos(p) * np.cos(pl) * np.cos(l - pn))))


@dataclass
class Orbit:
    name: str
    line1: str
    line2: str

    def __post_init__(self):
        self.sat = Satrec.twoline2rv(self.line1, self.line2)

    def ecef(self, times: pd.DatetimeIndex) -> np.ndarray:
        jd, fr = julian(times)
        err, r, _ = self.sat.sgp4_array(jd, fr)
        r = np.where(err[:, None] == 0, r, np.nan)
        return rot_z(r, gmst_rad(jd, fr))

    def subpoint(self, times: pd.DatetimeIndex) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
        return ecef_to_latlon(self.ecef(times))


def look_angles(sat_ecef: np.ndarray, lat: float, lon: float, alt_km: float = 0.0):
    """Азимут, угол места (°) и наклонная дальность (км) со станции на спутник."""
    sta = geodetic_to_ecef(lat, lon, alt_km)
    rho = sat_ecef - sta
    phi, lam = np.radians(lat), np.radians(lon)
    e = -np.sin(lam) * rho[:, 0] + np.cos(lam) * rho[:, 1]
    n = -np.sin(phi) * np.cos(lam) * rho[:, 0] - np.sin(phi) * np.sin(lam) * rho[:, 1] + np.cos(phi) * rho[:, 2]
    u = np.cos(phi) * np.cos(lam) * rho[:, 0] + np.cos(phi) * np.sin(lam) * rho[:, 1] + np.sin(phi) * rho[:, 2]
    rng = np.sqrt(e ** 2 + n ** 2 + u ** 2)
    el = np.degrees(np.arctan2(u, np.sqrt(e ** 2 + n ** 2)))
    az = np.mod(np.degrees(np.arctan2(e, n)), 360)
    return az, el, rng, rho / rng[:, None]


def passes(orbit: Orbit, station: dict, start: pd.Timestamp, end: pd.Timestamp, step_s: int = 30,
           min_el: float = 5.0) -> pd.DataFrame:
    times = pd.date_range(start, end, freq=f"{step_s}s", tz="UTC")
    _, el, rng, _ = look_angles(orbit.ecef(times), station["lat"], station["lon"], station.get("alt_m", 0) / 1000)
    up = np.nan_to_num(el, nan=-90) >= min_el
    edges = np.diff(np.concatenate([[0], up.astype(int), [0]]))
    rows = []
    for a, b in zip(np.flatnonzero(edges == 1), np.flatnonzero(edges == -1)):
        k = a + int(np.argmax(el[a:b]))
        rows.append({"satellite": orbit.name, "station": station["id"], "aos": times[a], "los": times[b - 1],
                     "tca": times[k], "max_el": round(float(el[k]), 1), "min_range_km": round(float(rng[a:b].min()), 0),
                     "duration_s": int((b - a) * step_s)})
    return pd.DataFrame(rows)


def haversine_km(lat1, lon1, lat2, lon2):
    p1, p2 = np.radians(lat1), np.radians(lat2)
    a = np.sin((p2 - p1) / 2) ** 2 + np.cos(p1) * np.cos(p2) * np.sin(np.radians(lon2 - lon1) / 2) ** 2
    return 2 * 6371.0 * np.arcsin(np.sqrt(np.clip(a, 0, 1)))


HALF_SWATH_KM = {"MODIS": 1165.0, "VIIRS": 1520.0}


def next_overpass(orbit: Orbit, lat: float, lon: float, start: pd.Timestamp, horizon_h: float = 24,
                  half_swath_km: float = 1520.0, step_s: int = 20) -> pd.Timestamp | None:
    """Ближайший момент, когда точка попадёт в полосу обзора спутника (после start)."""
    times = pd.date_range(start, start + pd.Timedelta(hours=horizon_h), freq=f"{step_s}s")
    la, lo, _ = orbit.subpoint(times)
    d = haversine_km(la, lo, lat, lon)
    inside = np.flatnonzero(np.nan_to_num(d, nan=1e9) <= half_swath_km)
    if len(inside) == 0:
        return None
    # первое непрерывное «окно» видимости; внутри него — момент минимального угла обзора
    breaks = np.flatnonzero(np.diff(inside) > 1)
    run = inside[: (breaks[0] + 1) if len(breaks) else len(inside)]
    return times[int(run[np.argmin(d[run])])]


def line_of_sight(a: np.ndarray, b: np.ndarray, grazing_km: float = 100.0) -> np.ndarray:
    """Не закрыта ли линия a→b Землёй (с запасом на атмосферу), для каждой строки."""
    d = b - a
    t = np.clip(-(a * d).sum(1) / np.maximum((d * d).sum(1), 1e-9), 0, 1)
    closest = a + d * t[:, None]
    return np.linalg.norm(closest, axis=1) > R_E + grazing_km

In [ ]:
%%writefile helios/comms/stations.py
"""Наземные станции и ретрансляторы для моделирования связи.

Координаты реальных станций — открытые (приблизительно). Станции в Казахстане помечены
как «условные» (illustrative): реальной приёмной станции MODIS/VIIRS там может не быть,
это сценарий «что если поставить свою станцию прямого приёма».
"""

from __future__ import annotations

from .orbits import geomagnetic_lat

STATIONS: list[dict] = [
    {"id": "SVALSAT", "name": "Svalbard SvalSat (KSAT)", "lat": 78.229, "lon": 15.408, "alt_m": 500,
     "kind": "rf", "bands": ["S", "X", "Ka"], "dish_m": 11.3, "real": True},
    {"id": "TROLLSAT", "name": "TrollSat, Antarctica (KSAT)", "lat": -72.012, "lon": 2.535, "alt_m": 1270,
     "kind": "rf", "bands": ["S", "X", "Ka"], "dish_m": 7.3, "real": True},
    {"id": "MCMURDO", "name": "McMurdo, Antarctica (NASA NEN)", "lat": -77.839, "lon": 166.667, "alt_m": 100,
     "kind": "rf", "bands": ["S", "X"], "dish_m": 10.0, "real": True},
    {"id": "FAIRBANKS", "name": "Fairbanks CDA, Alaska (NOAA)", "lat": 64.972, "lon": -147.511, "alt_m": 300,
     "kind": "rf", "bands": ["S", "X"], "dish_m": 13.0, "real": True},
    {"id": "WALLOPS", "name": "Wallops, Virginia (NASA)", "lat": 37.940, "lon": -75.466, "alt_m": 10,
     "kind": "rf", "bands": ["S", "X"], "dish_m": 11.0, "real": True},
    {"id": "WHITE_SANDS", "name": "White Sands Complex, NM (TDRS ground)", "lat": 32.500, "lon": -106.608, "alt_m": 1450,
     "kind": "rf", "bands": ["S", "Ka"], "dish_m": 18.0, "real": True, "relay_ground": "TDRS 13"},
    {"id": "ASTANA_DB", "name": "Астана, станция прямого приёма (условная)", "lat": 51.169, "lon": 71.449, "alt_m": 350,
     "kind": "rf", "bands": ["X"], "dish_m": 3.0, "real": False},
    {"id": "ALMATY_DB", "name": "Алматы, станция прямого приёма (условная)", "lat": 43.238, "lon": 76.946, "alt_m": 850,
     "kind": "rf", "bands": ["X"], "dish_m": 3.0, "real": False},
    {"id": "OGS_TABLE_MTN", "name": "Table Mountain OGS-1, CA (LCRD)", "lat": 34.381, "lon": -117.680, "alt_m": 2286,
     "kind": "optical", "aperture_m": 1.0, "real": True, "relay_ground": "STPSAT-6 (LCRD)"},
    {"id": "OGS_HALEAKALA", "name": "Haleakalā OGS-2, Hawaii (LCRD)", "lat": 20.708, "lon": -156.257, "alt_m": 3055,
     "kind": "optical", "aperture_m": 0.6, "real": True, "relay_ground": "STPSAT-6 (LCRD)"},
    {"id": "OGS_TENERIFE", "name": "ESA Optical Ground Station, Tenerife", "lat": 28.300, "lon": -16.511, "alt_m": 2393,
     "kind": "optical", "aperture_m": 1.0, "real": True},
    {"id": "OGS_ASSY_TURGEN", "name": "Ассы-Тургень (Казахстан), оптическая станция (условная)", "lat": 43.225,
     "lon": 77.871, "alt_m": 2750, "kind": "optical", "aperture_m": 1.0, "real": False},
]

for _s in STATIONS:
    _s["geomag_lat"] = round(geomagnetic_lat(_s["lat"], _s["lon"]), 1)

BY_ID = {s["id"]: s for s in STATIONS}


def rf_stations() -> list[dict]:
    return [s for s in STATIONS if s["kind"] == "rf"]


def optical_stations() -> list[dict]:
    return [s for s in STATIONS if s["kind"] == "optical"]

In [ ]:
%%writefile helios/comms/physics.py
"""Физико-математическая модель связи «радио ↔ лазер» (идея ментора, честная физика).

Формулы со слайда ментора верны по форме; числа и выводы мы проверили:

1. Плазменная частота f_p ≈ 8.98·√N_e Гц. Днём N_e(F2) ~ 1e12 м⁻³ → f_p ≈ 9 МГц.
   Поглощение α ∝ N_e·ν/f². Вспышка на Солнце (рентген) «глушит» КВ-радио (3–30 МГц) на
   дневной стороне, а для X-диапазона (8 ГГц) поглощение ~1e-5 дБ — НИЧТОЖНО. Поэтому
   реальные угрозы радиоканалу S/X/Ka — это: дождь (Ka!), ионосферные мерцания у полярных
   станций при геомагнитных бурях (Kp), шум Солнца, когда оно рядом с лучом антенны.
   Лазеру ионосфера не мешает вовсе, но его режут облака и турбулентность.
   → Отсюда гибрид: радио + лазер + разнесённые станции + ретранслятор на ГСО (LCRD).
2. Дифракция: θ ≈ 1.22·λ/D. λ=1064 нм, D=15 см → 8.6 мкрад (верно).
3. Гауссов пучок: w(R) = w0·√(1+(λR/πw0²)²), P_R = P_T·η·T_atm·(1−exp(−2r²/w²)).
   100 Вт с 800 км на приёмник r=0.5 м: ~3.8 Вт оптики → ~2 Вт электричества при КПД 55%.
   Для СВЯЗИ 100 Вт не нужно: детектору хватает ~100 фотонов на бит (≈ −50 дБм на 1 Гбит/с).
4. SNR приёмника с фильтром Фабри–Перо: SNR = (M·R·P_s)² / (2qBM²F(R·P_s + R·P_bg + I_d) + 4kTB/R_L).
   Солнце прямо в поле зрения даже через фильтр 0.1 нм даёт фон ~0.05 Вт на апертуру 0.8 м² —
   поэтому в реальных терминалах есть «угол исключения Солнца».
"""

from __future__ import annotations

import math

import numpy as np

C = 299_792_458.0
K_B = 1.380649e-23
H_PLANCK = 6.62607015e-34
Q_E = 1.602176634e-19
EPS0 = 8.8541878128e-12
M_E = 9.1093837015e-31
R_E_KM = 6378.137
MU_EARTH = 398_600.4418  # км³/с²


# ============================================================ 1. Ионосфера
def plasma_frequency_hz(ne_m3: float) -> float:
    return math.sqrt(ne_m3 * Q_E ** 2 / (EPS0 * M_E)) / (2 * math.pi)


def flare_d_region_ne(xray_flux_wm2: float) -> float:
    """Электронная плотность D-слоя при вспышке (грубая масштабная модель): B1→3e8, X1→3e10 м⁻³."""
    return 3e8 * math.sqrt(max(xray_flux_wm2, 1e-8) / 1e-7)


def ionospheric_absorption_db(f_hz: float, ne_m3: float, nu_hz: float = 5e6, path_km: float = 20.0) -> float:
    """Неотклоняющее поглощение: κ = e²/(2ε0·m·c)·N·ν/(ω²+ν²) [Нп/м]."""
    omega = 2 * math.pi * f_hz
    kappa = Q_E ** 2 / (2 * EPS0 * M_E * C) * ne_m3 * nu_hz / (omega ** 2 + nu_hz ** 2)
    return 8.686 * kappa * path_km * 1000.0


# ============================================================ 2–3. Оптика пучка
def airy_half_angle_rad(lambda_m: float, d_m: float) -> float:
    return 1.22 * lambda_m / d_m


def beam_radius_m(r_m: float, lambda_m: float, w0_m: float) -> float:
    return w0_m * math.sqrt(1.0 + (lambda_m * r_m / (math.pi * w0_m ** 2)) ** 2)


def captured_fraction(r_rx_m: float, w_m: float) -> float:
    return 1.0 - math.exp(-2.0 * r_rx_m ** 2 / w_m ** 2)


def laser_received_power_w(p_t_w: float, r_m: float, lambda_m: float = 1550e-9, w0_m: float = 0.05,
                           r_rx_m: float = 0.2, eta_t: float = 0.8, eta_r: float = 0.8,
                           t_atm: float = 1.0, pointing_loss_db: float = 3.0) -> float:
    w = beam_radius_m(r_m, lambda_m, w0_m)
    return p_t_w * eta_t * eta_r * t_atm * captured_fraction(r_rx_m, w) * 10 ** (-pointing_loss_db / 10)


def atmospheric_transmission(el_deg: float, tau_zenith: float = 0.15) -> float:
    return math.exp(-tau_zenith / max(math.sin(math.radians(max(el_deg, 1.0))), 0.02))


def solar_spectral_irradiance(lambda_nm: float) -> float:
    """Спектральная освещённость Солнца у Земли, Вт/м²/нм (Планк, 5772 K)."""
    lam = lambda_nm * 1e-9
    t = 5772.0
    b = 2 * H_PLANCK * C ** 2 / lam ** 5 / (math.exp(H_PLANCK * C / (lam * K_B * t)) - 1)
    omega_sun = 6.794e-5  # телесный угол диска Солнца, ср
    return b * omega_sun * 1e-9


def optical_snr(p_signal_w: float, p_background_w: float, bandwidth_hz: float = 1e9, responsivity: float = 0.9,
                apd_gain: float = 10.0, excess_noise: float = 4.0, dark_current_a: float = 1e-9,
                temp_k: float = 300.0, r_load: float = 50.0) -> float:
    m, r = apd_gain, responsivity
    signal = (m * r * p_signal_w) ** 2
    shot = 2 * Q_E * bandwidth_hz * m ** 2 * excess_noise * (r * p_signal_w + r * p_background_w + dark_current_a)
    thermal = 4 * K_B * temp_k * bandwidth_hz / r_load
    return signal / (shot + thermal)


def photons_required_w(rate_bps: float, photons_per_bit: float = 100.0, lambda_m: float = 1550e-9) -> float:
    return photons_per_bit * H_PLANCK * C / lambda_m * rate_bps


# ============================================================ Радиоканал
# ITU-R P.838-3, горизонтальная поляризация: f, k_H, α_H
_P838 = np.array([
    [1, 0.0000259, 0.9691], [2, 0.0000847, 1.0664], [4, 0.0001071, 1.6009], [6, 0.0007056, 1.5900],
    [8, 0.004115, 1.3905], [10, 0.01217, 1.2571], [12, 0.02386, 1.1825], [15, 0.04481, 1.1233],
    [20, 0.09164, 1.0568], [25, 0.1571, 0.9991], [30, 0.2403, 0.9485], [35, 0.3374, 0.9047],
    [40, 0.4431, 0.8673],
])


def fspl_db(f_hz: float | np.ndarray, d_m: float | np.ndarray) -> np.ndarray:
    return 20 * np.log10(4 * np.pi * np.asarray(d_m) * f_hz / C)


def rain_coeffs(f_ghz: float) -> tuple[float, float]:
    lf = np.log(_P838[:, 0])
    k = float(np.exp(np.interp(np.log(f_ghz), lf, np.log(_P838[:, 1]))))
    a = float(np.interp(np.log(f_ghz), lf, _P838[:, 2]))
    return k, a


def rain_attenuation_db(f_ghz: float, rain_mm_h: np.ndarray, el_deg: np.ndarray, lat: float, alt_km: float = 0.0) -> np.ndarray:
    """ITU-R P.618 (упрощённо): γ=kR^α, наклонный путь до высоты дождя, редукция по горизонтали."""
    k, a = rain_coeffs(f_ghz)
    rain = np.maximum(np.asarray(rain_mm_h, dtype=float), 0.0)
    el = np.radians(np.maximum(np.asarray(el_deg, dtype=float), 5.0))
    h_rain = 5.0 - 0.075 * max(abs(lat) - 23.0, 0.0) + 0.36  # P.839 (старая форма)
    ls = np.maximum(h_rain - alt_km, 0.0) / np.sin(el)
    gamma = k * rain ** a
    lg = ls * np.cos(el)
    r = 1.0 / (1.0 + 0.78 * np.sqrt(np.maximum(lg * gamma, 0) / f_ghz) - 0.38 * (1 - np.exp(-2 * lg)))
    return gamma * ls * np.clip(r, 0.1, 1.5)


def fso_rain_attenuation_db(rain_mm_h: np.ndarray, el_deg: np.ndarray, lat: float, alt_km: float = 0.0) -> np.ndarray:
    """Дождь на лазере 1550 нм (ITU-R P.1814, модель Carbonneau): γ = 1.076·R^0.67 дБ/км до высоты дождя.
    Слабый дождь лазер переживает, ливень 20+ мм/ч его гасит — в отличие от S/X-диапазонов."""
    rain = np.maximum(np.asarray(rain_mm_h, dtype=float), 0.0)
    el = np.radians(np.maximum(np.asarray(el_deg, dtype=float), 5.0))
    h_rain = 5.0 - 0.075 * max(abs(lat) - 23.0, 0.0) + 0.36
    return 1.076 * rain ** 0.67 * np.maximum(h_rain - alt_km, 0.0) / np.sin(el)


def cloud_attenuation_db(f_ghz: float, lwc_mm: np.ndarray, el_deg: np.ndarray) -> np.ndarray:
    """ITU-R P.840 (упрощённо): A = K_l(f)·L / sin(el), K_l ≈ 6e-4·f² дБ/мм."""
    return 6e-4 * f_ghz ** 2 * np.asarray(lwc_mm) / np.sin(np.radians(np.maximum(el_deg, 5.0)))


def gas_attenuation_db(f_ghz: float, el_deg: np.ndarray, rh: np.ndarray) -> np.ndarray:
    zenith = 0.035 + 0.0025 * f_ghz + (0.25 * np.exp(-((f_ghz - 22.2) / 6) ** 2) + 0.004 * f_ghz) * np.asarray(rh) / 100
    return zenith / np.sin(np.radians(np.maximum(el_deg, 5.0)))


def scintillation_s4(f_ghz: float, kp: np.ndarray, geomag_lat: float) -> np.ndarray:
    """Индекс мерцаний S4: L-диапазон ~0.05 в спокойствие, до ~0.6 в бурю у полярных станций; ∝ f^-1.5."""
    kp = np.asarray(kp, dtype=float)
    gm = abs(geomag_lat)
    zone = 1.0 if gm >= 60 else (0.35 if gm >= 50 else (0.6 if gm <= 20 else 0.15))
    s4_l = zone * (0.05 + 0.55 / (1 + np.exp(-(kp - 6.0) * 1.2)))
    return s4_l * (1.575 / f_ghz) ** 1.5


def scintillation_fade_db(s4: np.ndarray, rng: np.random.Generator | None = None) -> np.ndarray:
    """Мгновенное замирание по Накагами-m (m = 1/S4²); без rng — 99%-ная глубина ≈ 27.5·S4^1.26."""
    s4 = np.clip(np.asarray(s4, dtype=float), 1e-3, 1.0)
    if rng is None:
        return 27.5 * s4 ** 1.26 / 2
    m = np.maximum(1.0 / s4 ** 2, 0.5)
    power = rng.gamma(shape=m, scale=1.0 / m)
    return np.maximum(-10 * np.log10(np.maximum(power, 1e-4)), 0.0)


def solar_noise_increase_db(f_ghz: float, sep_deg: np.ndarray, dish_m: float, f107: float, xray_flux: float,
                            t_sys_k: float = 150.0) -> np.ndarray:
    """Рост шумовой температуры, когда Солнце рядом с лучом антенны (радиовсплески при вспышках)."""
    lam = C / (f_ghz * 1e9)
    bw = 70 * lam / dish_m  # ширина луча по половинной мощности, °
    t_quiet = 6000 + 1.2e5 * (2.8 / f_ghz) ** 1.8 * (f107 / 100.0)  # яркостная температура диска, К
    burst = 1 + 200 * max(xray_flux / 1e-4, 0) * (2.8 / f_ghz) ** 1.5  # X1 → x200 на S-диапазоне
    fill = min(1.0, (0.53 / bw) ** 2)
    gain = np.exp(-4 * math.log(2) * (np.asarray(sep_deg) / bw) ** 2)
    dt = t_quiet * burst * fill * gain
    return 10 * np.log10(1 + dt / t_sys_k)


# Параметры каналов «тяжёлых данных»: частота, ЭИИМ спутника, скорость, требуемое Eb/N0, Tсист.
# Подобраны так, чтобы запас при угле места 5° на 11-м антенне был ~6 дБ (типовой проектный запас).
RF_CHANNELS = {
    "S": {"f_ghz": 2.25, "eirp_dbw": 0.0, "rate_bps": 20e6, "ebn0_req_db": 4.0, "t_sys_k": 150.0},
    "X": {"f_ghz": 8.2, "eirp_dbw": 10.0, "rate_bps": 150e6, "ebn0_req_db": 4.5, "t_sys_k": 150.0},
    "Ka": {"f_ghz": 26.5, "eirp_dbw": 19.0, "rate_bps": 600e6, "ebn0_req_db": 5.0, "t_sys_k": 250.0},
}
# Короткие пакеты тревоги идут в робастном режиме на низкой скорости → запас больше на 10·lg(R/R_alert)
ALERT_RATE_BPS = 64e3


def alert_gain_db(channel: str) -> float:
    rate = OPTICAL["rate_bps"] if channel == "OPT" else RF_CHANNELS[channel]["rate_bps"]
    return 10 * math.log10(rate / ALERT_RATE_BPS)


def g_over_t_db(f_ghz: float, dish_m: float, eff: float = 0.6, t_sys_k: float = 150.0) -> float:
    lam = C / (f_ghz * 1e9)
    gain = 10 * math.log10(eff * (math.pi * dish_m / lam) ** 2)
    return gain - 10 * math.log10(t_sys_k)


def rf_margin_db(band: str, range_km: np.ndarray, el_deg: np.ndarray, dish_m: float, lat: float, alt_km: float,
                 rain_mm_h: np.ndarray, lwc_mm: np.ndarray, rh: np.ndarray, kp: np.ndarray, geomag_lat: float,
                 sun_sep_deg: np.ndarray, f107: float = 120.0, xray_flux: float = 1e-7,
                 rng: np.random.Generator | None = None, scint_mult: np.ndarray | float = 1.0) -> np.ndarray:
    """Запас радиолинии (дБ) = Eb/N0(принятое) − Eb/N0(требуемое)."""
    ch = RF_CHANNELS[band]
    f = ch["f_ghz"]
    losses = (
        fspl_db(f * 1e9, np.asarray(range_km) * 1000)
        + rain_attenuation_db(f, rain_mm_h, el_deg, lat, alt_km)
        + cloud_attenuation_db(f, lwc_mm, el_deg)
        + gas_attenuation_db(f, el_deg, rh)
        + scintillation_fade_db(scintillation_s4(f, kp, geomag_lat), rng) * scint_mult
        + 1.5  # поляризация, наведение, реализация
    )
    noise_rise = solar_noise_increase_db(f, sun_sep_deg, dish_m, f107, xray_flux, ch["t_sys_k"])
    cn0 = ch["eirp_dbw"] - losses + g_over_t_db(f, dish_m, t_sys_k=ch["t_sys_k"]) - noise_rise + 228.6
    return cn0 - 10 * np.log10(ch["rate_bps"]) - ch["ebn0_req_db"]


# ============================================================ Лазерные каналы
OPTICAL = {"lambda_m": 1550e-9, "p_t_w": 2.0, "w0_m": 0.05, "rate_bps": 1.2e9, "photons_per_bit": 100.0}


def optical_ground_margin_db(range_km: np.ndarray, el_deg: np.ndarray, cloud_frac: np.ndarray, aperture_m: float,
                             sun_sep_deg: np.ndarray, rng: np.random.Generator | None = None,
                             rain_mm_h: np.ndarray | float = 0.0, lat: float = 46.0, alt_km: float = 0.0) -> np.ndarray:
    """Запас оптической линии «спутник/ГСО → наземная станция»: облака режут линию почти полностью, ливень — тоже."""
    o = OPTICAL
    rng_m = np.asarray(range_km, dtype=float) * 1000
    lam, w0 = o["lambda_m"], o["w0_m"]
    w = w0 * np.sqrt(1 + (lam * rng_m / (np.pi * w0 ** 2)) ** 2)
    capture = 1 - np.exp(-2 * (aperture_m / 2) ** 2 / w ** 2)
    el = np.maximum(np.asarray(el_deg, dtype=float), 1.0)
    t_atm = np.exp(-0.12 / np.sin(np.radians(el)))
    turb_db = 1.0 + 6.0 * np.exp(-(el - 5) / 12)  # турбулентность хуже у горизонта
    cloud = np.asarray(cloud_frac, dtype=float)
    # оптически толстое облако ослабляет лазер на 60–120 дБ: линия просто «закрыта»
    if rng is None:
        cloud_db = 150.0 * np.clip((cloud - 0.5) / 0.3, 0, 1)
    else:
        n = len(np.atleast_1d(cloud))
        cloud_db = np.where(rng.random(n) < cloud, rng.uniform(60.0, 120.0, n), 0.0)
    rain_db = fso_rain_attenuation_db(rain_mm_h, el, lat, alt_km)
    p_rx = o["p_t_w"] * 0.64 * t_atm * capture * 10 ** (-(3.0 + turb_db + cloud_db + rain_db) / 10)
    sun_pen = np.where(np.asarray(sun_sep_deg) < 3.0, 10.0, 0.0)  # угол исключения Солнца
    need = photons_required_w(o["rate_bps"], o["photons_per_bit"], lam)
    return 10 * np.log10(np.maximum(p_rx, 1e-30) / need) - sun_pen


def isl_optical_margin_db(range_km: np.ndarray, sun_sep_deg: np.ndarray, rx_aperture_m: float = 0.108) -> np.ndarray:
    """Межспутниковый лазер LEO→ГСО (как ILLUMA-T→LCRD): вакуум, только дифракция и наведение."""
    o = OPTICAL
    rng_m = np.asarray(range_km, dtype=float) * 1000
    w = o["w0_m"] * np.sqrt(1 + (o["lambda_m"] * rng_m / (np.pi * o["w0_m"] ** 2)) ** 2)
    p_rx = o["p_t_w"] * 0.64 * (1 - np.exp(-2 * (rx_aperture_m / 2) ** 2 / w ** 2)) * 10 ** (-4.0 / 10)
    need = photons_required_w(o["rate_bps"], o["photons_per_bit"], o["lambda_m"])
    return 10 * np.log10(p_rx / need) - np.where(np.asarray(sun_sep_deg) < 5.0, 15.0, 0.0)


# ============================================================ Проектирование миссии
def orbital_period_min(alt_km: float) -> float:
    a = R_E_KM + alt_km
    return 2 * math.pi * math.sqrt(a ** 3 / MU_EARTH) / 60


def aperture_for_gsd_m(gsd_m: float, alt_km: float, wavelength_um: float) -> float:
    """Минимальный диаметр объектива по дифракции: D = 1.22·λ·H / GSD."""
    return 1.22 * wavelength_um * 1e-6 * alt_km * 1000 / gsd_m


def swath_km(alt_km: float, max_scan_deg: float) -> float:
    """Ширина полосы обзора на сферической Земле для угла сканирования ±max_scan."""
    rs = R_E_KM + alt_km
    s = math.radians(max_scan_deg)
    nadir_angle = math.asin(min(1.0, rs / R_E_KM * math.sin(s)))
    return 2 * R_E_KM * (nadir_angle - s)


def max_slant_range_km(alt_km: float, min_el_deg: float) -> float:
    el = math.radians(min_el_deg)
    rs = R_E_KM + alt_km
    return -R_E_KM * math.sin(el) + math.sqrt((R_E_KM * math.sin(el)) ** 2 + rs ** 2 - R_E_KM ** 2)


def max_pass_minutes(alt_km: float, min_el_deg: float) -> float:
    el = math.radians(min_el_deg)
    rs = R_E_KM + alt_km
    central = math.acos(R_E_KM / rs * math.cos(el)) - el
    return orbital_period_min(alt_km) * (2 * central) / (2 * math.pi)


def mission_design_table() -> list[dict]:
    """Высоты орбит, на которых тепловизор «добивает» до поверхности, и нужный объектив."""
    rows = []
    for name, alt, scan in (("ISS / ECOSTRESS", 410, 25.5), ("Terra/Aqua MODIS", 705, 55.0),
                            ("Suomi-NPP / NOAA-20/21 VIIRS", 829, 56.3), ("Средняя орбита (MEO)", 8000, 10.0),
                            ("Геостационар (GOES ABI, Himawari)", 35786, 8.7)):
        rows.append({
            "orbit": name, "alt_km": alt, "period_min": round(orbital_period_min(alt), 1),
            "swath_km": round(swath_km(alt, scan), 0),
            "aperture_cm_for_375m_at_4um": round(100 * aperture_for_gsd_m(375, alt, 4.0), 1),
            "aperture_cm_for_1km_at_11um": round(100 * aperture_for_gsd_m(1000, alt, 11.0), 1),
            "aperture_cm_for_30m_at_11um": round(100 * aperture_for_gsd_m(30, alt, 11.0), 1),
            "max_pass_min_el10": round(max_pass_minutes(alt, 10), 1) if alt < 20000 else None,
            "max_range_km_el5": round(max_slant_range_km(alt, 5), 0),
        })
    return rows


# ============================================================ Симулятор «радио vs лазер»
def simulate_space_link(xray_flux_wm2: float = 1e-6, kp: float = 3.0, rain_mm_h: float = 0.0, cloud_frac: float = 0.2,
                        distance_km: float = 800.0, elevation_deg: float = 30.0, laser_power_w: float = 2.0,
                        receiver_radius_m: float = 0.2, station_geomag_lat: float = 46.0) -> dict:
    """Интерактив для сайта: условия → запасы радиолиний S/X/Ka и лазера + какой канал выбрать."""
    out = {"inputs": dict(xray_flux_wm2=xray_flux_wm2, kp=kp, rain_mm_h=rain_mm_h, cloud_frac=cloud_frac,
                          distance_km=distance_km, elevation_deg=elevation_deg, laser_power_w=laser_power_w)}
    ne = flare_d_region_ne(xray_flux_wm2)
    out["ionosphere"] = {
        "d_region_ne_m3": ne,
        "plasma_freq_F2_MHz": round(plasma_frequency_hz(1e12) / 1e6, 2),
        "absorption_db": {name: round(ionospheric_absorption_db(f, ne), 6)
                          for name, f in (("HF_10MHz", 10e6), ("S_2.25GHz", 2.25e9), ("X_8.2GHz", 8.2e9),
                                          ("laser_1550nm", C / 1550e-9))},
    }
    el, rng_km = np.array([elevation_deg]), np.array([distance_km])
    margins = {}
    for band in ("S", "X", "Ka"):
        m = rf_margin_db(band, rng_km, el, 11.0, 50.0, 0.3, np.array([rain_mm_h]), np.array([cloud_frac * 0.4 + rain_mm_h * 0.05]),
                         np.array([70.0]), np.array([kp]), station_geomag_lat, np.array([60.0]), 120.0, xray_flux_wm2)
        margins[band] = round(float(m[0]), 2)
    opt = optical_ground_margin_db(rng_km, el, np.array([cloud_frac]), 1.0, np.array([60.0]),
                                   rain_mm_h=np.array([rain_mm_h]), lat=50.0, alt_km=0.3)
    margins["laser_ground"] = round(float(opt[0]), 2)
    margins["laser_ISL_to_GEO"] = round(float(isl_optical_margin_db(np.array([38000.0]), np.array([60.0]))[0]), 2)
    out["margins_db"] = margins
    out["alert_mode_margins_db"] = {k: round(v + alert_gain_db({"laser_ground": "OPT", "laser_ISL_to_GEO": "OPT"}.get(k, k)), 1)
                                    for k, v in margins.items()}
    out["p_clear_line_of_sight_laser"] = round(1.0 - cloud_frac, 2)
    w = beam_radius_m(distance_km * 1000, 1064e-9, 0.075)
    p_opt = laser_received_power_w(laser_power_w, distance_km * 1000, 1064e-9, 0.075, receiver_radius_m,
                                   t_atm=atmospheric_transmission(elevation_deg) * (1 - min(cloud_frac, 1.0)), pointing_loss_db=0)
    out["laser_beam"] = {"spot_radius_m": round(w, 2), "divergence_urad": round(airy_half_angle_rad(1064e-9, 0.15) * 1e6, 2),
                         "optical_received_w": round(p_opt, 4), "pv_electric_w_GaAs55pct": round(0.55 * p_opt, 4)}
    # из «живых» каналов (запас > 3 дБ) берём самый быстрый: лазер > Ka > X > S
    speed = {"laser_ground": 1.2e9, "laser_ISL_to_GEO": 1.2e9, "Ka": 600e6, "X": 150e6, "S": 20e6}
    usable = [k for k, v in margins.items() if v > 3.0]
    out["recommended_channel"] = max(usable, key=lambda k: (speed[k], margins[k])) if usable else "store_and_forward"
    alert_ok = [k for k, v in out["alert_mode_margins_db"].items() if v > 3.0]
    out["alert_channel"] = max(alert_ok, key=lambda k: out["alert_mode_margins_db"][k]) if alert_ok else "store_and_forward"
    return out

In [ ]:
%%writefile helios/comms/packets.py
"""Компактный пакет тревоги (24 байта) — то, что спутник шлёт первым, по любому живому каналу.

Ментор: «сначала короткие сообщения, потом длинные». Полный снимок весит сотни МБ,
а пакет тревоги — 24 байта: его можно протолкнуть даже через узкий S-band/ретранслятор
за доли секунды.

Формат (big-endian):
  B  version | B priority | B sat_id | B flags | I unix_time | i lat*1e5 | i lon*1e5 |
  H frp*10 (МВт) | H cluster_n | B p_escalation*255 | B p_real*255 | H crc16
"""

from __future__ import annotations

import struct

PRIORITIES = {"SUPPRESS": 0, "ROUTINE": 1, "HIGH": 2, "CRITICAL": 3, "CONFIRMED": 4}
PRIORITY_NAMES = {v: k for k, v in PRIORITIES.items()}
SAT_IDS = {"MODIS_T": 1, "MODIS_A": 2, "VIIRS_SNPP": 3, "VIIRS_NOAA20": 4, "VIIRS_NOAA21": 5}
SAT_NAMES = {v: k for k, v in SAT_IDS.items()}
FLAG_NIGHT, FLAG_STATIC_NEAR, FLAG_GLINT, FLAG_CONFIRMED = 1, 2, 4, 8
_FMT = ">BBBBIiiHHBB"
PACKET_BYTES = struct.calcsize(_FMT) + 2


def crc16_ccitt(data: bytes, crc: int = 0xFFFF) -> int:
    for byte in data:
        crc ^= byte << 8
        for _ in range(8):
            crc = ((crc << 1) ^ 0x1021) if crc & 0x8000 else (crc << 1)
            crc &= 0xFFFF
    return crc


def encode(priority: str, stream: str, unix_time: int, lat: float, lon: float, frp_mw: float,
           cluster_n: int, p_escalation: float, p_real: float, flags: int = 0) -> bytes:
    body = struct.pack(
        _FMT, 1, PRIORITIES.get(priority, 1), SAT_IDS.get(stream, 0), flags & 0xFF,
        int(unix_time) & 0xFFFFFFFF, int(round(lat * 1e5)), int(round(lon * 1e5)),
        min(int(round(frp_mw * 10)), 65535), min(int(cluster_n), 65535),
        int(round(max(0.0, min(1.0, p_escalation)) * 255)), int(round(max(0.0, min(1.0, p_real)) * 255)),
    )
    return body + struct.pack(">H", crc16_ccitt(body))


def decode(packet: bytes) -> dict:
    if len(packet) != PACKET_BYTES:
        raise ValueError(f"пакет должен быть {PACKET_BYTES} байт, получено {len(packet)}")
    body, crc = packet[:-2], struct.unpack(">H", packet[-2:])[0]
    if crc16_ccitt(body) != crc:
        raise ValueError("CRC не совпал — пакет повреждён")
    v, pr, sat, flags, t, lat, lon, frp, n, pe, preal = struct.unpack(_FMT, body)
    return {
        "version": v, "priority": PRIORITY_NAMES.get(pr, "ROUTINE"), "stream": SAT_NAMES.get(sat, "?"),
        "flags": flags, "unix_time": t, "lat": lat / 1e5, "lon": lon / 1e5, "frp_mw": frp / 10,
        "cluster_n": n, "p_escalation": round(pe / 255, 3), "p_real": round(preal / 255, 3),
    }

In [ ]:
%%writefile helios/comms/link_sim.py
"""Симулятор каналов связи на реальных входах (обучающие данные для ИИ №2 и планировщика).

Реальное: орбиты (TLE CelesTrak → SGP4), погода у станций (Open-Meteo ERA5/прогноз:
осадки, облачность нижнего/среднего/верхнего яруса, влажность), геомагнитный индекс Kp
и радиопоток F10.7 (GFZ Potsdam), вспышки (NOAA SWPC).
Смоделированное: запас каждой линии по физике (physics.py) + случайные, но ПЛАВНЫЕ во
времени (AR(1)) дождевые ячейки, облака на луче и всплески мерцаний. Открытой телеметрии
спутников нет, поэтому разметка «физическая», а не с борта.

Окна моделирования: свежие дни + реальные геомагнитные бури (май 2024, Kp 9; октябрь 2024).
Для старых окон геометрия пролётов считается по текущим TLE — она статистически
представительна (те же орбиты), но не повторяет точное положение спутников в тот день.
"""

from __future__ import annotations

from datetime import datetime, timedelta, timezone

import numpy as np
import pandas as pd
from scipy.stats import norm

from ..apis import celestrak, open_meteo, space_weather
from ..utils import get_logger
from . import physics
from .orbits import Orbit, look_angles, sun_ecef_unit, unix_seconds
from .stations import STATIONS

log = get_logger("helios.linksim")

OBSERVERS = ["TERRA", "AQUA", "SUOMI NPP", "NOAA 20", "NOAA 21"]
CHANNELS = ["S", "X", "Ka", "OPT"]
HOURLY = ["precipitation", "cloud_cover", "cloud_cover_low", "cloud_cover_mid", "cloud_cover_high", "relative_humidity_2m"]
STORM_WINDOWS = [("2024-05-08", "2024-05-15"), ("2024-10-07", "2024-10-12")]


class Environment:
    """Погода у станций и космическая погода на интервале [start, end]."""

    def __init__(self, start: pd.Timestamp, end: pd.Timestamp, mode: str = "archive", seed: int = 0):
        self.start, self.end, self.mode = start, end, mode
        self.rng = np.random.default_rng(seed)
        lats = [s["lat"] for s in STATIONS]
        lons = [s["lon"] for s in STATIONS]
        try:
            if mode == "archive":
                frames = open_meteo.archive(lats, lons, start.date(), end.date(), hourly=HOURLY)
            else:
                frames = open_meteo.forecast(lats, lons, hourly=HOURLY, days=3)
        except Exception as exc:
            log.warning("Open-Meteo недоступен (%s): климатическая погода", exc)
            idx = pd.date_range(start.floor("h"), end.ceil("h"), freq="h", tz="UTC")
            frames = [pd.DataFrame({"precipitation": self.rng.exponential(0.1, len(idx)),
                                    **{c: self.rng.uniform(0, 100, len(idx)) for c in HOURLY[1:5]},
                                    "relative_humidity_2m": 70.0}, index=idx) for _ in STATIONS]
        self.weather = {s["id"]: f for s, f in zip(STATIONS, frames)}
        self.kp = self._gfz("Kp", 2.0)
        self.f107 = self._gfz("Fobs", 120.0)
        self.xray = self._xray()

    def _gfz(self, index: str, default: float) -> pd.Series:
        try:
            df = space_weather.gfz_index(index, self.start - timedelta(days=2), self.end + timedelta(days=1))
            s = df.set_index("time")[index.lower()].astype(float)
            s = s[s > -1]
            if len(s):
                return s
        except Exception as exc:
            log.info("GFZ %s: %s", index, exc)
        if index == "Kp":
            try:
                k = space_weather.kp_recent().set_index("time")["kp"]
                if len(k):
                    return k
            except Exception:
                pass
        return pd.Series([default], index=[self.start])

    def _xray(self) -> pd.Series:
        """Поток рентгена: реальный (SWPC, последние сутки) или модель вспышек ~ F10.7 (Пуассон)."""
        idx = pd.date_range(self.start.floor("min"), self.end.ceil("min"), freq="min", tz="UTC")
        f107 = float(np.nanmean(self.f107.values)) if len(self.f107) else 120.0
        flux = np.full(len(idx), 1e-7 * (f107 / 100.0) ** 2)
        days = (self.end - self.start).total_seconds() / 86400
        for _ in range(self.rng.poisson(max(0.3, 3.0 * f107 / 150.0) * days)):
            peak = 1e-6 * (self.rng.pareto(0.8) + 1.0)  # от C1, степенной закон
            t0 = self.rng.integers(0, len(idx))
            dur = int(self.rng.integers(10, 60))
            tail = peak * np.exp(-np.arange(min(dur * 3, len(idx) - t0)) / dur)
            flux[t0:t0 + len(tail)] = np.maximum(flux[t0:t0 + len(tail)], tail)
        series = pd.Series(np.minimum(flux, 2e-3), index=idx)
        try:
            real = space_weather.xray_flux_recent().set_index("time")["flux"]
            common = series.index.intersection(real.index)
            if len(common):
                series.loc[common] = real.loc[common].to_numpy()
        except Exception:
            pass
        return series

    @staticmethod
    def _interp(series: pd.Series, times: pd.DatetimeIndex) -> np.ndarray:
        s = series.dropna()
        if len(s) == 0:
            return np.zeros(len(times))
        return np.interp(unix_seconds(times), unix_seconds(s.index), s.to_numpy(dtype=float))

    def at(self, station_id: str, times: pd.DatetimeIndex) -> dict[str, np.ndarray]:
        w = self.weather[station_id]
        g = lambda c: np.clip(self._interp(w[c], times) / 100.0, 0, 1) if c in w else np.zeros(len(times))
        low, mid, high = g("cloud_cover_low"), g("cloud_cover_mid"), g("cloud_cover_high")
        return {
            "precip": np.maximum(self._interp(w["precipitation"], times), 0.0),
            "cloud": g("cloud_cover"),
            # для лазера важны плотные облака: нижний ярус целиком, средний почти, перистые — слабо
            "cloud_opt": 1 - (1 - low) * (1 - 0.8 * mid) * (1 - 0.15 * high),
            "rh": self._interp(w["relative_humidity_2m"], times),
            "kp": self._interp(self.kp, times),
            "f107": self._interp(self.f107, times),
            "xray": self._interp(self.xray, times),
        }


def ar1(n: int, rho: float, rng: np.random.Generator) -> np.ndarray:
    """Гауссов AR(1) с единичной дисперсией: плавные во времени случайные процессы."""
    x = np.empty(n)
    x[0] = rng.normal()
    s = np.sqrt(1 - rho ** 2)
    eps = rng.normal(size=n)
    for i in range(1, n):
        x[i] = rho * x[i - 1] + s * eps[i]
    return x


def channel_margins(station: dict, rng_km, el, sun_sep, v: dict, rng: np.random.Generator | None,
                    stochastic: bool = True) -> dict[str, np.ndarray]:
    """Запасы всех каналов станции (дБ). Стохастика — коррелированная во времени (AR(1))."""
    n = len(el)
    if stochastic and rng is not None:
        rain = v["precip"] * np.exp(0.9 * ar1(n, 0.95, rng) - 0.4) * (ar1(n, 0.97, rng) > -1.0)
        scint = np.exp(0.6 * ar1(n, 0.9, rng))
        blocked = norm.cdf(ar1(n, 0.985, rng)) < v["cloud_opt"]
        cloud_for_opt = blocked.astype(float)
    else:
        rain, scint, cloud_for_opt = v["precip"], 1.0, v["cloud_opt"]
    lwc = v["cloud"] * 0.25 + 0.08 * rain
    out = {}
    if station["kind"] == "rf":
        for band in station["bands"]:
            out[band] = physics.rf_margin_db(
                band, rng_km, el, station["dish_m"], station["lat"], station["alt_m"] / 1000, rain, lwc,
                v["rh"], v["kp"], station["geomag_lat"], sun_sep, float(np.mean(v["f107"])),
                float(np.max(v["xray"])), None, scint,
            )
    else:
        out["OPT"] = physics.optical_ground_margin_db(rng_km, el, cloud_for_opt, station["aperture_m"], sun_sep, None,
                                                      rain_mm_h=rain, lat=station["lat"], alt_km=station["alt_m"] / 1000)
    return out


def _window_rows(start: pd.Timestamp, end: pd.Timestamp, tles: dict, step_s: int, horizon_s: int, min_el: float,
                 rng: np.random.Generator, seed: int) -> list[pd.DataFrame]:
    env = Environment(start, end, "archive", seed)
    times = pd.date_range(start, end, freq=f"{step_s}s", tz="UTC")
    sun = sun_ecef_unit(times)
    h = horizon_s // step_s
    rows = []
    for sat in OBSERVERS:
        if sat not in tles:
            continue
        ecef = Orbit(sat, *tles[sat]).ecef(times)
        for st in STATIONS:
            _, el, rng_km, los = look_angles(ecef, st["lat"], st["lon"], st["alt_m"] / 1000)
            up = np.nan_to_num(el, nan=-90) >= min_el
            if not up.any():
                continue
            vals = env.at(st["id"], times)
            sun_sep = np.degrees(np.arccos(np.clip((los * sun).sum(1), -1, 1)))
            edges = np.diff(np.concatenate([[0], up.astype(int), [0]]))
            for a, b in zip(np.flatnonzero(edges == 1), np.flatnonzero(edges == -1)):
                if b - a < 4:
                    continue
                sl = slice(a, b)
                v = {k: x[sl] for k, x in vals.items()}
                margins = channel_margins(st, rng_km[sl], el[sl], sun_sep[sl], v, rng)
                el_rate = np.gradient(el[sl]) * (60.0 / step_s)
                t_since = np.arange(b - a) * step_s / 60.0
                t_to = (b - a - 1 - np.arange(b - a)) * step_s / 60.0
                # прогноз погоды, который наземный ИИ №3 закладывает на борт перед пролётом (с ошибкой)
                fc_precip = np.maximum(v["precip"] * np.exp(rng.normal(0, 0.5)) + rng.normal(0, 0.1), 0)
                fc_cloud = np.clip(v["cloud_opt"] + rng.normal(0, 0.12), 0, 1)
                pass_id = f"{sat}|{st['id']}|{times[a]:%Y%m%d%H%M}"
                for ch, m in margins.items():
                    meas = m + rng.normal(0, 0.5, len(m))  # измеренный запас по пилот-тону (с шумом)
                    trend = np.concatenate([[0, 0], meas[2:] - meas[:-2]])
                    fut = np.array([m[i + 1:i + 1 + h].min() if i + 1 < len(m) else m[i] for i in range(len(m))])
                    valid = t_to >= horizon_s / 60.0  # плановый уход за горизонт — не «обрыв»
                    rows.append(pd.DataFrame({
                        "time": times[sl], "pass_id": pass_id, "satellite": sat, "station": st["id"], "channel": ch,
                        "el": el[sl], "el_rate": el_rate, "range_km": rng_km[sl], "t_since_aos": t_since,
                        "t_to_los": t_to, "max_el_pass": float(el[sl].max()), "sun_sep": sun_sep[sl],
                        "geomag_abs": abs(st["geomag_lat"]), "alt_km": st["alt_m"] / 1000, "fc_precip": fc_precip,
                        "fc_cloud": fc_cloud, "rh": v["rh"], "kp": v["kp"],
                        "log_xray": np.log10(np.maximum(v["xray"], 1e-9)), "f107": v["f107"],
                        "margin_now": meas, "margin_trend": trend, "margin_true": m,
                        "outage_next": (fut < 0).astype(np.int8),
                    })[valid])
    return rows


def simulate_dataset(days: int = 10, step_s: int = 30, end: pd.Timestamp | None = None, seed: int = 0,
                     horizon_s: int = 300, min_el: float = 5.0, include_storms: bool = True) -> pd.DataFrame:
    """Строки «спутник × станция × канал × момент» с признаками и меткой outage_next (5 мин)."""
    rng = np.random.default_rng(seed)
    end = end or pd.Timestamp(datetime.now(timezone.utc) - timedelta(days=6)).floor("h")
    windows = [(end - pd.Timedelta(days=days), end)]
    if include_storms:
        windows += [(pd.Timestamp(a, tz="UTC"), pd.Timestamp(b, tz="UTC")) for a, b in STORM_WINDOWS]
    tles = celestrak.fetch_all(roles=("observer",))
    rows = []
    for i, (a, b) in enumerate(windows):
        log.info("link sim window %s .. %s", a, b)
        rows += _window_rows(a, b, tles, step_s, horizon_s, min_el, rng, seed + i)
    data = pd.concat(rows, ignore_index=True)
    for ch in CHANNELS:
        data[f"ch_{ch}"] = (data["channel"] == ch).astype(np.float32)
    log.info("link dataset: %d rows, outage rate %.3f (%s), max Kp %.1f", len(data), data["outage_next"].mean(),
             data.groupby("channel")["outage_next"].mean().round(3).to_dict(), data["kp"].max())
    return data

In [ ]:
%%writefile helios/comms/scheduler.py
"""Predictive Connection Scheduler: куда и когда передавать данные в ближайшие часы.

Для каждого спутника: пролёты над станциями (SGP4), прогноз погоды над антеннами
(Open-Meteo), космическая погода сейчас (SWPC/GFZ) → физический запас каждого канала →
ИИ №2 оценивает риск обрыва → план: лучший канал на каждый пролёт, объём данных и
«окна без связи» (когда ни станция, ни ретранслятор недоступны) — их знаем заранее.
"""

from __future__ import annotations

import numpy as np
import pandas as pd

from ..apis import celestrak
from ..utils import get_logger
from . import physics
from .link_sim import OBSERVERS, Environment, channel_margins
from .orbits import Orbit, line_of_sight, look_angles, sun_ecef_unit
from .stations import STATIONS, optical_stations

log = get_logger("helios.scheduler")

# Условный ретранслятор на ГСО над Индийским океаном (как LCRD, но на 76° в.д.) — сценарий «что если»
HYPOTHETICAL_RELAY_LON = 76.0


def geo_ecef(lon_deg: float, n: int) -> np.ndarray:
    r = 42164.0
    return np.tile([r * np.cos(np.radians(lon_deg)), r * np.sin(np.radians(lon_deg)), 0.0], (n, 1))


def _ai2_probs(df: pd.DataFrame) -> np.ndarray:
    try:
        from ..models.link_ai import predict

        return predict(df)
    except Exception as exc:  # модель ещё не обучена — физика без ИИ
        log.info("AI#2 недоступен (%s): риск по запасу линии", exc)
        return 1 / (1 + np.exp(df["margin_now"].to_numpy() - 1.0))


def plan(hours: int = 24, start: pd.Timestamp | None = None, step_s: int = 30,
         include_hypothetical_relay: bool = True) -> dict:
    start = (start or pd.Timestamp.now(tz="UTC")).floor("min")
    end = start + pd.Timedelta(hours=hours)
    times = pd.date_range(start, end, freq=f"{step_s}s")
    env = Environment(start, end, mode="forecast")
    tles = celestrak.fetch_all()
    sun = sun_ecef_unit(times)
    pass_rows, gaps = [], []
    relays = {name: Orbit(name, *tles[name]).ecef(times) for name in ("STPSAT-6 (LCRD)", "TDRS 13") if name in tles}
    if include_hypothetical_relay:
        relays["GEO-76E (условный)"] = geo_ecef(HYPOTHETICAL_RELAY_LON, len(times))
    ogs_clear = np.max([1 - env.at(s["id"], times)["cloud_opt"] for s in optical_stations()], axis=0)
    for sat in OBSERVERS:
        if sat not in tles:
            continue
        ecef = Orbit(sat, *tles[sat]).ecef(times)
        any_link = np.zeros(len(times), dtype=bool)
        for st in STATIONS:
            _, el, rng_km, los = look_angles(ecef, st["lat"], st["lon"], st["alt_m"] / 1000)
            up = np.nan_to_num(el, nan=-90) >= 5.0
            if not up.any():
                continue
            vals = env.at(st["id"], times)
            sun_sep = np.degrees(np.arccos(np.clip((los * sun).sum(1), -1, 1)))
            edges = np.diff(np.concatenate([[0], up.astype(int), [0]]))
            for a, b in zip(np.flatnonzero(edges == 1), np.flatnonzero(edges == -1)):
                sl = slice(a, b)
                v = {k: x[sl] for k, x in vals.items()}
                margins = channel_margins(st, rng_km[sl], el[sl], sun_sep[sl], v, None, stochastic=False)
                n = b - a
                best, best_score = None, -1.0
                row = {"satellite": sat, "station": st["id"], "station_name": st["name"], "aos": times[a],
                       "los": times[b - 1], "max_el": round(float(el[sl].max()), 1), "duration_min": round(n * step_s / 60, 1)}
                for ch, m in margins.items():
                    feat = pd.DataFrame({
                        "el": el[sl], "el_rate": np.gradient(el[sl]) * (60.0 / step_s), "range_km": rng_km[sl],
                        "t_since_aos": np.arange(n) * step_s / 60, "t_to_los": (n - 1 - np.arange(n)) * step_s / 60,
                        "max_el_pass": float(el[sl].max()), "sun_sep": sun_sep[sl], "geomag_abs": abs(st["geomag_lat"]),
                        "alt_km": st["alt_m"] / 1000, "fc_precip": v["precip"], "fc_cloud": v["cloud_opt"], "rh": v["rh"],
                        "kp": v["kp"], "log_xray": np.log10(np.maximum(v["xray"], 1e-9)), "f107": v["f107"],
                        "margin_now": m, "margin_trend": np.concatenate([[0, 0], m[2:] - m[:-2]]),
                        **{f"ch_{c}": float(c == ch) for c in ("S", "X", "Ka", "OPT")},
                    })
                    p_out = _ai2_probs(feat)
                    avail = (m > 0) * (1 - p_out)
                    rate = physics.OPTICAL["rate_bps"] if ch == "OPT" else physics.RF_CHANNELS[ch]["rate_bps"]
                    gb = float((avail * rate * step_s).sum() / 8e9)
                    row[f"{ch}_availability"] = round(float(avail.mean()), 3)
                    row[f"{ch}_volume_gb"] = round(gb, 2)
                    alert_ok = (m + physics.alert_gain_db(ch)) > 0
                    any_link[sl] |= alert_ok
                    if gb > best_score:
                        best, best_score = ch, gb
                row["recommended_channel"] = best
                row["expected_volume_gb"] = round(best_score, 2)
                pass_rows.append(row)
        relay_cov, real_ok, all_ok = {}, np.zeros(len(times), bool), np.zeros(len(times), bool)
        for rname, recef in relays.items():
            vis = line_of_sight(ecef, recef)
            # лазерный ретранслятор полезен, только если на Земле хотя бы одна оптическая станция без облаков
            ok = vis & (ogs_clear > 0.5) if "LCRD" in rname or "условный" in rname else vis
            relay_cov[rname] = round(float(ok.mean()), 3)
            all_ok |= ok
            if "условный" not in rname:
                real_ok |= ok

        def _gaps(link: np.ndarray) -> tuple[float, list]:
            down = ~link
            e = np.diff(np.concatenate([[0], down.astype(int), [0]]))
            segs = [(times[a], times[b - 1], (b - a) * step_s / 60) for a, b in zip(np.flatnonzero(e == 1), np.flatnonzero(e == -1))]
            return round(float(down.sum() * step_s / 60), 1), segs

        ground_min, segs = _gaps(any_link)          # реалистично: только наземные станции
        real_min, real_segs = _gaps(any_link | real_ok)
        all_min, _ = _gaps(any_link | all_ok)
        gaps.append({
            "satellite": sat, "hours": hours,
            "minutes_without_any_link": ground_min,
            "longest_gap_min": round(max((s[2] for s in segs), default=0.0), 1),
            "next_gap_start": segs[0][0] if segs else None, "next_gap_end": segs[0][1] if segs else None,
            "minutes_without_link_with_real_relays": real_min,
            "minutes_without_link_with_relay_76E": all_min,
            "relay_coverage": relay_cov,
            "gaps": [{"start": str(a), "end": str(b), "minutes": round(m, 1)} for a, b, m in segs[:60]],
        })
    passes = pd.DataFrame(pass_rows).sort_values("aos").reset_index(drop=True) if pass_rows else pd.DataFrame()
    return {"start": start, "end": end, "passes": passes, "blackouts": gaps}

In [ ]:
%%writefile helios/comms/delivery.py
"""Smart Alert Delivery + Collaborative Satellite Alert Network (симуляция на реальном пожарном дне).

1. Берём реальный день 2025 г. с максимумом пожаров в регионе (архив FIRMS).
2. ИИ №1 на «борту» оценивает каждое обнаружение → пакеты тревоги по приоритетам.
3. Сравниваем, когда тревога окажется на Земле:
   A. Как сейчас (NRT FIRMS/LANCE): реальная медианная задержка гранул по NASA CMR.
   B. ИИ на борту + «сохранить и передать» на ближайшую станцию сети (короткий пакет).
   C. B + станции прямого приёма в Казахстане (условные Астана/Алматы).
   D. B + ретранслятор на ГСО (реальные LCRD/TDRS + условный узел 76° в.д.).
4. Сеть спутников: для КРИТИЧНЫХ тревог ищем, какой спутник следующим пролетит над очагом,
   и проверяем по реальным данным, подтвердил ли он пожар.

Геометрия связи считается по текущим TLE: для каждого пролёта находим ближайший «такой же»
пролёт того же спутника над тем же местом сейчас — так положение спутника над очагом в момент
съёмки физически верное, а погода у станций — сегодняшний прогноз.
"""

from __future__ import annotations

import numpy as np
import pandas as pd

from ..apis import celestrak
from ..apis.celestrak import STREAM_TO_SAT
from ..config import PROCESSED, REPORTS_DIR
from ..utils import get_logger, save_json
from . import physics
from .link_sim import Environment, channel_margins
from .orbits import HALF_SWATH_KM, Orbit, haversine_km, line_of_sight, look_angles, next_overpass, sun_ecef_unit
from .scheduler import HYPOTHETICAL_RELAY_LON, geo_ecef
from .stations import STATIONS, optical_stations

log = get_logger("helios.delivery")

NASA_NETWORK = {"SVALSAT", "TROLLSAT", "MCMURDO", "FAIRBANKS", "WALLOPS", "WHITE_SANDS"}
KZ_DB = {"ASTANA_DB", "ALMATY_DB", "OGS_ASSY_TURGEN"}  # условные станции в Казахстане
REAL_RELAYS = ("STPSAT-6 (LCRD)", "TDRS 13")
DEFAULT_NRT_LATENCY_MIN = {"MODIS_T": 150.0, "MODIS_A": 150.0, "VIIRS_SNPP": 150.0, "VIIRS_NOAA20": 120.0, "VIIRS_NOAA21": 120.0}


def replay_day(det: pd.DataFrame, year: int = 2025) -> pd.Timestamp:
    k = det[det["keep_veg"] & (det["acq_dt"].dt.year == year)]
    day = k["acq_dt"].dt.floor("D").value_counts().idxmax()
    return day


def measured_nrt_latency() -> dict:
    try:
        from ..apis.earthdata import granule_latency

        lat = granule_latency(hours=48)
        if len(lat):
            return lat.groupby("stream")["latency_min"].median().round(1).to_dict()
    except Exception as exc:
        log.info("CMR latency: %s", exc)
    return {}


def _first_contact(sat_ecef: np.ndarray, times: pd.DatetimeIndex, stations: list[dict], env: Environment, sun: np.ndarray):
    """Первый момент (индекс) и станция, когда короткий пакет проходит (запас в режиме тревоги > 0)."""
    best_i, best_st, best_ch = None, None, None
    for st in stations:
        _, el, rng_km, los = look_angles(sat_ecef, st["lat"], st["lon"], st["alt_m"] / 1000)
        up = np.flatnonzero(np.nan_to_num(el, nan=-90) >= 5.0)
        if len(up) == 0 or (best_i is not None and up[0] >= best_i):
            continue
        v = env.at(st["id"], times[up])
        sun_sep = np.degrees(np.arccos(np.clip((los[up] * sun[up]).sum(1), -1, 1)))
        margins = channel_margins(st, rng_km[up], el[up], sun_sep, v, None, stochastic=False)
        for ch, m in margins.items():
            ok = np.flatnonzero(m + physics.alert_gain_db(ch) > 0)
            if len(ok) and (best_i is None or up[ok[0]] < best_i):
                best_i, best_st, best_ch = int(up[ok[0]]), st["id"], ch
    return best_i, best_st, best_ch


def run_delivery_simulation(year: int = 2025, horizon_h: int = 6, step_s: int = 20) -> dict:
    from ..models import edge_detector

    det = pd.read_parquet(PROCESSED / "detections_filtered.parquet")
    day = replay_day(det, year)
    day_det = det[(det["acq_dt"] >= day) & (det["acq_dt"] < day + pd.Timedelta(days=1))].reset_index(drop=True)
    scored = edge_detector.score_detections(day_det)
    alerts = edge_detector.make_alerts(scored, min_priority="ROUTINE")
    log.info("replay day %s: %d detections -> %d alerts", day.date(), len(day_det), len(alerts))
    tles = celestrak.fetch_all()
    now = pd.Timestamp.now(tz="UTC").floor("min")
    env = Environment(now, now + pd.Timedelta(hours=36), mode="forecast")
    nrt_lat = {**DEFAULT_NRT_LATENCY_MIN, **measured_nrt_latency()}
    ogs = [s for s in optical_stations() if s["real"]]  # реальные OGS: Table Mountain, Haleakalā, Tenerife
    rows = []
    for (stream, overpass), grp in alerts.groupby(["stream", "overpass"], observed=True):
        sat = STREAM_TO_SAT.get(str(stream))
        if sat not in tles:
            continue
        orbit = Orbit(sat, *tles[sat])
        lat0, lon0 = float(grp["lat"].mean()), float(grp["lon"].mean())
        sensor = "MODIS" if str(stream).startswith("MODIS") else "VIIRS"
        # тот же спутник над тем же местом, в ближайшие сутки (с той же стороны суток: день/ночь)
        t0 = next_overpass(orbit, lat0, lon0, now, horizon_h=24, half_swath_km=HALF_SWATH_KM[sensor])
        if t0 is None:
            continue
        times = pd.date_range(t0, t0 + pd.Timedelta(hours=horizon_h), freq=f"{step_s}s")
        ecef = orbit.ecef(times)
        sun = sun_ecef_unit(times)
        nasa = [s for s in STATIONS if s["id"] in NASA_NETWORK]
        kz = [s for s in STATIONS if s["id"] in KZ_DB]
        iB, stB, chB = _first_contact(ecef, times, nasa + ogs, env, sun)
        iC, stC, chC = _first_contact(ecef, times, nasa + ogs + kz, env, sun)
        # ретрансляторы на ГСО: прямая видимость LEO→ГСО (+ для лазерного — ясное небо у какой-либо OGS)
        clear = np.max([1 - env.at(s["id"], times)["cloud_opt"] for s in ogs], axis=0)
        relays = {n: Orbit(n, *tles[n]).ecef(times) for n in REAL_RELAYS if n in tles}
        relays["GEO-76E (условный)"] = geo_ecef(HYPOTHETICAL_RELAY_LON, len(times))
        first_relay = {}
        for rname, recef in relays.items():
            vis = line_of_sight(ecef, recef)
            ok = vis & (clear > 0.5) if rname != "TDRS 13" else vis
            idx = np.flatnonzero(ok)
            if len(idx):
                first_relay[rname] = int(idx[0])
        lat_B = (iB * step_s / 60 if iB is not None else horizon_h * 60) + 1 / 60
        lat_C = (iC * step_s / 60 if iC is not None else horizon_h * 60) + 1 / 60
        real = {k: v for k, v in first_relay.items() if k in REAL_RELAYS}
        r1 = min(real.items(), key=lambda kv: kv[1]) if real else (None, None)
        r2 = min(first_relay.items(), key=lambda kv: kv[1]) if first_relay else (None, None)
        lat_D1 = min(lat_B, r1[1] * step_s / 60 + 0.5 / 60) if r1[1] is not None else lat_B
        lat_D2 = min(lat_B, r2[1] * step_s / 60 + 0.5 / 60) if r2[1] is not None else lat_B
        for _, a in grp.iterrows():
            rows.append({
                "alert_id": a["alert_id"], "priority": a["priority"], "stream": str(stream), "lat": a["lat"], "lon": a["lon"],
                "acq_dt": a["acq_dt"], "t0_equiv": t0, "frp_sum": a["frp_sum"], "cluster_n": a["cluster_n"],
                "p_real": a["p_real"], "p_escalation": a["p_escalation"], "packet_hex": a["packet_hex"],
                "latency_A_nrt_min": nrt_lat.get(str(stream), 150.0),
                "latency_B_storeforward_min": round(lat_B, 2), "station_B": stB, "channel_B": chB,
                "latency_C_kz_station_min": round(lat_C, 2), "station_C": stC, "channel_C": chC,
                "latency_D1_real_relay_min": round(lat_D1, 2), "relay_D1": r1[0] if lat_D1 < lat_B else stB,
                "latency_D2_relay_76E_min": round(lat_D2, 2), "relay_D2": r2[0] if lat_D2 < lat_B else stB,
            })
    out = pd.DataFrame(rows)
    if out.empty:
        return {"alerts": 0}
    window = det[(det["acq_dt"] >= day) & (det["acq_dt"] < day + pd.Timedelta(days=2))]
    out = collaborative_verification(out, window, tles, now)
    out.to_parquet(PROCESSED / "delivery_alerts.parquet", index=False)

    def stats(col: str, sub: pd.DataFrame) -> dict:
        s = sub[col]
        return {"median_min": round(float(s.median()), 2), "p90_min": round(float(s.quantile(0.9)), 2),
                "share_within_15min": round(float((s <= 15).mean()), 3), "share_within_60min": round(float((s <= 60).mean()), 3)}

    summary = {"replay_day": str(day.date()), "detections": int(len(day_det)), "alerts": int(len(out)),
               "alerts_by_priority": out["priority"].value_counts().to_dict(), "strategies": {}}
    for name, col in (("A_today_nrt", "latency_A_nrt_min"), ("B_onboard_ai_store_forward", "latency_B_storeforward_min"),
                      ("C_plus_kz_direct_stations", "latency_C_kz_station_min"),
                      ("D1_plus_real_geo_relays", "latency_D1_real_relay_min"),
                      ("D2_plus_geo_relay_76E", "latency_D2_relay_76E_min")):
        summary["strategies"][name] = {"all": stats(col, out),
                                       "critical": stats(col, out[out["priority"] == "CRITICAL"]) if (out["priority"] == "CRITICAL").any() else None}
    crit = out[out["priority"] == "CRITICAL"]
    summary["collaborative_network"] = {
        "critical_alerts": int(len(crit)),
        "confirmed_by_other_satellite_12h": round(float(crit["confirmed"].mean()), 3) if len(crit) else None,
        "median_minutes_to_next_look": round(float(crit["next_look_min"].median()), 1) if len(crit) else None,
        "median_minutes_to_confirmation": round(float(crit.loc[crit["confirmed"], "confirm_delay_min"].median()), 1)
        if len(crit) and crit["confirmed"].any() else None,
    }
    save_json(summary, REPORTS_DIR / "delivery_summary.json")
    log.info("delivery: %s", summary)
    return summary


def collaborative_verification(alerts: pd.DataFrame, det: pd.DataFrame, tles: dict, now: pd.Timestamp) -> pd.DataFrame:
    """Для каждой тревоги: какой спутник посмотрит следующим и подтвердили ли пожар реальные данные."""
    kept = det[det["keep_veg"]][["lat", "lon", "acq_dt", "stream"]]
    out = alerts.copy()
    out["confirmed"] = False
    out["confirm_stream"] = None
    out["confirm_delay_min"] = np.nan
    out["next_look_sat"] = None
    out["next_look_min"] = np.nan
    for i, a in out.iterrows():
        win = kept[(kept["acq_dt"] > a["acq_dt"] + pd.Timedelta(minutes=10)) &
                   (kept["acq_dt"] <= a["acq_dt"] + pd.Timedelta(hours=12)) & (kept["stream"].astype(str) != a["stream"])]
        if len(win):
            d = haversine_km(win["lat"].to_numpy(), win["lon"].to_numpy(), a["lat"], a["lon"])
            near = win[d <= 5.0]
            if len(near):
                first = near.sort_values("acq_dt").iloc[0]
                out.at[i, "confirmed"] = True
                out.at[i, "confirm_stream"] = str(first["stream"])
                out.at[i, "confirm_delay_min"] = round((first["acq_dt"] - a["acq_dt"]).total_seconds() / 60, 1)
        if a["priority"] == "CRITICAL":
            best = None
            t_ref = a["t0_equiv"] + pd.Timedelta(minutes=10)
            for stream, sat in STREAM_TO_SAT.items():
                if stream == a["stream"] or sat not in tles:
                    continue
                t = next_overpass(Orbit(sat, *tles[sat]), a["lat"], a["lon"], t_ref, horizon_h=12,
                                  half_swath_km=HALF_SWATH_KM["MODIS" if stream.startswith("MODIS") else "VIIRS"], step_s=60)
                if t is not None and (best is None or t < best[1]):
                    best = (sat, t)
            if best:
                out.at[i, "next_look_sat"] = best[0]
                out.at[i, "next_look_min"] = round((best[1] - a["t0_equiv"]).total_seconds() / 60, 1)
    return out

### Модели ИИ

In [ ]:
%%writefile helios/models/__init__.py
"""ИИ-модели HELIOS-SHIELD: №1 edge_detector (борт), №2 link_ai (борт), №3 ground_forecast (Земля)."""

In [ ]:
%%writefile helios/models/features.py
"""Панель признаков «ячейка H3 res-5 × неделя» для ИИ №3 (прогноз риска) и Time Machine.

Все признаки недели t используют ТОЛЬКО данные до конца недели t (без заглядывания вперёд):
* активность: cell-days за t, t-1, t-2, t-3, суммы за 4 и 8 недель, FRP;
* соседи: активность в кольцах k=1 и k=2 (распространение, общая «пожарная погода»);
* климатология: доля ПРОШЛЫХ лет, когда в эту неделю года (±1) ячейка горела;
* давность: недель с последнего пожара;
* погода NASA POWER (родительская ячейка res-3): t° макс, влажность, ветер, осадки 7/28 сут,
  влажность почвы GWETTOP, облачность + аномалии к климатологии;
* сезон (sin/cos недели года), координаты, число работающих VIIRS (усилие наблюдения).
"""

from __future__ import annotations

from functools import cached_property

import h3
import numpy as np
import pandas as pd
import scipy.sparse as sp

from ..config import PROCESSED, settings
from ..utils import get_logger

log = get_logger("helios.features")

FEATURES = [
    "a_t", "a_t1", "a_t2", "a_t3", "sum4", "sum8", "frp4_log", "nb1_2w", "nb2_2w", "nb1_8w",
    "weeks_since_fire", "clim_woy", "clim_woy_next", "years_burned_share",
    "t2m_max", "rh2m", "ws10m_max", "precip_7d", "precip_28d", "gwet", "cloud",
    "t2m_anom", "precip28_anom", "gwet_anom",
    "woy_sin", "woy_cos", "lat", "lon", "n_viirs",
]

VIIRS_START = {"VIIRS_SNPP": "2012-01-19", "VIIRS_NOAA20": "2018-01-05", "VIIRS_NOAA21": "2023-03-01"}
VIIRS_END = {"VIIRS_SNPP": "2026-11-02"}


class Panel:
    """Матрицы неделя × ячейка + быстрая сборка строк признаков."""

    def __init__(self, cell_weekly: pd.DataFrame | None = None, weather: pd.DataFrame | None = None):
        cw = cell_weekly if cell_weekly is not None else pd.read_parquet(PROCESSED / "cell_weekly_r5.parquet")
        self.weather_daily = weather if weather is not None else (
            pd.read_parquet(PROCESSED / "weather_daily_r3.parquet")
            if (PROCESSED / "weather_daily_r3.parquet").exists() else None
        )
        cw = cw.copy()
        cw["week"] = pd.to_datetime(cw["week"]).dt.normalize()
        self.cells = np.array(sorted(cw["h3_r5"].unique()))
        start = pd.Timestamp("2001-01-01")  # понедельник
        end = max(cw["week"].max(), pd.Timestamp(f"{settings()['history']['end_year']}-12-29"))
        self.weeks = pd.date_range(start, end, freq="W-MON")
        self.cidx = {c: i for i, c in enumerate(self.cells)}
        W, C = len(self.weeks), len(self.cells)
        widx = pd.Index(self.weeks)
        cw = cw[cw["week"] >= start]
        wi = widx.get_indexer(cw["week"])
        ok = wi >= 0
        ci = cw["h3_r5"].map(self.cidx).to_numpy()
        self.A = np.zeros((W, C), dtype=np.float32)
        self.F = np.zeros((W, C), dtype=np.float32)
        np.add.at(self.A, (wi[ok], ci[ok]), cw["union_celldays"].to_numpy(np.float32)[ok])
        np.add.at(self.F, (wi[ok], ci[ok]), cw["frp_sum"].to_numpy(np.float32)[ok])
        self.B = self.A > 0
        latlon = np.array([h3.cell_to_latlng(c) for c in self.cells], dtype=np.float32)
        self.lat, self.lon = latlon[:, 0], latlon[:, 1]
        log.info("panel: %d weeks x %d cells, positives %.2f%%", W, C, 100 * self.B.mean())

    # ------------------------------------------------------------------ вспомогательные матрицы
    def _adjacency(self, k: int) -> sp.csr_matrix:
        rows, cols = [], []
        for i, c in enumerate(self.cells):
            for nb in h3.grid_ring(c, k) if k > 0 else []:
                j = self.cidx.get(nb)
                if j is not None:
                    rows.append(i)
                    cols.append(j)
        C = len(self.cells)
        return sp.csr_matrix((np.ones(len(rows), np.float32), (rows, cols)), shape=(C, C))

    @cached_property
    def nb1(self) -> sp.csr_matrix:
        return self._adjacency(1)

    @cached_property
    def nb2(self) -> sp.csr_matrix:
        return self._adjacency(2)

    @cached_property
    def weeks_since(self) -> np.ndarray:
        W, C = self.A.shape
        out = np.zeros((W, C), dtype=np.float32)
        last = np.full(C, -10_000, dtype=np.int64)
        for w in range(W):
            last = np.where(self.B[w], w, last)
            out[w] = np.minimum(w - last, 520)
        return out

    @cached_property
    def iso(self) -> tuple[np.ndarray, np.ndarray]:
        cal = pd.DatetimeIndex(self.weeks).isocalendar()
        return cal["year"].to_numpy(int), np.minimum(cal["week"].to_numpy(int), 52)

    @cached_property
    def clim(self) -> dict[int, np.ndarray]:
        """clim[Y][woy-1, cell] — доля лет ДО Y с пожаром в неделю woy (±1)."""
        years, woys = self.iso
        uniq = np.unique(years)
        P = np.zeros((len(uniq), 52, len(self.cells)), dtype=np.float32)
        yi = {y: i for i, y in enumerate(uniq)}
        for w in range(len(self.weeks)):
            P[yi[years[w]], woys[w] - 1] = np.maximum(P[yi[years[w]], woys[w] - 1], self.B[w])
        P = np.maximum.reduce([P, np.roll(P, 1, axis=1), np.roll(P, -1, axis=1)])  # окно ±1 неделя
        out, cum, n = {}, np.zeros((52, len(self.cells)), np.float32), 0
        for y in uniq:
            out[int(y)] = cum / max(n, 1)
            cum = cum + P[yi[y]]
            n += 1
        out[int(uniq[-1]) + 1] = cum / max(n, 1)
        return out

    @cached_property
    def years_burned(self) -> dict[int, np.ndarray]:
        years, _ = self.iso
        uniq = np.unique(years)
        out, cum, n = {}, np.zeros(len(self.cells), np.float32), 0
        for y in uniq:
            out[int(y)] = cum / max(n, 1)
            cum = cum + self.B[years == y].any(axis=0)
            n += 1
        out[int(uniq[-1]) + 1] = cum / max(n, 1)
        return out

    @cached_property
    def weather_weekly(self) -> dict[str, np.ndarray] | None:
        """Погода по неделям для каждой res-5 ячейки (берём родительскую res-3)."""
        if self.weather_daily is None or self.weather_daily.empty:
            return None
        res_w = settings()["h3"]["res_weather"]
        wd = self.weather_daily.copy()
        wd["date"] = pd.to_datetime(wd["date"])
        wd = wd.sort_values(["cell", "date"])
        g = wd.groupby("cell")
        wd["precip_7d"] = g["PRECTOTCORR"].transform(lambda s: s.rolling(7, min_periods=1).sum())
        wd["precip_28d"] = g["PRECTOTCORR"].transform(lambda s: s.rolling(28, min_periods=1).sum())
        wd["week"] = wd["date"] - pd.to_timedelta(wd["date"].dt.weekday, unit="D")
        # значение на конец недели (воскресенье) + средние за неделю
        agg = wd.groupby(["cell", "week"]).agg(
            t2m_max=("T2M_MAX", "mean"), rh2m=("RH2M", "mean"), ws10m_max=("WS10M", "max"),
            precip_7d=("precip_7d", "last"), precip_28d=("precip_28d", "last"),
            gwet=("GWETTOP", "mean"), cloud=("CLOUD_AMT", "mean"),
        ).reset_index()
        agg["woy"] = np.minimum(agg["week"].dt.isocalendar().week.astype(int), 52)
        for col, name in (("t2m_max", "t2m_anom"), ("precip_28d", "precip28_anom"), ("gwet", "gwet_anom")):
            agg[name] = agg[col] - agg.groupby(["cell", "woy"])[col].transform("mean")
        wcells = sorted(agg["cell"].unique())
        wci = {c: i for i, c in enumerate(wcells)}
        parent = np.array([wci.get(h3.cell_to_parent(c, res_w), -1) for c in self.cells])
        # ячейки без своей res-3 точки (на границе) — ближайшая точка погоды
        if (parent < 0).any():
            wl = np.array([h3.cell_to_latlng(c) for c in wcells])
            for i in np.flatnonzero(parent < 0):
                parent[i] = int(np.argmin((wl[:, 0] - self.lat[i]) ** 2 + (wl[:, 1] - self.lon[i]) ** 2))
        widx = pd.Index(self.weeks)
        out = {}
        cols = ["t2m_max", "rh2m", "ws10m_max", "precip_7d", "precip_28d", "gwet", "cloud",
                "t2m_anom", "precip28_anom", "gwet_anom"]
        wi = widx.get_indexer(agg["week"])
        ok = wi >= 0
        ci = agg["cell"].map(wci).to_numpy()
        for col in cols:
            M = np.full((len(self.weeks), len(wcells)), np.nan, dtype=np.float32)
            M[wi[ok], ci[ok]] = agg[col].to_numpy(np.float32)[ok]
            out[col] = M[:, parent]
        return out

    @cached_property
    def n_viirs(self) -> np.ndarray:
        """Сколько VIIRS реально есть в ДАННЫХ этой недели (по coverage), а не по датам запуска."""
        w = pd.DatetimeIndex(self.weeks)
        n = np.zeros(len(w), dtype=np.float32)
        for s, start in VIIRS_START.items():  # запасной вариант для будущих недель (NRT)
            end = pd.Timestamp(VIIRS_END.get(s, "2100-01-01"))
            n += ((w >= pd.Timestamp(start)) & (w < end)).astype(np.float32)
        path = PROCESSED / "stream_coverage.parquet"
        if path.exists():
            cov = pd.read_parquet(path)
            cov = cov[cov["stream"].astype(str).str.startswith("VIIRS") & (cov["coverage"] >= 0.5)]
            per_month = cov.groupby(cov["month"].astype(str)).size()
            months = w.to_period("M").astype(str)
            known = months.isin(per_month.index)
            n = np.where(known, per_month.reindex(months).fillna(0).to_numpy(np.float32), n)
        return n.astype(np.float32)

    # ------------------------------------------------------------------ строки признаков
    def rows(self, week_idx: np.ndarray, horizon: int = 1, neg_rate: float | None = None,
             seed: int = 0, allow_future: bool = False) -> tuple[pd.DataFrame, np.ndarray, np.ndarray]:
        """Признаки для недель week_idx (все ячейки). Возвращает (X, y, weight).
        allow_future=True — для прогноза вперёд: целевая неделя ещё не наступила (y = 0)."""
        A, F = self.A, self.F
        years, woys = self.iso
        C = len(self.cells)
        rng = np.random.default_rng(seed)
        frames, ys, ws = [], [], []
        ww = self.weather_weekly
        for w in week_idx:
            future = w + horizon >= len(self.weeks)
            if w < 8 or (future and not allow_future):
                continue
            y = np.zeros(C, dtype=np.int8) if future else self.B[w + horizon].astype(np.int8)
            keep = np.ones(C, dtype=bool)
            weight = np.ones(C, dtype=np.float32)
            if neg_rate is not None:
                keep = (y == 1) | (rng.random(C) < neg_rate)
                weight = np.where(y == 1, 1.0, 1.0 / neg_rate).astype(np.float32)
            two = A[w] + A[w - 1]
            yr = int(years[w])
            woy_next = min(int((pd.Timestamp(self.weeks[w]) + pd.Timedelta(weeks=horizon)).isocalendar().week), 52)
            yr = min(yr, max(self.clim))
            d = {
                "a_t": A[w], "a_t1": A[w - 1], "a_t2": A[w - 2], "a_t3": A[w - 3],
                "sum4": A[w - 3:w + 1].sum(0), "sum8": A[w - 7:w + 1].sum(0),
                "frp4_log": np.log1p(F[w - 3:w + 1].sum(0)),
                "nb1_2w": self.nb1 @ two, "nb2_2w": self.nb2 @ two,
                "nb1_8w": self.nb1 @ A[w - 7:w + 1].sum(0),
                "weeks_since_fire": self.weeks_since[w],
                "clim_woy": self.clim[yr][int(woys[w]) - 1],
                "clim_woy_next": self.clim[yr][woy_next - 1],
                "years_burned_share": self.years_burned[yr],
                "woy_sin": np.full(C, np.sin(2 * np.pi * woy_next / 52), np.float32),
                "woy_cos": np.full(C, np.cos(2 * np.pi * woy_next / 52), np.float32),
                "lat": self.lat, "lon": self.lon,
                "n_viirs": np.full(C, self.n_viirs[w], np.float32),
            }
            for col in ("t2m_max", "rh2m", "ws10m_max", "precip_7d", "precip_28d", "gwet", "cloud",
                        "t2m_anom", "precip28_anom", "gwet_anom"):
                d[col] = ww[col][w] if ww is not None else np.full(C, np.nan, np.float32)
            df = pd.DataFrame({k: np.asarray(v, dtype=np.float32)[keep] for k, v in d.items()})[FEATURES]
            df["week_idx"] = w
            df["cell_idx"] = np.flatnonzero(keep)
            frames.append(df)
            ys.append(y[keep])
            ws.append(weight[keep])
        X = pd.concat(frames, ignore_index=True)
        return X, np.concatenate(ys), np.concatenate(ws)

    def week_index(self, start: str, end: str) -> np.ndarray:
        w = pd.DatetimeIndex(self.weeks)
        return np.flatnonzero((w >= pd.Timestamp(start)) & (w <= pd.Timestamp(end)))

In [ ]:
%%writefile helios/models/edge_detector.py
"""ИИ №1 (на борту спутника): «это настоящий пожар и станет ли он большим?»

Крошечная нейросеть (MLP ~1.6 тыс. параметров, int8 ≈ несколько КБ) работает без интернета
на слабом бортовом процессоре. Вход — только то, что есть на борту в момент съёмки:
* свойства пикселя: яркостные температуры 4/11 мкм, ΔT, FRP, площадь пикселя, угол скана,
  день/ночь по Солнцу, угол блика, уверенность продукта;
* контекст пролёта: сколько горячих пикселей рядом в той же сцене и их суммарная FRP;
* бортовая таблица истории (26 лет сжаты в ~сотни КБ): как часто эта ячейка H3 горит в этот
  месяц, известный ли это факел/завод, сколько лет здесь бывают пожары.

Два выхода:
  p_real        — это природный/растительный пожар (а не факел, блик или шум);
  p_escalation  — пожар разрастётся: ≥6 «сгоревших гексагонов» в радиусе 10 км за 72 ч.
Метки берутся из будущего (что реально произошло дальше) — модель учится предсказывать
развитие пожара по первому снимку. История для бортовой таблицы — только до 2018 г.,
обучение — 2019–2023, проверка — 2024, честный тест — 2025.
"""

from __future__ import annotations

import json
import time
from functools import lru_cache

import h3
import numpy as np
import pandas as pd
from scipy.spatial import cKDTree
from sklearn.metrics import average_precision_score, roc_auc_score

from ..config import MODELS_DIR, PROCESSED, REPORTS_DIR, settings
from ..utils import get_logger, save_json

log = get_logger("helios.ai1")

FEATURES = [
    "is_viirs", "is_night", "bright_mir", "bright_tir", "dt45", "log_frp", "log_pixel_km2", "scan_deg",
    "p_conf", "glint_n", "ctx_log_n", "ctx_log_frp", "hist_clim_month", "hist_static", "hist_static_score",
    "hist_years_r7", "month_sin", "month_cos", "lat_n",
]
ONNX_PATH = MODELS_DIR / "edge_detector.onnx"
ONNX_INT8_PATH = MODELS_DIR / "edge_detector_int8.onnx"
JSON_PATH = MODELS_DIR / "edge_detector_weights.json"
TABLE_PATH = MODELS_DIR / "onboard_history.npz"
META_PATH = MODELS_DIR / "edge_detector_meta.json"


# ----------------------------------------------------------------------------- бортовая таблица
def build_onboard_table(det: pd.DataFrame, until_year: int) -> dict:
    """Сжатая история до until_year: климатология res-5 по месяцам, годы res-7, факелы res-8."""
    from ..processing.persistence import build_static_registry

    hist = det[det["acq_dt"].dt.year <= until_year]
    kept = hist[hist["keep_veg"]]
    years = kept["acq_dt"].dt.year
    n_years = max(1, int(years.nunique()))
    m = kept.assign(month=kept["acq_dt"].dt.month, year=years)[["h3_r5", "month", "year"]].drop_duplicates()
    clim = m.groupby(["h3_r5", "month"]).size().div(n_years).clip(upper=1.0)
    r5_cells = np.array(sorted(clim.index.get_level_values(0).unique()))
    clim_mat = np.zeros((len(r5_cells), 12), dtype=np.uint8)
    pos = {c: i for i, c in enumerate(r5_cells)}
    for (c, mo), v in clim.items():
        clim_mat[pos[c], mo - 1] = int(round(v * 255))
    yr7 = kept.assign(year=years)[["h3_r7", "year"]].drop_duplicates().groupby("h3_r7").size()
    reg = build_static_registry(hist)
    static = reg[reg["is_static"]]
    static_cells, static_score = [], []
    for c, s in zip(static["h3_r8"], static["static_score"]):
        for nb in h3.grid_disk(c, 1):
            static_cells.append(nb)
            static_score.append(s)
    st = pd.DataFrame({"c": static_cells, "s": static_score}).groupby("c")["s"].max()
    table = {
        "until_year": until_year,
        "n_years": n_years,
        "r5_cells": np.array([h3.str_to_int(c) for c in r5_cells], dtype=np.uint64),
        "r5_clim_u8": clim_mat,
        "r7_cells": np.array([h3.str_to_int(c) for c in yr7.index], dtype=np.uint64),
        "r7_years_u8": np.minimum(yr7.to_numpy(), 255).astype(np.uint8),
        "r8_static_cells": np.array([h3.str_to_int(c) for c in st.index], dtype=np.uint64),
        "r8_static_score_u8": (st.to_numpy() * 255).round().astype(np.uint8),
    }
    return table


def save_table(table: dict, path=TABLE_PATH) -> int:
    path.parent.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(path, **{k: (np.asarray(v) if not isinstance(v, int) else np.array([v])) for k, v in table.items()})
    return path.stat().st_size


@lru_cache(maxsize=1)
def load_table(path_str: str = str(TABLE_PATH)) -> dict:
    z = np.load(path_str)
    return {k: (int(z[k][0]) if z[k].shape == (1,) else z[k]) for k in z.files}


def _lookup(keys: np.ndarray, table_keys: np.ndarray, values: np.ndarray, default=0):
    order = np.argsort(table_keys)
    tk, tv = table_keys[order], values[order]
    idx = np.searchsorted(tk, keys)
    idx = np.clip(idx, 0, len(tk) - 1)
    hit = tk[idx] == keys if len(tk) else np.zeros(len(keys), bool)
    out = np.where(hit[..., None] if tv.ndim > 1 else hit, tv[idx], default)
    return out, hit


# ----------------------------------------------------------------------------- признаки
def overpass_context(df: pd.DataFrame, radius_km: float = 5.0, window_min: float = 5.0) -> tuple[np.ndarray, np.ndarray]:
    """Сколько горячих пикселей в той же сцене (тот же спутник, ±5 мин) в радиусе 5 км, и их FRP."""
    n = np.zeros(len(df), dtype=np.float32)
    frp_sum = np.zeros(len(df), dtype=np.float32)
    t = (df["acq_dt"] - pd.Timestamp("2000-01-01", tz="UTC")).dt.total_seconds().to_numpy() / 60.0
    lat0 = float(df["lat"].mean()) if len(df) else 0.0
    x = np.radians(df["lon"].to_numpy()) * 6371 * np.cos(np.radians(lat0))
    y = np.radians(df["lat"].to_numpy()) * 6371
    frp = df["frp"].fillna(0).to_numpy(np.float32)
    keys = df["stream"].astype(str).to_numpy() + "_" + df["acq_dt"].dt.year.astype(str).to_numpy()
    for k in np.unique(keys):
        idx = np.flatnonzero(keys == k)
        pts = np.column_stack([x[idx], y[idx], t[idx] * (radius_km / window_min)])
        tree = cKDTree(pts)
        nbrs = tree.query_ball_point(pts, r=radius_km)
        n[idx] = np.array([len(a) - 1 for a in nbrs], dtype=np.float32)
        frp_sum[idx] = np.array([frp[idx[a]].sum() for a in nbrs], dtype=np.float32)
    return n, frp_sum


def build_features(df: pd.DataFrame, table: dict) -> pd.DataFrame:
    is_viirs = (df["sensor"].astype(str) == "VIIRS").to_numpy(np.float32)
    ctx_n, ctx_frp = overpass_context(df)
    month = df["acq_dt"].dt.month.to_numpy()
    r5 = np.array([h3.str_to_int(c) for c in df["h3_r5"]], dtype=np.uint64)
    r7 = np.array([h3.str_to_int(c) for c in df["h3_r7"]], dtype=np.uint64)
    r8 = np.array([h3.str_to_int(c) for c in df["h3_r8"]], dtype=np.uint64)
    clim_rows, _ = _lookup(r5, table["r5_cells"], table["r5_clim_u8"])
    clim = clim_rows[np.arange(len(df)), month - 1].astype(np.float32) / 255.0 if len(df) else np.zeros(0, np.float32)
    yrs, _ = _lookup(r7, table["r7_cells"], table["r7_years_u8"])
    sscore, shit = _lookup(r8, table["r8_static_cells"], table["r8_static_score_u8"])
    out = pd.DataFrame({
        "is_viirs": is_viirs,
        "is_night": (df["daynight_sza"].astype(str) == "N").to_numpy(np.float32),
        "bright_mir": df["bright_mir"].fillna(300).to_numpy(np.float32),
        "bright_tir": df["bright_tir"].fillna(280).to_numpy(np.float32),
        "dt45": (df["bright_mir"] - df["bright_tir"]).fillna(15).to_numpy(np.float32),
        "log_frp": np.log1p(df["frp"].fillna(0).to_numpy(np.float32)),
        "log_pixel_km2": np.log(np.maximum(df["pixel_km2"].fillna(0.2).to_numpy(np.float32), 0.05)),
        "scan_deg": df["scan_deg"].fillna(0).to_numpy(np.float32),
        "p_conf": df["p_fire_raw"].fillna(0.5).to_numpy(np.float32),
        "glint_n": (np.minimum(df["glint_deg"].fillna(90).to_numpy(np.float32), 90.0) / 90.0),
        "ctx_log_n": np.log1p(ctx_n),
        "ctx_log_frp": np.log1p(ctx_frp),
        "hist_clim_month": clim,
        "hist_static": shit.astype(np.float32),
        "hist_static_score": sscore.astype(np.float32) / 255.0,
        "hist_years_r7": yrs.astype(np.float32) / max(1, table["n_years"]),
        "month_sin": np.sin(2 * np.pi * month / 12).astype(np.float32),
        "month_cos": np.cos(2 * np.pi * month / 12).astype(np.float32),
        "lat_n": ((df["lat"].to_numpy() - 48.0) / 5.0).astype(np.float32),
    })
    return out[FEATURES]


# ----------------------------------------------------------------------------- метки из будущего
def escalation_counts(det: pd.DataFrame, celldays: pd.DataFrame, radius_km: float, days_ahead: int = 3) -> np.ndarray:
    """Число «сгоревших гексагонов» (любой спутник) в радиусе R за следующие 1..days_ahead суток."""
    from ..processing.harmonize import local_date

    cd = celldays[["date", "h3_r7"]].drop_duplicates()
    ll = np.array([h3.cell_to_latlng(c) for c in cd["h3_r7"].unique()])
    centers = dict(zip(cd["h3_r7"].unique(), map(tuple, ll)))
    cd = cd.assign(lat=[centers[c][0] for c in cd["h3_r7"]], lon=[centers[c][1] for c in cd["h3_r7"]])
    lat0 = 48.0
    to_xy = lambda la, lo: np.column_stack([np.radians(lo) * 6371 * np.cos(np.radians(lat0)), np.radians(la) * 6371])
    trees = {d: cKDTree(to_xy(g["lat"].to_numpy(), g["lon"].to_numpy())) for d, g in cd.groupby("date")}
    dates = local_date(det)
    counts = np.zeros(len(det), dtype=np.int32)
    xy = to_xy(det["lat"].to_numpy(), det["lon"].to_numpy())
    for d, idx in pd.Series(np.arange(len(det))).groupby(dates.to_numpy()).groups.items():
        idx = np.asarray(idx)
        for k in range(1, days_ahead + 1):
            tree = trees.get(pd.Timestamp(d) + pd.Timedelta(days=k))
            if tree is not None:
                counts[idx] += tree.query_ball_point(xy[idx], r=radius_km, return_length=True)
    return counts


# ----------------------------------------------------------------------------- модель
def _torch_model(n_in: int, mean: np.ndarray, std: np.ndarray):
    import torch
    from torch import nn

    class EdgeMLP(nn.Module):
        def __init__(self):
            super().__init__()
            self.register_buffer("mean", torch.tensor(mean, dtype=torch.float32))
            self.register_buffer("std", torch.tensor(std, dtype=torch.float32))
            self.body = nn.Sequential(nn.Linear(n_in, 32), nn.ReLU(), nn.Linear(32, 24), nn.ReLU())
            self.head = nn.Linear(24, 2)

        def forward(self, x):
            z = self.body((x - self.mean) / self.std)
            return torch.sigmoid(self.head(z))  # [p_real, p_escalation]

    return EdgeMLP()


def train() -> dict:
    import torch

    cfg = settings()["models"]["edge"]
    t0 = time.time()
    det = pd.read_parquet(PROCESSED / "detections_filtered.parquet")
    celldays = pd.read_parquet(PROCESSED / "celldays.parquet")
    table = build_onboard_table(det, cfg["history_until_year"])
    size = save_table(table)
    load_table.cache_clear()
    log.info("onboard history table: %.1f KB", size / 1024)

    y0, y1 = cfg["train_years"]
    sub = det[det["acq_dt"].dt.year.between(y0, cfg["test_year"])].reset_index(drop=True)
    X = build_features(sub, table)
    esc = escalation_counts(sub, celldays, cfg["escalation_radius_km"], days_ahead=max(1, cfg["escalation_window_h"] // 24))
    y_real = sub["keep_veg"].to_numpy(np.float32)
    y_esc = (y_real > 0) & (esc >= cfg["escalation_min_celldays"])
    y = np.column_stack([y_real, y_esc.astype(np.float32)])
    year = sub["acq_dt"].dt.year.to_numpy()
    tr, va, te = (year >= y0) & (year <= y1), year == cfg["val_year"], year == cfg["test_year"]
    rng = np.random.default_rng(0)
    tr_idx = np.flatnonzero(tr)
    if len(tr_idx) > 900_000:
        tr_idx = rng.choice(tr_idx, 900_000, replace=False)
    Xn = X.to_numpy(np.float32)
    mean, std = Xn[tr_idx].mean(0), Xn[tr_idx].std(0) + 1e-6
    log.info("AI#1 data: train %d, val %d, test %d; real=%.3f esc=%.3f", len(tr_idx), va.sum(), te.sum(),
             y_real[tr].mean(), y_esc[tr].mean())

    torch.manual_seed(0)
    model = _torch_model(len(FEATURES), mean, std)
    pos_w = torch.tensor([1.0, float(min(8.0, (1 - y_esc[tr].mean()) / max(y_esc[tr].mean(), 1e-3)) ** 0.5)])
    opt = torch.optim.AdamW(model.parameters(), lr=3e-3, weight_decay=1e-4)
    Xt, yt = torch.tensor(Xn[tr_idx]), torch.tensor(y[tr_idx])
    Xv, yv = torch.tensor(Xn[va]), y[va]
    best, best_state = -1.0, None
    for epoch in range(10):
        model.train()
        perm = torch.randperm(len(Xt))
        for i in range(0, len(perm), 2048):
            b = perm[i:i + 2048]
            p = model(Xt[b]).clamp(1e-6, 1 - 1e-6)
            loss = -(pos_w * (yt[b] * torch.log(p)) + (1 - yt[b]) * torch.log(1 - p)).mean()
            opt.zero_grad()
            loss.backward()
            opt.step()
        model.eval()
        with torch.no_grad():
            pv = model(Xv).numpy()
        score = roc_auc_score(yv[:, 0], pv[:, 0]) + average_precision_score(yv[:, 1], pv[:, 1])
        log.info("AI#1 epoch %d: loss %.4f val real-AUC %.4f esc-AP %.4f", epoch, float(loss),
                 roc_auc_score(yv[:, 0], pv[:, 0]), average_precision_score(yv[:, 1], pv[:, 1]))
        if score > best:
            best, best_state = score, {k: v.clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_state)
    model.eval()

    with torch.no_grad():
        p_val = model(Xv).numpy()
        p_test = model(torch.tensor(Xn[te])).numpy()
    real_val = p_val[:, 0] >= 0.5
    t_crit = float(np.quantile(p_val[real_val, 1], 0.95)) if real_val.any() else 0.7
    t_high = float(np.quantile(p_val[real_val, 1], 0.80)) if real_val.any() else 0.4

    export_onnx(model, len(FEATURES))
    int8_ok = quantize_int8()
    export_json(model, mean, std, t_crit, t_high)

    yte = y[te]
    sub_te = sub[te]
    metrics = {
        "test_year": cfg["test_year"],
        "n_test": int(te.sum()),
        "p_real": {
            "roc_auc": round(float(roc_auc_score(yte[:, 0], p_test[:, 0])), 4),
            "pr_auc": round(float(average_precision_score(yte[:, 0], p_test[:, 0])), 4),
            "baseline_confidence_only_auc": round(float(roc_auc_score(yte[:, 0], sub_te["p_fire_raw"])), 4),
        },
        "p_escalation": {
            "positives_share": round(float(yte[:, 1].mean()), 4),
            "roc_auc": round(float(roc_auc_score(yte[:, 1], p_test[:, 1])), 4),
            "pr_auc": round(float(average_precision_score(yte[:, 1], p_test[:, 1])), 4),
            "baseline_frp_only_auc": round(float(roc_auc_score(yte[:, 1], sub_te["frp"].fillna(0))), 4),
            "baseline_frp_only_pr_auc": round(float(average_precision_score(yte[:, 1], sub_te["frp"].fillna(0))), 4),
        },
        "thresholds": {"critical": round(t_crit, 4), "high": round(t_high, 4), "real": 0.5},
    }
    crit = (p_test[:, 0] >= 0.5) & (p_test[:, 1] >= t_crit)
    high = (p_test[:, 0] >= 0.5) & (p_test[:, 1] >= t_high)
    metrics["critical_precision"] = round(float(yte[crit, 1].mean()), 4) if crit.any() else None
    metrics["critical_recall"] = round(float(yte[crit, 1].sum() / max(yte[:, 1].sum(), 1)), 4)
    metrics["high_or_critical_recall"] = round(float(yte[high, 1].sum() / max(yte[:, 1].sum(), 1)), 4)
    metrics["model"] = model_footprint(int8_ok)
    metrics["onboard_table_kb"] = round(size / 1024, 1)
    metrics["train_seconds"] = round(time.time() - t0, 1)
    save_json({"features": FEATURES, **metrics}, META_PATH)
    save_json(metrics, REPORTS_DIR / "ai1_edge_detector_metrics.json")
    log.info("AI#1 test: %s", metrics)
    _session.cache_clear()
    return metrics


def export_onnx(model, n_in: int) -> None:
    import torch

    dummy = torch.zeros(1, n_in)
    MODELS_DIR.mkdir(parents=True, exist_ok=True)
    kwargs = dict(input_names=["features"], output_names=["probs"], dynamic_axes={"features": {0: "batch"}, "probs": {0: "batch"}},
                  opset_version=17)
    try:
        torch.onnx.export(model, dummy, str(ONNX_PATH), dynamo=False, **kwargs)
    except TypeError:
        torch.onnx.export(model, dummy, str(ONNX_PATH), **kwargs)


def quantize_int8() -> bool:
    try:
        from onnxruntime.quantization import QuantType, quantize_dynamic

        quantize_dynamic(str(ONNX_PATH), str(ONNX_INT8_PATH), weight_type=QuantType.QInt8)
        return True
    except Exception as exc:
        log.warning("int8 quantization failed: %s", exc)
        return False


def export_json(model, mean, std, t_crit: float, t_high: float) -> None:
    """Веса в JSON: модель можно запустить хоть на микроконтроллере, хоть в браузере."""
    sd = {k: v.detach().cpu().numpy().round(6).tolist() for k, v in model.state_dict().items()}
    save_json({"features": FEATURES, "mean": np.asarray(mean).round(6).tolist(), "std": np.asarray(std).round(6).tolist(),
               "layers": [{"W": sd["body.0.weight"], "b": sd["body.0.bias"], "act": "relu"},
                          {"W": sd["body.2.weight"], "b": sd["body.2.bias"], "act": "relu"},
                          {"W": sd["head.weight"], "b": sd["head.bias"], "act": "sigmoid"}],
               "outputs": ["p_real", "p_escalation"], "thresholds": {"critical": t_crit, "high": t_high, "real": 0.5}},
              JSON_PATH)


def mlp_numpy(x: np.ndarray, weights: dict) -> np.ndarray:
    """Эталонный инференс на чистом numpy (то же, что на борту)."""
    z = (x - np.array(weights["mean"])) / np.array(weights["std"])
    for layer in weights["layers"]:
        z = z @ np.array(layer["W"]).T + np.array(layer["b"])
        z = np.maximum(z, 0) if layer["act"] == "relu" else 1 / (1 + np.exp(-z))
    return z


def model_footprint(int8_ok: bool) -> dict:
    import onnxruntime as ort

    out = {"onnx_fp32_kb": round(ONNX_PATH.stat().st_size / 1024, 1)}
    if int8_ok and ONNX_INT8_PATH.exists():
        out["onnx_int8_kb"] = round(ONNX_INT8_PATH.stat().st_size / 1024, 1)
    sess = ort.InferenceSession(str(ONNX_PATH), providers=["CPUExecutionProvider"])
    x = np.random.default_rng(0).normal(size=(10_000, len(FEATURES))).astype(np.float32) * 0 + 1
    t0 = time.perf_counter()
    for _ in range(5):
        sess.run(None, {"features": x})
    out["latency_us_per_detection_cpu"] = round((time.perf_counter() - t0) / 5 / len(x) * 1e6, 3)
    w = json.loads(JSON_PATH.read_text(encoding="utf-8")) if JSON_PATH.exists() else None
    if w:
        out["parameters"] = int(sum(np.array(l["W"]).size + np.array(l["b"]).size for l in w["layers"]))
    return out


# ----------------------------------------------------------------------------- инференс
@lru_cache(maxsize=1)
def _session():
    import onnxruntime as ort

    path = ONNX_INT8_PATH if ONNX_INT8_PATH.exists() else ONNX_PATH
    return ort.InferenceSession(str(path), providers=["CPUExecutionProvider"])


def thresholds() -> dict:
    meta = json.loads(META_PATH.read_text(encoding="utf-8")) if META_PATH.exists() else {}
    return meta.get("thresholds", {"critical": 0.6, "high": 0.35, "real": 0.5})


def score_detections(df: pd.DataFrame) -> pd.DataFrame:
    """Бортовой инференс ИИ №1 для отфильтрованных точек (после add_geometry/apply_filters)."""
    if df.empty:
        return df.assign(p_real=[], p_escalation=[], priority=[])
    X = build_features(df, load_table()).to_numpy(np.float32)
    probs = _session().run(None, {"features": X})[0]
    th = thresholds()
    out = df.copy()
    out["p_real"] = probs[:, 0].round(4)
    out["p_escalation"] = probs[:, 1].round(4)
    pr = np.where(out["p_real"] < th["real"], "SUPPRESS",
                  np.where(out["p_escalation"] >= th["critical"], "CRITICAL",
                           np.where(out["p_escalation"] >= th["high"], "HIGH", "ROUTINE")))
    out["priority"] = pd.Categorical(pr, categories=["SUPPRESS", "ROUTINE", "HIGH", "CRITICAL"], ordered=True)
    return out


def make_alerts(scored: pd.DataFrame, min_priority: str = "ROUTINE") -> pd.DataFrame:
    """Склеить пиксели одного пролёта и одной ячейки res-7 в один алерт + 24-байтный пакет."""
    from ..comms import packets

    if scored.empty:
        return pd.DataFrame()
    s = scored[scored["priority"] >= min_priority].copy()
    if s.empty:
        return pd.DataFrame()
    s["overpass"] = s["acq_dt"].dt.floor("10min")
    s["prio_rank"] = s["priority"].cat.codes
    g = s.groupby(["stream", "overpass", "h3_r7"], observed=True)
    alerts = g.agg(
        lat=("lat", "mean"), lon=("lon", "mean"), acq_dt=("acq_dt", "min"), frp_sum=("frp", "sum"),
        cluster_n=("frp", "size"), p_real=("p_real", "max"), p_escalation=("p_escalation", "max"),
        prio_rank=("prio_rank", "max"), night=("daynight_sza", lambda x: (x.astype(str) == "N").any()),
        h3_r5=("h3_r5", "first"),
    ).reset_index()
    cats = scored["priority"].cat.categories
    alerts["priority"] = [cats[i] for i in alerts["prio_rank"]]
    alerts["packet_hex"] = [
        packets.encode(p, st, int(t.timestamp()), la, lo, f, n, pe, prl,
                       flags=packets.FLAG_NIGHT if nt else 0).hex()
        for p, st, t, la, lo, f, n, pe, prl, nt in zip(
            alerts["priority"], alerts["stream"].astype(str), alerts["acq_dt"], alerts["lat"], alerts["lon"],
            alerts["frp_sum"], alerts["cluster_n"], alerts["p_escalation"], alerts["p_real"], alerts["night"])
    ]
    alerts["alert_id"] = [f"{st}-{t:%Y%m%d%H%M}-{c}" for st, t, c in zip(alerts["stream"].astype(str), alerts["overpass"], alerts["h3_r7"])]
    return alerts.drop(columns=["prio_rank"]).sort_values(["acq_dt", "priority"], ascending=[True, False]).reset_index(drop=True)

In [ ]:
%%writefile helios/models/link_ai.py
"""ИИ №2 (на борту): «пропадёт ли этот канал в ближайшие 5 минут?» → выбор канала заранее.

Вход (то, что знает спутник): геометрия пролёта (угол места, скорость его изменения, время до
ухода за горизонт), измеренный запас линии по пилот-тону и его тренд, прогноз погоды над
станцией (его заранее закладывает наземный ИИ №3), Kp, рентген, F10.7, угол на Солнце.
Выход: вероятность обрыва для канала S / X / Ka / лазер. Канал с минимальным риском и
достаточной скоростью выбирается ДО того, как связь пропадёт; при риске на всех каналах
короткие пакеты тревоги уходят первыми.

Сравнение: правило «запас < 2 дБ → обрыв» (реактивное) и физика по прогнозу погоды.
"""

from __future__ import annotations

import json
import time
from functools import lru_cache

import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, roc_auc_score

from ..config import MODELS_DIR, PROCESSED, REPORTS_DIR, settings
from ..utils import get_logger, save_json

log = get_logger("helios.ai2")

FEATURES = [
    "el", "el_rate", "range_km", "t_since_aos", "t_to_los", "max_el_pass", "sun_sep", "geomag_abs", "alt_km",
    "fc_precip", "fc_cloud", "rh", "kp", "log_xray", "f107", "margin_now", "margin_trend",
    "ch_S", "ch_X", "ch_Ka", "ch_OPT",
]
ONNX_PATH = MODELS_DIR / "link_ai.onnx"
META_PATH = MODELS_DIR / "link_ai_meta.json"
JSON_PATH = MODELS_DIR / "link_ai_weights.json"


def _model(n_in: int, mean, std):
    import torch
    from torch import nn

    class LinkMLP(nn.Module):
        def __init__(self):
            super().__init__()
            self.register_buffer("mean", torch.tensor(mean, dtype=torch.float32))
            self.register_buffer("std", torch.tensor(std, dtype=torch.float32))
            self.net = nn.Sequential(nn.Linear(n_in, 32), nn.ReLU(), nn.Linear(32, 16), nn.ReLU(), nn.Linear(16, 1))

        def forward(self, x):
            return torch.sigmoid(self.net((x - self.mean) / self.std))

    return LinkMLP()


def train(days: int | None = None) -> dict:
    import torch

    from ..comms.link_sim import simulate_dataset

    cfg = settings()["models"]["link"]
    t0 = time.time()
    data = simulate_dataset(days=days or cfg["sim_days"], step_s=cfg["step_s"], min_el=cfg["min_elevation_deg"])
    data.to_parquet(PROCESSED / "link_dataset.parquet", index=False)
    # Задача ИИ №2 — РАННЕЕ предупреждение: канал сейчас жив (запас > 0), а через ≤5 мин пропадёт.
    data = data[data["margin_now"] > 0].reset_index(drop=True)
    # разбиение по пролётам (25% пролётов — тест, модель их не видела)
    passes = data["pass_id"].unique()
    test_passes = set(np.random.default_rng(3).choice(passes, size=len(passes) // 4, replace=False))
    te = data["pass_id"].isin(test_passes).to_numpy()
    tr = ~te
    X = data[FEATURES].to_numpy(np.float32)
    y = data["outage_next"].to_numpy(np.float32)
    mean, std = X[tr].mean(0), X[tr].std(0) + 1e-6
    torch.manual_seed(1)
    model = _model(len(FEATURES), mean, std)
    opt = torch.optim.AdamW(model.parameters(), lr=3e-3, weight_decay=1e-4)
    Xt, yt = torch.tensor(X[tr]), torch.tensor(y[tr])[:, None]
    pos_w = float(min(10.0, (1 - y[tr].mean()) / max(y[tr].mean(), 1e-3)) ** 0.5)
    for epoch in range(25):
        perm = torch.randperm(len(Xt))
        for i in range(0, len(perm), 1024):
            b = perm[i:i + 1024]
            p = model(Xt[b]).clamp(1e-6, 1 - 1e-6)
            loss = -(pos_w * yt[b] * torch.log(p) + (1 - yt[b]) * torch.log(1 - p)).mean()
            opt.zero_grad()
            loss.backward()
            opt.step()
    model.eval()
    with torch.no_grad():
        p_te = model(torch.tensor(X[te])).numpy()[:, 0]
    d_te = data[te]
    yt_ = y[te]
    rule = -d_te["margin_now"].to_numpy()  # реактивное правило: чем меньше запас сейчас, тем ближе обрыв

    def recall_at_far(score: np.ndarray, far: float = 0.05) -> float:
        thr = np.quantile(score[yt_ == 0], 1 - far)
        return round(float((score[yt_ == 1] > thr).mean()), 4) if (yt_ == 1).any() else 0.0

    metrics = {
        "rows_alive": int(len(data)), "test_rows": int(te.sum()), "outage_rate": round(float(y.mean()), 4),
        "outage_rate_by_channel": data.groupby("channel")["outage_next"].mean().round(4).to_dict(),
        "max_kp_in_data": round(float(data["kp"].max()), 2),
        "ai": {"roc_auc": round(float(roc_auc_score(yt_, p_te)), 4),
               "pr_auc": round(float(average_precision_score(yt_, p_te)), 4),
               "early_warning_recall_at_5pct_false_alarms": recall_at_far(p_te)},
        "baseline_margin_rule": {"roc_auc": round(float(roc_auc_score(yt_, rule)), 4),
                                 "pr_auc": round(float(average_precision_score(yt_, rule)), 4),
                                 "early_warning_recall_at_5pct_false_alarms": recall_at_far(rule)},
    }
    by_ch = {}
    for ch, g in d_te.assign(p=p_te, r=rule).groupby("channel"):
        if g["outage_next"].nunique() == 2:
            by_ch[ch] = {"ai_roc_auc": round(float(roc_auc_score(g["outage_next"], g["p"])), 4),
                         "ai_pr_auc": round(float(average_precision_score(g["outage_next"], g["p"])), 4),
                         "rule_pr_auc": round(float(average_precision_score(g["outage_next"], g["r"])), 4),
                         "positives": int(g["outage_next"].sum())}
    metrics["by_channel"] = by_ch

    import torch.onnx

    MODELS_DIR.mkdir(parents=True, exist_ok=True)
    kwargs = dict(input_names=["features"], output_names=["p_outage"], dynamic_axes={"features": {0: "batch"}, "p_outage": {0: "batch"}},
                  opset_version=17)
    try:
        torch.onnx.export(model, torch.zeros(1, len(FEATURES)), str(ONNX_PATH), dynamo=False, **kwargs)
    except TypeError:
        torch.onnx.export(model, torch.zeros(1, len(FEATURES)), str(ONNX_PATH), **kwargs)
    sd = {k: v.detach().numpy().round(6).tolist() for k, v in model.state_dict().items()}
    save_json({"features": FEATURES, "mean": mean.round(6).tolist(), "std": std.round(6).tolist(),
               "layers": [{"W": sd["net.0.weight"], "b": sd["net.0.bias"], "act": "relu"},
                          {"W": sd["net.2.weight"], "b": sd["net.2.bias"], "act": "relu"},
                          {"W": sd["net.4.weight"], "b": sd["net.4.bias"], "act": "sigmoid"}]}, JSON_PATH)
    metrics["model"] = {"onnx_kb": round(ONNX_PATH.stat().st_size / 1024, 1),
                        "parameters": int(sum(np.array(v).size for k, v in sd.items() if k.startswith("net")))}
    metrics["train_seconds"] = round(time.time() - t0, 1)
    save_json({"features": FEATURES, **metrics}, META_PATH)
    save_json(metrics, REPORTS_DIR / "ai2_link_metrics.json")
    _session.cache_clear()
    log.info("AI#2: %s", metrics)
    return metrics


@lru_cache(maxsize=1)
def _session():
    import onnxruntime as ort

    return ort.InferenceSession(str(ONNX_PATH), providers=["CPUExecutionProvider"])


def predict(df: pd.DataFrame) -> np.ndarray:
    """P(обрыв в ближайшие 5 мин) для строк с колонками FEATURES."""
    X = df.reindex(columns=FEATURES).fillna(0).to_numpy(np.float32)
    return _session().run(None, {"features": X})[0][:, 0]


def metrics() -> dict:
    return json.loads(META_PATH.read_text(encoding="utf-8")) if META_PATH.exists() else {}

In [ ]:
%%writefile helios/models/ground_forecast.py
"""ИИ №3 (Земля, «суперкомпьютер»): прогноз пожарного риска на 1–4 недели по ячейкам H3.

Модель: HistGradientBoosting (sklearn) на панели «ячейка res-5 × неделя».
Обучение 2013–2022, валидация 2023, честный тест 2024–2025 (модель этих лет не видела).
Сравнение с базовыми линиями:
* сезонная климатология (как «обычно горит в эту неделю»),
* персистентность (горело на этой неделе → будет гореть на следующей).
"""

from __future__ import annotations

import json
import time

import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import average_precision_score, brier_score_loss, roc_auc_score

from ..config import MODELS_DIR, REPORTS_DIR, settings
from ..utils import get_logger, save_json
from .features import FEATURES, Panel

log = get_logger("helios.ai3")

HORIZONS = (1, 2, 3, 4)
_PANEL: Panel | None = None


def panel() -> Panel:
    global _PANEL
    if _PANEL is None:
        _PANEL = Panel()
    return _PANEL


def _metrics(y: np.ndarray, p: np.ndarray) -> dict:
    if y.sum() == 0 or y.sum() == len(y):
        return {"n": int(len(y)), "positives": int(y.sum())}
    order = np.argsort(-p)
    top10 = order[: max(1, len(p) // 10)]
    return {
        "n": int(len(y)),
        "positives": int(y.sum()),
        "roc_auc": round(float(roc_auc_score(y, p)), 4),
        "pr_auc": round(float(average_precision_score(y, p)), 4),
        "brier": round(float(brier_score_loss(y, np.clip(p, 0, 1))), 5),
        "recall_top10pct": round(float(y[top10].sum() / y.sum()), 4),
    }


def fit_model(pn: Panel, horizon: int, train_end: str, val_year: int | None = None,
              train_start: str = "2013-01-01") -> HistGradientBoostingClassifier:
    rate = settings()["models"]["ground"]["negative_sample_rate"]
    Xtr, ytr, wtr = pn.rows(pn.week_index(train_start, train_end), horizon, neg_rate=rate, seed=horizon)
    model = HistGradientBoostingClassifier(
        max_iter=500, learning_rate=0.06, max_leaf_nodes=63, min_samples_leaf=80,
        l2_regularization=1.0, early_stopping=True, validation_fraction=0.1, n_iter_no_change=25,
        random_state=horizon,
    )
    model.fit(Xtr[FEATURES], ytr, sample_weight=wtr)
    log.info("AI#3 h=%d: trained on %d rows (%d positives), %d iters", horizon, len(ytr), int(ytr.sum()), model.n_iter_)
    return model


def evaluate(pn: Panel, model, horizon: int, start: str, end: str) -> dict:
    X, y, _ = pn.rows(pn.week_index(start, end), horizon)
    p_ai = model.predict_proba(X[FEATURES])[:, 1]
    return {
        "ai": _metrics(y, p_ai),
        "baseline_climatology": _metrics(y, X["clim_woy_next"].to_numpy()),
        "baseline_persistence": _metrics(y, (X["a_t"].to_numpy() > 0).astype(float) * 0.9 + X["clim_woy_next"].to_numpy() * 0.1),
    }


def train() -> dict:
    cfg = settings()["models"]["ground"]
    pn = panel()
    tr0, tr1 = cfg["train_years"]
    report = {"train": [tr0, tr1], "val": cfg["val_year"], "test": cfg["test_years"], "horizons": {}}
    MODELS_DIR.mkdir(parents=True, exist_ok=True)
    for h in HORIZONS:
        t0 = time.time()
        model = fit_model(pn, h, f"{tr1}-12-31", train_start=f"{tr0}-01-01")
        val = evaluate(pn, model, h, f"{cfg['val_year']}-01-01", f"{cfg['val_year']}-12-31")
        test = evaluate(pn, model, h, f"{cfg['test_years'][0]}-01-01", f"{cfg['test_years'][-1]}-12-31")
        joblib.dump({"model": model, "features": FEATURES, "horizon": h, "train_end": f"{tr1}-12-31"},
                    MODELS_DIR / f"ground_forecast_h{h}.joblib")
        report["horizons"][f"h{h}"] = {"val": val, "test": test, "seconds": round(time.time() - t0, 1)}
        log.info("AI#3 h=%d test: AI %s | clim %s", h, test["ai"], test["baseline_climatology"])
    # важность признаков (перестановочная, на подвыборке теста) для горизонта 1 недели
    try:
        from sklearn.inspection import permutation_importance

        m1 = joblib.load(MODELS_DIR / "ground_forecast_h1.joblib")["model"]
        X, y, _ = pn.rows(pn.week_index(f"{cfg['test_years'][0]}-04-01", f"{cfg['test_years'][0]}-10-31"), 1, neg_rate=0.2)
        imp = permutation_importance(m1, X[FEATURES], y, n_repeats=3, random_state=0, scoring="average_precision")
        report["feature_importance_h1"] = dict(sorted(
            {f: round(float(v), 4) for f, v in zip(FEATURES, imp.importances_mean)}.items(), key=lambda kv: -kv[1]))
    except Exception as exc:
        log.warning("permutation importance: %s", exc)
    save_json(report, REPORTS_DIR / "ai3_ground_forecast_metrics.json")
    return report


def load(horizon: int = 1) -> dict:
    return joblib.load(MODELS_DIR / f"ground_forecast_h{horizon}.joblib")


def predict_week(week_start: str | pd.Timestamp, horizon: int = 1, model_bundle: dict | None = None) -> pd.DataFrame:
    """Риск на неделю (week_start + horizon) для всех ячеек по данным до конца week_start."""
    pn = panel()
    w = int(pd.Index(pn.weeks).get_indexer([pd.Timestamp(week_start).normalize()], method="ffill")[0])
    bundle = model_bundle or load(horizon)
    X, y, _ = pn.rows(np.array([w]), horizon, allow_future=True)
    prob = bundle["model"].predict_proba(X[bundle["features"]])[:, 1]
    out = pd.DataFrame({
        "h3_r5": pn.cells[X["cell_idx"].to_numpy()],
        "lat": X["lat"].to_numpy(), "lon": X["lon"].to_numpy(),
        "as_of_week": pn.weeks[w], "target_week": pd.Timestamp(pn.weeks[w]) + pd.Timedelta(weeks=horizon),
        "horizon_weeks": horizon,
        "p_fire": prob.round(4),
        "p_climatology": X["clim_woy_next"].to_numpy().round(4),
        "recent_celldays_4w": X["sum4"].to_numpy(),
        "precip_28d": X["precip_28d"].to_numpy().round(1),
        "gwet": X["gwet"].to_numpy().round(3),
        "actual": y,
    })
    return out


# ----------------------------------------------------------------------------- задержки спутников
def satellite_delay_report(hours: int = 24) -> pd.DataFrame:
    """Какой спутник «опаздывает»: задержка гранул NRT по CMR + робастная аномалия."""
    from ..apis.earthdata import granule_latency, latency_report

    lat = granule_latency(hours=hours)
    rep = latency_report(lat)
    if len(rep):
        save_json(json.loads(rep.to_json(orient="records", date_format="iso")), REPORTS_DIR / "satellite_delays.json")
    return rep

In [ ]:
%%writefile helios/models/time_machine.py
"""Fire Time Machine — проверка ИИ на прошлом.

Пользователь выбирает дату (например, 1 июня 2024). Модель видит ТОЛЬКО данные до этой даты
и прогнозирует следующие 1–4 недели; затем «проигрываем историю» и сравниваем с тем, что
реально сгорело, а рядом — прогноз простого сезонного алгоритма (климатология).
Модели ИИ №3 обучены на 2013–2022, поэтому любые даты 2024–2025 — честная проверка
(для более ранних дат можно включить strict=True: модель переобучится до этой даты).
"""

from __future__ import annotations

import numpy as np
import pandas as pd

from ..config import PROCESSED, REPORTS_DIR
from ..utils import get_logger, save_json
from .ground_forecast import HORIZONS, _metrics, fit_model, load, panel, predict_week

log = get_logger("helios.timemachine")

DEFAULT_DATES = ["2024-04-01", "2024-06-03", "2024-08-26", "2025-04-07", "2025-07-07", "2025-09-01"]


def scenario(as_of: str, horizons=HORIZONS, strict: bool = False) -> tuple[dict, pd.DataFrame]:
    as_of_ts = pd.Timestamp(as_of)
    frames, res = [], {"as_of": as_of, "horizons": {}}
    for h in horizons:
        bundle = load(h)
        if pd.Timestamp(bundle["train_end"]) >= as_of_ts:
            if not strict:
                res.setdefault("warnings", []).append(f"h{h}: модель обучена до {bundle['train_end']} — включите strict")
            else:
                bundle = {"model": fit_model(panel(), h, str((as_of_ts - pd.Timedelta(days=7)).date())),
                          "features": bundle["features"], "train_end": str(as_of_ts.date())}
        pred = predict_week(as_of_ts, h, bundle)
        y = pred["actual"].to_numpy()
        res["horizons"][f"h{h}"] = {
            "target_week": str(pred["target_week"].iloc[0].date()) if pd.notna(pred["target_week"].iloc[0]) else None,
            "ai": _metrics(y, pred["p_fire"].to_numpy()),
            "seasonal_baseline": _metrics(y, pred["p_climatology"].to_numpy()),
            "cells_burned": int(y.sum()),
        }
        frames.append(pred.assign(as_of=as_of_ts))
    return res, pd.concat(frames, ignore_index=True)


def run_default_scenarios(dates=DEFAULT_DATES) -> list[dict]:
    results, frames = [], []
    for d in dates:
        try:
            r, df = scenario(d)
            results.append(r)
            # для фронтенда храним только информативные ячейки (риск или факт пожара)
            frames.append(df[(df["p_fire"] >= 0.02) | (df["actual"] == 1) | (df["p_climatology"] >= 0.05)])
            log.info("Time Machine %s: %s", d, {k: (v["ai"].get("pr_auc"), v["seasonal_baseline"].get("pr_auc"))
                                                for k, v in r["horizons"].items()})
        except Exception as exc:
            log.warning("Time Machine %s: %s", d, exc)
    if frames:
        pd.concat(frames, ignore_index=True).to_parquet(PROCESSED / "time_machine.parquet", index=False)
    save_json(results, REPORTS_DIR / "time_machine.json")
    return results

### База данных и оркестрация

In [ ]:
%%writefile helios/db/__init__.py
"""База данных: Supabase."""

In [ ]:
%%writefile helios/db/supabase_sync.py
"""Выгрузка результатов в Supabase (PostgREST).

Запись идёт одним из двух способов:
1. SUPABASE_SERVICE_ROLE_KEY задан — обычный upsert через REST (обходит RLS);
2. иначе — RPC helios_ingest(token, table, rows) с HELIOS_INGEST_TOKEN из .env
   (функция SECURITY DEFINER проверяет sha256-хэш токена и пишет только в разрешённые таблицы).
Чтение для фронтенда — ключом SUPABASE_PUBLISHABLE_KEY (RLS: всем можно читать).
"""

from __future__ import annotations

import json
import math
from datetime import datetime, timezone

import numpy as np
import pandas as pd

from ..config import MODELS_DIR, PROCESSED, REPORTS_DIR, env, settings
from ..utils import get_logger, http

log = get_logger("helios.supabase")


def _records(df: pd.DataFrame) -> list[dict]:
    """DataFrame → JSON-записи (NaN → null, даты → ISO, numpy → python)."""
    return json.loads(df.to_json(orient="records", date_format="iso", default_handler=str))


class SupabaseClient:
    def __init__(self):
        self.url = (env("SUPABASE_URL") or "").rstrip("/")
        self.service_key = env("SUPABASE_SERVICE_ROLE_KEY")
        self.public_key = env("SUPABASE_PUBLISHABLE_KEY") or env("SUPABASE_ANON_KEY")
        self.token = env("HELIOS_INGEST_TOKEN")
        if not self.url or not (self.service_key or self.public_key):
            raise RuntimeError("SUPABASE_URL и ключ не заданы в .env")

    def _headers(self, write: bool = False) -> dict:
        key = self.service_key if (write and self.service_key) else self.public_key or self.service_key
        h = {"apikey": key, "Content-Type": "application/json"}
        if key and not key.startswith("sb_"):
            h["Authorization"] = f"Bearer {key}"
        return h

    def ping(self) -> str:
        r = http().get(f"{self.url}/rest/v1/regions", params={"select": "id,name", "limit": 5}, headers=self._headers(), timeout=30)
        r.raise_for_status()
        mode = "service_role" if self.service_key else ("ingest-token" if self.token else "read-only")
        return f"{len(r.json())} регион(ов), запись: {mode}"

    def select(self, table: str, params: dict | None = None) -> list[dict]:
        r = http().get(f"{self.url}/rest/v1/{table}", params=params or {"select": "*"}, headers=self._headers(), timeout=60)
        r.raise_for_status()
        return r.json()

    def upsert(self, table: str, rows: list[dict], chunk: int = 500, mode: str = "upsert") -> int:
        total = 0
        for i in range(0, max(len(rows), 1), chunk):
            part = rows[i:i + chunk]
            if not part and mode != "replace":
                break
            m = mode if i == 0 else "upsert"
            if self.service_key:
                if m == "replace":
                    http().delete(f"{self.url}/rest/v1/{table}", params={"select": "*"}, headers=self._headers(True), timeout=120)
                h = {**self._headers(True), "Prefer": "resolution=merge-duplicates,return=minimal"}
                r = http().post(f"{self.url}/rest/v1/{table}", headers=h, data=json.dumps(part), timeout=120)
                r.raise_for_status()
                total += len(part)
            else:
                if not self.token:
                    raise RuntimeError("нужен HELIOS_INGEST_TOKEN или SUPABASE_SERVICE_ROLE_KEY для записи")
                r = http().post(f"{self.url}/rest/v1/rpc/helios_ingest", headers=self._headers(),
                                data=json.dumps({"p_token": self.token, "p_table": table, "p_rows": part, "p_mode": m}),
                                timeout=120)
                if r.status_code >= 400:
                    raise RuntimeError(f"{table}: {r.status_code} {r.text[:300]}")
                total += int(r.json() or 0)
        return total


# ----------------------------------------------------------------------------- таблицы
def _tbl_reference() -> dict[str, list[dict]]:
    from ..apis.celestrak import SATELLITES, STREAM_TO_SAT
    from ..comms.stations import STATIONS

    sat_rows = []
    for stream, name in STREAM_TO_SAT.items():
        norad, role, sensor = SATELLITES[name]
        sat_rows.append({"id": stream, "name": name, "norad": norad, "sensor": sensor, "role": role,
                         "active": stream != "VIIRS_SNPP",
                         "note": "Suomi-NPP: NRT-данные прекратились 2026-11-02" if stream == "VIIRS_SNPP" else None})
    for name, (norad, role, sensor) in SATELLITES.items():
        if role.startswith("relay"):
            sat_rows.append({"id": name, "name": name, "norad": norad, "sensor": sensor, "role": role, "active": True, "note": None})
    st_rows = [{"id": s["id"], "name": s["name"], "lat": s["lat"], "lon": s["lon"], "alt_m": s["alt_m"], "kind": s["kind"],
                "bands": s.get("bands"), "dish_m": s.get("dish_m"), "aperture_m": s.get("aperture_m"),
                "geomag_lat": s["geomag_lat"], "is_real": s["real"]} for s in STATIONS]
    reg = settings()["region"]
    return {"regions": [{"id": reg["id"], "name": reg["name"], "bbox": reg["bbox"]}],
            "satellites": sat_rows, "ground_stations": st_rows}


def _tbl_history() -> dict[str, pd.DataFrame]:
    out = {}
    reg = settings()["region"]["id"]
    if (PROCESSED / "calendar_monthly.parquet").exists():
        c = pd.read_parquet(PROCESSED / "calendar_monthly.parquet")
        c = c.rename(columns={k: k.lower() for k in c.columns if k.startswith("celldays_")})
        c["month"] = pd.to_datetime(c["month"]).dt.strftime("%Y-%m-01")
        c["region_id"] = reg
        cols = ["region_id", "month", "year", "mon", "harmonized", "harmonized_day", "harmonized_night", "lo95", "hi95",
                "raw_detections", "raw_union_celldays", "celldays_modis_a", "celldays_modis_t", "celldays_viirs_snpp",
                "celldays_viirs_noaa20", "anomaly_z", "pct_rank", "streams"]
        out["burning_calendar"] = c.reindex(columns=cols)
    if (PROCESSED / "calibration.parquet").exists():
        out["calibration"] = pd.read_parquet(PROCESSED / "calibration.parquet")
    if (PROCESSED / "static_registry.parquet").exists():
        r = pd.read_parquet(PROCESSED / "static_registry.parquet")
        out["static_sources"] = r[r["is_static"]][["h3_r8", "lat", "lon", "n", "static_score", "static_type_share",
                                                   "max_months_in_year", "years_with_6m", "night_share", "frp_median",
                                                   "first_year", "last_year", "ev_type", "ev_temporal", "ev_hdbscan"]]
    if (PROCESSED / "fire_dna_r7.parquet").exists():
        d = pd.read_parquet(PROCESSED / "fire_dna_r7.parquet")
        d = d[d["n_det"] >= 5]
        out["fire_dna"] = d[["h3_r7", "lat", "lon", "dna_class", "label_ru", "confidence", "p_industrial", "p_agricultural",
                             "p_wildfire", "p_sporadic", "n_det", "n_years", "first_year", "last_year", "peak_doy",
                             "night_share", "frp_median"]].round(4)
    if (PROCESSED / "cell_weekly_r5.parquet").exists():
        w = pd.read_parquet(PROCESSED / "cell_weekly_r5.parquet")
        w["month"] = pd.to_datetime(w["week"]).dt.strftime("%Y-%m-01")
        m = w.groupby(["h3_r5", "month"]).agg(union_celldays=("union_celldays", "sum"),
                                              harmonized_celldays=("harmonized_celldays", "sum"),
                                              frp_sum=("frp_sum", "sum")).reset_index().round(3)
        out["cell_activity_monthly"] = m
    if (PROCESSED / "season_shift_r4.parquet").exists():
        out["season_shift_cells"] = pd.read_parquet(PROCESSED / "season_shift_r4.parquet")
    return out


def _tbl_ai() -> dict[str, pd.DataFrame]:
    out = {}
    if (PROCESSED / "time_machine.parquet").exists():
        t = pd.read_parquet(PROCESSED / "time_machine.parquet")
        t = t.rename(columns={"horizon_weeks": "horizon_weeks"})
        t["as_of"] = pd.to_datetime(t["as_of"]).dt.strftime("%Y-%m-%d")
        t["target_week"] = pd.to_datetime(t["target_week"]).dt.strftime("%Y-%m-%d")
        out["time_machine"] = t[["as_of", "horizon_weeks", "h3_r5", "target_week", "lat", "lon", "p_fire", "p_climatology", "actual"]]
    if (PROCESSED / "risk_latest.parquet").exists():
        r = pd.read_parquet(PROCESSED / "risk_latest.parquet")
        for c in ("as_of_week", "target_week"):
            r[c] = pd.to_datetime(r[c]).dt.strftime("%Y-%m-%d")
        out["risk_forecast"] = r
    if (PROCESSED / "blind_spots_hist.parquet").exists():
        import h3

        b = pd.read_parquet(PROCESSED / "blind_spots_hist.parquet")
        b = b[pd.to_datetime(b["week"]) >= "2024-01-01"]
        if len(b):
            ll = np.array([h3.cell_to_latlng(c) for c in b["h3_r5"]])
            b = b.assign(h3=b["h3_r5"], week=pd.to_datetime(b["week"]).dt.strftime("%Y-%m-%d"), source="history",
                         message="Облачно в часы пролётов при высоком ожидаемом риске — возможны пропущенные очаги",
                         lat=ll[:, 0].round(4), lon=ll[:, 1].round(4))
            out["blind_spots"] = b[["h3", "week", "source", "expected_fire_p", "cloud_frac", "blind_score",
                                    "fire_next_week", "message", "lat", "lon"]].round(4)
    if (PROCESSED / "delivery_alerts.parquet").exists():
        a = pd.read_parquet(PROCESSED / "delivery_alerts.parquet")
        a.columns = [c.lower() for c in a.columns]
        a = a.assign(source="replay")
        keep = ["alert_id", "source", "stream", "acq_dt", "lat", "lon", "priority", "p_real", "p_escalation", "frp_sum",
                "cluster_n", "packet_hex", "latency_a_nrt_min", "latency_b_storeforward_min", "latency_c_kz_station_min",
                "latency_d1_real_relay_min", "latency_d2_relay_76e_min", "station_b", "relay_d1", "confirmed",
                "confirm_stream", "confirm_delay_min", "next_look_sat", "next_look_min"]
        out["alerts_replay"] = a.reindex(columns=keep)
    return out


def _tbl_nrt() -> dict[str, pd.DataFrame]:
    out = {}
    if (PROCESSED / "nrt_scored.parquet").exists():
        s = pd.read_parquet(PROCESSED / "nrt_scored.parquet")
        s["id"] = (s["stream"].astype(str) + "_" + s["acq_dt"].dt.strftime("%Y%m%d%H%M") + "_" +
                   s["lat"].round(4).astype(str) + "_" + s["lon"].round(4).astype(str))
        out["fire_detections_nrt"] = s.drop_duplicates("id")[[
            "id", "acq_dt", "lat", "lon", "stream", "sensor", "frp", "conf_cat", "p_fire", "keep_veg", "is_static",
            "noise_reason", "h3_r7", "h3_r5", "p_real", "p_escalation", "priority"]].astype({"stream": str, "sensor": str,
                                                                                          "conf_cat": str, "noise_reason": str,
                                                                                          "priority": str})
    if (PROCESSED / "nrt_alerts.parquet").exists():
        a = pd.read_parquet(PROCESSED / "nrt_alerts.parquet")
        if len(a):
            a = a.assign(source="nrt", stream=a["stream"].astype(str))
            out["alerts_nrt"] = a.reindex(columns=["alert_id", "source", "stream", "acq_dt", "lat", "lon", "h3_r7",
                                                   "priority", "p_real", "p_escalation", "frp_sum", "cluster_n", "packet_hex"])
    for name in ("comm_passes", "comm_blackouts", "satellite_delays", "spread_forecasts", "space_weather", "blind_spots_nrt"):
        if (PROCESSED / f"{name}.parquet").exists():
            df = pd.read_parquet(PROCESSED / f"{name}.parquet")
            df.columns = [c.lower() for c in df.columns]  # имена колонок Postgres — в нижнем регистре
            out[name] = df
    return out


def _model_registry() -> list[dict]:
    rows = []
    for name, kind, report, artifact in (
        ("ai1_edge_detector", "onboard MLP (ONNX int8)", "ai1_edge_detector_metrics.json", "edge_detector_int8.onnx"),
        ("ai2_link_predictor", "onboard MLP (ONNX)", "ai2_link_metrics.json", "link_ai.onnx"),
        ("ai3_ground_forecast", "HistGradientBoosting x4 horizons", "ai3_ground_forecast_metrics.json", "ground_forecast_h1..h4.joblib"),
    ):
        path = REPORTS_DIR / report
        if path.exists():
            rows.append({"name": name, "kind": kind, "version": "0.1.0", "metrics": json.loads(path.read_text(encoding="utf-8")),
                         "artifact": f"artifacts/models/{artifact}",
                         "trained_at": datetime.fromtimestamp(path.stat().st_mtime, tz=timezone.utc).isoformat()})
    return rows


def _reports() -> list[dict]:
    rows = []
    for path in sorted(REPORTS_DIR.glob("*.json")):
        try:
            rows.append({"key": path.stem, "payload": json.loads(path.read_text(encoding="utf-8")),
                         "updated_at": datetime.now(timezone.utc).isoformat()})
        except Exception:
            continue
    return rows


def _tbl_world() -> dict[str, pd.DataFrame]:
    """Мировой контур: регионы, страны по месяцам, регионы по годам, текущий прогноз ИИ №3 (p ≥ 0,05)."""
    import h3
    import polars as pl

    from ..world.aggregate import AGG
    from ..world.regions import region_index

    out: dict[str, pd.DataFrame] = {}
    idx = region_index()
    out["world_regions"] = pd.DataFrame([{"region_id": r["id"], "iso2": r["iso2"], "name": r["name"],
                                          "name_ru": r["names"].get("ru"), "kind": r["type"], "bbox": r["bbox"],
                                          "area_km2": r["area_km2"]} for r in idx])
    am = pl.read_parquet(AGG / "area_month.parquet")
    month_date = (pl.date(2000 + pl.col("month") // 12, pl.col("month") % 12 + 1, 1)).alias("month")
    c = am.filter(pl.col("level") == "country").select(
        pl.col("area").alias("iso2"), month_date, pl.col("hex_any").alias("hexdays"), pl.col("hex_m").alias("hex_modis"),
        pl.col("hex_v").alias("hex_viirs"), (pl.col("n_mc") + pl.col("n_vc")).alias("detections"),
        pl.col("n_r").alias("detections_raw"), pl.col("n_low").alias("low_conf"), pl.col("n_static").alias("static_src"),
        pl.col("frp_sum").round(1))
    out["world_country_month"] = c.to_pandas()
    iso = {r["id"]: r["iso2"] for r in idx}
    r = (am.filter(pl.col("level") == "region").with_columns((2000 + pl.col("month") // 12).alias("year"))
         .group_by("area", "year").agg(pl.col("hex_any").sum().alias("hexdays"), pl.col("hex_m").sum().alias("hex_modis"),
                                        pl.col("hex_v").sum().alias("hex_viirs"),
                                        (pl.col("n_mc") + pl.col("n_vc")).sum().alias("detections"),
                                        pl.col("frp_sum").sum().round(1).alias("frp_sum")))
    r = r.with_columns(pl.col("area").replace_strict(iso, default="").alias("iso2")).rename({"area": "region_id"})
    out["world_region_year"] = r.to_pandas()
    risk = AGG.parent / "forecast" / "risk_latest.parquet"
    if risk.exists():
        k = pl.read_parquet(risk).filter(pl.col("p") >= 0.05)
        cells = [h3.int_to_str(int(x)) for x in k["h3"].to_list()]
        ll = [h3.cell_to_latlng(x) for x in cells]
        out["world_risk_latest"] = pd.DataFrame({
            "h3": cells, "horizon_weeks": k["horizon"].to_numpy().astype(int), "p_fire": k["p"].round(4).to_numpy(),
            "p_climatology": k["clim"].round(4).to_numpy(), "lat": [round(a, 3) for a, _ in ll],
            "lon": [round(b, 3) for _, b in ll], "as_of": k["as_of"].to_list()})
    return out


TABLE_OF = {"alerts_replay": "alerts", "alerts_nrt": "alerts", "blind_spots_nrt": "blind_spots"}
# группа -> (ключи выгрузки, функция-построитель); строим только то, что нужно (ежечасный refresh — лёгкий)
GROUPS = {
    "reference": (["regions", "satellites", "ground_stations"], _tbl_reference),
    "history": (["burning_calendar", "calibration", "static_sources", "fire_dna", "cell_activity_monthly",
                 "season_shift_cells"], _tbl_history),
    "ai": (["time_machine", "risk_forecast", "blind_spots", "alerts_replay"], _tbl_ai),
    "nrt": (["fire_detections_nrt", "alerts_nrt", "comm_passes", "comm_blackouts", "satellite_delays",
             "spread_forecasts", "space_weather", "blind_spots_nrt"], _tbl_nrt),
    "model_registry": (["model_registry"], lambda: {"model_registry": _model_registry()}),
    "reports": (["reports"], lambda: {"reports": _reports()}),
    "world": (["world_regions", "world_country_month", "world_region_year", "world_risk_latest"], _tbl_world),
}
REPLACE_TABLES = {"world_risk_latest"}  # прогноз — целиком заменяется свежим


def _wanted(key: str, only: list[str] | None) -> bool:
    return not only or key in only or TABLE_OF.get(key, key) in only


def sync_all(only: list[str] | None = None) -> dict:
    client = SupabaseClient()
    log.info("Supabase: %s", client.ping())
    plan: dict[str, list[dict]] = {}
    for keys, builder in GROUPS.values():
        if not any(_wanted(k, only) for k in keys):
            continue
        for k, v in builder().items():
            plan[k] = v if isinstance(v, list) else _records(v)
    order = [k for keys, _ in GROUPS.values() for k in keys if k in plan]
    result = {}
    for key in order:
        table = TABLE_OF.get(key, key)
        if not _wanted(key, only):
            continue
        rows = plan[key]
        if not rows:
            continue
        try:
            n = client.upsert(table, rows, mode="replace" if table in REPLACE_TABLES else "upsert")
            result[key] = n
            log.info("Supabase %-24s %6d rows", key, n)
        except Exception as exc:
            result[key] = f"ERROR: {exc}"
            log.warning("Supabase %s: %s", key, exc)
    return result

In [ ]:
%%writefile helios/analysis.py
"""Сводка по ячейке/точке для ИИ-аналитика (Gemini) и карточки региона на сайте.

Все числа берутся из пайплайна; Gemini только формулирует текст (принцип Kiln-Watch).
"""

from __future__ import annotations

import json
from functools import lru_cache

import h3
import pandas as pd

from .config import PROCESSED, REPORTS_DIR, settings


@lru_cache(maxsize=1)
def _tables() -> dict:
    def rd(name):
        path = PROCESSED / name
        return pd.read_parquet(path) if path.exists() else None

    dna = rd("fire_dna_r7.parquet")
    if dna is not None:
        dna["h3_r5"] = [h3.cell_to_parent(c, 5) for c in dna["h3_r7"]]
    return {
        "weekly": rd("cell_weekly_r5.parquet"),
        "risk": rd("risk_latest.parquet"),
        "dna": dna,
        "shift": rd("season_shift_r4.parquet"),
        "weather": rd("weather_daily_r3.parquet"),
        "nrt": rd("nrt_scored.parquet"),
        "blind": rd("blind_spots_nrt.parquet"),
    }


def reload() -> None:
    _tables.cache_clear()


def resolve_cell(h3_cell: str | None = None, lat: float | None = None, lon: float | None = None, res: int = 5) -> str:
    if h3_cell:
        r = h3.get_resolution(h3_cell)
        return h3.cell_to_parent(h3_cell, res) if r > res else h3_cell
    if lat is None or lon is None:
        raise ValueError("нужна ячейка h3 или координаты lat/lon")
    return h3.latlng_to_cell(float(lat), float(lon), res)


def cell_stats(h3_cell: str | None = None, lat: float | None = None, lon: float | None = None) -> dict:
    t = _tables()
    c5 = resolve_cell(h3_cell, lat, lon, 5)
    clat, clon = h3.cell_to_latlng(c5)
    out: dict = {"h3_r5": c5, "center": [round(clat, 4), round(clon, 4)], "region": settings()["region"]["name"]}
    w = t["weekly"]
    if w is not None:
        g = w[w["h3_r5"] == c5].copy()
        g["year"] = pd.to_datetime(g["week"]).dt.year
        by_year = g.groupby("year")["union_celldays"].sum()
        out["burned_hexagons_by_year_last5"] = {int(y): int(v) for y, v in by_year.tail(5).items()}
        out["fire_weeks_total_2001_2025"] = int((g["union_celldays"] > 0).sum())
        if len(g):
            peak_week = pd.to_datetime(g["week"]).dt.isocalendar().week.value_counts().idxmax()
            out["typical_peak_iso_week"] = int(peak_week)
    r = t["risk"]
    if r is not None:
        rr = r[r["h3_r5"] == c5].sort_values("horizon_weeks")
        if len(rr):
            out["forecast_as_of_week"] = str(pd.Timestamp(rr["as_of_week"].iloc[0]).date())
            out["risk_probability_next_7d"] = round(float(rr["p_fire"].iloc[0]), 4)
            out["risk_by_horizon_weeks"] = {int(h): round(float(p), 4) for h, p in zip(rr["horizon_weeks"], rr["p_fire"])}
            out["climatology_probability"] = round(float(rr["p_climatology"].iloc[0]), 4)
    d = t["dna"]
    if d is not None:
        kids = d[d["h3_r5"] == c5]
        if len(kids):
            counts = kids["dna_class"].value_counts()
            top = counts.idxmax()
            row = kids[kids["dna_class"] == top].sort_values("n_det", ascending=False).iloc[0]
            out["fire_dna"] = {"class": top, "label_ru": row["label_ru"], "confidence": float(row["confidence"]),
                               "cells_by_class": {k: int(v) for k, v in counts.items()}}
    s = t["shift"]
    if s is not None:
        c4 = h3.cell_to_parent(c5, 4)
        ss = s[s["h3_r4"] == c4]
        if len(ss) and pd.notna(ss["days_per_decade"].iloc[0]):
            out["season_shift_days_per_decade"] = float(ss["days_per_decade"].iloc[0])
            out["season_shift_significant"] = bool(ss["significant"].iloc[0])
    wx = t["weather"]
    if wx is not None:
        c3 = h3.cell_to_parent(c5, settings()["h3"]["res_weather"])
        ww = wx[wx["cell"] == c3].sort_values("date").tail(30)
        if len(ww):
            last7 = ww.tail(7)
            out["weather"] = {"as_of": str(pd.Timestamp(ww["date"].iloc[-1]).date()),
                              "t2m_max": round(float(last7["T2M_MAX"].mean()), 1), "rh2m": round(float(last7["RH2M"].mean()), 0),
                              "ws10m": round(float(last7["WS10M"].mean()), 1), "precip_7d": round(float(last7["PRECTOTCORR"].sum()), 1),
                              "soil_wetness": round(float(last7["GWETTOP"].mean()), 2), "source": "NASA POWER"}
    n = t["nrt"]
    if n is not None:
        nn = n[n["h3_r5"] == c5]
        out["detections_last_7d"] = int(nn["keep_veg"].sum())
        out["static_source_detections_last_7d"] = int(nn["is_static"].sum())
        if len(nn) and "priority" in nn:
            out["max_priority_last_7d"] = str(nn["priority"].max())
    b = t["blind"]
    if b is not None:
        c3 = h3.cell_to_parent(c5, settings()["h3"]["res_weather"])
        bb = b[b["h3"] == c3]
        out["blind_spot"] = bool(len(bb) and bb["message"].iloc[0])
    rep = REPORTS_DIR / "ai3_ground_forecast_metrics.json"
    if rep.exists():
        m = json.loads(rep.read_text(encoding="utf-8"))["horizons"]["h1"]["test"]
        out["model_quality_test_2024_2025"] = {"roc_auc": m["ai"].get("roc_auc"), "pr_auc": m["ai"].get("pr_auc"),
                                               "baseline_climatology_pr_auc": m["baseline_climatology"].get("pr_auc")}
    return out

In [ ]:
%%writefile helios/pipeline.py
"""Оркестрация: история (фильтры + H3 + гармонизация), погода, аналитика, NRT.

Все результаты кладутся в data/processed (parquet) и artifacts/reports (json) —
оттуда их берут обучение ИИ, API-сервер и выгрузка в Supabase.
"""

from __future__ import annotations

from datetime import date

import h3
import numpy as np
import pandas as pd

from .config import PROCESSED, REPORTS_DIR, settings
from .utils import get_logger, save_json

log = get_logger("helios.pipeline")

SLIM_COLS = [
    "lat", "lon", "acq_dt", "sensor", "stream", "bright_mir", "bright_tir", "frp", "scan", "track",
    "conf_pct", "conf_cat", "type", "sza", "daynight_sza", "scan_deg", "pixel_km2", "glint_deg",
    "h3_r8", "h3_r7", "h3_r5", "p_fire_raw", "p_fire", "neighbors", "cross_confirmed",
    "is_low_conf", "is_glint_risk", "is_isolated", "is_saa", "is_static", "noise_reason", "keep_veg",
]


def p(name: str):
    return PROCESSED / name


# ----------------------------------------------------------------------------- история
def build_history() -> dict:
    from .apis.firms import load_country_archive
    from .processing import harmonize
    from .processing.filters import add_geometry, apply_filters, filter_report
    from .processing.persistence import build_static_registry, static_cell_set

    raw = load_country_archive()
    log.info("geometry for %d detections", len(raw))
    geo = add_geometry(raw)
    registry = build_static_registry(geo)
    registry.to_parquet(p("static_registry.parquet"), index=False)
    cells = static_cell_set(registry, buffer_k=1)
    filtered = apply_filters(geo, cells)
    filtered[SLIM_COLS].to_parquet(p("detections_filtered.parquet"), index=False)
    frep = filter_report(filtered)
    save_json(frep, REPORTS_DIR / "filter_report.json")

    res = harmonize.run(filtered)
    res["celldays"].to_parquet(p("celldays.parquet"), index=False)
    res["coverage"].assign(month=res["coverage"]["month"].astype(str)).to_parquet(p("stream_coverage.parquet"), index=False)
    res["calibration"].to_parquet(p("calibration.parquet"), index=False)
    cal = res["calendar"].copy()
    cal["month"] = cal["month"].astype(str)
    cal.to_parquet(p("calendar_monthly.parquet"), index=False)
    save_json(res["validation"], REPORTS_DIR / "harmonization_validation.json")
    save_json(res["to_ref"], REPORTS_DIR / "calibration_to_aqua.json")

    weekly = harmonize.harmonized_cell_weekly(res["celldays"], res["to_ref"])
    weekly.to_parquet(p("cell_weekly_r5.parquet"), index=False)
    log.info("history ready: %d detections kept of %d", int(filtered["keep_veg"].sum()), len(filtered))
    return {"filter_report": frep, "validation": res["validation"]}


# ----------------------------------------------------------------------------- погода
def weather_points() -> dict[str, tuple[float, float]]:
    """Центры H3 res-3 ячеек страны (≈ 0.5–1° сетка, как у MERRA-2 в NASA POWER)."""
    from .apis.boundaries import region_cells

    res = settings()["h3"]["res_weather"]
    return {c: h3.cell_to_latlng(c) for c in sorted(region_cells(res))}


def fetch_weather(start: date | None = None, end: date | None = None) -> pd.DataFrame:
    from .apis.nasa_power import fetch_grid

    cfg = settings()["history"]
    start = start or date(cfg["start_year"] if cfg["start_year"] > 2000 else 2001, 1, 1)
    end = end or date(cfg["end_year"], 12, 31)
    pts = weather_points()
    log.info("NASA POWER: %d points, %s..%s", len(pts), start, end)
    df = fetch_grid(pts, start, end, workers=4)
    if not df.empty:
        df.to_parquet(p("weather_daily_r3.parquet"), index=False)
    return df


# ----------------------------------------------------------------------------- аналитика
def run_analytics() -> dict:
    from .models import time_machine
    from .processing import blind_spots, fire_dna, season_shift

    det = pd.read_parquet(p("detections_filtered.parquet"))
    registry = pd.read_parquet(p("static_registry.parquet"))
    cd = pd.read_parquet(p("celldays.parquet"))

    dna = fire_dna.classify_cells(det, registry)
    dna.to_parquet(p("fire_dna_r7.parquet"), index=False)

    shift_region = season_shift.region_season_shift(cd)
    shift_cells = season_shift.cell_season_shift(cd)
    shift_cells.to_parquet(p("season_shift_r4.parquet"), index=False)
    save_json(shift_region, REPORTS_DIR / "season_shift_region.json")

    blind = blind_spots.historical_blind_spots()
    blind_validation = None
    if blind is not None and len(blind):
        blind_validation = blind.attrs.get("validation")
        blind.to_parquet(p("blind_spots_hist.parquet"), index=False)

    tm = time_machine.run_default_scenarios()
    summary = {
        "fire_dna_counts": dna["dna_class"].value_counts().to_dict(),
        "season_shift_verdict": shift_region.get("verdict"),
        "season_shift_cells_significant": int(shift_cells["significant"].sum()) if "significant" in shift_cells else 0,
        "blind_spot_rows": 0 if blind is None else int(len(blind)),
        "blind_spot_validation": blind_validation,
        "time_machine": tm,
    }
    save_json(summary, REPORTS_DIR / "analytics_summary.json")
    return summary


# ----------------------------------------------------------------------------- текущий год
def update_current_year() -> dict:
    """Догрузить текущий год: погода NASA POWER + пожары (Area API с MAP_KEY, иначе NRT 7 дней)."""
    from .apis.firms import fetch_area_range, fetch_open_nrt, map_key
    from .apis.nasa_power import fetch_grid
    from .processing import harmonize
    from .processing.filters import add_geometry, apply_filters
    from .processing.persistence import static_cell_set

    today = pd.Timestamp.now(tz="UTC").date()
    year_start = date(today.year, 1, 1)
    info = {"year": today.year}
    try:
        w = fetch_grid(weather_points(), year_start, today - pd.Timedelta(days=3), workers=4)
        if len(w):
            w.to_parquet(p("weather_daily_r3_current.parquet"), index=False)
            info["weather_days"] = int(w["date"].nunique())
    except Exception as exc:
        log.warning("POWER current year: %s", exc)
    if map_key():
        fires = fetch_area_range(year_start, today, quality="nrt")
        info["fires_source"] = "FIRMS Area API (MAP_KEY)"
        info["history_gap"] = None
    else:
        fires = fetch_open_nrt("7d")
        info["fires_source"] = "FIRMS open NRT 7d (без MAP_KEY)"
        info["history_gap"] = f"{year_start}..{today - pd.Timedelta(days=8)}: добавьте FIRMS_MAP_KEY для полной истории"
    if len(fires):
        from .apis.boundaries import in_region

        fires = fires[in_region(fires["lat"].to_numpy(), fires["lon"].to_numpy())].reset_index(drop=True)
        registry = pd.read_parquet(p("static_registry.parquet"))
        filt = apply_filters(add_geometry(fires), static_cell_set(registry, buffer_k=1))
        cd = harmonize.celldays(filt)
        to_ref = __import__("json").loads((REPORTS_DIR / "calibration_to_aqua.json").read_text(encoding="utf-8"))
        weekly = harmonize.harmonized_cell_weekly(cd, to_ref)
        weekly.to_parquet(p("cell_weekly_r5_current.parquet"), index=False)
        info["fires_kept"] = int(filt["keep_veg"].sum())
    save_json(info, REPORTS_DIR / "current_year_update.json")
    return info


def forecast_now() -> pd.DataFrame:
    """Текущий прогноз риска ИИ №3 (последняя неделя с данными) на 1–4 недели вперёд."""
    from .models import ground_forecast
    from .models.features import Panel

    cw = pd.read_parquet(p("cell_weekly_r5.parquet"))
    wx = pd.read_parquet(p("weather_daily_r3.parquet"))
    if p("cell_weekly_r5_current.parquet").exists():
        cw = pd.concat([cw, pd.read_parquet(p("cell_weekly_r5_current.parquet"))], ignore_index=True)
    if p("weather_daily_r3_current.parquet").exists():
        wx = pd.concat([wx, pd.read_parquet(p("weather_daily_r3_current.parquet"))], ignore_index=True)
    ground_forecast._PANEL = Panel(cw, wx)
    last_week = pd.to_datetime(cw["week"]).max()
    frames = [ground_forecast.predict_week(last_week, h) for h in ground_forecast.HORIZONS]
    risk = pd.concat(frames, ignore_index=True).drop(columns=["actual"])
    risk = risk.rename(columns={"horizon_weeks": "horizon_weeks"})
    risk.to_parquet(p("risk_latest.parquet"), index=False)
    ground_forecast._PANEL = None
    log.info("forecast_now: as of %s, %d cells", last_week.date(), risk["h3_r5"].nunique())
    return risk


# ----------------------------------------------------------------------------- NRT (оперативный контур)
NRT_EXTRAS = ("spread", "satellite_delays", "comm_plan", "space_weather", "blind_spots_now")


def run_nrt(period: str = "48h", extras: tuple[str, ...] | set[str] | None = None) -> dict:
    """Свежие пожары: FIRMS NRT -> фильтр (реестр факелов) -> ИИ №1 -> тревоги,
    + распространение по ветру, задержки спутников, план связи, космическая погода, слепые зоны.
    extras — какие дополнительные продукты считать (по умолчанию все)."""
    from .apis.boundaries import in_region
    from .apis.firms import fetch_open_nrt
    from .models import edge_detector
    from .processing.filters import add_geometry, apply_filters
    from .processing.persistence import static_cell_set

    extras = set(NRT_EXTRAS if extras is None else extras)
    out: dict = {}
    nrt = fetch_open_nrt(period)
    if len(nrt):
        nrt = nrt[in_region(nrt["lat"].to_numpy(), nrt["lon"].to_numpy())].reset_index(drop=True)
    if len(nrt):
        registry = pd.read_parquet(p("static_registry.parquet"))
        filt = apply_filters(add_geometry(nrt), static_cell_set(registry, buffer_k=1))
        scored = edge_detector.score_detections(filt)
        scored.to_parquet(p("nrt_scored.parquet"), index=False)
        alerts = edge_detector.make_alerts(scored)
        alerts.to_parquet(p("nrt_alerts.parquet"), index=False)
        out.update({
            "detections": int(len(filt)), "kept_veg": int(filt["keep_veg"].sum()), "static": int(filt["is_static"].sum()),
            "by_stream": filt["stream"].astype(str).value_counts().to_dict(),
            "alerts": alerts["priority"].astype(str).value_counts().to_dict() if len(alerts) else {},
            "latest_acq": str(filt["acq_dt"].max()),
        })
        if "spread" in extras:
            _safe(out, "spread", lambda: _spread_for(scored))
    else:
        log.warning("NRT: нет свежих точек в регионе")
        out["detections"] = 0
    for key, fn in (("satellite_delays", _satellite_delays), ("comm_plan", _comm_plan),
                    ("space_weather", _space_weather), ("blind_spots_now", _blind_now)):
        if key in extras:
            _safe(out, key, fn)
    out["computed_at"] = pd.Timestamp.now(tz="UTC").isoformat()
    save_json(out, REPORTS_DIR / "nrt_summary.json")
    log.info("NRT: %s", {k: v for k, v in out.items() if k != "comm_plan"})
    return out


# ----------------------------------------------------------------------------- автообновление (каждый час)
NRT_SYNC_TABLES = ["fire_detections_nrt", "alerts_nrt", "spread_forecasts", "satellite_delays", "space_weather",
                   "comm_passes", "comm_blackouts", "blind_spots_nrt", "reports"]


def _age_hours(path) -> float:
    import time

    return (time.time() - path.stat().st_mtime) / 3600 if path.exists() else 1e9


def refresh(period: str = "48h", forecast_every_h: float = 24.0, comm_every_h: float = 3.0) -> dict:
    """Ежечасное автообновление: свежие пожары -> фильтр -> ИИ №1 -> тревоги -> Supabase.
    Тяжёлые продукты пересчитываются реже: план связи и слепые зоны — раз в 3 ч,
    погода/пожары текущего года и прогноз риска ИИ №3 — раз в сутки."""
    import os
    import time

    from .db.supabase_sync import sync_all

    lock = PROCESSED / ".refresh.lock"
    if lock.exists() and _age_hours(lock) < 2:
        log.warning("refresh: предыдущий запуск ещё идёт (lock %s) — пропускаю", lock)
        return {"skipped": "locked"}
    lock.write_text(f"{os.getpid()} {time.time():.0f}", encoding="utf-8")
    t0 = time.time()
    out: dict = {"started_at": pd.Timestamp.now(tz="UTC").isoformat()}
    try:
        extras = {"spread", "satellite_delays", "space_weather"}
        if _age_hours(p("comm_passes.parquet")) >= comm_every_h:
            extras.add("comm_plan")
        if _age_hours(p("blind_spots_nrt.parquet")) >= comm_every_h:
            extras.add("blind_spots_now")
        tables = list(NRT_SYNC_TABLES)
        if _age_hours(p("risk_latest.parquet")) >= forecast_every_h:
            _safe(out, "current_year", update_current_year)
            _safe(out, "forecast_cells", lambda: int(forecast_now()["h3_r5"].nunique()))
            tables.append("risk_forecast")
            extras.add("blind_spots_now")  # слепые зоны зависят от свежего прогноза риска
            from .frontend_export import global_overview

            _safe(out, "globe_overview", lambda: bool(global_overview()))  # глобус фронтенда: пожары мира за сутки
            from .config import MODELS_DIR

            if (MODELS_DIR / "world_forecast_oper_h1.txt").exists():  # прогноз ИИ №3 для всего мира (лента NRT 7 дней)
                from .world.forecast import forecast_now as world_forecast_now

                _safe(out, "world_forecast", lambda: world_forecast_now()["rows"])
                tables.append("world_risk_latest")
        _safe(out, "nrt", lambda: run_nrt(period, extras))
        out["extras"] = sorted(extras)
        _safe(out, "sync", lambda: sync_all(only=tables))
    finally:
        lock.unlink(missing_ok=True)
    out["seconds"] = round(time.time() - t0, 1)
    save_json(out, REPORTS_DIR / "refresh_last.json")
    log.info("refresh done in %.0f s", out["seconds"])
    return out


def _safe(out: dict, key: str, fn) -> None:
    try:
        out[key] = fn()
    except Exception as exc:
        log.warning("%s: %s", key, exc)
        out[key] = f"error: {exc}"


def _spread_for(scored: pd.DataFrame):
    from .processing.spread import forecast_spread

    k = scored[scored["keep_veg"]]
    if k.empty:
        return 0
    k = k.assign(h3_r6=[h3.cell_to_parent(c, 6) for c in k["h3_r7"]])
    cl = k.groupby("h3_r6").agg(lat=("lat", "mean"), lon=("lon", "mean"), frp_sum=("frp", "sum"), n=("frp", "size")).reset_index()
    cl = cl.sort_values("frp_sum", ascending=False).head(40).rename(columns={"h3_r6": "cluster_id"})
    sp = forecast_spread(cl)
    if len(sp):
        sp["computed_at"] = pd.Timestamp.now(tz="UTC").floor("min")
        sp.to_parquet(p("spread_forecasts.parquet"), index=False)
    return int(len(sp))


def _satellite_delays():
    from .models.ground_forecast import satellite_delay_report

    rep = satellite_delay_report(hours=24)
    if len(rep):
        rep["computed_at"] = pd.Timestamp.now(tz="UTC").floor("min")
        rep.to_parquet(p("satellite_delays.parquet"), index=False)
        return rep[["stream", "median_latency_min", "status"]].to_dict("records")
    return []


def _comm_plan():
    from .comms.scheduler import plan

    res = plan(hours=24)
    passes = res["passes"]
    if len(passes):
        cols = {c: c.lower() for c in passes.columns}
        passes = passes.rename(columns=cols)
        keep = ["satellite", "station", "aos", "los", "max_el", "duration_min", "s_availability", "x_availability",
                "ka_availability", "opt_availability", "recommended_channel", "expected_volume_gb"]
        passes.reindex(columns=keep).to_parquet(p("comm_passes.parquet"), index=False)
    bl = pd.DataFrame(res["blackouts"])
    if len(bl):
        bl["computed_at"] = pd.Timestamp.now(tz="UTC").floor("min")
        bl.to_parquet(p("comm_blackouts.parquet"), index=False)
    return {"passes": int(len(passes)),
            "blackouts": [{k: b[k] for k in ("satellite", "minutes_without_any_link", "longest_gap_min",
                                             "minutes_without_link_with_real_relays", "minutes_without_link_with_relay_76E")}
                          for b in res["blackouts"]]}


def _space_weather():
    from .apis.space_weather import current_conditions

    cur = current_conditions()
    df = pd.DataFrame([{"time": pd.Timestamp.now(tz="UTC").floor("min"), "kp": cur.get("kp"), "xray_flux": cur.get("xray_flux"),
                        "flare_class": cur.get("flare_class"), "f107": cur.get("f107")}])
    df.to_parquet(p("space_weather.parquet"), index=False)
    return cur


def _blind_now():
    from .processing.blind_spots import recent_blind_spots

    risk = pd.read_parquet(p("risk_latest.parquet")) if p("risk_latest.parquet").exists() else None
    if risk is not None:
        risk = risk[risk["horizon_weeks"] == 1]
    seen = set()
    if p("nrt_scored.parquet").exists():
        s = pd.read_parquet(p("nrt_scored.parquet"))
        seen = {h3.cell_to_parent(c, settings()["h3"]["res_weather"]) for c in s.loc[s["keep_veg"], "h3_r7"]}
    b = recent_blind_spots(risk, seen)
    if len(b):
        b2 = b.assign(h3=b["h3_r3"], week=pd.Timestamp.now(tz="UTC").strftime("%Y-%m-%d"), source="nrt",
                      expected_fire_p=b["expected_fire_p"], cloud_frac=b["cloud_at_overpasses"],
                      fire_next_week=None)[["h3", "week", "source", "expected_fire_p", "cloud_frac", "blind_score",
                                            "fire_next_week", "message", "lat", "lon"]]
        b2.to_parquet(p("blind_spots_nrt.parquet"), index=False)
    return {"cells": int(len(b)), "blind": int(b["is_blind"].sum()) if len(b) else 0}

In [ ]:
%%writefile helios/cli.py
"""Командная строка HELIOS-SHIELD.

    python -m helios.cli check-apis        # проверить все API и ключи
    python -m helios.cli fetch-history     # скачать 26 лет FIRMS (MODIS+VIIRS) по стране
    python -m helios.cli build-history     # фильтрация шума + H3 + гармонизация + календарь
    python -m helios.cli fetch-weather     # NASA POWER (погода + облачность) по сетке H3
    python -m helios.cli train-edge        # ИИ №1 (борт): критичность обнаружения
    python -m helios.cli train-link        # ИИ №2 (борт): прогноз потери связи
    python -m helios.cli train-ground      # ИИ №3 (Земля): прогноз пожарного риска
    python -m helios.cli analytics         # Fire DNA, сдвиг сезона, слепые зоны, Time Machine
    python -m helios.cli update-current    # текущий год: погода POWER + пожары (MAP_KEY или NRT 7д)
    python -m helios.cli forecast          # текущий прогноз риска ИИ №3 на 1–4 недели
    python -m helios.cli simulate-delivery # умная доставка алертов (окна связи, ретранслятор)
    python -m helios.cli export-frontend   # данные в формате фронтенда Kyzyl Aspan (API /v1)
    python -m helios.cli nrt               # свежие пожары (NRT) -> фильтр -> ИИ №1 -> алерты
    python -m helios.cli refresh           # ежечасное автообновление: NRT -> ИИ №1 -> Supabase (лог: data/logs/refresh.log)
    python -m helios.cli sync-supabase     # выгрузка результатов в Supabase
    python -m helios.cli all               # всё по порядку
"""

from __future__ import annotations

import argparse
import sys
import time

from .config import ensure_dirs
from .utils import get_logger

log = get_logger("helios.cli")


def _run(step: str, fn, *args, **kwargs):
    t0 = time.time()
    log.info("=== %s ===", step)
    result = fn(*args, **kwargs)
    log.info("=== %s: готово за %.1f c ===", step, time.time() - t0)
    return result


def cmd_check_apis(_):
    from .apis.health import check_all

    report = check_all()
    for name, info in report.items():
        mark = "OK " if info.get("ok") else "-- "
        print(f"{mark}{name:22s} {info.get('detail', '')}")


def cmd_fetch_history(args):
    from .apis.firms import download_country_archive

    _run("FIRMS archive", download_country_archive, workers=args.workers)


def cmd_build_history(_):
    from .pipeline import build_history

    _run("History (filters + H3 + harmonization)", build_history)


def cmd_fetch_weather(_):
    from .pipeline import fetch_weather

    _run("NASA POWER weather", fetch_weather)


def cmd_train_edge(_):
    from .models.edge_detector import train

    _run("AI #1 edge detector", train)


def cmd_train_link(_):
    from .models.link_ai import train

    _run("AI #2 link predictor", train)


def cmd_train_ground(_):
    from .models.ground_forecast import train

    _run("AI #3 ground forecast", train)


def cmd_analytics(_):
    from .pipeline import run_analytics

    _run("Analytics", run_analytics)


def cmd_update_current(_):
    from .pipeline import update_current_year

    _run("Current year (POWER + FIRMS)", update_current_year)


def cmd_forecast(_):
    from .pipeline import forecast_now

    _run("AI #3 forecast now", forecast_now)


def cmd_export_frontend(_):
    from .frontend_export import export_all

    _run("Export for Kyzyl Aspan frontend", export_all)


def cmd_simulate_delivery(_):
    from .comms.delivery import run_delivery_simulation

    _run("Smart alert delivery", run_delivery_simulation)


def cmd_nrt(args):
    from .pipeline import run_nrt

    _run("NRT pipeline", run_nrt, period=args.period)


def cmd_refresh(args):
    """Ежечасное автообновление (его запускает Планировщик заданий Windows)."""
    import logging
    from logging.handlers import RotatingFileHandler

    from .config import DATA
    from .pipeline import refresh

    log_dir = DATA / "logs"
    log_dir.mkdir(parents=True, exist_ok=True)
    handler = RotatingFileHandler(log_dir / "refresh.log", maxBytes=2_000_000, backupCount=5, encoding="utf-8")
    handler.setFormatter(logging.Formatter("%(asctime)s %(levelname)s %(name)s: %(message)s"))
    logging.getLogger().addHandler(handler)
    _run("Hourly refresh", refresh, period=args.period)


def cmd_sync(args):
    from .db.supabase_sync import sync_all

    _run("Supabase sync", sync_all, only=args.only)


def cmd_all(args):
    for fn in (cmd_fetch_history, cmd_build_history, cmd_fetch_weather, cmd_train_edge,
               cmd_train_link, cmd_train_ground, cmd_analytics, cmd_update_current, cmd_forecast,
               cmd_simulate_delivery, cmd_export_frontend):
        try:
            fn(args)
        except Exception as exc:  # один упавший шаг (например, сеть) не должен ронять весь прогон
            log.exception("шаг %s упал: %s", fn.__name__, exc)
    cmd_nrt(args)
    cmd_sync(args)


def main(argv: list[str] | None = None) -> int:
    import os

    if sys.stdout is None or sys.stderr is None:  # запуск через pythonw (Планировщик, без окна консоли)
        devnull = open(os.devnull, "w", encoding="utf-8")
        sys.stdout = sys.stdout or devnull
        sys.stderr = sys.stderr or devnull
    ensure_dirs()
    p = argparse.ArgumentParser(prog="helios", description="HELIOS-SHIELD pipeline")
    sub = p.add_subparsers(dest="cmd", required=True)
    commands = {
        "check-apis": cmd_check_apis,
        "fetch-history": cmd_fetch_history,
        "build-history": cmd_build_history,
        "fetch-weather": cmd_fetch_weather,
        "train-edge": cmd_train_edge,
        "train-link": cmd_train_link,
        "train-ground": cmd_train_ground,
        "analytics": cmd_analytics,
        "update-current": cmd_update_current,
        "forecast": cmd_forecast,
        "simulate-delivery": cmd_simulate_delivery,
        "export-frontend": cmd_export_frontend,
        "nrt": cmd_nrt,
        "refresh": cmd_refresh,
        "sync-supabase": cmd_sync,
        "all": cmd_all,
    }
    for name, fn in commands.items():
        sp = sub.add_parser(name)
        sp.set_defaults(func=fn)
        sp.add_argument("--workers", type=int, default=4)
        sp.add_argument("--period", default="48h", choices=["24h", "48h", "7d"])
        sp.add_argument("--only", nargs="*", default=None, help="таблицы для sync-supabase")
    args = p.parse_args(argv)
    args.func(args)
    return 0


if __name__ == "__main__":
    sys.exit(main())

### Весь мир: архив всех стран, регионы, гексагоны, ИИ на мировых данных

In [ ]:
%%writefile helios/world/__init__.py
"""Мировой контур HELIOS: архив NASA FIRMS по всем странам (MODIS с 2011 г., VIIRS с 2012 г.),
регионы (admin-1), гексагоны H3 разных разрешений и обучение ИИ на мировых данных."""

In [ ]:
%%writefile helios/world/fetch.py
"""Мировой архив NASA FIRMS: годовые zip «все страны», параллельная докачка частями.

Сервер FIRMS отдаёт ~80 КБ/с на одно соединение, но поддерживает HTTP Range. Поэтому файл качается
кусками по 4 МБ в несколько потоков; у каждого куска свой срок, повторы с паузой, а список готовых
кусков пишется в `.state` — после обрыва или перезапуска докачка продолжается с того же места.

    python -m helios.world.fetch              # MODIS 2011–2025 + VIIRS S-NPP 2012–2025
    python -m helios.world.fetch --noaa20     # + VIIRS NOAA-20 2018–2025
"""

from __future__ import annotations

import argparse
import json
import queue
import threading
import time
import zipfile
from pathlib import Path

import requests

from ..config import RAW
from ..utils import get_logger

log = get_logger("helios.world.fetch")

BASE = "https://firms.modaps.eosdis.nasa.gov/data/country/zips"
GLOBAL_DIR = RAW / "global"
CHUNK = 4 * 1024 * 1024
CHUNK_DEADLINE_S = 240.0  # кусок, который «капает» дольше, перезапрашиваем новым соединением
UA = {"User-Agent": "HELIOS-SHIELD research pipeline (NASA Space Apps)"}


def jobs(noaa20: bool = False) -> list[tuple[str, int]]:
    out = [("modis", y) for y in range(2011, 2026)]
    out += [("modis", y) for y in range(2010, 1999, -1)]  # 2000–2010: только MODIS (история «26 лет»)
    out += [("viirs-snpp", y) for y in range(2025, 2011, -1)]
    if noaa20:
        out += [("viirs-jpss1", y) for y in range(2025, 2017, -1)]
    return out


def zip_path(sensor: str, year: int) -> Path:
    return GLOBAL_DIR / f"{sensor}_{year}_all_countries.zip"


def _content_length(url: str) -> int | None:
    for k in range(8):
        try:
            r = requests.head(url, timeout=30, allow_redirects=True, headers=UA)
            if r.status_code == 404:
                return None
            r.raise_for_status()
            return int(r.headers["Content-Length"])
        except Exception as e:  # сеть / 5xx — ждём и повторяем
            log.warning("HEAD %s: %s", url.rsplit("/", 1)[-1], e)
            time.sleep(min(60, 2 ** k))
    raise RuntimeError(f"HEAD не удался: {url}")


def _fetch_range(sess: requests.Session, url: str, a: int, b: int) -> bytes:
    t0 = time.time()
    buf = bytearray()
    with sess.get(url, headers={**UA, "Range": f"bytes={a}-{b}"}, timeout=(20, 60), stream=True) as r:
        if r.status_code != 206:
            raise IOError(f"HTTP {r.status_code}")
        for block in r.iter_content(256 * 1024):
            buf.extend(block)
            if time.time() - t0 > CHUNK_DEADLINE_S:
                raise TimeoutError("кусок качается слишком медленно")
    if len(buf) != b - a + 1:
        raise IOError(f"получено {len(buf)} из {b - a + 1} байт")
    return bytes(buf)


def download(sensor: str, year: int, threads: int = 8) -> Path | None:
    final = zip_path(sensor, year)
    if final.exists():
        return final
    GLOBAL_DIR.mkdir(parents=True, exist_ok=True)
    url = f"{BASE}/{final.name}"
    size = _content_length(url)
    if size is None:
        log.warning("на сервере нет %s", final.name)
        return None
    part, state_f = final.with_name(final.name + ".part"), final.with_name(final.name + ".state")
    n = (size + CHUNK - 1) // CHUNK
    done: set[int] = set()
    if part.exists() and part.stat().st_size == size and state_f.exists():
        st = json.loads(state_f.read_text())
        if st.get("size") == size:
            done = set(st["done"])
    else:
        with open(part, "wb") as f:
            f.truncate(size)
    todo: queue.Queue[int] = queue.Queue()
    for i in range(n):
        if i not in done:
            todo.put(i)
    lock = threading.Lock()
    failed: list[int] = []
    got = [0]
    t0 = time.time()
    log.info("%s: %.0f МБ, кусков %d (готово %d)", final.name, size / 2**20, n, len(done))

    with open(part, "r+b") as fh:

        def worker() -> None:
            sess = requests.Session()
            while True:
                try:
                    i = todo.get_nowait()
                except queue.Empty:
                    return
                a, b = i * CHUNK, min(size, (i + 1) * CHUNK) - 1
                for k in range(80):
                    try:
                        data = _fetch_range(sess, url, a, b)
                        with lock:
                            fh.seek(a)
                            fh.write(data)
                            done.add(i)
                            got[0] += len(data)
                            if len(done) % 4 == 0 or len(done) == n:
                                fh.flush()
                                state_f.write_text(json.dumps({"size": size, "done": sorted(done)}))
                        break
                    except Exception:
                        sess.close()
                        sess = requests.Session()
                        time.sleep(min(45, 1.5 ** min(k, 9)))
                else:
                    with lock:
                        failed.append(i)

        workers = [threading.Thread(target=worker, daemon=True) for _ in range(threads)]
        for w in workers:
            w.start()
        last = time.time()
        while any(w.is_alive() for w in workers):
            time.sleep(2)
            if time.time() - last > 60:
                last = time.time()
                el = max(time.time() - t0, 1)
                log.info("  %s: %d/%d кусков, %.2f МБ/с", final.name, len(done), n, got[0] / el / 2**20)
        fh.flush()
        state_f.write_text(json.dumps({"size": size, "done": sorted(done)}))

    if failed or len(done) != n:
        raise RuntimeError(f"{final.name}: не скачаны куски {sorted(set(range(n)) - done)[:10]}…")
    with zipfile.ZipFile(part) as z:  # центральный каталог читается — архив цел
        bad = z.testzip()
        if bad:
            raise RuntimeError(f"{final.name}: битый файл внутри архива {bad}")
    part.replace(final)
    state_f.unlink(missing_ok=True)
    el = time.time() - t0
    log.info("готово %s за %.0f с (%.2f МБ/с)", final.name, el, got[0] / max(el, 1) / 2**20)
    return final


def _job(sensor: str, year: int, threads: int) -> Path | None:
    for attempt in range(3):
        try:
            return download(sensor, year, threads)
        except Exception as e:
            log.error("%s %d: %s (попытка %d)", sensor, year, e, attempt + 1)
            time.sleep(30)
    return None


def main() -> None:
    from concurrent.futures import ThreadPoolExecutor

    ap = argparse.ArgumentParser()
    ap.add_argument("--noaa20", action="store_true")
    ap.add_argument("--threads", type=int, default=8, help="соединений на файл")
    ap.add_argument("--files", type=int, default=2, help="файлов одновременно")
    args = ap.parse_args()
    with ThreadPoolExecutor(args.files) as ex:
        list(ex.map(lambda j: _job(j[0], j[1], args.threads), jobs(args.noaa20)))
    log.info("загрузка мирового архива завершена")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile helios/world/ingest.py
"""Годовой zip FIRMS «все страны» → компактный Parquet: одна строка = одно обнаружение.

Схема (≈18 байт на точку после zstd):
    lat, lon       float32   центр пикселя
    day            uint16    дни с 2000-01-01
    hhmm           uint16    время съёмки UTC
    sat            uint8     0 Terra, 1 Aqua, 2 Suomi-NPP, 3 NOAA-20, 4 NOAA-21
    conf           uint8     0 low, 1 nominal, 2 high (MODIS: <30 / 30–79 / ≥80)
    frp            float32   мощность излучения, МВт
    bt, bt2        float32   яркостная температура (MODIS 4/11 мкм, VIIRS I4/I5), К
    scan, track    float32   размер пикселя поперёк/вдоль трассы, км
    night          uint8     1 — ночной пролёт
    type           uint8     поле FIRMS: 0 растительность, 1 вулкан, 2 статичный источник, 3 офшор
    country        uint16    индекс страны FIRMS (world/countries_firms.json)
    h3             uint64    ячейка H3 res 8 (~0,74 км²); строки отсортированы по ней

    python -m helios.world.ingest            # все скачанные zip, которых ещё нет в Parquet
    python -m helios.world.ingest --watch    # ждать новые zip от загрузчика
"""

from __future__ import annotations

import argparse
import json
import re
import threading
import time
import zipfile
from datetime import date
from pathlib import Path

import numpy as np
import polars as pl
from h3ronpy import vector as h3v

from ..config import PROCESSED
from ..utils import get_logger
from .fetch import GLOBAL_DIR

log = get_logger("helios.world.ingest")

WORLD = PROCESSED / "world"
DET_DIR = WORLD / "det"
COUNTRIES_JSON = WORLD / "countries_firms.json"
EPOCH = date(2000, 1, 1)
SAT_CODE = {"Terra": 0, "T": 0, "Aqua": 1, "A": 1, "N": 2, "NPP": 2, "Suomi NPP": 2,
            "1": 3, "N20": 3, "NOAA-20": 3, "J1": 3, "2": 4, "N21": 4, "NOAA-21": 4, "J2": 4}
_lock = threading.Lock()


def _country_index(name: str) -> int:
    """Стабильный индекс страны (имя из файла FIRMS) — один список на все годы и сенсоры."""
    with _lock:
        names: list[str] = json.loads(COUNTRIES_JSON.read_text(encoding="utf-8")) if COUNTRIES_JSON.exists() else []
        if name not in names:
            names.append(name)
            COUNTRIES_JSON.parent.mkdir(parents=True, exist_ok=True)
            COUNTRIES_JSON.write_text(json.dumps(names, ensure_ascii=False, indent=0), encoding="utf-8")
        return names.index(name)


def parquet_path(zip_file: Path) -> Path:
    m = re.match(r"(.+)_(\d{4})_all_countries\.zip", zip_file.name)
    assert m, zip_file.name
    return DET_DIR / f"{m.group(1)}_{m.group(2)}.parquet"


def _read_member(z: zipfile.ZipFile, name: str, viirs: bool) -> pl.DataFrame | None:
    raw = z.read(name)
    if len(raw) < 64:
        return None
    bt, bt2 = ("bright_ti4", "bright_ti5") if viirs else ("brightness", "bright_t31")
    df = pl.read_csv(raw, columns=["latitude", "longitude", bt, "scan", "track", "acq_date", "acq_time",
                                   "satellite", "confidence", bt2, "frp", "daynight", "type"],
                     schema_overrides={"acq_time": pl.Utf8, "satellite": pl.Utf8, "confidence": pl.Utf8,
                                       "type": pl.Utf8}, ignore_errors=True)
    if df.is_empty():
        return None
    country = re.sub(r"^.*?_\d{4}_", "", Path(name).stem)
    conf = pl.col("confidence").str.strip_chars().str.to_lowercase()
    if viirs:
        conf_code = pl.when(conf == "h").then(2).when(conf == "n").then(1).otherwise(0)
    else:
        c = conf.cast(pl.Float32, strict=False).fill_null(0)
        conf_code = pl.when(c >= 80).then(2).when(c >= 30).then(1).otherwise(0)
    return df.select(
        pl.col("latitude").cast(pl.Float32).alias("lat"),
        pl.col("longitude").cast(pl.Float32).alias("lon"),
        (pl.col("acq_date").str.to_date("%Y-%m-%d", strict=False) - pl.lit(EPOCH)).dt.total_days().cast(pl.UInt16).alias("day"),
        pl.col("acq_time").str.strip_chars().cast(pl.UInt16, strict=False).fill_null(0).alias("hhmm"),
        pl.col("satellite").str.strip_chars().replace_strict(SAT_CODE, default=255, return_dtype=pl.UInt8).alias("sat"),
        conf_code.cast(pl.UInt8).alias("conf"),
        pl.col("frp").cast(pl.Float32, strict=False).fill_null(0).alias("frp"),
        pl.col(bt).cast(pl.Float32, strict=False).alias("bt"),
        pl.col(bt2).cast(pl.Float32, strict=False).alias("bt2"),
        pl.col("scan").cast(pl.Float32, strict=False).alias("scan"),
        pl.col("track").cast(pl.Float32, strict=False).alias("track"),
        (pl.col("daynight").str.strip_chars() == "N").cast(pl.UInt8).alias("night"),
        pl.col("type").cast(pl.Float32, strict=False).fill_null(0).cast(pl.UInt8).alias("type"),
        pl.lit(_country_index(country), dtype=pl.UInt16).alias("country"),
    ).filter(pl.col("day").is_not_null())


def ingest_zip(zip_file: Path) -> Path:
    out = parquet_path(zip_file)
    if out.exists():
        return out
    t0 = time.time()
    viirs = zip_file.name.startswith("viirs")
    frames = []
    with zipfile.ZipFile(zip_file) as z:
        for name in z.namelist():
            if name.endswith(".csv"):
                df = _read_member(z, name, viirs)
                if df is not None:
                    frames.append(df)
    df = pl.concat(frames, how="vertical")
    cells = h3v.coordinates_to_cells(df["lat"].to_numpy().astype(np.float64), df["lon"].to_numpy().astype(np.float64), 8)
    df = df.with_columns(pl.Series("h3", np.asarray(cells.to_numpy()), dtype=pl.UInt64)).sort("h3")
    DET_DIR.mkdir(parents=True, exist_ok=True)
    tmp = out.with_suffix(".tmp")
    df.write_parquet(tmp, compression="zstd", compression_level=3, row_group_size=1_000_000, statistics=True)
    tmp.replace(out)
    log.info("%s → %s: %d точек, %.0f МБ, %.0f с", zip_file.name, out.name, df.height, out.stat().st_size / 2**20,
             time.time() - t0)
    return out


def pending() -> list[Path]:
    return [z for z in sorted(GLOBAL_DIR.glob("*_all_countries.zip")) if not parquet_path(z).exists()]


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--watch", action="store_true", help="ждать новые zip, пока работает загрузчик")
    ap.add_argument("--delete-zip", action="store_true", help="удалять zip после конвертации")
    args = ap.parse_args()
    idle = 0
    while True:
        todo = pending()
        for z in todo:
            try:
                ingest_zip(z)
                if args.delete_zip:
                    z.unlink()
            except Exception as e:
                log.exception("ошибка %s: %s", z.name, e)
        if not args.watch:
            break
        busy = any(GLOBAL_DIR.glob("*.part"))
        idle = 0 if (todo or busy) else idle + 1
        if idle > 20:  # 10 минут без новых файлов и без активной загрузки
            break
        time.sleep(30)
    log.info("конвертация завершена: %d файлов Parquet", len(list(DET_DIR.glob("*.parquet"))))


if __name__ == "__main__":
    main()

In [ ]:
%%writefile helios/world/regions.py
"""Регионы мира (admin-1, Natural Earth 1:10m, общественное достояние).

* `build_regions()` — индекс регионов (id, страна, названия на 7 языках, рамка, площадь) и упрощённая
  геометрия по странам для карты: `data/processed/world/regions/<ISO2>.geojson`.
* `assign_points(lat, lon)` — к какому региону относится точка (центр гексагона): «внутри полигона»,
  а для прибрежных пикселей — ближайший регион не дальше ~25 км.
"""

from __future__ import annotations

import json
from functools import lru_cache

import numpy as np
import shapely
from shapely.geometry import mapping, shape

from ..config import RAW
from ..utils import get_logger
from .ingest import WORLD

log = get_logger("helios.world.regions")

NE_ADM1 = RAW / "boundaries" / "ne_10m_admin_1_states_provinces.geojson"
NE_ADM0 = RAW / "boundaries" / "ne_10m_admin_0_countries.geojson"
REGIONS_DIR = WORLD / "regions"
INDEX_JSON = REGIONS_DIR / "index.json"
LANGS = ("en", "ru", "zh", "de", "es", "fr", "pt")


def download_boundaries() -> None:
    """Natural Earth 1:10m (общественное достояние): страны и регионы admin-1."""
    import requests

    base = "https://raw.githubusercontent.com/nvkelso/natural-earth-vector/master/geojson"
    for path in (NE_ADM1, NE_ADM0):
        if path.exists():
            continue
        path.parent.mkdir(parents=True, exist_ok=True)
        r = requests.get(f"{base}/{path.name}", timeout=600)
        r.raise_for_status()
        path.write_bytes(r.content)


def _iso2_fix() -> dict[str, str]:
    """adm0_a3 → ISO2 (у Natural Earth для Франции, Норвегии, Косово и др. iso_a2 = -99)."""
    d = json.loads(NE_ADM0.read_text(encoding="utf-8"))
    out = {}
    for f in d["features"]:
        p = f["properties"]
        code = p.get("ISO_A2_EH") or p.get("ISO_A2")
        if code and len(code) == 2 and code.isalpha():
            for k in ("ADM0_A3", "SOV_A3", "GU_A3", "ADM0_A3_US"):
                if p.get(k):
                    out.setdefault(p[k], code)
    return out


def area_km2(g) -> float:
    """Площадь по синусоидальной (равновеликой) проекции: x = lon·cos(lat), y = lat."""
    def f(xy):
        out = xy.copy()
        out[:, 0] = xy[:, 0] * np.cos(np.radians(xy[:, 1])) * 111.32
        out[:, 1] = xy[:, 1] * 110.95
        return out
    return float(shapely.area(shapely.transform(g, f)))


def _round_coords(geom: dict, nd: int = 4) -> dict:
    def r(c):
        if isinstance(c[0], (float, int)):
            return [round(c[0], nd), round(c[1], nd)]
        return [r(x) for x in c]

    return {"type": geom["type"], "coordinates": r(geom["coordinates"])}


def build_regions() -> list[dict]:
    """Индекс + геометрия по странам. Возвращает индекс (список регионов)."""
    download_boundaries()
    fix = _iso2_fix()
    d = json.loads(NE_ADM1.read_text(encoding="utf-8"))
    REGIONS_DIR.mkdir(parents=True, exist_ok=True)
    index, by_country = [], {}
    for i, f in enumerate(d["features"]):
        p = f["properties"]
        iso2 = p.get("iso_a2") if (p.get("iso_a2") or "").isalpha() else fix.get(p.get("adm0_a3"), "")
        if not iso2:
            continue
        g = shape(f["geometry"])
        if g.is_empty:
            continue
        g = shapely.make_valid(g)
        w, s, e, n = g.bounds
        names = {lang: (p.get(f"name_{lang}") or "").strip() for lang in LANGS}
        names["en"] = names["en"] or (p.get("name") or "").strip()
        rid = p.get("adm1_code") or f"{iso2}-{i}"
        rec = {
            "idx": len(index), "id": rid, "iso2": iso2.upper(), "name": names["en"] or rid,
            "names": {k: v for k, v in names.items() if v}, "type": p.get("type_en") or p.get("type") or "",
            "bbox": [round(w, 3), round(s, 3), round(e, 3), round(n, 3)],
            "area_km2": round(area_km2(g), 1),
            "center": [round(float(p.get("longitude") or (w + e) / 2), 3), round(float(p.get("latitude") or (s + n) / 2), 3)],
        }
        index.append(rec)
        # для карты: упрощение с допуском по размеру региона (≈1–2 км у мелких, до ~5 км у огромных)
        tol = float(np.clip(max(e - w, n - s) / 400, 0.004, 0.05))
        gs = shapely.simplify(g, tol, preserve_topology=True)
        by_country.setdefault(rec["iso2"], []).append({
            "type": "Feature", "id": rec["idx"],
            "properties": {"id": rid, "idx": rec["idx"], "name": rec["name"], "names": rec["names"], "type": rec["type"]},
            "geometry": _round_coords(mapping(gs)),
        })
        _geoms_cache.append(g)
    for iso2, feats in by_country.items():
        (REGIONS_DIR / f"{iso2}.geojson").write_text(
            json.dumps({"type": "FeatureCollection", "features": feats}, ensure_ascii=False, separators=(",", ":")),
            encoding="utf-8")
    INDEX_JSON.write_text(json.dumps(index, ensure_ascii=False, separators=(",", ":")), encoding="utf-8")
    log.info("регионы: %d в %d странах", len(index), len(by_country))
    return index


_geoms_cache: list = []


@lru_cache(maxsize=1)
def region_index() -> list[dict]:
    if not INDEX_JSON.exists():
        return build_regions()
    return json.loads(INDEX_JSON.read_text(encoding="utf-8"))


@lru_cache(maxsize=1)
def _tree() -> tuple[shapely.STRtree, list]:
    idx = region_index()
    fix = _iso2_fix()
    d = json.loads(NE_ADM1.read_text(encoding="utf-8"))
    by_id = {r["id"]: r["idx"] for r in idx}
    geoms: list = [None] * len(idx)
    for i, f in enumerate(d["features"]):
        p = f["properties"]
        iso2 = p.get("iso_a2") if (p.get("iso_a2") or "").isalpha() else fix.get(p.get("adm0_a3"), "")
        rid = p.get("adm1_code") or f"{iso2}-{i}"
        if rid in by_id:
            geoms[by_id[rid]] = shapely.make_valid(shape(f["geometry"]))
    return shapely.STRtree(geoms), geoms


def assign_points(lat: np.ndarray, lon: np.ndarray, max_dist_deg: float = 0.25) -> np.ndarray:
    """Индекс региона (region_index()[i]['idx']) для каждой точки, -1 — океан / вне регионов."""
    tree, _ = _tree()
    pts = shapely.points(np.asarray(lon, dtype=float), np.asarray(lat, dtype=float))
    out = np.full(len(pts), -1, dtype=np.int32)
    inp, hit = tree.query(pts, predicate="within")
    out[inp[::-1]] = hit[::-1]  # при пересечениях берём первый
    miss = np.flatnonzero(out < 0)
    if len(miss):
        mi, mh = tree.query_nearest(pts[miss], max_distance=max_dist_deg, all_matches=False)
        out[miss[mi]] = mh
    return out


if __name__ == "__main__":
    build_regions()

In [ ]:
%%writefile helios/world/aggregate.py
"""Мировые агрегаты HELIOS (DuckDB, вне памяти) из Parquet-архива обнаружений.

Что строится (data/processed/world/agg/):
    static_r8.parquet       реестр постоянных источников тепла (факелы, заводы, вулканы) — res 8
    cellday_r{7..3}.parquet «горящие гексагоно-сутки»: битовая маска потоков (спутник × день/ночь),
                            чистые / все обнаружения, шум, FRP
    cellmonth_r{7..3}.parquet то же помесячно + центр ячейки — для карты гексагонов любого масштаба
    cells_r6.parquet        ячейки res 6 → регион (admin-1) и страна
    area_month.parquet      помесячно по регионам, странам и миру: cell-days по потокам × день/ночь
    coverage.parquet        доля суток месяца, когда поток вообще работал (глобально)

Сутки — местные солнечные (UTC + долгота/15 ч): ночной пролёт в 01:30 относится к своим суткам.
Чистое обнаружение = не low-confidence, поле FIRMS type = 0 и не в буфере постоянного источника.
"""

from __future__ import annotations

import json
import time

import duckdb
import h3
import numpy as np
import polars as pl
from h3ronpy import vector as h3v

from ..config import CACHE
from ..utils import get_logger
from .ingest import DET_DIR, WORLD
from .regions import assign_points, region_index

log = get_logger("helios.world.aggregate")

AGG = WORLD / "agg"
RES_LIST = (7, 6, 5, 4, 3, 2)
STREAM_OF_SAT = {0: "MODIS_T", 1: "MODIS_A", 2: "VIIRS_SNPP", 3: "VIIRS_NOAA20", 4: "VIIRS_NOAA21"}
MODIS_MASK = 0b0000001111  # биты (sat*2 + night) для sat 0,1
VIIRS_MASK = 0b1111110000  # sat 2,3,4
EPOCH = "DATE '2000-01-01'"


def con(memory_gb: int = 9) -> duckdb.DuckDBPyConnection:
    tmp = CACHE / "duckdb_tmp"
    tmp.mkdir(parents=True, exist_ok=True)
    c = duckdb.connect()
    c.execute(f"SET memory_limit='{memory_gb}GB'; SET threads=8; SET preserve_insertion_order=false; "
              f"SET temp_directory='{tmp.as_posix()}'; SET max_temp_directory_size='60GiB'")
    return c


def parent(col: str, r: int) -> str:
    """Родитель ячейки H3 на разрешении r битовыми операциями (без расширений)."""
    return (f"(({col} & ~(15::UBIGINT << 52)) | ({r}::UBIGINT << 52) | "
            f"((1::UBIGINT << {45 - 3 * r}) - 1::UBIGINT))")


def det_glob() -> str:
    return (DET_DIR / "*.parquet").as_posix()


LDAY = "(day::INTEGER + floor(((hhmm // 100) + (hhmm % 100) / 60.0 + lon / 15.0) / 24.0))::INTEGER"
MONTH_OF = f"((year({EPOCH} + {{d}}) - 2000) * 12 + month({EPOCH} + {{d}}) - 1)"


def _copy(c, sql: str, out) -> None:
    tmp = out.with_suffix(".tmp.parquet")
    c.execute(f"COPY ({sql}) TO '{tmp.as_posix()}' (FORMAT PARQUET, COMPRESSION ZSTD, ROW_GROUP_SIZE 500000)")
    tmp.replace(out)


# ----------------------------------------------------------------------------- постоянные источники
def build_static(c) -> pl.DataFrame:
    """Res-8 ячейки: поле type∈{1,2,3} у половины точек ИЛИ тепло ≥10 месяцев в году / ≥6 месяцев 3+ лет."""
    t0 = time.time()
    g = det_glob()
    # 1) простые счётчики по ячейке (без медиан и count DISTINCT — они держат все значения группы)
    c.execute(f"""
        CREATE OR REPLACE TEMP TABLE r8 AS
        SELECT h3, count(*)::INTEGER AS n, avg((type IN (1,2,3))::DOUBLE) AS st_share,
               avg(night::DOUBLE) AS night_share, avg(frp)::FLOAT AS frp_mean,
               year({EPOCH} + min(day)::INTEGER) AS first_year, year({EPOCH} + max(day)::INTEGER) AS last_year
        FROM read_parquet('{g}') GROUP BY h3 HAVING count(*) >= 4""")
    # 2) уникальные (год, месяц) только для кандидатов: GROUP BY без агрегатов = DISTINCT, потом счётчики
    c.execute(f"""
        CREATE OR REPLACE TEMP TABLE r8ym AS
        SELECT h3, y, count(*)::INTEGER AS months FROM (
            SELECT p.h3, year({EPOCH} + p.day::INTEGER) AS y, month({EPOCH} + p.day::INTEGER) AS m
            FROM read_parquet('{g}') p SEMI JOIN (SELECT h3 FROM r8 WHERE n >= 6) k ON p.h3 = k.h3
            GROUP BY 1, 2, 3
        ) GROUP BY 1, 2""")
    c.execute("""
        CREATE OR REPLACE TEMP TABLE r8m AS
        SELECT h3, count(*)::INTEGER AS n_years, max(months) AS max_months, sum((months >= 6)::INTEGER) AS years_6m
        FROM r8ym GROUP BY h3""")
    reg = c.sql("""
        SELECT r8.*, coalesce(n_years, 1) AS n_years, coalesce(max_months, 0) AS max_months,
               coalesce(years_6m, 0) AS years_6m, (st_share >= 0.5) AS ev_type,
               ((coalesce(max_months, 0) >= 10) OR (coalesce(years_6m, 0) >= 3)) AND coalesce(n_years, 1) >= 3 AS ev_temporal
        FROM r8 LEFT JOIN r8m USING (h3)""").pl()
    reg = reg.with_columns((pl.col("ev_type") | pl.col("ev_temporal")).alias("is_static"))
    st = reg.filter(pl.col("is_static"))
    coords = h3v.cells_to_coordinates(st["h3"].to_numpy())
    st = st.with_columns(pl.Series("lat", np.asarray(coords.column("lat")), dtype=pl.Float32),
                         pl.Series("lon", np.asarray(coords.column("lng")), dtype=pl.Float32))
    AGG.mkdir(parents=True, exist_ok=True)
    st.write_parquet(AGG / "static_r8.parquet", compression="zstd")
    log.info("постоянные источники: %d из %d кандидатов (type=%d, временные=%d), %.0f с", st.height, reg.height,
             int(reg["ev_type"].sum()), int(reg["ev_temporal"].sum()), time.time() - t0)
    return st


def _static_buffer(c, st: pl.DataFrame) -> None:
    cells: set[int] = set()
    for x in st["h3"].to_list():
        cells.update(h3.str_to_int(k) for k in h3.grid_disk(h3.int_to_str(x), 1))
    buf = pl.DataFrame({"h3": np.fromiter(cells, dtype=np.uint64)})
    c.register("static_buf_df", buf.to_arrow())
    c.execute("CREATE OR REPLACE TEMP TABLE static_buf AS SELECT h3::UBIGINT AS h3 FROM static_buf_df")
    log.info("буфер постоянных источников: %d ячеек res 8", buf.height)


# ----------------------------------------------------------------------------- гексагоно-сутки
def build_cellday(c) -> None:
    t0 = time.time()
    bit = "(1::USMALLINT << (sat::INTEGER * 2 + night::INTEGER))"
    sql = f"""
        SELECT {parent('h3', 7)} AS h3, {LDAY} AS lday,
               bit_or(CASE WHEN clean THEN {bit} ELSE 0::USMALLINT END) AS bits_c,
               bit_or({bit}) AS bits_r,
               sum((clean AND sat < 2)::INTEGER)::UINTEGER AS n_mc,
               sum((clean AND sat >= 2)::INTEGER)::UINTEGER AS n_vc,
               sum((conf = 0)::INTEGER)::UINTEGER AS n_low,
               sum((conf > 0 AND NOT clean)::INTEGER)::UINTEGER AS n_static,
               count(*)::UINTEGER AS n_r,
               max(CASE WHEN clean THEN frp END)::FLOAT AS frp_max,
               sum(CASE WHEN clean THEN frp ELSE 0 END)::FLOAT AS frp_sum
        FROM (SELECT p.h3, p.day, p.hhmm, p.lon, p.sat, p.night, p.conf, p.frp,
                     (p.conf > 0 AND p.type = 0 AND s.h3 IS NULL) AS clean
              FROM read_parquet('{det_glob()}') p LEFT JOIN static_buf s ON p.h3 = s.h3
              WHERE p.sat < 5)
        GROUP BY 1, 2"""
    _copy(c, sql, AGG / "cellday_r7.parquet")
    log.info("cellday r7: %.0f с", time.time() - t0)
    for r in RES_LIST[1:]:
        t1 = time.time()
        _copy(c, f"""
            SELECT {parent('h3', r)} AS h3, lday, bit_or(bits_c) AS bits_c, bit_or(bits_r) AS bits_r,
                   sum(n_mc)::UINTEGER AS n_mc, sum(n_vc)::UINTEGER AS n_vc, sum(n_low)::UINTEGER AS n_low,
                   sum(n_static)::UINTEGER AS n_static, sum(n_r)::UINTEGER AS n_r,
                   max(frp_max) AS frp_max, sum(frp_sum) AS frp_sum
            FROM read_parquet('{(AGG / f"cellday_r{r + 1}.parquet").as_posix()}') GROUP BY 1, 2""",
              AGG / f"cellday_r{r}.parquet")
        log.info("cellday r%d: %.0f с", r, time.time() - t1)


def _centers(cells: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    xy = h3v.cells_to_coordinates(cells)
    return np.asarray(xy.column("lat"), dtype=np.float32), np.asarray(xy.column("lng"), dtype=np.float32)


def build_cellmonth(c) -> None:
    for r in RES_LIST:
        t0 = time.time()
        src = (AGG / f"cellday_r{r}.parquet").as_posix()
        cells = c.execute(f"SELECT DISTINCT h3 FROM read_parquet('{src}')").fetchnumpy()["h3"].astype(np.uint64)
        lat, lon = _centers(cells)
        c.register("centers_df", pl.DataFrame({"h3": cells, "lat": lat, "lon": lon}).to_arrow())
        mon = MONTH_OF.format(d="lday")
        _copy(c, f"""
            SELECT m.*, k.lat, k.lon FROM (
                SELECT h3, {mon}::USMALLINT AS month,
                       sum(((bits_c & {MODIS_MASK}) > 0)::INTEGER)::USMALLINT AS d_cm,
                       sum(((bits_c & {VIIRS_MASK}) > 0)::INTEGER)::USMALLINT AS d_cv,
                       sum((bits_c > 0)::INTEGER)::USMALLINT AS d_ca,
                       sum(((bits_r & {MODIS_MASK}) > 0)::INTEGER)::USMALLINT AS d_rm,
                       sum(((bits_r & {VIIRS_MASK}) > 0)::INTEGER)::USMALLINT AS d_rv,
                       sum((bits_r > 0)::INTEGER)::USMALLINT AS d_ra,
                       sum(n_mc + n_vc)::UINTEGER AS n_c, sum(n_r)::UINTEGER AS n_r,
                       max(frp_max) AS frp_max
                FROM read_parquet('{src}') GROUP BY 1, 2
            ) m JOIN centers_df k ON m.h3 = k.h3::UBIGINT ORDER BY m.h3, m.month""",
              AGG / f"cellmonth_r{r}.parquet")
        c.unregister("centers_df")
        log.info("cellmonth r%d: %d ячеек, %.0f с", r, len(cells), time.time() - t0)


# ----------------------------------------------------------------------------- регионы и ряды
def build_cells_r6(c) -> pl.DataFrame:
    t0 = time.time()
    cells = c.execute(f"SELECT DISTINCT h3 FROM read_parquet('{(AGG / 'cellday_r6.parquet').as_posix()}')"
                      ).fetchnumpy()["h3"].astype(np.uint64)
    lat, lon = _centers(cells)
    reg = assign_points(lat, lon)
    idx = region_index()
    iso = np.array([r["iso2"] for r in idx] + [""], dtype=object)
    df = pl.DataFrame({"h3": cells, "lat": lat, "lon": lon, "region": reg.astype(np.int32),
                       "iso2": iso[reg]})
    df.write_parquet(AGG / "cells_r6.parquet", compression="zstd")
    log.info("ячейки r6 → регионы: %d ячеек, вне регионов %.1f%%, %.0f с", df.height,
             100 * float((reg < 0).mean()), time.time() - t0)
    return df


def build_area_month(c) -> None:
    """Помесячно по регионам → странам → миру: cell-days каждого потока (спутник × день/ночь) на res 6."""
    t0 = time.time()
    cells = pl.read_parquet(AGG / "cells_r6.parquet")
    c.register("cells_df", cells.select("h3", "region").to_arrow())
    mon = MONTH_OF.format(d="lday")
    bits = ", ".join(f"sum(((bits_c >> {k}) & 1)::INTEGER)::UINTEGER AS c{k}" for k in range(10))
    reg = c.sql(f"""
        SELECT k.region, {mon}::USMALLINT AS month, {bits},
               sum((bits_r > 0)::INTEGER)::UINTEGER AS r_any,
               sum((bits_c > 0)::INTEGER)::UINTEGER AS hex_any,
               sum(((bits_c & {MODIS_MASK}) > 0)::INTEGER)::UINTEGER AS hex_m,
               sum(((bits_c & {VIIRS_MASK}) > 0)::INTEGER)::UINTEGER AS hex_v,
               sum(n_mc)::UBIGINT AS n_mc, sum(n_vc)::UBIGINT AS n_vc, sum(n_low)::UBIGINT AS n_low,
               sum(n_static)::UBIGINT AS n_static, sum(n_r)::UBIGINT AS n_r, sum(frp_sum)::DOUBLE AS frp_sum,
               count(DISTINCT d.h3)::UINTEGER AS cells
        FROM read_parquet('{(AGG / 'cellday_r6.parquet').as_posix()}') d JOIN cells_df k ON d.h3 = k.h3::UBIGINT
        GROUP BY 1, 2""").pl()
    c.unregister("cells_df")
    idx = region_index()
    rid = np.array([r["id"] for r in idx] + ["OCEAN"], dtype=object)
    iso = np.array([r["iso2"] for r in idx] + ["OCEAN"], dtype=object)
    r_arr = reg["region"].to_numpy()
    reg = reg.with_columns(pl.Series("area", rid[r_arr]), pl.Series("iso2", iso[r_arr])).drop("region")
    num = [x for x in reg.columns if x not in ("area", "iso2", "month")]
    country = reg.group_by("iso2", "month").agg([pl.col(x).sum() for x in num]).rename({"iso2": "area"})
    world = reg.group_by("month").agg([pl.col(x).sum() for x in num]).with_columns(pl.lit("WORLD").alias("area"))
    out = pl.concat([reg.drop("iso2").with_columns(pl.lit("region").alias("level")),
                     country.with_columns(pl.lit("country").alias("level")),
                     world.with_columns(pl.lit("world").alias("level"))], how="diagonal")
    out = out.filter(pl.col("area") != "OCEAN").sort("area", "month")
    out.write_parquet(AGG / "area_month.parquet", compression="zstd")
    log.info("area_month: %d строк (%d регионов, %d стран), %.0f с", out.height,
             out.filter(pl.col("level") == "region")["area"].n_unique(),
             out.filter(pl.col("level") == "country")["area"].n_unique(), time.time() - t0)


def build_coverage(c) -> None:
    """Глобальная доля суток месяца, когда поток что-то видел (признак работы сенсора)."""
    mon = MONTH_OF.format(d="d")
    cov = c.sql(f"""
        SELECT sat, {mon}::USMALLINT AS month, count(DISTINCT d)::INTEGER AS active_days FROM (
            SELECT DISTINCT sat, day::INTEGER AS d FROM read_parquet('{det_glob()}')
        ) GROUP BY 1, 2 ORDER BY 1, 2""").pl()
    days = cov.select(pl.col("month")).to_series().map_elements(
        lambda m: [31, 29 if (2000 + m // 12) % 4 == 0 else 28, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31][m % 12],
        return_dtype=pl.Int32)
    cov = cov.with_columns((pl.col("active_days") / days).clip(upper_bound=1.0).alias("coverage"),
                           pl.col("sat").replace_strict(STREAM_OF_SAT, return_dtype=pl.Utf8).alias("stream"))
    cov.write_parquet(AGG / "coverage.parquet")
    log.info("coverage: %d строк", cov.height)


def build_all(skip_static: bool = False) -> dict:
    t0 = time.time()
    c = con()
    AGG.mkdir(parents=True, exist_ok=True)
    if skip_static and (AGG / "static_r8.parquet").exists():
        st = pl.read_parquet(AGG / "static_r8.parquet")
    else:
        st = build_static(c)
    _static_buffer(c, st)
    build_cellday(c)
    build_cellmonth(c)
    build_coverage(c)
    build_cells_r6(c)
    build_area_month(c)
    files = sorted(p.name for p in DET_DIR.glob("*.parquet"))
    summary = {"det_files": files, "static_cells": st.height, "seconds": round(time.time() - t0)}
    (AGG / "summary.json").write_text(json.dumps(summary, indent=1), encoding="utf-8")
    log.info("мировые агрегаты готовы за %.0f с", time.time() - t0)
    return summary


if __name__ == "__main__":
    import sys

    build_all(skip_static="--skip-static" in sys.argv)

In [ ]:
%%writefile helios/world/nrt.py
"""Свежие пожары всего мира за 7 дней (открытые ленты NASA FIRMS NRT, без ключа) для текущего прогноза.

Ленты: MODIS C6.1, VIIRS NOAA-20, VIIRS NOAA-21 (и S-NPP, пока лента жива). Файлы по 10–100 МБ, сервер
медленный — качаем частями в несколько потоков (как архив). Фильтр: low confidence и буфер мирового
реестра постоянных источников (в NRT нет поля type).
"""

from __future__ import annotations

import concurrent.futures as cf
import time
from pathlib import Path

import h3
import numpy as np
import polars as pl
import requests
from h3ronpy import vector as h3v

from ..config import RAW
from ..utils import get_logger
from .aggregate import AGG
from .fetch import UA, _fetch_range

log = get_logger("helios.world.nrt")

BASE = "https://firms.modaps.eosdis.nasa.gov/data/active_fire"
FEEDS = [
    ("modis-c6.1", "MODIS_C6_1_Global_7d.csv", 0),
    ("noaa-20-viirs-c2", "J1_VIIRS_C2_Global_7d.csv", 3),
    ("noaa-21-viirs-c2", "J2_VIIRS_C2_Global_7d.csv", 4),
    ("suomi-npp-viirs-c2", "SUOMI_VIIRS_C2_Global_7d.csv", 2),
]
NRT_DIR = RAW / "global_nrt"
CHUNK = 2 * 1024 * 1024


def _download(url: str, dest: Path, threads: int = 8, max_age_h: float = 3.0) -> Path | None:
    if dest.exists() and time.time() - dest.stat().st_mtime < max_age_h * 3600:
        return dest
    head = requests.head(url, timeout=30, headers=UA, allow_redirects=True)
    if head.status_code != 200:
        log.warning("NRT нет файла %s (%s)", dest.name, head.status_code)
        return None
    size = int(head.headers.get("Content-Length", 0))
    dest.parent.mkdir(parents=True, exist_ok=True)
    if size <= 0 or head.headers.get("accept-ranges", "").lower() != "bytes":
        r = requests.get(url, timeout=600, headers=UA)
        r.raise_for_status()
        dest.write_bytes(r.content)
        return dest
    parts = [(a, min(size, a + CHUNK) - 1) for a in range(0, size, CHUNK)]
    buf = bytearray(size)

    def job(ab):
        sess = requests.Session()
        for k in range(30):
            try:
                data = _fetch_range(sess, url, ab[0], ab[1])
                buf[ab[0]:ab[1] + 1] = data
                return
            except Exception:
                time.sleep(min(30, 1.5 ** k))
        raise IOError(f"кусок {ab} не скачался")

    with cf.ThreadPoolExecutor(threads) as ex:
        list(ex.map(job, parts))
    tmp = dest.with_suffix(".tmp")
    tmp.write_bytes(bytes(buf))
    tmp.replace(dest)
    log.info("NRT %s: %.1f МБ", dest.name, size / 2**20)
    return dest


def _static_buffer() -> np.ndarray:
    st = pl.read_parquet(AGG / "static_r8.parquet", columns=["h3"])
    cells: set[int] = set()
    for x in st["h3"].to_list():
        cells.update(h3.str_to_int(k) for k in h3.grid_disk(h3.int_to_str(x), 1))
    return np.fromiter(cells, dtype=np.uint64)


def global_nrt_frame() -> pl.DataFrame:
    frames = []
    for folder, name, sat in FEEDS:
        try:
            p = _download(f"{BASE}/{folder}/csv/{name}", NRT_DIR / name)
        except Exception as e:
            log.warning("NRT %s: %s", name, e)
            continue
        if p is None or p.stat().st_size < 200:
            continue
        df = pl.read_csv(p, columns=["latitude", "longitude", "acq_date", "acq_time", "confidence", "frp", "daynight"],
                         schema_overrides={"acq_time": pl.Utf8, "confidence": pl.Utf8}, ignore_errors=True)
        conf = pl.col("confidence").str.strip_chars().str.to_lowercase()
        low = (conf == "l") if sat >= 2 else (conf.cast(pl.Float32, strict=False).fill_null(0) < 30)
        frames.append(df.select(
            pl.col("latitude").cast(pl.Float64).alias("lat"), pl.col("longitude").cast(pl.Float64).alias("lon"),
            pl.col("acq_date").str.to_date("%Y-%m-%d", strict=False).alias("date"),
            pl.col("acq_time").str.strip_chars().cast(pl.Int32, strict=False).fill_null(0).alias("hhmm"),
            low.alias("low"), pl.col("frp").cast(pl.Float32, strict=False).fill_null(0).alias("frp"),
            pl.lit(sat, dtype=pl.UInt8).alias("sat"), (pl.col("daynight") == "N").alias("night")))
    if not frames:
        raise RuntimeError("ни одна лента NRT не скачалась")
    return pl.concat(frames)


def global_nrt_week() -> tuple[np.ndarray, np.ndarray, np.ndarray, str]:
    """Ячейки res 5 с чистым огнём за последние 7 дней: (ячейки, дней с огнём, обнаружений, дата «на сегодня»)."""
    df = global_nrt_frame().filter(~pl.col("low") & pl.col("date").is_not_null())
    as_of = df["date"].max()
    r8 = np.asarray(h3v.coordinates_to_cells(df["lat"].to_numpy(), df["lon"].to_numpy(), 8).to_numpy(), np.uint64)
    clean = ~np.isin(r8, _static_buffer())
    df = df.with_columns(pl.Series("r8", r8)).filter(pl.Series(clean))
    r5 = np.asarray(h3v.coordinates_to_cells(df["lat"].to_numpy(), df["lon"].to_numpy(), 5).to_numpy(), np.uint64)
    hours = (df["hhmm"].to_numpy() // 100) + (df["hhmm"].to_numpy() % 100) / 60.0 + df["lon"].to_numpy() / 15.0
    lday = (df["date"].cast(pl.Int32).to_numpy() + np.floor(hours / 24.0)).astype(np.int64)
    g = pl.DataFrame({"h3": r5, "lday": lday}).group_by("h3").agg(pl.col("lday").n_unique().alias("days"),
                                                                 pl.len().alias("n"))
    log.info("NRT мир: %d чистых обнаружений, %d ячеек res 5, на %s", df.height, g.height, as_of)
    return (g["h3"].to_numpy().astype(np.uint64), g["days"].to_numpy(), g["n"].to_numpy(), str(as_of))

In [ ]:
%%writefile helios/world/forecast.py
"""ИИ №3 на мировых данных: вероятность пожара в каждой ячейке H3 res 5 (~252 км²) на 1–4 недели вперёд.

Данные: все чистые обнаружения NASA FIRMS (MODIS c 2000 г., VIIRS S-NPP c 2012 г.) во всех странах →
недельная матрица «дней с огнём» (ячейка × неделя). Модели LightGBM, по одной на горизонт:

  full  — вся история ячейки (последние недели, 13/52 недели, давность огня, соседи, регион, климатология);
          для «Машины времени» и честной оценки (обучение 2013–2022, валидация 2023, тест 2024–2025);
  oper  — только то, что есть в любой момент «сейчас»: последние 7 дней (открытая лента NRT) +
          климатология архива прошлых лет + соседи + сезон; этой моделью считается текущий прогноз.

Оценка — на ВСЕХ ячейках мира и неделях 2024–2025 с настоящей долей пожаров (без прореживания),
против сезонного алгоритма (доля прошлых лет с огнём в те же ±2 недели).

    python -m helios.world.forecast            # обучение + оценка + прогнозы «сейчас» и «Машина времени»
"""

from __future__ import annotations

import json
import time
from datetime import date, timedelta

import h3
import lightgbm as lgb
import numpy as np
import polars as pl
from sklearn.metrics import average_precision_score, roc_auc_score

from ..config import MODELS_DIR, REPORTS_DIR
from ..utils import get_logger
from .aggregate import AGG, con, parent
from .ingest import WORLD

log = get_logger("helios.world.forecast")

EPOCH = date(2000, 1, 1)
HORIZONS = (1, 2, 3, 4)
TRAIN_WEEKS = ("2013-01-07", "2022-12-26")
VAL_WEEKS = ("2023-01-02", "2023-12-25")
TEST_WEEKS = ("2024-01-01", "2025-12-22")
VIIRS_WEEK = (date(2012, 1, 20) - EPOCH).days // 7
FULL = ["d0", "d1", "d2", "d3", "s4", "s13", "s52", "s52_prev", "since", "n4", "nb_d0", "nb_max_d0", "nb_s4",
        "reg_d0", "reg_s4", "clim_frac", "clim_days", "ly_d", "ly_win", "nb_clim", "reg_clim",
        "woy_sin", "woy_cos", "lat", "lon_sin", "lon_cos", "viirs_on"]
OPER = ["d0", "n0", "nb_d0", "nb_max_d0", "reg_d0", "clim_frac", "clim_days", "ly_d", "ly_win", "nb_clim", "reg_clim",
        "woy_sin", "woy_cos", "lat", "lon_sin", "lon_cos", "viirs_on"]
TM_DATES = ("2024-04-01", "2024-07-15", "2024-09-02", "2025-03-17", "2025-08-04", "2025-10-13")
OUT_DIR = WORLD / "forecast"


def week_of(iso: str) -> int:
    return (date.fromisoformat(iso) - EPOCH).days // 7


def week_date(w: int) -> date:
    return EPOCH + timedelta(days=7 * int(w))


class Panel:
    """Плотные матрицы ячейка × неделя: D (дней с огнём, 0–7), N (чистых обнаружений, ≤255), их суммы."""

    def __init__(self, min_weeks: int = 0) -> None:
        t0 = time.time()
        c = con()
        df = c.sql(f"""
            SELECT h3, (lday // 7)::INTEGER AS w, count(*)::INTEGER AS d, sum(n_mc + n_vc)::INTEGER AS n
            FROM read_parquet('{(AGG / 'cellday_r5.parquet').as_posix()}') WHERE bits_c > 0 AND lday >= 0
            GROUP BY 1, 2""").pl()
        cells = np.unique(df["h3"].to_numpy())
        self.cells = cells.astype(np.uint64)
        self.C = len(cells)
        self.last_week = int(df["w"].max())
        self.W = max(self.last_week + 1 + 8, min_weeks)
        ci = np.searchsorted(cells, df["h3"].to_numpy())
        w = df["w"].to_numpy()
        self.D = np.zeros((self.C, self.W), np.uint8)
        self.N = np.zeros((self.C, self.W), np.uint8)
        self.D[ci, w] = np.minimum(df["d"].to_numpy(), 7)
        self.N[ci, w] = np.minimum(df["n"].to_numpy(), 255)
        self.CS = np.zeros((self.C, self.W + 1), np.uint16)
        np.cumsum(self.D, axis=1, dtype=np.uint16, out=self.CS[:, 1:])
        # последняя неделя с огнём ≤ w: отсортированные (ячейка, неделя)
        order = np.lexsort((w, ci))
        self.fire_key = (ci[order].astype(np.int64) << 20) | w[order].astype(np.int64)
        # центры, соседи (кольцо 1), родитель res 3
        from h3ronpy import vector as h3v

        xy = h3v.cells_to_coordinates(self.cells)
        self.lat = np.asarray(xy.column("lat"), np.float32)
        self.lon = np.asarray(xy.column("lng"), np.float32)
        self.nb = np.full((self.C, 6), -1, np.int64)
        for i, x in enumerate(self.cells.tolist()):
            ring = [h3.str_to_int(k) for k in h3.grid_ring(h3.int_to_str(x), 1)]
            idx = np.searchsorted(cells, ring)
            ok = (idx < self.C) & (cells[np.minimum(idx, self.C - 1)] == np.asarray(ring, np.uint64))
            k = idx[ok][:6]
            self.nb[i, : len(k)] = k
        p3 = c.sql(f"SELECT {parent('h::UBIGINT', 3)} AS p FROM (SELECT unnest(?::UBIGINT[]) AS h)",
                   params=[self.cells.tolist()]).fetchnumpy()["p"]
        self.par_ids, self.par = np.unique(p3, return_inverse=True)
        R = np.zeros((len(self.par_ids), self.W), np.int32)
        np.add.at(R, self.par, self.D.astype(np.int32))
        self.R = R
        self.RCS = np.zeros((len(self.par_ids), self.W + 1), np.int32)
        np.cumsum(R, axis=1, out=self.RCS[:, 1:])
        self.par_size = np.bincount(self.par).astype(np.float32)
        log.info("панель: %d ячеек × %d недель, %.0f с", self.C, self.W, time.time() - t0)

    # ------------------------------------------------------------------ признаки
    def _win(self, CS: np.ndarray, rows: np.ndarray, a: np.ndarray, b: np.ndarray) -> np.ndarray:
        """Сумма по неделям [a, b] (включительно) через накопленные суммы; вне диапазона — 0."""
        a = np.clip(a, 0, self.W)
        b = np.clip(b + 1, 0, self.W)
        return CS[rows, b].astype(np.int32) - CS[rows, np.minimum(a, b)].astype(np.int32)

    def features(self, ci: np.ndarray, w: np.ndarray, h: int, names: list[str]) -> np.ndarray:
        ci = np.asarray(ci, np.int64)
        w = np.asarray(w, np.int64)
        t = w + h
        D = self.D
        out: dict[str, np.ndarray] = {}
        g = lambda k: D[ci, np.clip(w - k, 0, self.W - 1)].astype(np.float32) * (w - k >= 0)  # noqa: E731
        out["d0"], out["d1"], out["d2"], out["d3"] = g(0), g(1), g(2), g(3)
        out["n0"] = np.log1p(self.N[ci, w].astype(np.float32))
        if "s4" in names:
            out["s4"] = self._win(self.CS, ci, w - 3, w).astype(np.float32)
            out["s13"] = self._win(self.CS, ci, w - 12, w).astype(np.float32)
            out["s52"] = self._win(self.CS, ci, w - 51, w).astype(np.float32)
            out["s52_prev"] = self._win(self.CS, ci, w - 103, w - 52).astype(np.float32)
            n4 = sum(self.N[ci, np.clip(w - k, 0, self.W - 1)].astype(np.float32) for k in range(4))
            out["n4"] = np.log1p(n4)
            key = (ci << 20) | w
            pos = np.searchsorted(self.fire_key, key, "right") - 1
            okp = pos >= 0
            last_ci = np.where(okp, self.fire_key[np.maximum(pos, 0)] >> 20, -1)
            last_w = np.where(okp, self.fire_key[np.maximum(pos, 0)] & ((1 << 20) - 1), -10_000)
            out["since"] = np.where(last_ci == ci, np.minimum(w - last_w, 520), 520).astype(np.float32)
        nbm = self.nb[ci]
        valid = nbm >= 0
        nbi = np.where(valid, nbm, 0)
        nb_d0 = D[nbi, w[:, None]].astype(np.float32) * valid
        out["nb_d0"] = nb_d0.mean(axis=1)
        out["nb_max_d0"] = nb_d0.max(axis=1)
        if "nb_s4" in names:
            s4n = (self.CS[nbi, np.clip(w + 1, 0, self.W)[:, None]].astype(np.int32)
                   - self.CS[nbi, np.clip(w - 3, 0, self.W)[:, None]].astype(np.int32)) * valid
            out["nb_s4"] = s4n.mean(axis=1).astype(np.float32)
        p = self.par[ci]
        size = self.par_size[p]
        out["reg_d0"] = self.R[p, w].astype(np.float32) / size
        if "reg_s4" in names:
            out["reg_s4"] = self._win(self.RCS, p, w - 3, w).astype(np.float32) / size
        # климатология: те же ±2 недели целевой недели в прошлые 1–5 лет (только прошлое)
        pres, days, nbp, regp, avail = [], [], [], [], []
        for k in range(1, 6):
            a, b = t - 52 * k - 2, t - 52 * k + 2
            ok = (a >= 0) & (b <= w)
            s = self._win(self.CS, ci, a, b) * ok
            pres.append((s > 0).astype(np.float32))
            days.append(s.astype(np.float32))
            sn = np.stack([self._win(self.CS, nbi[:, j], a, b) for j in range(6)], 1) * valid * ok[:, None]
            nbp.append((sn > 0).mean(axis=1).astype(np.float32))
            regp.append(self._win(self.RCS, p, a, b).astype(np.float32) * ok / size)
            avail.append(ok.astype(np.float32))
        av = np.maximum(np.sum(avail, axis=0), 1.0)
        out["clim_frac"] = np.sum(pres, axis=0) / av
        out["clim_days"] = np.sum(days, axis=0) / av
        out["nb_clim"] = np.sum(nbp, axis=0) / av
        out["reg_clim"] = np.sum(regp, axis=0) / av
        out["ly_d"] = D[ci, np.clip(t - 52, 0, self.W - 1)].astype(np.float32) * (t - 52 <= w)
        out["ly_win"] = days[0]
        doy = np.array([(week_date(x).timetuple().tm_yday) for x in np.unique(t)], np.float32)
        doy_map = dict(zip(np.unique(t).tolist(), doy.tolist()))
        ang = 2 * np.pi * np.array([doy_map[x] for x in t.tolist()], np.float32) / 365.25
        out["woy_sin"], out["woy_cos"] = np.sin(ang), np.cos(ang)
        out["lat"] = self.lat[ci]
        lon = np.radians(self.lon[ci])
        out["lon_sin"], out["lon_cos"] = np.sin(lon), np.cos(lon)
        out["viirs_on"] = (w >= VIIRS_WEEK).astype(np.float32)
        return np.column_stack([out[n] for n in names]).astype(np.float32)

    def target(self, ci: np.ndarray, w: np.ndarray, h: int) -> np.ndarray:
        return (self.D[ci, w + h] > 0).astype(np.uint8)


# ----------------------------------------------------------------------------- обучение
def _sample(pn: Panel, w0: int, w1: int, h: int, n_pos: int, n_neg: int, rng) -> tuple[np.ndarray, np.ndarray, np.ndarray, float]:
    """Все/часть положительных (ячейка, неделя) + случайные отрицательные с весом (их реальная доля)."""
    sub = pn.D[:, w0 + h:w1 + h + 1]
    pc, pw = np.nonzero(sub)
    pw = pw + w0
    total_pos = len(pc)
    if total_pos > n_pos:
        k = rng.choice(total_pos, n_pos, replace=False)
        pc, pw = pc[k], pw[k]
    nc = rng.integers(0, pn.C, n_neg * 2)
    nw = rng.integers(w0, w1 + 1, n_neg * 2)
    neg = pn.D[nc, nw + h] == 0
    nc, nw = nc[neg][:n_neg], nw[neg][:n_neg]
    total_neg = pn.C * (w1 - w0 + 1) - total_pos
    w_pos = total_pos / max(len(pc), 1)
    w_neg = total_neg / max(len(nc), 1)
    ci = np.concatenate([pc, nc])
    ww = np.concatenate([pw, nw])
    wt = np.concatenate([np.full(len(pc), w_pos), np.full(len(nc), w_neg)]).astype(np.float32)
    return ci, ww, wt, total_pos / (total_pos + total_neg)


def _metrics(y: np.ndarray, p: np.ndarray, weeks: np.ndarray) -> dict:
    """ROC-AUC, PR-AUC и доля пожаров, пойманных топ-10% ячеек каждой недели."""
    out = {"roc_auc": round(float(roc_auc_score(y, p)), 4), "pr_auc": round(float(average_precision_score(y, p)), 4)}
    caught = total = 0
    for wk in np.unique(weeks):
        m = weeks == wk
        yy, pp = y[m], p[m]
        if yy.sum() == 0:
            continue
        k = max(1, int(0.1 * len(pp)))
        top = np.argpartition(-pp, k - 1)[:k]
        caught += int(yy[top].sum())
        total += int(yy.sum())
    out["top10_capture"] = round(caught / max(total, 1), 4)
    return out


def train_all(n_pos: int = 2_500_000, n_neg: int = 3_500_000, rounds: int = 600) -> dict:
    t0 = time.time()
    pn = Panel(min_weeks=(date.today() - EPOCH).days // 7 + 8)  # запас недель для прогноза «сейчас»
    rng = np.random.default_rng(42)
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    tr0, tr1 = week_of(TRAIN_WEEKS[0]), week_of(TRAIN_WEEKS[1])
    va0, va1 = week_of(VAL_WEEKS[0]), week_of(VAL_WEEKS[1])
    te0, te1 = week_of(TEST_WEEKS[0]), week_of(TEST_WEEKS[1])
    report: dict = {"cells": pn.C, "weeks": [TRAIN_WEEKS, VAL_WEEKS, TEST_WEEKS], "resolution": 5,
                    "cell_area_km2": round(h3.average_hexagon_area(5, "km^2"), 1), "variants": {}}
    from .regions import assign_points, region_index

    reg = assign_points(pn.lat, pn.lon)
    iso = np.array([r["iso2"] for r in region_index()] + [""], dtype=object)[reg].astype(str)
    iso_codes, iso_idx = np.unique(iso, return_inverse=True)
    for variant, names in (("full", FULL), ("oper", OPER)):
        report["variants"][variant] = {}
        for h in HORIZONS:
            t1 = time.time()
            ci, ww, wt, base = _sample(pn, tr0, tr1 - h, h, n_pos, n_neg, rng)
            X = pn.features(ci, ww, h, names)
            y = pn.target(ci, ww, h)
            vci, vww, vwt, _ = _sample(pn, va0, va1 - h, h, 400_000, 600_000, rng)
            Xv, yv = pn.features(vci, vww, h, names), pn.target(vci, vww, h)
            params = {"objective": "binary", "learning_rate": 0.06, "num_leaves": 127, "min_data_in_leaf": 200,
                      "feature_fraction": 0.85, "bagging_fraction": 0.8, "bagging_freq": 1, "lambda_l2": 1.0,
                      "num_threads": 10, "verbose": -1, "max_bin": 127}
            dtr = lgb.Dataset(X, y, weight=wt, feature_name=names, free_raw_data=True)
            dva = lgb.Dataset(Xv, yv, weight=vwt, reference=dtr)
            model = lgb.train(params, dtr, rounds, valid_sets=[dva], callbacks=[lgb.early_stopping(40, verbose=False)])
            del X, dtr
            # тест: все ячейки × все недели 2024–2025
            ps, ys, cs, ws_ = [], [], [], []
            for wk in range(te0, te1 - h + 1, 2):  # каждая вторая неделя 2024–2025: несмещённо и вдвое быстрее
                cia = np.arange(pn.C)
                wka = np.full(pn.C, wk)
                Xt = pn.features(cia, wka, h, names)
                ps.append(model.predict(Xt, num_threads=10).astype(np.float32))
                cs.append(Xt[:, names.index("clim_frac")].astype(np.float32))
                ys.append(pn.target(cia, wka, h))
                ws_.append(wka.astype(np.int32))
            p, y_t, clim, wks = np.concatenate(ps), np.concatenate(ys), np.concatenate(cs), np.concatenate(ws_)
            m_ai, m_clim = _metrics(y_t, p, wks), _metrics(y_t, clim, wks)
            by_c = {}
            cell_iso = np.tile(iso_idx.astype(np.int32), len(np.unique(wks)))
            for code in ("KZ", "RU", "BR", "AU", "US", "CD", "AO", "IN", "ID", "CA", "AR", "ZM"):
                if code not in iso_codes:
                    continue
                m = cell_iso == int(np.searchsorted(iso_codes, code))
                if m.any() and y_t[m].sum() > 50:
                    by_c[code] = {"ai_pr_auc": round(float(average_precision_score(y_t[m], p[m])), 4),
                                  "clim_pr_auc": round(float(average_precision_score(y_t[m], clim[m])), 4),
                                  "ai_roc_auc": round(float(roc_auc_score(y_t[m], p[m])), 4)}
            gain = model.feature_importance("gain")
            imp = sorted(zip(names, (gain / gain.sum()).round(4).tolist()), key=lambda x: -x[1])[:10]
            model.save_model(str(MODELS_DIR / f"world_forecast_{variant}_h{h}.txt"))
            rec = {"test": {"ai": m_ai, "baseline_climatology": m_clim, "base_rate": round(float(y_t.mean()), 5),
                            "rows": int(len(y_t))},
                   "train_rows": int(len(ci)), "train_base_rate": round(base, 5), "iterations": model.best_iteration,
                   "by_country": by_c, "importance": imp, "seconds": round(time.time() - t1)}
            report["variants"][variant][f"h{h}"] = rec
            log.info("ИИ №3 мир %s h=%d: %s vs клим %s (%.0f с)", variant, h, m_ai, m_clim, time.time() - t1)
            del p, y_t, clim, wks, ps, ys, cs, ws_
    report["seconds"] = round(time.time() - t0)
    REPORTS_DIR.mkdir(parents=True, exist_ok=True)
    (REPORTS_DIR / "world_ai3_metrics.json").write_text(json.dumps(report, ensure_ascii=False, indent=1), encoding="utf-8")
    time_machine(pn)
    try:
        report["now"] = forecast_now(pn)
    except Exception as e:  # без сети — только «Машина времени»
        log.warning("прогноз «сейчас» не посчитан: %s", e)
    return report


# ----------------------------------------------------------------------------- прогнозы
def _predict_rows(pn: Panel, w: int, variant: str, min_p: float = 0.003) -> pl.DataFrame:
    names = FULL if variant == "full" else OPER
    rows = []
    for h in HORIZONS:
        model = lgb.Booster(model_file=str(MODELS_DIR / f"world_forecast_{variant}_h{h}.txt"))
        ci = np.arange(pn.C)
        X = pn.features(ci, np.full(pn.C, w), h, names)
        p = model.predict(X, num_threads=10)
        clim = X[:, names.index("clim_frac")]
        actual = pn.D[ci, w + h] > 0 if w + h <= pn.last_week else np.zeros(pn.C, bool)
        keep = (p >= min_p) | actual
        rows.append(pl.DataFrame({"h3": pn.cells[keep], "horizon": np.full(int(keep.sum()), h, np.int8),
                                  "p": p[keep].astype(np.float32), "clim": clim[keep].astype(np.float32),
                                  "actual": actual[keep].astype(np.int8)}))
    return pl.concat(rows)


def time_machine(pn: Panel) -> list[dict]:
    """«Как будто сегодня» прошлая дата: прогноз полной моделью по всем ячейкам мира + что сгорело на самом деле."""
    out, meta = [], []
    for d in TM_DATES:
        w = week_of(d)
        rec = {"as_of": d, "horizons": {}}
        ci = np.arange(pn.C)
        for h in HORIZONS:
            model = lgb.Booster(model_file=str(MODELS_DIR / f"world_forecast_full_h{h}.txt"))
            X = pn.features(ci, np.full(pn.C, w), h, FULL)
            p = model.predict(X, num_threads=10).astype(np.float32)
            clim = X[:, FULL.index("clim_frac")]
            y = pn.target(ci, np.full(pn.C, w), h)
            if y.sum() > 0:
                rec["horizons"][str(h)] = {
                    "ai": {"roc_auc": round(float(roc_auc_score(y, p)), 4), "pr_auc": round(float(average_precision_score(y, p)), 4)},
                    "baseline": {"roc_auc": round(float(roc_auc_score(y, clim)), 4),
                                 "pr_auc": round(float(average_precision_score(y, clim)), 4)},
                    "target_week": str(week_date(w + h)), "fires": int(y.sum())}
            keep = (p >= 0.003) | (y == 1)
            out.append(pl.DataFrame({"h3": pn.cells[keep], "horizon": np.full(int(keep.sum()), h, np.int8),
                                     "p": p[keep], "clim": clim[keep].astype(np.float32), "actual": y[keep].astype(np.int8),
                                     "as_of": [d] * int(keep.sum())}))
        meta.append(rec)
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    pl.concat(out).write_parquet(OUT_DIR / "time_machine.parquet", compression="zstd")
    (OUT_DIR / "time_machine.json").write_text(json.dumps(meta, ensure_ascii=False), encoding="utf-8")
    log.info("Машина времени: %s", [(m["as_of"], m["horizons"].get("1", {}).get("ai")) for m in meta])
    return meta


def forecast_now(pn: Panel | None = None) -> dict:
    """Текущий прогноз оперативной моделью: последние 7 дней — открытая лента FIRMS NRT (весь мир)."""
    from .nrt import global_nrt_week

    cells, days, n_det, as_of = global_nrt_week()
    w_now = week_of(as_of)
    # недели между концом архива и «сейчас» остаются пустыми: оперативная модель их и не использует —
    # ей нужны только текущая неделя (NRT) и те же недели прошлых лет (архив)
    pn = pn if pn is not None and pn.W > w_now + 5 else Panel(min_weeks=w_now + 8)
    idx = np.searchsorted(pn.cells, cells)
    ok = (idx < pn.C) & (pn.cells[np.minimum(idx, pn.C - 1)] == cells)
    pn.D[:, w_now] = 0
    pn.N[:, w_now] = 0
    pn.D[idx[ok], w_now] = np.minimum(days[ok], 7)
    pn.N[idx[ok], w_now] = np.minimum(n_det[ok], 255)
    np.cumsum(pn.D, axis=1, dtype=np.uint16, out=pn.CS[:, 1:])
    pn.R[:, w_now] = np.bincount(pn.par, weights=pn.D[:, w_now].astype(np.float64), minlength=len(pn.par_ids)).astype(np.int32)
    np.cumsum(pn.R, axis=1, out=pn.RCS[:, 1:])
    os_ = OUT_DIR
    os_.mkdir(parents=True, exist_ok=True)
    df = _predict_rows(pn, w_now, "oper").with_columns(pl.lit(as_of).alias("as_of"))
    df.write_parquet(OUT_DIR / "risk_latest.parquet", compression="zstd")
    meta = {"as_of": as_of, "week_start": str(week_date(w_now)), "cells_with_fire_last_7d": int(ok.sum()),
            "new_cells_never_burned": int((~ok).sum()), "rows": df.height}
    (OUT_DIR / "risk_latest.json").write_text(json.dumps(meta), encoding="utf-8")
    log.info("прогноз сейчас: %s", meta)
    return meta


if __name__ == "__main__":
    import sys

    if "--now" in sys.argv:
        forecast_now()
    else:
        train_all()

In [ ]:
%%writefile helios/world/edge.py
"""ИИ №1 v2 на мировых данных: бортовая оценка каждого горячего пикселя — «настоящий ли пожар?» и
«разрастётся ли за 72 ч?». Только признаки, которые спутник считает сам в момент съёмки, + компактная
бортовая таблица истории (по данным до 2017 г. включительно — тест 2024–2025 честный).

Признаки: сенсор, ночь, яркостные температуры (MIR/TIR) и их разность, FRP, площадь пикселя, размер
скана, уверенность продукта, местное солнечное время, месяц, широта, число горячих пикселей рядом
(в той же ячейке H3 res 7 за сутки), бортовая история: климатология месяца для ячейки res 5 и
«известный постоянный источник» (реестр по данным ≤ 2017).

Метки (из того, что произошло на самом деле):
  real        — поле FIRMS type = 0 и точка не в мировом реестре постоянных источников (все годы);
  escalation  — real и ≥ 3 чистых обнаружения в той же ячейке res 7 за следующие 3 суток.

    python -m helios.world.edge
"""

from __future__ import annotations

import json
import time

import h3
import numpy as np
import polars as pl
from sklearn.metrics import average_precision_score, roc_auc_score

from ..config import MODELS_DIR, REPORTS_DIR
from ..utils import get_logger, save_json
from .aggregate import AGG, EPOCH, LDAY, con, parent
from .ingest import DET_DIR

log = get_logger("helios.world.edge")

FEATURES = ["is_viirs", "is_night", "bright_mir", "bright_tir", "dt45", "log_frp", "log_pixel_km2", "scan_km",
            "p_conf", "hour_sin", "hour_cos", "month_sin", "month_cos", "lat_n", "ctx_log_n",
            "hist_clim_month", "hist_static", "hist_years"]
JSON_PATH = MODELS_DIR / "edge_detector_world_weights.json"
ONNX_PATH = MODELS_DIR / "edge_detector_world.onnx"
HIST_UNTIL = 2017
YEARS = (2018, 2025)
TRAIN, VAL, TEST = (2018, 2022), 2023, (2024, 2025)


def _static_set_until(c, year: int) -> None:
    """Реестр постоянных источников только по данным ≤ year (для бортовой таблицы)."""
    from datetime import date

    g = (DET_DIR / "*.parquet").as_posix()
    d_end = (date(year + 1, 1, 1) - date(2000, 1, 1)).days
    c.execute(f"""
        CREATE OR REPLACE TEMP TABLE h8 AS
        SELECT h3, count(*) AS n, avg((type IN (1,2,3))::DOUBLE) AS st
        FROM read_parquet('{g}') WHERE day < {d_end} GROUP BY h3 HAVING count(*) >= 6""")
    c.execute(f"""
        CREATE OR REPLACE TEMP TABLE hist8 AS
        SELECT h3 FROM h8 WHERE st >= 0.5
        UNION
        SELECT h3 FROM (
            SELECT h3, count(*) AS ny FROM (
                SELECT p.h3, year({EPOCH} + p.day::INTEGER) AS y FROM read_parquet('{g}') p
                SEMI JOIN (SELECT h3 FROM h8 WHERE st < 0.5 AND n >= 30) k ON p.h3 = k.h3
                WHERE p.day < {d_end} GROUP BY 1, 2
            ) GROUP BY 1
        ) WHERE ny >= 3""")
    cells = c.sql("SELECT h3 FROM hist8").fetchnumpy()["h3"]
    ring: set[int] = set()
    for x in cells.tolist():
        ring.update(h3.str_to_int(k) for k in h3.grid_disk(h3.int_to_str(int(x)), 1))
    c.register("hist_ring_df", pl.DataFrame({"h3": np.fromiter(ring, dtype=np.uint64)}).to_arrow())
    c.execute("CREATE OR REPLACE TEMP TABLE hist_ring AS SELECT h3::UBIGINT AS h3 FROM hist_ring_df")
    log.info("бортовой реестр ≤ %d: %d ячеек (с буфером %d)", year, len(cells), len(ring))


def build_dataset(per_file: dict[str, int] | None = None) -> pl.DataFrame:
    t0 = time.time()
    per_file = per_file or {"modis": 180_000, "viirs-snpp": 360_000}
    c = con()
    files = []
    for p in sorted(DET_DIR.glob("*.parquet")):
        kind, year = p.stem.rsplit("_", 1)
        if YEARS[0] <= int(year) <= YEARS[1]:
            files.append((p.as_posix(), per_file["modis" if kind == "modis" else "viirs-snpp"]))
    parts = " UNION ALL ".join(
        f"SELECT * FROM (SELECT * FROM read_parquet('{f}') USING SAMPLE reservoir({k} ROWS) REPEATABLE (3))" for f, k in files)
    c.execute(f"""
        CREATE OR REPLACE TEMP TABLE s AS
        SELECT row_number() OVER () AS id, *, {parent('h3', 7)} AS r7, {parent('h3', 5)} AS r5, {LDAY} AS lday,
               month({EPOCH} + day::INTEGER) AS mon, year({EPOCH} + day::INTEGER) AS yr
        FROM ({parts})""")
    # реестр всех лет (метка) и реестр ≤ 2017 (бортовой признак)
    st = pl.read_parquet(AGG / "static_r8.parquet", columns=["h3"])
    ring: set[int] = set()
    for x in st["h3"].to_list():
        ring.update(h3.str_to_int(k) for k in h3.grid_disk(h3.int_to_str(x), 1))
    c.register("label_ring_df", pl.DataFrame({"h3": np.fromiter(ring, dtype=np.uint64)}).to_arrow())
    c.execute("CREATE OR REPLACE TEMP TABLE label_ring AS SELECT h3::UBIGINT AS h3 FROM label_ring_df")
    _static_set_until(c, HIST_UNTIL)
    cd7 = (AGG / "cellday_r7.parquet").as_posix()
    cm5 = (AGG / "cellmonth_r5.parquet").as_posix()
    cm7 = (AGG / "cellmonth_r7.parquet").as_posix()
    last_m = (HIST_UNTIL - 2000) * 12 + 11
    df = c.sql(f"""
        WITH ctx AS (
            SELECT s.id, coalesce(max(c.n_r), 0) AS ctx_n FROM s LEFT JOIN read_parquet('{cd7}') c
            ON c.h3 = s.r7 AND c.lday = s.lday GROUP BY s.id),
        fut AS (
            SELECT s.id, coalesce(sum(c.n_mc + c.n_vc), 0) AS fut_n FROM s LEFT JOIN read_parquet('{cd7}') c
            ON c.h3 = s.r7 AND c.lday BETWEEN s.lday + 1 AND s.lday + 3 GROUP BY s.id),
        clim AS (
            SELECT h3, mon, count(*)::DOUBLE / {HIST_UNTIL - 2000 + 1} AS clim FROM (
                SELECT h3, (month % 12) + 1 AS mon, month // 12 AS y FROM read_parquet('{cm5}')
                WHERE month <= {last_m} AND d_ca > 0 GROUP BY 1, 2, 3) GROUP BY 1, 2),
        yrs AS (
            SELECT h3, count(*)::DOUBLE / {HIST_UNTIL - 2000 + 1} AS years FROM (
                SELECT h3, month // 12 AS y FROM read_parquet('{cm7}')
                WHERE month <= {last_m} AND d_ca > 0 GROUP BY 1, 2) GROUP BY 1)
        SELECT s.id, s.sat, s.night, s.bt, s.bt2, s.frp, s.scan, s.track, s.conf, s.hhmm, s.lon, s.lat, s.mon, s.yr,
               (s.type = 0 AND lr.h3 IS NULL) AS y_real, (hr.h3 IS NOT NULL) AS hist_static,
               ctx.ctx_n, fut.fut_n, coalesce(clim.clim, 0) AS hist_clim, coalesce(yrs.years, 0) AS hist_years
        FROM s
        LEFT JOIN label_ring lr ON lr.h3 = s.h3
        LEFT JOIN hist_ring hr ON hr.h3 = s.h3
        LEFT JOIN ctx ON ctx.id = s.id
        LEFT JOIN fut ON fut.id = s.id
        LEFT JOIN clim ON clim.h3 = s.r5 AND clim.mon = s.mon
        LEFT JOIN yrs ON yrs.h3 = s.r7""").pl()
    log.info("датасет ИИ №1 мир: %d строк, %.0f с", df.height, time.time() - t0)
    return df


def features(df: pl.DataFrame) -> np.ndarray:
    viirs = (df["sat"].to_numpy() >= 2).astype(np.float32)
    bt = df["bt"].fill_null(330).to_numpy().astype(np.float32)
    bt2 = df["bt2"].fill_null(295).to_numpy().astype(np.float32)
    scan = df["scan"].fill_null(1).to_numpy().astype(np.float32)
    track = df["track"].fill_null(1).to_numpy().astype(np.float32)
    hh = df["hhmm"].to_numpy()
    hour = ((hh // 100) + (hh % 100) / 60.0 + df["lon"].to_numpy() / 15.0) % 24
    mon = df["mon"].to_numpy()
    conf = df["conf"].to_numpy()
    cols = {
        "is_viirs": viirs, "is_night": df["night"].to_numpy().astype(np.float32), "bright_mir": bt, "bright_tir": bt2,
        "dt45": bt - bt2, "log_frp": np.log1p(np.maximum(df["frp"].to_numpy(), 0)).astype(np.float32),
        "log_pixel_km2": np.log(np.maximum(scan * track, 0.01)), "scan_km": scan,
        "p_conf": np.choose(conf.astype(int), [0.3, 0.6, 0.9]).astype(np.float32),
        "hour_sin": np.sin(2 * np.pi * hour / 24).astype(np.float32), "hour_cos": np.cos(2 * np.pi * hour / 24).astype(np.float32),
        "month_sin": np.sin(2 * np.pi * (mon - 1) / 12).astype(np.float32), "month_cos": np.cos(2 * np.pi * (mon - 1) / 12).astype(np.float32),
        "lat_n": (df["lat"].to_numpy() / 90.0).astype(np.float32), "ctx_log_n": np.log1p(df["ctx_n"].to_numpy()).astype(np.float32),
        "hist_clim_month": df["hist_clim"].to_numpy().astype(np.float32),
        "hist_static": df["hist_static"].to_numpy().astype(np.float32),
        "hist_years": df["hist_years"].to_numpy().astype(np.float32),
    }
    return np.column_stack([cols[f] for f in FEATURES]).astype(np.float32)


def train(epochs: int = 8) -> dict:
    import torch

    from ..models.edge_detector import _torch_model

    t0 = time.time()
    df = build_dataset()
    X = features(df)
    y_real = df["y_real"].to_numpy().astype(np.float32)
    y_esc = ((y_real > 0) & (df["fut_n"].to_numpy() >= 3)).astype(np.float32)
    y = np.column_stack([y_real, y_esc])
    yr = df["yr"].to_numpy()
    tr = (yr >= TRAIN[0]) & (yr <= TRAIN[1])
    va = yr == VAL
    te = (yr >= TEST[0]) & (yr <= TEST[1])
    mean, std = X[tr].mean(0), X[tr].std(0) + 1e-6
    # калибровочная выборка для бортового int8 (helios/onboard/export_c.py)
    np.save(MODELS_DIR / "edge_world_calib.npy", X[np.random.default_rng(1).choice(len(X), min(60_000, len(X)), replace=False)])
    torch.manual_seed(0)
    model = _torch_model(len(FEATURES), mean, std)
    pos_w = torch.tensor([1.0, float(min(8.0, (1 - y_esc[tr].mean()) / max(y_esc[tr].mean(), 1e-3)) ** 0.5)])
    opt = torch.optim.AdamW(model.parameters(), lr=3e-3, weight_decay=1e-4)
    Xt, yt = torch.tensor(X[tr]), torch.tensor(y[tr])
    Xv, yv = torch.tensor(X[va]), y[va]
    best, best_state = -1.0, None
    for epoch in range(epochs):
        model.train()
        perm = torch.randperm(len(Xt))
        for i in range(0, len(perm), 4096):
            b = perm[i:i + 4096]
            p = model(Xt[b]).clamp(1e-6, 1 - 1e-6)
            loss = -(pos_w * (yt[b] * torch.log(p)) + (1 - yt[b]) * torch.log(1 - p)).mean()
            opt.zero_grad()
            loss.backward()
            opt.step()
        model.eval()
        with torch.no_grad():
            pv = model(Xv).numpy()
        score = roc_auc_score(yv[:, 0], pv[:, 0]) + average_precision_score(yv[:, 1], pv[:, 1])
        log.info("ИИ №1 мир, эпоха %d: real-AUC %.4f esc-AP %.4f", epoch, roc_auc_score(yv[:, 0], pv[:, 0]),
                 average_precision_score(yv[:, 1], pv[:, 1]))
        if score > best:
            best, best_state = score, {k: v.clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        p_val = model(Xv).numpy()
        p_te = model(torch.tensor(X[te])).numpy()
    real_val = p_val[:, 0] >= 0.5
    t_crit = float(np.quantile(p_val[real_val, 1], 0.95))
    t_high = float(np.quantile(p_val[real_val, 1], 0.80))
    sd = {k: v.detach().numpy() for k, v in model.state_dict().items()}
    layers = [{"W": sd["body.0.weight"].round(6).tolist(), "b": sd["body.0.bias"].round(6).tolist(), "act": "relu"},
              {"W": sd["body.2.weight"].round(6).tolist(), "b": sd["body.2.bias"].round(6).tolist(), "act": "relu"},
              {"W": sd["head.weight"].round(6).tolist(), "b": sd["head.bias"].round(6).tolist(), "act": "sigmoid"}]
    save_json({"features": FEATURES, "mean": mean.round(6).tolist(), "std": std.round(6).tolist(), "layers": layers,
               "outputs": ["p_real", "p_escalation"], "thresholds": {"critical": t_crit, "high": t_high, "real": 0.5},
               "history_until": HIST_UNTIL}, JSON_PATH)
    try:
        torch.onnx.export(model, torch.zeros(1, len(FEATURES)), str(ONNX_PATH), dynamo=False, input_names=["features"],
                          output_names=["probs"], dynamic_axes={"features": {0: "batch"}, "probs": {0: "batch"}}, opset_version=17)
    except TypeError:
        torch.onnx.export(model, torch.zeros(1, len(FEATURES)), str(ONNX_PATH), input_names=["features"],
                          output_names=["probs"], dynamic_axes={"features": {0: "batch"}, "probs": {0: "batch"}}, opset_version=17)
    yte = y[te]
    conf_te = X[te][:, FEATURES.index("p_conf")]
    frp_te = X[te][:, FEATURES.index("log_frp")]
    params = int(sum(v.size for k, v in sd.items() if k.startswith(("body", "head")) and v.ndim >= 1))
    metrics = {
        "scope": "world", "train_years": list(TRAIN), "val_year": VAL, "test_years": list(TEST), "history_until": HIST_UNTIL,
        "n_train": int(tr.sum()), "n_test": int(te.sum()),
        "p_real": {"roc_auc": round(float(roc_auc_score(yte[:, 0], p_te[:, 0])), 4),
                   "pr_auc": round(float(average_precision_score(yte[:, 0], p_te[:, 0])), 4),
                   "baseline_confidence_only_auc": round(float(roc_auc_score(yte[:, 0], conf_te)), 4),
                   "real_share": round(float(yte[:, 0].mean()), 4)},
        "p_escalation": {"positives_share": round(float(yte[:, 1].mean()), 4),
                         "roc_auc": round(float(roc_auc_score(yte[:, 1], p_te[:, 1])), 4),
                         "pr_auc": round(float(average_precision_score(yte[:, 1], p_te[:, 1])), 4),
                         "baseline_frp_only_auc": round(float(roc_auc_score(yte[:, 1], frp_te)), 4),
                         "baseline_frp_only_pr_auc": round(float(average_precision_score(yte[:, 1], frp_te)), 4)},
        "thresholds": {"critical": round(t_crit, 4), "high": round(t_high, 4), "real": 0.5},
        "model": {"parameters": params, "onnx_fp32_kb": round(ONNX_PATH.stat().st_size / 1024, 1)},
        "seconds": round(time.time() - t0),
    }
    REPORTS_DIR.mkdir(parents=True, exist_ok=True)
    (REPORTS_DIR / "ai1_world_metrics.json").write_text(json.dumps(metrics, ensure_ascii=False, indent=1), encoding="utf-8")
    log.info("ИИ №1 мир, тест 2024–2025: %s", json.dumps(metrics, ensure_ascii=False)[:500])
    return metrics


if __name__ == "__main__":
    train()

### Бортовой ИИ для любого спутника (C99, int8)

In [ ]:
%%writefile helios/onboard/__init__.py
"""Бортовой ИИ HELIOS для любого спутника: целочисленная (int8) версия ИИ №1 и ИИ №2 на чистом C99,
без float, без динамической памяти и без стандартной библиотеки, с самопроверкой бит-в-бит."""

In [ ]:
%%writefile helios/onboard/quant.py
"""Квантование MLP в int8 и эталонный целочисленный инференс (бит-в-бит как в C).

Схема (как в TFLite Micro, но проще и без float на борту):
  вход   z = (x − mean) / std  →  x_q = clip(round(z / s_i), −127, 127); s_i — свой масштаб признака:
         ≥ 4σ, а у тяжёлых хвостов (ливень, запас линии в бурю) — до 99,99-го процентиля |z|;
         масштаб входа вносится в веса первого слоя, поэтому на борту ничего не меняется
  слой   acc_j = Σ W_q[j,i]·x_q[i] + B_q[j]          (int32; W_q — int8 с масштабом на строку)
  ReLU   y_j = clip(rshift_round(acc_j · M_j, 24), 0, 127)   (M_j — множитель Q24, int32)
  выход  logit_q8 = rshift_round(acc_j · M_j, 24)  →  p_q15 = сигмоида по таблице 257 точек + интерполяция
rshift_round — симметричное округление только сдвигами неотрицательных чисел: одинаково на любом
компиляторе и процессоре (сдвиг отрицательных чисел в C зависит от реализации).
"""

from __future__ import annotations

import json
from dataclasses import dataclass
from pathlib import Path

import numpy as np

S_IN = 4.0 / 127.0
Q = 24
LUT_LO, LUT_STEP_SHIFT, LUT_N = -2048, 4, 257  # logit Q8 ∈ [−8, 8], шаг 1/16


@dataclass
class QLayer:
    w: np.ndarray  # int8 [n_out, n_in]
    b: np.ndarray  # int32 [n_out]
    m: np.ndarray  # int32 [n_out] — множитель Q24
    act: str       # "relu" | "sigmoid"
    s_in: float
    s_out: float   # масштаб выхода (для ReLU); для сигмоиды — 1/256 (логит в Q8)


@dataclass
class QModel:
    name: str
    features: list[str]
    outputs: list[str]
    mean: np.ndarray
    std: np.ndarray
    s_x: np.ndarray  # масштаб каждого входа (z на единицу int8)
    layers: list[QLayer]
    lut: np.ndarray  # uint16 [257]


def load_float(path: Path) -> dict:
    d = json.loads(Path(path).read_text(encoding="utf-8"))
    return {
        "features": d["features"], "outputs": d.get("outputs", ["p"]),
        "mean": np.asarray(d["mean"], np.float64), "std": np.asarray(d["std"], np.float64),
        "layers": [(np.asarray(L["W"], np.float64), np.asarray(L["b"], np.float64), L["act"]) for L in d["layers"]],
        "thresholds": d.get("thresholds"),
    }


def standardize(model: dict, x_raw: np.ndarray) -> np.ndarray:
    return (np.asarray(x_raw, np.float64) - model["mean"]) / np.where(model["std"] > 0, model["std"], 1.0)


def float_forward(model: dict, z: np.ndarray, collect: bool = False):
    a = np.asarray(z, np.float64)
    acts = []
    for w, b, act in model["layers"]:
        a = a @ w.T + b
        if act == "relu":
            a = np.maximum(a, 0.0)
            acts.append(a)
        elif act == "sigmoid":
            a = 1.0 / (1.0 + np.exp(-a))
    return (a, acts) if collect else a


def sigmoid_lut() -> np.ndarray:
    x = (LUT_LO + np.arange(LUT_N) * (1 << LUT_STEP_SHIFT)) / 256.0
    return np.round(32767.0 / (1.0 + np.exp(-x))).astype(np.uint16)


def quantize(name: str, model: dict, z_cal: np.ndarray, pct: float = 99.99) -> QModel:
    """z_cal — стандартизованные входы реальных данных для калибровки масштабов активаций."""
    rng = np.maximum(4.0, np.percentile(np.abs(z_cal), 99.99, axis=0))
    s_x = rng / 127.0
    z_cal = np.clip(z_cal, -rng, rng)
    _, acts = float_forward(model, z_cal, collect=True)
    layers, s_in, k = [], 1.0, 0
    for li, (w, b, act) in enumerate(model["layers"]):
        if li == 0:
            w = w * s_x[None, :]  # x_real = x_q · s_x  →  масштаб входа внутри весов
        sw = np.maximum(np.abs(w).max(axis=1), 1e-12) / 127.0
        wq = np.clip(np.round(w / sw[:, None]), -127, 127).astype(np.int8)
        bq = np.round(b / (s_in * sw)).astype(np.int64)
        if act == "relu":
            s_out = max(float(np.percentile(acts[k], pct)), 1e-6) / 127.0
            k += 1
            mult = s_in * sw / s_out
        else:
            s_out = 1.0 / 256.0
            mult = s_in * sw * 256.0
        mq = np.round(mult * (1 << Q)).astype(np.int64)
        assert np.all(np.abs(bq) < 2**31) and np.all(np.abs(mq) < 2**31), "переполнение int32"
        layers.append(QLayer(wq, bq.astype(np.int32), mq.astype(np.int32), act, s_in, s_out))
        s_in = s_out
    return QModel(name, model["features"], model["outputs"], model["mean"], model["std"], s_x, layers, sigmoid_lut())


def quantize_input(z: np.ndarray, s_x: np.ndarray) -> np.ndarray:
    return np.clip(np.round(np.asarray(z, np.float64) / s_x), -127, 127).astype(np.int8)


def _rshift_round(v: np.ndarray, s: int) -> np.ndarray:
    v = v.astype(np.int64)
    half = np.int64(1) << (s - 1)
    pos = (v + half) >> s
    neg = -(((-v) + half) >> s)
    return np.where(v >= 0, pos, neg)


def int_forward(qm: QModel, x_q: np.ndarray) -> np.ndarray:
    """Эталон бит-в-бит: int8 входы [N, n_in] → вероятности Q15 (uint16) [N, n_out]."""
    a = np.asarray(x_q, np.int64)
    for L in qm.layers:
        acc = a @ L.w.astype(np.int64).T + L.b.astype(np.int64)
        assert np.all(np.abs(acc) < 2**31), "переполнение аккумулятора"
        y = _rshift_round(acc * L.m.astype(np.int64), Q)
        if L.act == "relu":
            a = np.clip(y, 0, 127)
        else:
            t = np.clip(y - LUT_LO, 0, (LUT_N - 1) << LUT_STEP_SHIFT)
            idx = np.minimum(t >> LUT_STEP_SHIFT, LUT_N - 2)
            frac = t - (idx << LUT_STEP_SHIFT)
            lut = qm.lut.astype(np.int64)
            lo, hi = lut[idx], lut[idx + 1]
            a = lo + (((hi - lo) * frac + (1 << (LUT_STEP_SHIFT - 1))) >> LUT_STEP_SHIFT)
    return a.astype(np.uint16)


def param_bytes(qm: QModel) -> dict:
    w = sum(L.w.size for L in qm.layers)
    b = sum(L.b.size for L in qm.layers)
    return {"weights_int8": int(w), "bias_int32": int(b), "rom_bytes": int(w + 4 * b * 2 + 2 * LUT_N),
            "max_width": int(max(max(L.w.shape) for L in qm.layers)),
            "macs": int(sum(L.w.size for L in qm.layers))}

In [ ]:
%%writefile helios/onboard/export_c.py
"""Экспорт бортовых ИИ №1 и №2 в переносимый C99 + самопроверка и кросс-компиляция.

    python -m helios.onboard.export_c            # artifacts/onboard_c/: исходники, тесты, отчёт
    python -m helios.onboard.export_c --no-build # только сгенерировать (например, в Kaggle без компилятора)

Код без float (флаг HO_NO_FLOAT), без malloc, без рекурсии и без стандартной библиотеки — собирается
для микроконтроллеров кубсатов (ARM Cortex-M0), радиационно-стойких LEON3/4 (SPARC, ESA), RAD750
(PowerPC), RISC-V и любых Linux-бортовых компьютеров. Самопроверка сверяет C с эталоном на Python
бит-в-бит, а с исходной float-моделью — по ошибке вероятности и совпадению решений.
"""

from __future__ import annotations

import argparse
import json
import shutil
import struct
import subprocess
import sys
import time
from pathlib import Path

import numpy as np

from ..config import ARTIFACTS, MODELS_DIR, PROCESSED
from ..utils import get_logger
from . import quant as Qm

log = get_logger("helios.onboard")

OUT = ARTIFACTS / "onboard_c"
N_TEST = 256


# ----------------------------------------------------------------------------- данные калибровки
def _ai1_inputs(n: int = 20000, world: bool = False) -> np.ndarray:
    """Реальные признаки ИИ №1 (как на борту): мировая калибровочная выборка или отфильтрованные точки KZ."""
    calib = MODELS_DIR / "edge_world_calib.npy"
    if world and calib.exists():
        x = np.load(calib).astype(np.float64)
        return x[np.random.default_rng(5).permutation(len(x))[:n]]
    import pandas as pd

    from ..models.edge_detector import build_features, load_table

    df = pd.read_parquet(PROCESSED / "detections_filtered.parquet")
    df = df.sample(n=min(n, len(df)), random_state=5).reset_index(drop=True)
    return build_features(df, load_table()).to_numpy(np.float64)


def _ai2_inputs(features: list[str], n: int = 20000) -> np.ndarray:
    import pandas as pd

    df = pd.read_parquet(PROCESSED / "link_dataset.parquet", columns=features)
    return df.sample(n=min(n, len(df)), random_state=5)[features].to_numpy(np.float64)


# ----------------------------------------------------------------------------- генерация C
def _arr(name: str, ctype: str, values, per_line: int = 16) -> str:
    vals = [str(int(v)) for v in np.asarray(values).ravel()]
    lines = [", ".join(vals[i:i + per_line]) for i in range(0, len(vals), per_line)]
    return f"static const {ctype} {name}[{len(vals)}] = {{\n    " + ",\n    ".join(lines) + "\n};\n"


def _farr(name: str, values) -> str:
    vals = [f"{float(v):.9g}f" for v in np.asarray(values).ravel()]
    return f"static const float {name}[{len(vals)}] = {{ " + ", ".join(vals) + " };\n"


HEADER = r"""/*
 * HELIOS-SHIELD onboard AI — portable C99, integer-only inference (int8 weights, int32 accumulators).
 * Generated by helios/onboard/export_c.py from the models trained in the HELIOS pipeline / Kaggle notebook.
 *
 *  - no floating point in the inference path (define HO_NO_FLOAT to drop the optional float helpers);
 *  - no dynamic memory, no recursion, no libc: only <stdint.h>; stack use < 200 bytes;
 *  - deterministic and bit-exact on every compiler and CPU (rounding uses shifts of non-negative values only).
 */
#ifndef HELIOS_ONBOARD_H
#define HELIOS_ONBOARD_H

#include <stdint.h>

#ifdef __cplusplus
extern "C" {
#endif

#define HO_AI1_N_IN  %(ai1_in)d
#define HO_AI1_N_OUT %(ai1_out)d
#define HO_AI2_N_IN  %(ai2_in)d
#define HO_AI2_N_OUT %(ai2_out)d
#define HO_PACKET_BYTES 24

/* Probabilities are Q15: 0 … 32767 ≙ 0 … 1. Inputs are standardized features as int8; every feature has its own
 * range (>= 4 sigma, wider for heavy-tailed ones such as rain fade) — use the quantize helpers below. */

/* AI #1 (edge detector): out[0] = P(real vegetation fire), out[1] = P(grows within 72 h). Returns 0 on success. */
int ho_ai1_infer(const int8_t x_q[HO_AI1_N_IN], uint16_t out_q15[HO_AI1_N_OUT]);

/* Alert priority from AI #1: 0 suppress, 1 routine, 2 high, 3 critical (thresholds chosen on validation data). */
uint8_t ho_ai1_priority(const uint16_t p_q15[HO_AI1_N_OUT]);

/* AI #2 (link predictor): out[0] = P(the current radio / laser link drops within 5 minutes). */
int ho_ai2_infer(const int8_t x_q[HO_AI2_N_IN], uint16_t out_q15[HO_AI2_N_OUT]);

/* Integer-only feature scaling for CPUs without an FPU: raw features given in milli-units (value x 1000). */
void ho_ai1_quantize_milli(const int32_t raw_milli[HO_AI1_N_IN], int8_t x_q[HO_AI1_N_IN]);
void ho_ai2_quantize_milli(const int32_t raw_milli[HO_AI2_N_IN], int8_t x_q[HO_AI2_N_IN]);

#ifndef HO_NO_FLOAT
/* Same scaling for CPUs with an FPU (LEON3/4, RAD750, Cortex-M4F/M7/R5, x86/ARM Linux). */
void ho_ai1_quantize(const float raw[HO_AI1_N_IN], int8_t x_q[HO_AI1_N_IN]);
void ho_ai2_quantize(const float raw[HO_AI2_N_IN], int8_t x_q[HO_AI2_N_IN]);
#endif

/* 24-byte alert packet, big-endian, CRC16-CCITT — byte-identical to helios/comms/packets.py. */
typedef struct {
    uint8_t priority;   /* 0 suppress … 3 critical, 4 confirmed by another satellite */
    uint8_t sat_id;     /* 1 Terra, 2 Aqua, 3 Suomi NPP, 4 NOAA-20, 5 NOAA-21 */
    uint8_t flags;      /* 1 night, 2 static source nearby, 4 sun glint, 8 confirmed */
    uint32_t unix_time;
    int32_t lat_e5;     /* latitude x 1e5 */
    int32_t lon_e5;     /* longitude x 1e5 */
    uint16_t frp_x10;   /* fire radiative power, 0.1 MW */
    uint16_t cluster_n; /* hot pixels in the cluster */
    uint8_t p_esc_u8;   /* P(escalation) x 255 */
    uint8_t p_real_u8;  /* P(real fire) x 255 */
} ho_alert_t;

void ho_packet_encode(const ho_alert_t *a, uint8_t out[HO_PACKET_BYTES]);
uint16_t ho_crc16_ccitt(const uint8_t *data, uint32_t n);

#ifdef __cplusplus
}
#endif
#endif /* HELIOS_ONBOARD_H */
"""

MLP_H = r"""/* Integer MLP engine shared by the HELIOS onboard models (see helios_onboard.h). */
#ifndef HO_MLP_H
#define HO_MLP_H

#include <stdint.h>

#define HO_Q 24          /* requantization multipliers are Q24 */
#define HO_LUT_LO (-2048) /* sigmoid table covers logits -8 … +8 in Q8 */
#define HO_LUT_SHIFT 4    /* table step 1/16 */
#define HO_LUT_N 257
#define HO_MAX_WIDTH 64

enum { HO_ACT_RELU = 0, HO_ACT_SIGMOID = 1 };

typedef struct {
    uint16_t n_in;
    uint16_t n_out;
    uint8_t act;
    const int8_t *w;   /* [n_out][n_in] */
    const int32_t *b;  /* [n_out], scale s_in * s_w */
    const int32_t *m;  /* [n_out], Q24 multiplier to the next scale */
} ho_layer_t;

int ho_mlp_run(const ho_layer_t *layers, uint8_t n_layers, const uint16_t *lut, const int8_t *x, uint16_t *out_q15);
int32_t ho_rshift_round(int64_t v, int s);

#endif /* HO_MLP_H */
"""

MLP_C = r"""#include "ho_mlp.h"

/* Rounding right shift, half away from zero, using shifts of non-negative values only (portable). */
int32_t ho_rshift_round(int64_t v, int s)
{
    const int64_t half = (int64_t)1 << (s - 1);
    if (v >= 0) {
        return (int32_t)((v + half) >> s);
    }
    return (int32_t)(-(((-v) + half) >> s));
}

int ho_mlp_run(const ho_layer_t *layers, uint8_t n_layers, const uint16_t *lut, const int8_t *x, uint16_t *out_q15)
{
    int8_t buf[2][HO_MAX_WIDTH];
    const int8_t *in = x;
    uint8_t cur = 0u;
    uint8_t l;
    for (l = 0u; l < n_layers; ++l) {
        const ho_layer_t *L = &layers[l];
        uint16_t j;
        if (L->n_in > HO_MAX_WIDTH || L->n_out > HO_MAX_WIDTH) {
            return -1;
        }
        for (j = 0u; j < L->n_out; ++j) {
            const int8_t *w = &L->w[(uint32_t)j * L->n_in];
            int32_t acc = L->b[j];
            int32_t y;
            uint16_t i;
            for (i = 0u; i < L->n_in; ++i) {
                acc += (int32_t)w[i] * (int32_t)in[i];
            }
            y = ho_rshift_round((int64_t)acc * (int64_t)L->m[j], HO_Q);
            if (L->act == HO_ACT_RELU) {
                if (y < 0) {
                    y = 0;
                }
                if (y > 127) {
                    y = 127;
                }
                buf[cur][j] = (int8_t)y;
            } else {
                int32_t t = y - HO_LUT_LO;
                int32_t idx;
                int32_t frac;
                int32_t lo;
                int32_t hi;
                if (t < 0) {
                    t = 0;
                }
                if (t > ((HO_LUT_N - 1) << HO_LUT_SHIFT)) {
                    t = (HO_LUT_N - 1) << HO_LUT_SHIFT;
                }
                idx = t >> HO_LUT_SHIFT;
                if (idx > HO_LUT_N - 2) {
                    idx = HO_LUT_N - 2;
                }
                frac = t - (idx << HO_LUT_SHIFT);
                lo = (int32_t)lut[idx];
                hi = (int32_t)lut[idx + 1];
                out_q15[j] = (uint16_t)(lo + (((hi - lo) * frac + (1 << (HO_LUT_SHIFT - 1))) >> HO_LUT_SHIFT));
            }
        }
        if (L->act == HO_ACT_RELU) {
            in = buf[cur];
            cur ^= 1u;
        }
    }
    return 0;
}
"""

PACKET_C = r"""#include "helios_onboard.h"

uint16_t ho_crc16_ccitt(const uint8_t *data, uint32_t n)
{
    uint16_t crc = 0xFFFFu;
    uint32_t i;
    for (i = 0u; i < n; ++i) {
        uint8_t b;
        crc ^= (uint16_t)((uint16_t)data[i] << 8);
        for (b = 0u; b < 8u; ++b) {
            crc = (crc & 0x8000u) ? (uint16_t)((crc << 1) ^ 0x1021u) : (uint16_t)(crc << 1);
        }
    }
    return crc;
}

static void put16(uint8_t *p, uint16_t v) { p[0] = (uint8_t)(v >> 8); p[1] = (uint8_t)v; }
static void put32(uint8_t *p, uint32_t v) { put16(p, (uint16_t)(v >> 16)); put16(p + 2, (uint16_t)v); }

void ho_packet_encode(const ho_alert_t *a, uint8_t out[HO_PACKET_BYTES])
{
    uint16_t crc;
    out[0] = 1u; /* format version */
    out[1] = a->priority;
    out[2] = a->sat_id;
    out[3] = a->flags;
    put32(&out[4], a->unix_time);
    put32(&out[8], (uint32_t)a->lat_e5);
    put32(&out[12], (uint32_t)a->lon_e5);
    put16(&out[16], a->frp_x10);
    put16(&out[18], a->cluster_n);
    out[20] = a->p_esc_u8;
    out[21] = a->p_real_u8;
    crc = ho_crc16_ccitt(out, 22u);
    put16(&out[22], crc);
}
"""


def _model_c(tag: str, qm: Qm.QModel) -> str:
    out = [f"/* {tag.upper()}: {qm.name} — inputs: {', '.join(qm.features)}; outputs: {', '.join(qm.outputs)} */\n"]
    for i, L in enumerate(qm.layers):
        out.append(_arr(f"{tag}_w{i}", "int8_t", L.w))
        out.append(_arr(f"{tag}_b{i}", "int32_t", L.b, 8))
        out.append(_arr(f"{tag}_m{i}", "int32_t", L.m, 8))
    rows = ",\n    ".join(
        f"{{{L.w.shape[1]}u, {L.w.shape[0]}u, {'HO_ACT_RELU' if L.act == 'relu' else 'HO_ACT_SIGMOID'}, {tag}_w{i}, {tag}_b{i}, {tag}_m{i}}}"
        for i, L in enumerate(qm.layers))
    out.append(f"static const ho_layer_t {tag}_layers[{len(qm.layers)}] = {{\n    {rows}\n}};\n")
    # целочисленное масштабирование: x_q = rshift_round((raw_milli - mean_milli) * K, 32), K = 2^32 / (1000 * std * S_IN)
    std = np.where(qm.std > 0, qm.std, 1.0)
    mean_milli = np.round(qm.mean * 1000).astype(np.int64)
    k = np.round((2.0 ** 32) / (1000.0 * std * qm.s_x)).astype(np.int64)
    out.append(_arr(f"{tag}_mean_milli", "int32_t", mean_milli, 8))
    out.append(_arr(f"{tag}_k", "int64_t", k, 6).replace("};", "};").replace("int64_t", "int64_t"))
    out.append("#ifndef HO_NO_FLOAT\n" + _farr(f"{tag}_mean", qm.mean) + _farr(f"{tag}_std", std)
               + _farr(f"{tag}_inv_s", 1.0 / qm.s_x) + "#endif\n")
    return "".join(out)


def _models_c(ai1: Qm.QModel, ai2: Qm.QModel, th: dict) -> str:
    th_q = {k: int(round(float(v) * 32767)) for k, v in th.items()}
    body = ['#include "helios_onboard.h"\n#include "ho_mlp.h"\n\n',
            _arr("ho_sigmoid_lut", "uint16_t", ai1.lut, 12),
            _model_c("ai1", ai1), _model_c("ai2", ai2)]
    body.append(f"""
#define HO_AI1_TH_REAL {th_q['real']}u
#define HO_AI1_TH_HIGH {th_q['high']}u
#define HO_AI1_TH_CRIT {th_q['critical']}u

int ho_ai1_infer(const int8_t x_q[HO_AI1_N_IN], uint16_t out_q15[HO_AI1_N_OUT])
{{
    return ho_mlp_run(ai1_layers, {len(ai1.layers)}u, ho_sigmoid_lut, x_q, out_q15);
}}

int ho_ai2_infer(const int8_t x_q[HO_AI2_N_IN], uint16_t out_q15[HO_AI2_N_OUT])
{{
    return ho_mlp_run(ai2_layers, {len(ai2.layers)}u, ho_sigmoid_lut, x_q, out_q15);
}}

uint8_t ho_ai1_priority(const uint16_t p_q15[HO_AI1_N_OUT])
{{
    if (p_q15[0] < HO_AI1_TH_REAL) {{
        return 0u;
    }}
    if (p_q15[1] >= HO_AI1_TH_CRIT) {{
        return 3u;
    }}
    if (p_q15[1] >= HO_AI1_TH_HIGH) {{
        return 2u;
    }}
    return 1u;
}}

static int8_t clamp_q(int32_t v)
{{
    if (v > 127) {{
        return 127;
    }}
    if (v < -127) {{
        return -127;
    }}
    return (int8_t)v;
}}

void ho_ai1_quantize_milli(const int32_t raw_milli[HO_AI1_N_IN], int8_t x_q[HO_AI1_N_IN])
{{
    uint16_t i;
    for (i = 0u; i < HO_AI1_N_IN; ++i) {{
        x_q[i] = clamp_q(ho_rshift_round(((int64_t)raw_milli[i] - ai1_mean_milli[i]) * ai1_k[i], 32));
    }}
}}

void ho_ai2_quantize_milli(const int32_t raw_milli[HO_AI2_N_IN], int8_t x_q[HO_AI2_N_IN])
{{
    uint16_t i;
    for (i = 0u; i < HO_AI2_N_IN; ++i) {{
        x_q[i] = clamp_q(ho_rshift_round(((int64_t)raw_milli[i] - ai2_mean_milli[i]) * ai2_k[i], 32));
    }}
}}

#ifndef HO_NO_FLOAT
static int8_t quantize_one(float raw, float mean, float std, float inv_s)
{{
    float v = (raw - mean) / std * inv_s;
    v = (v >= 0.0f) ? (v + 0.5f) : (v - 0.5f);
    if (v > 127.0f) {{
        v = 127.0f;
    }}
    if (v < -127.0f) {{
        v = -127.0f;
    }}
    return (int8_t)v;
}}

void ho_ai1_quantize(const float raw[HO_AI1_N_IN], int8_t x_q[HO_AI1_N_IN])
{{
    uint16_t i;
    for (i = 0u; i < HO_AI1_N_IN; ++i) {{
        x_q[i] = quantize_one(raw[i], ai1_mean[i], ai1_std[i], ai1_inv_s[i]);
    }}
}}

void ho_ai2_quantize(const float raw[HO_AI2_N_IN], int8_t x_q[HO_AI2_N_IN])
{{
    uint16_t i;
    for (i = 0u; i < HO_AI2_N_IN; ++i) {{
        x_q[i] = quantize_one(raw[i], ai2_mean[i], ai2_std[i], ai2_inv_s[i]);
    }}
}}
#endif
""")
    return "".join(body)


SELFTEST_C = r"""/* Host self-test: C == Python reference bit for bit; error vs the float model; packet bytes. */
#include <stdio.h>
#include <stdlib.h>
#include "helios_onboard.h"
#include "ho_test_vectors.h"

int main(void)
{
    int i, j, exact_bad = 0, prio_same = 0, a2_same = 0;
    long max_err1 = 0, max_err2 = 0;
    double sum_err1 = 0.0, sum_err2 = 0.0;
    uint8_t pkt[HO_PACKET_BYTES];
    ho_alert_t a;
    for (i = 0; i < HO_N_TEST; ++i) {
        uint16_t p[HO_AI1_N_OUT];
        uint16_t q[HO_AI2_N_OUT];
        if (ho_ai1_infer(t_ai1_x[i], p) != 0 || ho_ai2_infer(t_ai2_x[i], q) != 0) {
            printf("inference error\n");
            return 2;
        }
        for (j = 0; j < HO_AI1_N_OUT; ++j) {
            long e = labs((long)p[j] - (long)t_ai1_float[i][j]);
            if (p[j] != t_ai1_exact[i][j]) exact_bad++;
            if (e > max_err1) max_err1 = e;
            sum_err1 += (double)e;
        }
        if (ho_ai1_priority(p) == t_ai1_prio[i]) prio_same++;
        {
            long e = labs((long)q[0] - (long)t_ai2_float[i][0]);
            if (q[0] != t_ai2_exact[i][0]) exact_bad++;
            if (e > max_err2) max_err2 = e;
            sum_err2 += (double)e;
            if ((q[0] >= 16384) == (t_ai2_float[i][0] >= 16384)) a2_same++;
        }
    }
    a = t_packet_alert;
    ho_packet_encode(&a, pkt);
    for (j = 0; j < HO_PACKET_BYTES; ++j) {
        if (pkt[j] != t_packet_bytes[j]) {
            printf("packet mismatch at byte %d\n", j);
            return 3;
        }
    }
    printf("{\"vectors\": %d, \"bit_exact_mismatches\": %d, "
           "\"ai1_max_abs_err\": %.4f, \"ai1_mean_abs_err\": %.5f, \"ai1_priority_agreement\": %.4f, "
           "\"ai2_max_abs_err\": %.4f, \"ai2_mean_abs_err\": %.5f, \"ai2_decision_agreement\": %.4f, \"packet_ok\": true}\n",
           HO_N_TEST, exact_bad, max_err1 / 32767.0, sum_err1 / (HO_N_TEST * HO_AI1_N_OUT * 32767.0),
           (double)prio_same / HO_N_TEST, max_err2 / 32767.0, sum_err2 / (HO_N_TEST * 32767.0), (double)a2_same / HO_N_TEST);
    return exact_bad == 0 ? 0 : 1;
}
"""


def _priority_float(p: np.ndarray, th: dict) -> np.ndarray:
    return np.where(p[:, 0] < th["real"], 0, np.where(p[:, 1] >= th["critical"], 3, np.where(p[:, 1] >= th["high"], 2, 1)))


def _test_vectors(ai1: Qm.QModel, f1: dict, z1: np.ndarray, ai2: Qm.QModel, f2: dict, z2: np.ndarray, th: dict) -> tuple[str, dict]:
    x1, x2 = Qm.quantize_input(z1, ai1.s_x), Qm.quantize_input(z2, ai2.s_x)
    e1, e2 = Qm.int_forward(ai1, x1), Qm.int_forward(ai2, x2)
    pf1, pf2 = Qm.float_forward(f1, z1), Qm.float_forward(f2, z2)
    q1 = np.round(pf1 * 32767).astype(int)
    q2 = np.round(pf2 * 32767).astype(int)
    prio = _priority_float(pf1, th)
    from ..comms import packets

    alert = dict(priority="CRITICAL", stream="VIIRS_NOAA20", unix_time=1794700000, lat=51.12345, lon=71.43210,
                 frp_mw=187.4, cluster_n=12, p_escalation=0.91, p_real=0.987, flags=1)
    pkt = packets.encode(**alert)
    fields = struct.unpack(">BBBBIiiHHBB", pkt[:22])

    def mat(name, ctype, arr):
        rows = ",\n    ".join("{" + ", ".join(str(int(v)) for v in r) + "}" for r in np.asarray(arr))
        return f"static const {ctype} {name}[HO_N_TEST][{np.asarray(arr).shape[1]}] = {{\n    {rows}\n}};\n"

    h = [f"#define HO_N_TEST {len(x1)}\n#include <stdint.h>\n",
         mat("t_ai1_x", "int8_t", x1), mat("t_ai1_exact", "uint16_t", e1), mat("t_ai1_float", "uint16_t", q1),
         f"static const uint8_t t_ai1_prio[HO_N_TEST] = {{{', '.join(str(int(v)) for v in prio)}}};\n",
         mat("t_ai2_x", "int8_t", x2), mat("t_ai2_exact", "uint16_t", e2), mat("t_ai2_float", "uint16_t", q2),
         "static const ho_alert_t t_packet_alert = {"
         f"{fields[1]}u, {fields[2]}u, {fields[3]}u, {fields[4]}u, {fields[5]}, {fields[6]}, {fields[7]}u, {fields[8]}u, {fields[9]}u, {fields[10]}u}};\n",
         f"static const uint8_t t_packet_bytes[24] = {{{', '.join(str(b) for b in pkt)}}};\n"]
    py = {"ai1_max_abs_err": float(np.abs(e1.astype(int) - q1).max() / 32767), "ai2_max_abs_err": float(np.abs(e2.astype(int) - q2).max() / 32767)}
    return "".join(h), py


# ----------------------------------------------------------------------------- сборка и проверка
def _zig(args: list[str], cwd: Path) -> subprocess.CompletedProcess:
    return subprocess.run([sys.executable, "-m", "ziglang", *args], cwd=cwd, capture_output=True, text=True, timeout=600)


def _elf_sizes(path: Path) -> dict:
    """Размеры секций ELF-объекта (.text/.rodata/.data/.bss) без внешних утилит."""
    b = path.read_bytes()
    if b[:4] != b"\x7fELF":
        return {}
    is64, le = b[4] == 2, b[5] == 1
    e = "<" if le else ">"
    if is64:
        shoff, = struct.unpack_from(e + "Q", b, 0x28)
        shentsize, shnum, shstrndx = struct.unpack_from(e + "HHH", b, 0x3A)
    else:
        shoff, = struct.unpack_from(e + "I", b, 0x20)
        shentsize, shnum, shstrndx = struct.unpack_from(e + "HHH", b, 0x2E)
    secs = []
    for i in range(shnum):
        off = shoff + i * shentsize
        if is64:
            name, typ, _flags, _addr, soff, size = struct.unpack_from(e + "IIQQQQ", b, off)
        else:
            name, typ, _flags, _addr, soff, size = struct.unpack_from(e + "IIIIII", b, off)
        secs.append((name, typ, soff, size))
    stroff = secs[shstrndx][2]
    out = {"text": 0, "rodata": 0, "data": 0, "bss": 0}
    for name, _typ, _off, size in secs:
        end = b.index(b"\0", stroff + name)
        nm = b[stroff + name:end].decode()
        for k in out:
            if nm.startswith("." + k):
                out[k] += size
    return out


TARGETS = [
    ("ARM Cortex-M0 (CubeSat OBC, no FPU)", ["-target", "thumb-freestanding-eabi", "-mcpu=cortex_m0"], True),
    ("ARM Cortex-M7 (SAMRH71/SAMV71 rad-tolerant)", ["-target", "thumb-freestanding-eabihf", "-mcpu=cortex_m7"], False),
    ("ARM Cortex-R5 (TI Hercules / Zynq RPU)", ["-target", "arm-freestanding-eabi", "-mcpu=cortex_r5"], False),
    ("SPARC LEON3 (ESA GR712RC)", ["-target", "sparc-freestanding-none", "-mcpu=leon3"], False),
    ("PowerPC 750 (RAD750)", ["-target", "powerpc-freestanding-eabi", "-mcpu=750"], False),
    ("RISC-V RV32IMC (NOEL-V class)", ["-target", "riscv32-freestanding-none", "-mcpu=generic_rv32+m+c"], True),
    ("x86-64 Linux (onboard Linux / ground)", ["-target", "x86_64-linux-musl"], False),
]
SOURCES = ["ho_mlp.c", "ho_models.c", "ho_packet.c"]


def build_and_test(out: Path) -> dict:
    report: dict = {}
    exe = out / ("ho_selftest.exe" if sys.platform == "win32" else "ho_selftest")
    r = _zig(["cc", "-std=c99", "-O2", "-Wall", "-Wextra", "-Werror", "-o", exe.name, "ho_selftest.c", *SOURCES], out)
    if r.returncode != 0:
        report["host_build"] = {"ok": False, "stderr": r.stderr[-2000:]}
        return report
    run = subprocess.run([str(exe)], cwd=out, capture_output=True, text=True, timeout=60)
    try:
        report["selftest"] = {"ok": run.returncode == 0, **json.loads(run.stdout.strip().splitlines()[-1])}
    except Exception:
        report["selftest"] = {"ok": False, "stdout": run.stdout[-1000:], "stderr": run.stderr[-1000:]}
    builds = []
    tmp = out / "_cross"
    tmp.mkdir(exist_ok=True)
    for name, args, no_float in TARGETS:
        sizes = {"text": 0, "rodata": 0, "data": 0, "bss": 0}
        ok, err = True, ""
        for src in SOURCES:
            obj = tmp / (Path(src).stem + ".o")
            flags = ["-std=c99", "-Os", "-Wall", "-Werror", "-ffreestanding", *(["-DHO_NO_FLOAT"] if no_float else [])]
            rr = _zig(["cc", *args, *flags, "-c", src, "-o", str(obj)], out)
            if rr.returncode != 0:
                ok, err = False, rr.stderr.strip().splitlines()[-1] if rr.stderr.strip() else "error"
                break
            for k, v in _elf_sizes(obj).items():
                sizes[k] += v
        builds.append({"target": name, "ok": ok, "no_float": no_float, **({"bytes": sizes,
                       "rom_bytes": sizes["text"] + sizes["rodata"] + sizes["data"]} if ok else {"error": err[:300]})})
    shutil.rmtree(tmp, ignore_errors=True)
    report["cross"] = builds
    return report


def export(build: bool = True, ai1_json: Path | None = None, ai2_json: Path | None = None) -> dict:
    t0 = time.time()
    world_json = MODELS_DIR / "edge_detector_world_weights.json"
    ai1_json = ai1_json or (world_json if world_json.exists() else MODELS_DIR / "edge_detector_weights.json")
    f1 = Qm.load_float(ai1_json)
    f2 = Qm.load_float(ai2_json or MODELS_DIR / "link_ai_weights.json")
    th = f1.get("thresholds") or {"real": 0.5, "high": 0.67, "critical": 0.84}
    x1 = _ai1_inputs(world=Path(ai1_json).name == world_json.name)
    x2 = _ai2_inputs(f2["features"])
    z1, z2 = Qm.standardize(f1, x1), Qm.standardize(f2, x2)
    cal1, test1 = z1[N_TEST:], z1[:N_TEST]
    cal2, test2 = z2[N_TEST:], z2[:N_TEST]
    ai1 = Qm.quantize("AI #1 edge detector (" + ("world" if "world" in Path(ai1_json).name else "Kazakhstan") + ")", f1, cal1)
    ai2 = Qm.quantize("AI #2 link predictor", f2, cal2)
    # точность на всей калибровочной выборке (Python-эталон == C бит-в-бит)
    acc = {}
    for tag, qm, f, z in (("ai1", ai1, f1, cal1), ("ai2", ai2, f2, cal2)):
        pi = Qm.int_forward(qm, Qm.quantize_input(z, qm.s_x)).astype(np.float64) / 32767.0
        pf = Qm.float_forward(f, z)
        acc[tag] = {"rows": int(len(z)), "max_abs_err": round(float(np.abs(pi - pf).max()), 4),
                    "mean_abs_err": round(float(np.abs(pi - pf).mean()), 5)}
        if tag == "ai1":
            acc[tag]["priority_agreement"] = round(float((_priority_float(pi, th) == _priority_float(pf, th)).mean()), 4)
        else:
            acc[tag]["decision_agreement_at_0.5"] = round(float(((pi[:, 0] >= 0.5) == (pf[:, 0] >= 0.5)).mean()), 4)
    OUT.mkdir(parents=True, exist_ok=True)
    (OUT / "helios_onboard.h").write_text(HEADER % {"ai1_in": len(f1["features"]), "ai1_out": len(f1["outputs"]),
                                                    "ai2_in": len(f2["features"]), "ai2_out": len(f2["outputs"])}, encoding="utf-8")
    (OUT / "ho_mlp.h").write_text(MLP_H, encoding="utf-8")
    (OUT / "ho_mlp.c").write_text(MLP_C, encoding="utf-8")
    (OUT / "ho_packet.c").write_text(PACKET_C, encoding="utf-8")
    (OUT / "ho_models.c").write_text(_models_c(ai1, ai2, th), encoding="utf-8")
    tv, py_err = _test_vectors(ai1, f1, test1, ai2, f2, test2, th)
    (OUT / "ho_test_vectors.h").write_text('#include "helios_onboard.h"\n' + tv, encoding="utf-8")
    (OUT / "ho_selftest.c").write_text(SELFTEST_C, encoding="utf-8")
    features = {"ai1": f1["features"], "ai2": f2["features"]}
    report = {
        "generated": time.strftime("%Y-%m-%d %H:%M"), "quantization": "int8 weights (per-row scale), int32 accumulators, Q24 requantization, Q15 outputs",
        "ai1": {**Qm.param_bytes(ai1), **acc["ai1"], "thresholds": th},
        "ai2": {**Qm.param_bytes(ai2), **acc["ai2"]},
        "features": features, "python_reference_vs_float_on_test": py_err,
    }
    if build:
        report.update(build_and_test(OUT))
    report["seconds"] = round(time.time() - t0, 1)
    write_readme(report)
    (OUT / "report.json").write_text(json.dumps(report, ensure_ascii=False, indent=1), encoding="utf-8")
    log.info("onboard C: %s", json.dumps({k: report[k] for k in ("ai1", "ai2") if k in report}, ensure_ascii=False)[:600])
    return report


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--no-build", action="store_true")
    args = ap.parse_args()
    rep = export(build=not args.no_build)
    print(json.dumps({k: v for k, v in rep.items() if k != "features"}, ensure_ascii=False, indent=1))


if __name__ == "__main__":
    main()


def write_readme(report: dict, out: Path = OUT) -> Path:
    """README для инженеров бортового ПО (генерируется из отчёта: размеры, точность, признаки)."""
    a1, a2 = report.get("ai1", {}), report.get("ai2", {})
    st = report.get("selftest", {})
    rows = "\n".join(f"| {b['target']} | {'да' if b['ok'] else 'нет'} | {b.get('rom_bytes', '—')} | "
                     f"{'без float' if b.get('no_float') else 'с FPU-хелперами'} |" for b in report.get("cross", []))
    f1 = ", ".join(f"`{x}`" for x in report["features"]["ai1"])
    f2 = ", ".join(f"`{x}`" for x in report["features"]["ai2"])
    th = a1.get("thresholds", {})
    text = f"""# Бортовой ИИ HELIOS-SHIELD для любого спутника

Код в этой папке сгенерирован из обученных моделей (`python -m helios.onboard.export_c`, тот же шаг есть
в Kaggle-ноутбуке). Это две крошечные нейросети, которые работают прямо на спутнике, без связи с Землёй:

| | Что решает | Вход → выход |
|---|---|---|
| **ИИ №1** | настоящий ли это пожар и разрастётся ли он за 72 ч → приоритет тревоги | {len(report['features']['ai1'])} признаков пикселя → `p_real`, `p_escalation` |
| **ИИ №2** | пропадёт ли текущий радио- или лазерный канал в ближайшие 5 минут | {len(report['features']['ai2'])} признаков связи → `p_outage` |

Затем тревога упаковывается в **24 байта** (CRC16) и уходит по самому надёжному живому каналу.

## Почему это потянет любой спутник

* **Только целые числа:** веса int8, сумматоры int32, выход Q15. Флаг `HO_NO_FLOAT` убирает даже вспомогательные float-функции.
* **Без malloc, без рекурсии, без стандартной библиотеки:** нужен только `<stdint.h>`. Стек < 200 байт.
* **Детерминизм:** округление сделано сдвигами неотрицательных чисел — один и тот же результат на любом компиляторе и процессоре.
  Самопроверка: **{st.get('bit_exact_mismatches', '—')} расхождений** с эталоном на Python на {st.get('vectors', '—')} реальных векторах.
* **Размер:** ИИ №1 — {a1.get('macs', '—')} умножений на пиксель, ИИ №2 — {a2.get('macs', '—')}; всё вместе с пакетами — несколько КБ ПЗУ.

| Процессор (класс бортового компьютера) | собирается | ПЗУ, байт | режим |
|---|---|---|---|
{rows}

## Точность int8 против исходной float-модели

| | средняя ошибка вероятности | совпадение решений |
|---|---|---|
| ИИ №1 | {a1.get('mean_abs_err', '—')} | приоритет тревоги: {a1.get('priority_agreement', '—')} |
| ИИ №2 | {a2.get('mean_abs_err', '—')} | «обрыв / нет» при 0,5: {a2.get('decision_agreement_at_0.5', '—')} |

## Как вызывать

```c
#include "helios_onboard.h"

/* 1. признаки пикселя считает бортовое ПО (см. список ниже); без FPU — в тысячных долях */
int32_t raw_milli[HO_AI1_N_IN] = {{ /* ... */ }};
int8_t x[HO_AI1_N_IN];
uint16_t p[HO_AI1_N_OUT];
ho_ai1_quantize_milli(raw_milli, x);      /* или ho_ai1_quantize(float raw[]) на процессорах с FPU */
ho_ai1_infer(x, p);                       /* p[0] = P(настоящий пожар), p[1] = P(разрастётся), Q15 */
uint8_t prio = ho_ai1_priority(p);        /* 0 подавить, 1 обычная, 2 высокая, 3 критичная */

/* 2. пакет тревоги 24 байта */
ho_alert_t a = {{ prio, 4u /* NOAA-20 */, 1u /* ночь */, unix_time, lat_e5, lon_e5, frp_x10, cluster_n,
                 (uint8_t)(p[1] >> 7), (uint8_t)(p[0] >> 7) }};
uint8_t pkt[HO_PACKET_BYTES];
ho_packet_encode(&a, pkt);

/* 3. ИИ №2 выбирает канал: если P(обрыв за 5 мин) высока — шлём по другому каналу / ретранслятору */
```

Пороги приоритета (подобраны на валидационных данных): `real` {th.get('real', '—')}, `high` {round(th.get('high', 0), 3)}, `critical` {round(th.get('critical', 0), 3)}.

**Признаки ИИ №1** (в этом порядке): {f1}.

**Признаки ИИ №2**: {f2}.

## Сборка

```sh
# проверка на компьютере (zig: pip install ziglang; подойдёт и gcc/clang)
python -m ziglang cc -std=c99 -O2 -Wall -Wextra -Werror -o ho_selftest ho_selftest.c ho_mlp.c ho_models.c ho_packet.c
./ho_selftest

# бортовые процессоры (пример — объектные файлы для вашей прошивки)
python -m ziglang cc -target thumb-freestanding-eabi -mcpu=cortex_m0 -std=c99 -Os -DHO_NO_FLOAT -c ho_mlp.c ho_models.c ho_packet.c
python -m ziglang cc -target sparc-freestanding-none -mcpu=leon3 -std=c99 -Os -c ho_mlp.c ho_models.c ho_packet.c
```

Обычные кросс-компиляторы тоже подходят: `arm-none-eabi-gcc`, `sparc-gaisler-elf-gcc` (BCC), `powerpc-eabi-gcc`,
`riscv64-unknown-elf-gcc -march=rv32imc`.

## Встраивание в полётное ПО

* **NASA cFS:** приложение-обёртка подписывается на сообщения о новых горячих пикселях, вызывает `ho_ai1_*`
  и публикует 24-байтный пакет в очередь телеметрии высокого приоритета.
* **F´ (F Prime):** пассивный компонент с портами «пиксель → тревога» и «состояние канала → выбор канала».
* **RTEMS / FreeRTOS / bare metal:** функции реентерабельны и не держат состояния — можно звать из любой задачи.

Обновление моделей «по воздуху»: заменяются только массивы весов в `ho_models.c` (≈3 КБ) — тот же код.
"""
    p = out / "README.md"
    p.write_text(text, encoding="utf-8")
    return p

In [ ]:
import importlib, helios, json, shutil, pandas as pd, numpy as np
import matplotlib.pyplot as plt
from helios.config import ensure_dirs, REPORTS_DIR, PROCESSED
ensure_dirs()
pd.set_option('display.width', 200)
def show(name):
    print(json.dumps(json.loads((REPORTS_DIR / f'{name}.json').read_text(encoding='utf-8')), ensure_ascii=False, indent=1)[:4000])

## Проверка API (что доступно без ключей)

In [ ]:
from helios.apis.health import check_all
for name, info in check_all().items():
    print(('OK ' if info['ok'] else '-- ') + f"{name:24s} {info['detail']}")

# 1. Historical Intelligence
Годовой архив FIRMS по стране (научное качество, с полем `type`) → геометрия пикселя и Солнца → реестр постоянных
источников (4 улики) → единый фильтр шума → «сгоревшие гексагоны» H3 → калибровка VIIRS к шкале Aqua MODIS
(эмпирический Байес) → Burning Activity Calendar. Проверки: ступеньки 2012/2018 и тест «MODIS выключили».

In [ ]:
from helios.apis.firms import download_country_archive
_ = download_country_archive(workers=6)

In [ ]:
from helios.pipeline import build_history
res = build_history()
show('filter_report'); show('harmonization_validation')

In [ ]:
cal = pd.read_parquet(PROCESSED / 'calendar_monthly.parquet')
cal['month'] = pd.to_datetime(cal['month'])
fig, ax = plt.subplots(2, 1, figsize=(14, 8))
ax[0].plot(cal['month'], cal['raw_detections'], label='сырые точки (все спутники)', alpha=.6)
ax[0].plot(cal['month'], cal['harmonized'], label='гармонизировано (шкала Aqua MODIS)', lw=2)
ax[0].fill_between(cal['month'], cal['lo95'], cal['hi95'], alpha=.25, label='95% интервал')
for x, t in [('2012-01-19', 'Suomi-NPP'), ('2018-01-05', 'NOAA-20')]:
    ax[0].axvline(pd.Timestamp(x), ls='--', c='gray'); ax[0].text(pd.Timestamp(x), ax[0].get_ylim()[1]*.9, t)
ax[0].set_yscale('log'); ax[0].legend(); ax[0].set_title('Без гармонизации появление новых спутников выглядит как «рост пожаров»')
mat = cal.pivot_table(index='year', columns='mon', values='harmonized')
im = ax[1].imshow(np.log1p(mat.values), aspect='auto', cmap='inferno'); ax[1].set_yticks(range(len(mat.index)), mat.index)
ax[1].set_xticks(range(12), ['Я','Ф','М','А','М','И','И','А','С','О','Н','Д']); ax[1].set_title('Burning Activity Calendar (log)')
plt.tight_layout(); plt.show()

In [ ]:
reg = pd.read_parquet(PROCESSED / 'static_registry.parquet'); st = reg[reg.is_static]
det = pd.read_parquet(PROCESSED / 'detections_filtered.parquet', columns=['lat','lon','keep_veg','acq_dt'])
s = det[det.acq_dt.dt.year == 2023].sample(60000, random_state=0)
plt.figure(figsize=(12, 6)); plt.scatter(s.lon, s.lat, s=1, c=np.where(s.keep_veg, 'tab:orange', 'lightgray'))
plt.scatter(st.lon, st.lat, s=8, c='blue', label=f'постоянные источники ({len(st)})'); plt.legend()
plt.title('Пожары 2023 (оранжевые — природные/растительные) и найденные факелы/заводы (синие)'); plt.show()

### Погода NASA POWER по сетке H3 res-3 (для ИИ №3 и детектора слепых зон)

In [ ]:
from helios.pipeline import fetch_weather
w = fetch_weather(); print(w.shape)

# 2. Edge AI Simulation — ИИ №1 (на борту)
Крошечная MLP (≈1.5 тыс. параметров, ONNX int8 ≈ 6 КБ) + бортовая таблица истории (≈0.4 МБ, только до 2018 г.).
Метки из будущего: разросся ли пожар (≥6 «сгоревших гексагонов» в 10 км за 72 ч). Тест — 2025 год.

In [ ]:
from helios.models import edge_detector
m1 = edge_detector.train(); print(json.dumps(m1, ensure_ascii=False, indent=1))

### ИИ №2 (на борту): прогноз потери связи + физика «радио vs лазер»

In [ ]:
from helios.models import link_ai
m2 = link_ai.train(); print(json.dumps(m2, ensure_ascii=False, indent=1))
from helios.comms.physics import simulate_space_link, mission_design_table
for case in [dict(), dict(xray_flux_wm2=1e-4, kp=8, rain_mm_h=12, cloud_frac=0.85)]:
    r = simulate_space_link(**case); print(case, '->', r['margins_db'], '| канал:', r['recommended_channel'], '| тревоги:', r['alert_channel'])
pd.DataFrame(mission_design_table())

### ИИ №3 (Земля): прогноз пожарного риска на 1–4 недели + Time Machine, Fire DNA, сдвиг сезона, слепые зоны

In [ ]:
from helios.models import ground_forecast
m3 = ground_forecast.train()
for h, v in m3['horizons'].items():
    print(h, 'TEST AI', v['test']['ai'], '| климатология', v['test']['baseline_climatology'])
print('важность признаков:', list(m3.get('feature_importance_h1', {}).items())[:10])

In [ ]:
from helios.pipeline import run_analytics, forecast_now
summary = run_analytics(); show('analytics_summary')
risk = forecast_now(); risk.sort_values('p_fire', ascending=False).head(10)

# 3. Smart Alert Delivery
Реальный пожарный день → ИИ №1 → пакеты по 24 байта → когда тревога окажется на Земле при разных схемах связи
(как сейчас / ИИ на борту + сеть станций / + станции в Казахстане / + ретранслятор на ГСО) + подтверждение другим спутником.

In [ ]:
from helios.comms.delivery import run_delivery_simulation
d = run_delivery_simulation()
for k, v in d['strategies'].items(): print(f'{k:32s}', 'медиана, мин:', v['all']['median_min'], '| критичные:', (v['critical'] or {}).get('median_min'))
print(d['collaborative_network'])
al = pd.read_parquet(PROCESSED / 'delivery_alerts.parquet')
colors = {'ROUTINE': 'gold', 'HIGH': 'orange', 'CRITICAL': 'red'}
plt.figure(figsize=(12, 6))
for p_, g in al.groupby('priority'): plt.scatter(g.lon, g.lat, s=10 + 30 * (p_ == 'CRITICAL'), c=colors.get(str(p_), 'gray'), label=str(p_))
plt.legend(); plt.title(f"Тревоги ИИ №1 за {d['replay_day']}"); plt.show()

In [ ]:
from helios.comms.scheduler import plan
pl = plan(hours=24)
pd.DataFrame([{k: b[k] for k in ('satellite','minutes_without_any_link','longest_gap_min','minutes_without_link_with_real_relays','minutes_without_link_with_relay_76E')} for b in pl['blackouts']])

## Оперативный режим (свежие пожары NRT) + анализ Gemini

In [ ]:
from helios.pipeline import run_nrt
nrt = run_nrt('48h'); print(json.dumps(nrt, ensure_ascii=False, default=str, indent=1)[:2500])
from helios.analysis import cell_stats, reload; reload()
from helios.apis.gemini import analyze_region
stats = cell_stats(lat=48.34, lon=57.40); print(analyze_region(stats, 'Насколько опасно на этой неделе?')['text'])

# 4. Весь мир: все страны, регионы и ИИ на мировых данных
Годовые архивы FIRMS «все страны» (MODIS с 2000 г., VIIRS S-NPP с 2012 г.) → Parquet → реестр постоянных источников
мира → «горящие гексагоно-сутки» H3 res 2–7 по каждому спутнику и дню/ночи → ряды по 240 странам и 4 589 регионам →
ИИ №3 (LightGBM, весь мир, H3 res 5, 1–4 недели) и ИИ №1 v2 (бортовой, весь мир). Полный архив ~7 ГБ:
для быстрого прогона на Kaggle ограничьте годы в `WORLD_YEARS` (модели тогда учатся на меньшем периоде).

In [ ]:
from helios.world import fetch, ingest, regions, aggregate
WORLD_YEARS = range(2000, 2026)          # например range(2018, 2026) для быстрого прогона
for y in WORLD_YEARS:
    fetch.download('modis', y)
    if y >= 2012:
        fetch.download('viirs-snpp', y)
for z in ingest.pending():
    ingest.ingest_zip(z); z.unlink()       # zip больше не нужен — экономим диск Kaggle
regions.build_regions()
print(aggregate.build_all())

In [ ]:
from helios.world import edge, forecast
m1w = edge.train(); print(json.dumps({k: m1w[k] for k in ('p_real', 'p_escalation', 'thresholds')}, ensure_ascii=False, indent=1))
m3w = forecast.train_all()
for v, hs in m3w['variants'].items():
    for h, r in hs.items(): print(v, h, 'ИИ', r['test']['ai'], '| сезонный алгоритм', r['test']['baseline_climatology'])

# 5. Бортовой ИИ для любого спутника
ИИ №1 и ИИ №2 → int8 + целочисленный C99 без float, malloc и libc (5–7 КБ ПЗУ, < 200 байт стека).
Самопроверка сравнивает C с эталоном на Python бит-в-бит; кросс-компиляция — ARM Cortex-M0/M7/R5,
SPARC LEON3 (ESA), PowerPC 750 (RAD750), RISC-V. Архив `helios_onboard_c.zip` — готовый код для бортового ПО.

In [ ]:
from helios.onboard.export_c import export, OUT
rep = export(build=True)
print(json.dumps({k: rep[k] for k in ('ai1', 'ai2', 'selftest') if k in rep}, ensure_ascii=False, indent=1))
print(pd.DataFrame([{'target': b['target'], 'ok': b['ok'], 'ROM, bytes': b.get('rom_bytes')} for b in rep.get('cross', [])]))
shutil.make_archive('helios_onboard_c', 'zip', OUT)

## Экспорт: модели и отчёты (+ выгрузка в Supabase, если заданы секреты)

In [ ]:
import shutil, os
shutil.make_archive('helios_artifacts', 'zip', 'artifacts'); print('artifacts -> helios_artifacts.zip')
if os.environ.get('SUPABASE_URL') and (os.environ.get('HELIOS_INGEST_TOKEN') or os.environ.get('SUPABASE_SERVICE_ROLE_KEY')):
    from helios.db.supabase_sync import sync_all
    print(sync_all())
else:
    print('Supabase-секреты не заданы — выгрузку пропускаю')